# RSNA Knee: Offline Task-Specific Model Inference

**Architecture:** Task-trained DINOv2, CoAtNet, or timm MIL checkpoints. Equal-family rank blending is the default; D4-derived target weights are opt-in.
**Resolution & Slices:** Checkpoint-matched preprocessing; the default `v2` cache uses 518-pixel images and depth 32.
**Design Philosophy:**
1. **Embedded Source Snapshot:** Pipeline modules are embedded from the repository at notebook generation time and must be refreshed when `src/` changes.
2. **Weights-Only Zip:** Upload trained fold checkpoints (`*_best.pt`; EMA/SWA are fallback-only) while preserving their model-family/fold directories.
3. **Inference:** Test studies are cached once; horizontal flips remain disabled to preserve medial/lateral orientation.
4. **Hardware:** Mixed precision and multi-GPU wrapping are used when supported by the active Kaggle runtime.

---
### Kaggle Submission Checklist
- [x] **Internet:** Disabled in Notebook settings (mandatory for submission).
- [x] **Competition data:** Attach the RSNA Knee Abnormality Detection dataset.
- [x] **Model weights:** Attach the private dataset containing compatible task-trained checkpoints.


In [ ]:
# ==============================================================================
# STEP 1: OFFLINE WHEEL INSTALLATION
# ==============================================================================
import os
import sys
import glob

# Auto-detect and install any offline wheels (.whl) in /kaggle/input (e.g. dicomsdl, timm)
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
if wheels:
    print(f'[SETUP] Found {len(wheels)} offline wheel packages. Installing...')
    for whl in wheels:
        os.system(f'pip install "{whl}" --no-index --find-links /kaggle/input/ --quiet')
    print('[SETUP] Wheel installation complete.')
else:
    print('[SETUP] No .whl packages found. Proceeding with pre-installed environment packages.')


In [ ]:
# ==============================================================================
# STEP 2: AUTO-EXTRACT MODEL WEIGHTS (RSNA.ZIP)
# ==============================================================================
import zipfile
import shutil

WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Search for rsna.zip or any zip containing model weights
zip_candidates = glob.glob('/kaggle/input/**/rsna*.zip', recursive=True) + glob.glob('/kaggle/input/**/*.zip', recursive=True)
extracted_from_zip = False

if zip_candidates:
    target_zip = zip_candidates[0]
    print(f'[AUTO-DETECT] Found weights archive: {target_zip}')
    print(f'[AUTO-DETECT] Extracting weights to {WEIGHTS_DIR}...')
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall(WEIGHTS_DIR)
    extracted_from_zip = True
    print('[SUCCESS] Weights extracted successfully.')
else:
    print('[INFO] No .zip archive found. Checking for unzipped dataset folders in /kaggle/input...')

# Discover checkpoint files (.pt)
found_checkpoints = glob.glob(f'{WEIGHTS_DIR}/**/*.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pt', recursive=True)
print(f'[SUCCESS] Discovered {len(found_checkpoints)} checkpoint file(s):')
for ckpt in found_checkpoints[:10]:
    print(f'   - {ckpt}')


In [ ]:
# ==============================================================================
# CELL 3: EXTRACT EXACT CURRENT PIPELINE MODULES (NO CODE IN WEIGHTS ZIP)
# ==============================================================================
import os
import sys
import base64

REPO_ROOT = '/kaggle/working'
PIPELINE_MODULES = {
    'src/main.py': '...',
    'src/__init__.py': 'IiIiUlNOQSBLbmVlIEFibm9ybWFsaXR5IERldGVjdGlvbiBzb3VyY2UgcGFja2FnZSAodjIgcHJlcHJvY2Vzc2luZyBsYXllcikuIiIiCg==',
    'src/core/config.py': 'IiIiQ2VudHJhbCBjb25maWd1cmF0aW9uIGZvciBwcmVwcm9jZXNzaW5nLCB0cmFpbmluZywgYW5kIG1lbW9yeSBzYWZlZ3VhcmRzLg0KDQpUcmFpbmluZyBoeXBlcnBhcmFtZXRlcnMgYXJlIHN0YXJ0aW5nIHZhbHVlcyBhbmQgcmVxdWlyZSBlbXBpcmljYWwgY29tcGFyaXNvbg0Kb24gdGhlIGNvbXBldGl0aW9uJ3MgR29sZC1sYWJlbGVkIHZhbGlkYXRpb24gZGF0YS4NCiIiIg0KaW1wb3J0IG9zDQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIHJlcGxhY2UNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgSGFyZHdhcmUgU2FmZXR5ICYgQ2lyY3VpdCBCcmVha2VyIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgA0KIyBVc2UgODggR2lCIGFzIGFuIG9wZXJhdGlvbmFsIHRhcmdldCBvbiB0aGUgMTI4LUdpQiB1bmlmaWVkLW1lbW9yeSBob3N0Lg0KIyBJdCBpcyBhZHZpc29yeTsgdGhlIGhhcmQgY2VpbGluZyBhbmQgYXZhaWxhYmxlLW1lbW9yeSBndWFyZCBwcmVzZXJ2ZSByZWNvdmVyeSByb29tLg0KTUVNT1JZX1RBUkdFVF9HQiA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJSU05BX01FTU9SWV9UQVJHRVRfR0IiLCAiODguMCIpKQ0KQ0lSQ1VJVF9CUkVBS0VSX01BWF9SQU1fR0IgPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiUlNOQV9NQVhfUkFNX0dCIiwgIjEwMC4wIikpDQpNSU5fQVZBSUxBQkxFX1JBTV9HQiA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJSU05BX01JTl9BVkFJTEFCTEVfUkFNX0dCIiwgIjIwLjAiKSkNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgdHJhaW5pbmcgY29uc3RhbnRzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgA0KU0VFRCAgICAgICAgICAgICA9IDIwMjYNCkVQT0NIUyAgICAgICAgICAgPSAxMCAgICAgICAgICAjIEluaXRpYWwgYnVkZ2V0OyBjb252ZXJnZW5jZSBhbmQgZWFybHkgc3RvcHBpbmcgcmVxdWlyZSBlbXBpcmljYWwgdmFsaWRhdGlvbg0KQkFUQ0hfU0laRSAgICAgICA9IDI0DQpOVU1fV09SS0VSUyAgICAgID0gOA0KUFJFRkVUQ0hfRkFDVE9SICA9IDENCkVWQUxfQkFUQ0hfU0laRSAgPSA4DQpQUkVQUk9DRVNTX1dPUktFUlMgPSA4DQpHUkFEX0FDQ1VNICAgICAgID0gMSAgICAgICAgICAgIyBFZmZlY3RpdmUgYmF0Y2ggZXF1YWxzIEJBVENIX1NJWkUgd2hlbiBubyBmaW5hbCBwYXJ0aWFsIGdyb3VwIGV4aXN0cw0KTl9XSU5ET1dTX1RSQUlOICA9IDUgICAgICAgICAgICMgTnVtYmVyIG9mIGRlcHRoLXN0cmF0aWZpZWQgd2luZG93cyBzYW1wbGVkIHBlciBzbG90IGFuZCBlcG9jaA0KTFJfSEVBRCAgICAgICAgICA9IDJlLTMgICAgICAgICMgU3RhcnRpbmcgdmFsdWU7IG5vdCB0dW5lZCBieSBhIERHWCB2YWxpZGF0aW9uIHN3ZWVwDQpMUl9CQUNLQk9ORSAgICAgID0gMWUtNSAgICAgICAgIyBTdGFydGluZyB2YWx1ZTsgc2NhbGVkIHBlci1sYXllciBieSB0aGUgRElOT3YyIG9wdGltaXplcg0KV0VJR0hUX0RFQ0FZICAgICA9IDAuMDUNCkNPQVRORVRfTFJfSEVBRCAgICAgID0gMWUtMw0KQ09BVE5FVF9MUl9CQUNLQk9ORSAgPSAzZS01DQpDT0FUTkVUX1dFSUdIVF9ERUNBWSA9IDAuMDINCkNPQVRORVRfSU5QVVRfU0laRSAgID0gMzg0DQpDT0FUTkVUX0VOQ09ERV9DSFVOSyA9IDgNClVORlJFRVpFX0xBU1QgICAgPSA4ICAgICAgICAgICAjIFRvcCA4IGJsb2NrcyBmaW5lLXR1bmVkIHdpdGggTExSRDsgYmxvY2tzIDAtMyBmcm96ZW4gKHByZXNlcnZlcyAyLjVEIE1SSSBhZGFwdGF0aW9uKQ0KTE9SQV9SQU5LICAgICAgICA9IDAgICAgICAgICAgIyBaZXJvIGRpc2FibGVzIExvUkEgYWRhcHRlcnMNCkxPUkFfQUxQSEEgICAgICAgPSAzMiAgICAgICAgICAjIFNjYWxpbmcgdXNlZCBpZiBhIHBvc2l0aXZlIExvUkEgcmFuayBpcyBzZWxlY3RlZA0KVElNRV9CVURHRVRfSE9VUlMgPSA5OTk5LjAgICAgICMgREdYIFNwYXJrOiBubyBLYWdnbGUgdGltZSBjYXAg4oCUIGRpc2FibGVkDQpFQVJMWV9TVE9QX1BBVElFTkNFID0gMyAgICAgICAgIyBFYXJseSBzdG9wcGluZyBwYXRpZW5jZSBvbiB2YWxpZGF0aW9uIE1hY3JvLUFVQw0KDQojIFNXQSBkdXBsaWNhdGVzIG1vZGVsIHN0YXRlIGFuZCBwZXJmb3JtcyBhbiBleHRyYSBmdWxsIHRyYWluaW5nIHBhc3MuIEVuYWJsZQ0KIyBpdCBvbmx5IGZvciBhIG1lYXN1cmVkIGFibGF0aW9uLg0KU1dBX0VQT0NIUyAgICAgID0gMA0KDQojIEFzeW1tZXRyaWMgTG9zcyBkZWZhdWx0cyAoUmlkbmlrIGV0IGFsLiwgSUNDViAyMDIxKQ0KQVNMX0dBTU1BX05FRyAgID0gNC4wICAgICAgICAgICMgU2hhcmVkIG5lZ2F0aXZlIGZvY3VzaW5nIGV4cG9uZW50OyBjb21wYXJlIHdpdGggQkNFIG9uIG1hdGNoZWQgZm9sZHMNCkFTTF9HQU1NQV9QT1MgICA9IDAuMA0KQVNMX0NMSVAgICAgICAgID0gMC4wNSAgICAgICAgICMgTmVnYXRpdmUgcHJvYmFiaWxpdHkgc2hpZnQ7IGEgaHlwb3RoZXNpcyBmb3Igbm9pc3kgbGFiZWxzLCBub3QgYSB2ZXJpZmllZCBnYWluDQoNCiMgTGFiZWwgc21vb3RoaW5nIChTemVnZWR5IGV0IGFsLiAyMDE2OyBzdGFuZGFyZCBWaVQgcmVjaXBlKQ0KTEFCRUxfU01PT1RISU5HID0gMC4wICAgICAgICAgICMgRGlzYWJsZWQgaW4gdGhlIGRlZmF1bHQgdHJhaW5pbmcgcmVjaXBlDQoNCiMgQXVnbWVudGF0aW9uICh1bmNoYW5nZWQgZnJvbSBiYXNlbGluZSkNCkFVR19ST1RfREVHICA9IDguMA0KQVVHX1NDQUxFICAgID0gMC4wOA0KQVVHX1NISUZUICAgID0gMC4wNQ0KQVVHX0lOVEVOU0lUWSA9IDAuMQ0KDQojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgCB0YXJnZXQgLyBzbG90IHNjaGVtYSDilIDilIDilIDilIDilIDilIDilIDilIANClRBUkdFVFMgPSBbDQogICAgIkFDTCIsICJNQ0wiLCAiTWVkaWFsIE1lbmlzY3VzIiwgIkxhdGVyYWwgTWVuaXNjdXMiLCAiTWVkaWFsIE9BIiwNCiAgICAiTGF0ZXJhbCBPQSIsICJQRiBPQSIsICJFZmZ1c2lvbiIsICJTeW5vdml0aXMiLCAiQmFrZXIncyIsDQogICAgIkNvbnR1c2lvbiIsICJGcmFjdHVyZSIsDQpdDQoNCiMgKG5hbWUsIHBsYW5lLCBmbHVpZC13ZWlnaHRlZCwgZmF0LXN1cHByZXNzZWQpDQpTTE9UUyA9IFsNCiAgICAoIlNBR19GTFVJRF9GUyIsICAgIlNhZ2l0dGFsIiwgIFRydWUsICBUcnVlKSwNCiAgICAoIkNPUl9GTFVJRF9GUyIsICAgIkNvcm9uYWwiLCAgIFRydWUsICBUcnVlKSwNCiAgICAoIkFYX0ZMVUlEX0ZTIiwgICAgIkF4aWFsIiwgICAgIFRydWUsICBUcnVlKSwNCiAgICAoIlNBR19GTFVJRF9OT0ZTIiwgIlNhZ2l0dGFsIiwgIFRydWUsICBGYWxzZSksDQogICAgKCJDT1JfVDEiLCAgICAgICAgICJDb3JvbmFsIiwgICBGYWxzZSwgRmFsc2UpLA0KICAgICgiU0FHX1QxIiwgICAgICAgICAiU2FnaXR0YWwiLCAgRmFsc2UsIEZhbHNlKSwNCl0NCk5fU0xPVFMgPSBsZW4oU0xPVFMpDQoNCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAIGRhdGFzZXQgZmFjdHMgKHZlcmlmaWVkIGJ5IHRoZSAyMDI2IGludGVncml0eSBhdWRpdCkg4pSADQpDQU5PTl9PUklFTlQgICAgICAgPSB7IlNhZ2l0dGFsIjogIlBJIiwgIkNvcm9uYWwiOiAiTEkiLCAiQXhpYWwiOiAiTFAifQ0KTEFUX01JTl9PRkZTRVRfTU0gID0gMjAuMA0KQ09NUEVUSVRJT04gICAgICAgID0gInJzbmEta25lZS1hYm5vcm1hbGl0eS1kZXRlY3Rpb24iDQpST09UX0NBTkRJREFURVMgICAgPSBbDQogICAgZiIva2FnZ2xlL2lucHV0L2NvbXBldGl0aW9ucy97Q09NUEVUSVRJT059IiwNCiAgICBmIi9rYWdnbGUvaW5wdXQve0NPTVBFVElUSU9OfSIsDQogICAgImQ6L0tuZWVfUlNOQV9EYXRhIiwNCiAgICAiZDovS25lZV9SU05BL2RhdGEiLA0KICAgICJkOi9LbmVlX1JTTkEiLA0KXQ0KDQoNCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAIFByZUNmZyAocHJlcHJvY2Vzc2luZykg4pSA4pSA4pSA4pSA4pSA4pSA4pSADQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgUHJlQ2ZnOg0KICAgIG5hbWU6IHN0ciA9ICJ2MiINCiAgICAjIGluLXBsYW5lDQogICAgY3JvcF9tbTogZmxvYXQgICAgICA9IDEzMC4wDQogICAgaW1nX3NpemU6IGludCAgICAgICA9IDMzNg0KICAgIHBhZF9tb2RlOiBzdHIgICAgICAgPSAiYm9yZGVyX21lZGlhbiINCiAgICByZXNpemVfbW9kZTogc3RyICAgID0gImFyZWEiDQogICAgd2lkZV9jZW50ZXI6IGJvb2wgICA9IFRydWUNCiAgICB3aWRlX3JhdGlvOiBmbG9hdCAgID0gMS4zDQogICAgIyBpbnRlbnNpdHkNCiAgICBub3JtX2xvOiBmbG9hdCAgICAgID0gMS4wDQogICAgbm9ybV9oaTogZmxvYXQgICAgICA9IDk5LjUNCiAgICBjbGlwX25lZ2F0aXZlOiBib29sID0gVHJ1ZQ0KICAgICMgdGhyb3VnaC1wbGFuZQ0KICAgIG9yZGVyX21vZGU6IHN0ciAgICAgPSAicG9zaXRpb24iDQogICAgel9tb2RlOiBzdHIgICAgICAgICA9ICJtbSINCiAgICBzdGFja19kZXB0aDogaW50ICAgID0gMjQNCiAgICB6X3N0ZXBfbW06IGZsb2F0ICAgID0gNC4wDQogICAgYmFuZDogdHVwbGUgICAgICAgICA9ICgwLjIsIDAuOCkNCiAgICAjIDIuNUQgd2luZG93cw0KICAgIGdyb3VwOiBpbnQgICAgICAgICAgPSAzDQogICAgd2luX3N0cmlkZTogaW50ICAgICA9IDINCiAgICAjIGNhbm9uaWNhbGlzYXRpb24NCiAgICByZW9yaWVudDogYm9vbCAgICAgID0gVHJ1ZQ0KICAgIGxhdF9jYW5vbjogYm9vbCAgICAgPSBUcnVlDQogICAgIyBzbG90IGxvZ2ljDQogICAgc2xvdF9wcmVmZXJfMmQ6IGJvb2wgICAgPSBUcnVlDQogICAgc2xvdF9mc19wcmlvcml0eTogc3RyICAgPSAiY3N2Ig0KICAgIHNsb3RfY3N2X2ZhbGxiYWNrOiBib29sID0gVHJ1ZQ0KDQogICAgQHByb3BlcnR5DQogICAgZGVmIG5fd2luZG93cyhzZWxmKSAtPiBpbnQ6DQogICAgICAgIHJldHVybiBtYXgoMSwgKHNlbGYuc3RhY2tfZGVwdGggLSBzZWxmLmdyb3VwKSAvLyBzZWxmLndpbl9zdHJpZGUgKyAxKQ0KDQoNClBSRVNFVFMgPSB7DQogICAgInB1YmxpYyI6IFByZUNmZygNCiAgICAgICAgbmFtZT0icHVibGljIiwgcGFkX21vZGU9InB1YmxpYyIsIHJlc2l6ZV9tb2RlPSJiaWxpbmVhciIsDQogICAgICAgIHdpZGVfY2VudGVyPUZhbHNlLCBub3JtX2xvPTEuMCwgbm9ybV9oaT05OS4wLCBjbGlwX25lZ2F0aXZlPUZhbHNlLA0KICAgICAgICBvcmRlcl9tb2RlPSJub3JtYWwiLCB6X21vZGU9ImluZGV4Iiwgc3RhY2tfZGVwdGg9MTIsIGJhbmQ9KDAuMiwgMC44KSwNCiAgICAgICAgd2luX3N0cmlkZT0xLCByZW9yaWVudD1GYWxzZSwgc2xvdF9wcmVmZXJfMmQ9RmFsc2UsIHNsb3RfY3N2X2ZhbGxiYWNrPUZhbHNlLA0KICAgICksDQogICAgInYyIjogUHJlQ2ZnKG5hbWU9InYyIiwgaW1nX3NpemU9NTE4LCBzdGFja19kZXB0aD0zMiksDQp9DQoNCg0KZGVmIGdldF9jZmcobmFtZTogc3RyID0gInYyIiwgKipvdmVycmlkZXMpIC0+IFByZUNmZzoNCiAgICBiYXNlID0gUFJFU0VUU1tuYW1lXQ0KICAgIHJldHVybiByZXBsYWNlKGJhc2UsICoqb3ZlcnJpZGVzKSBpZiBvdmVycmlkZXMgZWxzZSBiYXNlDQoNCg0KZGVmIGNmZ19mcm9tX2RpY3QoZDogZGljdCkgLT4gUHJlQ2ZnOg0KICAgICIiIlJlYnVpbGQgYSBQcmVDZmcgZnJvbSBhIEpTT04tbG9hZGVkIGRpY3QgKGxpc3RzIOKGkiB0dXBsZXMpLiIiIg0KICAgIGQgPSBkaWN0KGQpDQogICAgaWYgImJhbmQiIGluIGQ6DQogICAgICAgIGRbImJhbmQiXSA9IHR1cGxlKGRbImJhbmQiXSkNCiAgICByZXR1cm4gUHJlQ2ZnKCoqZCkNCg0KDQojIExlZ2FjeSBhbGlhc2VzDQpDUk9QX01NICAgID0gUFJFU0VUU1sidjIiXS5jcm9wX21tDQpJTUdfU0laRSAgID0gUFJFU0VUU1sidjIiXS5pbWdfc2l6ZQ0KR1JPVVBfU0laRSA9IFBSRVNFVFNbInYyIl0uZ3JvdXANCg==',
    'src/core/__init__.py': '',
    'src/data/dataset.py': 'IiIidG9yY2ggRGF0YXNldCBvdmVyIHRoZSBtZW1tYXAgY2FjaGUuIEFsbCByZWFsIHdvcmsgaGFwcGVucyBpbiBwcmVwcm9jZXNzLmxvYWRlciAobnVtcHkpLiIiIg0KaW1wb3J0IGpzb24NCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHRvcmNoDQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gc3JjLmRhdGEucHJlcHJvY2VzcyBpbXBvcnQgY2FjaGUgYXMgY2FjaGVfbW9kDQpmcm9tIHNyYy5kYXRhLnByZXByb2Nlc3MgaW1wb3J0IGxvYWRlcg0KDQoNCmNsYXNzIFJTTkFEYXRhc2V0KERhdGFzZXQpOg0KICAgICIiImRmOiBTdHVkeUluc3RhbmNlVUlEICsgdGFyZ2V0IGNvbHVtbnMgKE5hTiA9IHVubGFiZWxlZCkgWysgJzx0YXJnZXQ+X3dlaWdodCcgY29sdW1uc10uDQoNCiAgICBSZXR1cm5zIChpbWdzIHVpbnQ4IFtTLFcsMyxILFddLCBzbG90X21hc2sgW1NdLCB3aW5fbWFzayBbUyxXXSwgdGFyZ2V0cyBbMTJdLCB3ZWlnaHRzIFsxMl0pLg0KICAgIFRocm91Z2hwdXQgbm90ZXM6IHRoZSBtZW1tYXAgaXMgb3BlbmVkIGxhemlseSBwZXIgd29ya2VyIChmb3JrLXNhZmUpOyBsYWJlbHMgYXJlIGRlbnNlIGFycmF5czsgdGhlIGF1Z21lbnRhdGlvbg0KICAgIFJORyBpcyBzZWVkZWQgZnJvbSAoc2VlZCwgZXBvY2gsIGluZGV4KSBzbyBydW5zIGFyZSByZXByb2R1Y2libGUgcmVnYXJkbGVzcyBvZiB3b3JrZXIgc2NoZWR1bGluZy4iIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkZiwgY2FjaGVfcHJlZml4LCBjZmc9Tm9uZSwgaXNfdHJhaW49RmFsc2UsIG5fd2luZG93c191c2U9Tm9uZSwgc2VlZD1jb25maWcuU0VFRCwgYXVnPVRydWUpOg0KICAgICAgICBzZWxmLnByZWZpeCwgc2VsZi5jZmcgPSBjYWNoZV9wcmVmaXgsIGNmZyBvciBjYWNoZV9tb2QuY2ZnX29mKGNhY2hlX3ByZWZpeCkgICAjIGRlZmF1bHQ6IHRoZSBjYWNoZSdzIG93biBjb25maWcNCiAgICAgICAgc2VsZi5pc190cmFpbiwgc2VsZi5uX3VzZSwgc2VsZi5zZWVkLCBzZWxmLmF1ZyA9IGlzX3RyYWluLCBuX3dpbmRvd3NfdXNlLCBzZWVkLCBhdWcNCiAgICAgICAgc2VsZi5lcG9jaCA9IHRvcmNoLnplcm9zKCgpLCBkdHlwZT10b3JjaC5pbnQ2NCkuc2hhcmVfbWVtb3J5XygpDQogICAgICAgIHdpdGggb3BlbihmJ3tjYWNoZV9wcmVmaXh9Lm1ldGEuanNvbicpIGFzIGZoOg0KICAgICAgICAgICAgc3R1ZGllcyA9IGpzb24ubG9hZChmaClbJ3N0dWRpZXMnXQ0KICAgICAgICBhdmFpbGFibGUgPSBzZXQoc3R1ZGllcykNCiAgICAgICAgYWxpZ25lZCA9IGRmW2RmWydTdHVkeUluc3RhbmNlVUlEJ10uYXN0eXBlKHN0cikuaXNpbihhdmFpbGFibGUpXS5yZXNldF9pbmRleChkcm9wPVRydWUpDQogICAgICAgIHNlbGYuaWRzID0gYWxpZ25lZFsnU3R1ZHlJbnN0YW5jZVVJRCddLmFzdHlwZShzdHIpLnRvbGlzdCgpDQogICAgICAgIHNlbGYucm93cywgc2VsZi5ZLCBzZWxmLld0ID0gbG9hZGVyLmxhYmVsX2FycmF5cyhhbGlnbmVkLCB7czogaSBmb3IgaSwgcyBpbiBlbnVtZXJhdGUoc3R1ZGllcyl9KQ0KICAgICAgICBzZWxmLl9jYWNoZSA9IE5vbmUNCg0KICAgIGRlZiBzZXRfZXBvY2goc2VsZiwgZSk6DQogICAgICAgIHNlbGYuZXBvY2guZmlsbF8oaW50KGUpKQ0KDQogICAgZGVmIF9fbGVuX18oc2VsZik6DQogICAgICAgIHJldHVybiBsZW4oc2VsZi5yb3dzKQ0KDQogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGspOg0KICAgICAgICBpZiBzZWxmLl9jYWNoZSBpcyBOb25lOg0KICAgICAgICAgICAgc2VsZi5fY2FjaGUgPSBjYWNoZV9tb2QuU3R1ZHlDYWNoZShzZWxmLnByZWZpeCwgJ3InKQ0KICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoW3NlbGYuc2VlZCwgaW50KHNlbGYuZXBvY2guaXRlbSgpKSwgaW50KGspXSkNCiAgICAgICAgaW1ncywgc2xvdCwgd20gPSBsb2FkZXIubWFrZV9zYW1wbGUoc2VsZi5fY2FjaGUsIGludChzZWxmLnJvd3Nba10pLCBzZWxmLmNmZywgc2VsZi5pc190cmFpbiwgc2VsZi5uX3VzZSwgcm5nLCBzZWxmLmF1ZykNCiAgICAgICAgcmV0dXJuICh0b3JjaC5mcm9tX251bXB5KGltZ3MpLCB0b3JjaC5mcm9tX251bXB5KHNsb3QpLmZsb2F0KCksIHRvcmNoLmZyb21fbnVtcHkod20pLmZsb2F0KCksDQogICAgICAgICAgICAgICAgdG9yY2guZnJvbV9udW1weShzZWxmLllba10pLCB0b3JjaC5mcm9tX251bXB5KHNlbGYuV3Rba10pKQ0K',
    'src/data/labels.py': 'IiIiTGFiZWwgdGFibGUgZm9yIHRyYWluaW5nOiBnb2xkIGxhYmVscyArIG9wdGlvbmFsIGV4dHJhIChyZXBvcnQtZGVyaXZlZCAvIHBzZXVkbykgbGFiZWxzIHdpdGggY29uZmlkZW5jZSB3ZWlnaHRzLg0KDQpDb250cmFjdCBjb25zdW1lZCBieSBSU05BRGF0YXNldDogU3R1ZHlJbnN0YW5jZVVJRCwgdGhlIDEyIFRBUkdFVFMgKE5hTiA9IHVubGFiZWxlZCAtPiBtYXNrZWQpLCBvcHRpb25hbA0KJzx0YXJnZXQ+X3dlaWdodCcgY29sdW1ucy4gOTguNyUgb2YgdHJhaW4gc3R1ZGllcyBoYXZlIG5vIGdvbGQgbGFiZWwsIHNvIHRoZSBleHRyYSB0YWJsZSBpcyB3aGVyZSBtb3N0IHN1cGVydmlzaW9uDQpjb21lcyBmcm9tOyBnb2xkIGFsd2F5cyB3aW5zIHdoZXJlIGl0IGV4aXN0cy4NCiIiIg0KaW1wb3J0IG9zDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQoNCg0KZGVmIF9zaGEyNTZfZmlsZShwYXRoKToNCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpDQogICAgd2l0aCBvcGVuKHBhdGgsICdyYicpIGFzIHN0cmVhbToNCiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICogMTAyNCksIGInJyk6DQogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGNodW5rKQ0KICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkNCg0KDQpkZWYgYnVpbGRfbGFiZWxzKHJvb3QsIGV4dHJhX2Nzdj1Ob25lLCBleHRyYV93ZWlnaHQ9MC41LCBvdXRfY3N2PU5vbmUpOg0KICAgICIiInJvb3Q6IGNvbXBldGl0aW9uIGZvbGRlci4gZXh0cmFfY3N2OiBzYW1lIHNjaGVtYSBhcyB0cmFpbi5jc3YgKHRhcmdldHMgbWF5IGJlIHNvZnQgcHJvYmFiaWxpdGllcyBpbiBbMCwxXSksDQogICAgb3B0aW9uYWxseSB3aXRoICc8dGFyZ2V0Pl93ZWlnaHQnIGNvbHVtbnMgKGV4dHJhY3RvciBjb25maWRlbmNlKS4gLT4gRGF0YUZyYW1lIChhbmQgQ1NWIGlmIG91dF9jc3YpLiIiIg0KICAgIGlmIG5vdCBucC5pc2Zpbml0ZShleHRyYV93ZWlnaHQpIG9yIG5vdCAwLjAgPD0gZXh0cmFfd2VpZ2h0IDw9IDEuMDoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImV4dHJhX3dlaWdodCBtdXN0IGJlIGZpbml0ZSBhbmQgaW4gWzAsIDFdLCBnb3Qge2V4dHJhX3dlaWdodCFyfSIpDQogICAgdHIgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4ocm9vdCwgJ3RyYWluLmNzdicpKQ0KICAgIGlmICdTdHVkeUluc3RhbmNlVUlEJyBub3QgaW4gdHIgb3IgdHJbJ1N0dWR5SW5zdGFuY2VVSUQnXS5pc25hKCkuYW55KCk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluLmNzdiBjb250YWlucyBhIG1pc3NpbmcgU3R1ZHlJbnN0YW5jZVVJRCIpDQogICAgb3V0ID0gcGQuRGF0YUZyYW1lKHsnU3R1ZHlJbnN0YW5jZVVJRCc6IHRyWydTdHVkeUluc3RhbmNlVUlEJ10uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCl9KQ0KICAgIGlmIG91dFsnU3R1ZHlJbnN0YW5jZVVJRCddLmVxKCcnKS5hbnkoKSBvciBvdXRbJ1N0dWR5SW5zdGFuY2VVSUQnXS5kdXBsaWNhdGVkKCkuYW55KCk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluLmNzdiBtdXN0IGhhdmUgbm9uLWVtcHR5LCB1bmlxdWUgU3R1ZHlJbnN0YW5jZVVJRCB2YWx1ZXMiKQ0KICAgIGdvbGQgPSB7dDogKHRyW3RdLmFzdHlwZShmbG9hdCkgaWYgdCBpbiB0ci5jb2x1bW5zIGVsc2UgcGQuU2VyaWVzKG5wLm5hbiwgaW5kZXg9dHIuaW5kZXgpKSBmb3IgdCBpbiBjb25maWcuVEFSR0VUU30NCiAgICBmb3IgdGFyZ2V0LCB2YWx1ZXMgaW4gZ29sZC5pdGVtcygpOg0KICAgICAgICBpZiBucC5pc2luZih2YWx1ZXMudG9fbnVtcHkoZHR5cGU9ZmxvYXQpKS5hbnkoKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJnb2xkIGxhYmVscyBmb3Ige3RhcmdldCFyfSBtdXN0IGJlIGZpbml0ZSBvciBtaXNzaW5nIikNCiAgICAgICAgZmluaXRlID0gdmFsdWVzW25wLmlzZmluaXRlKHZhbHVlcyldDQogICAgICAgIGlmIG5vdCBmaW5pdGUuYmV0d2VlbigwLjAsIDEuMCkuYWxsKCk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZ29sZCBsYWJlbHMgZm9yIHt0YXJnZXQhcn0gbXVzdCBiZSBpbiBbMCwgMV0iKQ0KICAgIGV4ID0gTm9uZQ0KICAgIGlmIGV4dHJhX2NzdiBhbmQgb3MucGF0aC5leGlzdHMoZXh0cmFfY3N2KToNCiAgICAgICAgZXhfcmF3ID0gcGQucmVhZF9jc3YoZXh0cmFfY3N2KQ0KICAgICAgICBpZiAnU3R1ZHlJbnN0YW5jZVVJRCcgbm90IGluIGV4X3Jhdy5jb2x1bW5zIG9yIGV4X3Jhd1snU3R1ZHlJbnN0YW5jZVVJRCddLmlzbmEoKS5hbnkoKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHRyYSBsYWJlbHMgbXVzdCBjb250YWluIG5vbi1udWxsIFN0dWR5SW5zdGFuY2VVSUQgdmFsdWVzOiB7ZXh0cmFfY3N2fSIpDQogICAgICAgIGV4X3Jhd1snU3R1ZHlJbnN0YW5jZVVJRCddID0gZXhfcmF3WydTdHVkeUluc3RhbmNlVUlEJ10uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkNCiAgICAgICAgaWYgZXhfcmF3WydTdHVkeUluc3RhbmNlVUlEJ10uZXEoJycpLmFueSgpIG9yIGV4X3Jhd1snU3R1ZHlJbnN0YW5jZVVJRCddLmR1cGxpY2F0ZWQoKS5hbnkoKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHRyYSBsYWJlbHMgbXVzdCBoYXZlIG5vbi1lbXB0eSwgdW5pcXVlIFN0dWR5SW5zdGFuY2VVSUQgdmFsdWVzOiB7ZXh0cmFfY3N2fSIpDQogICAgICAgIHVua25vd24gPSBzb3J0ZWQoc2V0KGV4X3Jhd1snU3R1ZHlJbnN0YW5jZVVJRCddKSAtIHNldChvdXRbJ1N0dWR5SW5zdGFuY2VVSUQnXSkpDQogICAgICAgIGlmIHVua25vd246DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgIGYiZXh0cmEgbGFiZWxzIGNvbnRhaW4ge2xlbih1bmtub3duKX0gU3R1ZHlJbnN0YW5jZVVJRCB2YWx1ZXMgYWJzZW50IGZyb20gdHJhaW4uY3N2OyAiDQogICAgICAgICAgICAgICAgZiJleGFtcGxlczoge3Vua25vd25bOjVdfSINCiAgICAgICAgICAgICkNCiAgICAgICAgZXggPSBleF9yYXcuc2V0X2luZGV4KCdTdHVkeUluc3RhbmNlVUlEJykucmVpbmRleChvdXRbJ1N0dWR5SW5zdGFuY2VVSUQnXSkNCiAgICBvdXRbJ3NvdXJjZSddID0gJ25vbmUnDQogICAgYW55X2dvbGQgPSBwZC5jb25jYXQoZ29sZCwgYXhpcz0xKS5ub3RuYSgpLmFueShheGlzPTEpLnZhbHVlcw0KICAgIG91dC5sb2NbYW55X2dvbGQsICdzb3VyY2UnXSA9ICdnb2xkJw0KICAgIGZvciB0IGluIGNvbmZpZy5UQVJHRVRTOg0KICAgICAgICBnID0gZ29sZFt0XS52YWx1ZXMNCiAgICAgICAgeSA9IGcuY29weSgpDQogICAgICAgIHcgPSBucC53aGVyZShucC5pc2Zpbml0ZShnKSwgMS4wLCAwLjApDQogICAgICAgIGlmIGV4IGlzIG5vdCBOb25lIGFuZCB0IGluIGV4LmNvbHVtbnM6DQogICAgICAgICAgICBlID0gZXhbdF0uYXN0eXBlKGZsb2F0KS52YWx1ZXMNCiAgICAgICAgICAgIGlmIG5wLmlzaW5mKGUpLmFueSgpOg0KICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHRyYSBsYWJlbHMgZm9yIHt0IXJ9IG11c3QgYmUgZmluaXRlIG9yIG1pc3NpbmciKQ0KICAgICAgICAgICAgZmluaXRlID0gZVtucC5pc2Zpbml0ZShlKV0NCiAgICAgICAgICAgIGlmIG5vdCBucC5sb2dpY2FsX2FuZChmaW5pdGUgPj0gMC4wLCBmaW5pdGUgPD0gMS4wKS5hbGwoKToNCiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXh0cmEgbGFiZWxzIGZvciB7dCFyfSBtdXN0IGJlIGluIFswLCAxXSIpDQogICAgICAgICAgICBldyA9IGV4W2Yne3R9X3dlaWdodCddLmFzdHlwZShmbG9hdCkuZmlsbG5hKDEuMCkudmFsdWVzIGlmIGYne3R9X3dlaWdodCcgaW4gZXguY29sdW1ucyBlbHNlIDEuMA0KICAgICAgICAgICAgaWYgbnAuaXNzY2FsYXIoZXcpOg0KICAgICAgICAgICAgICAgIHBhc3MNCiAgICAgICAgICAgIGVsaWYgbm90IG5wLmlzZmluaXRlKGV3KS5hbGwoKSBvciAoZXcgPCAwKS5hbnkoKSBvciAoZXcgPiAxKS5hbnkoKToNCiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXh0cmEgY29uZmlkZW5jZSB3ZWlnaHRzIGZvciB7dCFyfSBtdXN0IGJlIGZpbml0ZSBhbmQgaW4gWzAsIDFdIikNCiAgICAgICAgICAgIHVzZSA9IH5ucC5pc2Zpbml0ZShnKSAmIG5wLmlzZmluaXRlKGUpDQogICAgICAgICAgICB5ID0gbnAud2hlcmUodXNlLCBlLCBnKQ0KICAgICAgICAgICAgdyA9IG5wLndoZXJlKHVzZSwgZXh0cmFfd2VpZ2h0ICogZXcsIHcpDQogICAgICAgICAgICBvdXQubG9jW3VzZSAmIChvdXRbJ3NvdXJjZSddID09ICdub25lJyksICdzb3VyY2UnXSA9ICdleHRyYScNCiAgICAgICAgb3V0W3RdID0geQ0KICAgICAgICBvdXRbZid7dH1fd2VpZ2h0J10gPSB3DQogICAgaWYgb3V0X2NzdjoNCiAgICAgICAgdGVtcG9yYXJ5ID0gZiJ7b3V0X2Nzdn0udG1wIg0KICAgICAgICBvdXQudG9fY3N2KHRlbXBvcmFyeSwgaW5kZXg9RmFsc2UpDQogICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBvdXRfY3N2KQ0KICAgICAgICBtYW5pZmVzdCA9IHsNCiAgICAgICAgICAgICJzY2hlbWFfdmVyc2lvbiI6ICJ0cmFpbi1sYWJlbHMtdjIiLA0KICAgICAgICAgICAgInRyYWluX2Nzdl9zaGEyNTYiOiBfc2hhMjU2X2ZpbGUob3MucGF0aC5qb2luKHJvb3QsICd0cmFpbi5jc3YnKSksDQogICAgICAgICAgICAiZXh0cmFfY3N2X3NoYTI1NiI6ICgNCiAgICAgICAgICAgICAgICBfc2hhMjU2X2ZpbGUoZXh0cmFfY3N2KQ0KICAgICAgICAgICAgICAgIGlmIGV4dHJhX2NzdiBhbmQgb3MucGF0aC5leGlzdHMoZXh0cmFfY3N2KQ0KICAgICAgICAgICAgICAgIGVsc2UgTm9uZQ0KICAgICAgICAgICAgKSwNCiAgICAgICAgICAgICJsYWJlbHNfY3N2X3NoYTI1NiI6IF9zaGEyNTZfZmlsZShvdXRfY3N2KSwNCiAgICAgICAgICAgICJzdHVkaWVzIjogbGVuKG91dCksDQogICAgICAgICAgICAic291cmNlX2NvdW50cyI6IG91dFsnc291cmNlJ10udmFsdWVfY291bnRzKCkudG9fZGljdCgpLA0KICAgICAgICAgICAgInBlcl90YXJnZXQiOiB7DQogICAgICAgICAgICAgICAgdDogew0KICAgICAgICAgICAgICAgICAgICAibGFiZWxlZCI6IGludChvdXRbdF0ubm90bmEoKS5zdW0oKSksDQogICAgICAgICAgICAgICAgICAgICJmdWxsX3dlaWdodCI6IGludCgob3V0W2Yne3R9X3dlaWdodCddID09IDEuMCkuc3VtKCkpLA0KICAgICAgICAgICAgICAgICAgICAicG9zaXRpdmUiOiBpbnQoKG91dFt0XSA+IDApLnN1bSgpKSwNCiAgICAgICAgICAgICAgICB9DQogICAgICAgICAgICAgICAgZm9yIHQgaW4gY29uZmlnLlRBUkdFVFMNCiAgICAgICAgICAgIH0sDQogICAgICAgIH0NCiAgICAgICAgbWFuaWZlc3RfcGF0aCA9IGYie291dF9jc3Z9Lm1hbmlmZXN0Lmpzb24iDQogICAgICAgIG1hbmlmZXN0X3RtcCA9IGYie21hbmlmZXN0X3BhdGh9LnRtcCINCiAgICAgICAgd2l0aCBvcGVuKG1hbmlmZXN0X3RtcCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06DQogICAgICAgICAgICBpbXBvcnQganNvbg0KICAgICAgICAgICAganNvbi5kdW1wKG1hbmlmZXN0LCBzdHJlYW0sIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkNCiAgICAgICAgICAgIHN0cmVhbS5mbHVzaCgpDQogICAgICAgICAgICBvcy5mc3luYyhzdHJlYW0uZmlsZW5vKCkpDQogICAgICAgIG9zLnJlcGxhY2UobWFuaWZlc3RfdG1wLCBtYW5pZmVzdF9wYXRoKQ0KICAgIHJldHVybiBvdXQNCg==',
    'src/data/__init__.py': '',
    'src/data/preprocess/bench.py': 'IiIiQ2FsaWJyYXRpb246IG1lYXN1cmUgcmVhbCB0aHJvdWdocHV0IG9uIGEgU01BTEwgc2FtcGxlIG9mIHRoZSBhY3R1YWwgZGF0YXNldCwgdGhlbiBwcm9qZWN0IHRoZSBmdWxsIGNvc3QuDQoNClJ1biBpdCBmaXJzdCBpbiBhIGZyZXNoIEthZ2dsZSBzZXNzaW9uICh0YWtlcyB+MS0zIG1pbnV0ZXMpOiBpdCB0ZWxscyB5b3UgaG93IGxvbmcgdGhlIGluZGV4IHNjYW4gYW5kIHRoZSBjYWNoZSBidWlsZA0Kd2lsbCB0YWtlIG9uIFRISVMgbWFjaGluZS9maWxlc3lzdGVtLCBob3cgbXVjaCBzY3JhdGNoIGRpc2sgdGhlIGNhY2hlIG5lZWRzLCBhbmQgd2hldGhlciB0aGUgdHJhaW5pbmcgbG9vcCdzIGRhdGENCmxvYWRpbmcgY2FuIGtlZXAgYSBHUFUgZmVkLg0KIiIiDQppbXBvcnQgb3MNCmltcG9ydCB0aW1lDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQppbXBvcnQgc3JjLmNvcmUuY29uZmlnIGFzIGNvbmZpZw0KZnJvbSAuIGltcG9ydCBpbmRleCBhcyBwaXgNCmZyb20gLiBpbXBvcnQgc2xvdHMgYXMgcHNsb3RzDQpmcm9tIC4gaW1wb3J0IHBpcGVsaW5lDQpmcm9tIC4gaW1wb3J0IGNhY2hlIGFzIHBjYWNoZQ0KZnJvbSAuIGltcG9ydCBsb2FkZXINCmZyb20gLiBpbXBvcnQgcnVubmVyDQoNCg0KZGVmIF9pdGVtc19mb3Iocm9vdCwgc3BsaXQsIHN0dWRpZXMpOg0KICAgIGJhc2UgPSBvcy5wYXRoLmpvaW4ocm9vdCwgcGl4LlNQTElUX0RJUlNbc3BsaXRdKQ0KICAgIGl0ZW1zID0gW10NCiAgICBmb3Igc3QgaW4gc3R1ZGllczoNCiAgICAgICAgZm9yIGUgaW4gc29ydGVkKG9zLnNjYW5kaXIob3MucGF0aC5qb2luKGJhc2UsIHN0KSksIGtleT1sYW1iZGEgeDogeC5uYW1lKToNCiAgICAgICAgICAgIGlmIGUuaXNfZGlyKCk6DQogICAgICAgICAgICAgICAgaXRlbXMuYXBwZW5kKChzcGxpdCwgc3QsIGUubmFtZSwgZS5wYXRoKSkNCiAgICByZXR1cm4gaXRlbXMNCg0KDQpkZWYgY2FsaWJyYXRlKHJvb3QsIHNwbGl0PSd0cmFpbicsIG5fc3R1ZGllcz0yNCwgd29ya2Vycz1Ob25lLCBwcmVzZXQ9J3YyJywgc2NyYXRjaD1Ob25lLCBzdGVwX3NlY29uZHM9MS4wLA0KICAgICAgICAgICAgICBiYXRjaF9zaXplPWNvbmZpZy5CQVRDSF9TSVpFLCBuX3VzZT00KToNCiAgICBmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgUHJvY2Vzc1Bvb2xFeGVjdXRvcg0KICAgIGltcG9ydCBtdWx0aXByb2Nlc3NpbmcgYXMgbXANCiAgICBjZmcgPSBjb25maWcuZ2V0X2NmZyhwcmVzZXQpDQogICAgd29ya2VycyA9IHdvcmtlcnMgb3IgbWF4KDIsIChvcy5jcHVfY291bnQoKSBvciA0KSkNCiAgICBzY3JhdGNoID0gc2NyYXRjaCBvciBydW5uZXIucGlja19zY3JhdGNoKHZlcmJvc2U9RmFsc2UpDQogICAgYmFzZSA9IG9zLnBhdGguam9pbihyb290LCBwaXguU1BMSVRfRElSU1tzcGxpdF0pDQogICAgc3R1ZGllcyA9IHNvcnRlZChlLm5hbWUgZm9yIGUgaW4gb3Muc2NhbmRpcihiYXNlKSBpZiBlLmlzX2RpcigpKVs6bl9zdHVkaWVzXQ0KICAgIGl0ZW1zID0gX2l0ZW1zX2Zvcihyb290LCBzcGxpdCwgc3R1ZGllcykNCiAgICBuX3RvdGFsX3NlcmllcyA9IG5fdG90YWxfc3R1ZGllcyA9IE5vbmUNCiAgICBmb3IgbmFtZSwgYXR0ciBpbiAoKCd0cmFpbl9zZXJpZXMuY3N2JyBpZiBzcGxpdCA9PSAndHJhaW4nIGVsc2UgJ3Rlc3Rfc2VyaWVzLmNzdicsICdzZXJpZXMnKSwgKCd0cmFpbi5jc3YnIGlmIHNwbGl0ID09ICd0cmFpbicgZWxzZSAndGVzdC5jc3YnLCAnc3R1ZGllcycpKToNCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihyb290LCBuYW1lKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToNCiAgICAgICAgICAgIG4gPSBsZW4ocGQucmVhZF9jc3YocCwgdXNlY29scz1bMF0pKQ0KICAgICAgICAgICAgaWYgYXR0ciA9PSAnc2VyaWVzJzoNCiAgICAgICAgICAgICAgICBuX3RvdGFsX3NlcmllcyA9IG4NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgbl90b3RhbF9zdHVkaWVzID0gbg0KICAgIG5fdG90YWxfc2VyaWVzID0gbl90b3RhbF9zZXJpZXMgb3IgbGVuKGl0ZW1zKSAqIDQ0MDcgLy8gbWF4KGxlbihzdHVkaWVzKSwgMSkNCiAgICBuX3RvdGFsX3N0dWRpZXMgPSBuX3RvdGFsX3N0dWRpZXMgb3IgNDQwNw0KICAgIHJlcyA9IHt9DQogICAgIyAtLS0tIDEuIGluZGV4IHNjYW4NCiAgICB0ID0gdGltZS50aW1lKCkNCiAgICB0cnk6DQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz13b3JrZXJzLCBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdmb3JrJykpDQogICAgICAgIHJvd3MgPSBsaXN0KGV4Lm1hcChwaXguc2Nhbl9zZXJpZXMsIGl0ZW1zLCBjaHVua3NpemU9NCkpDQogICAgICAgIGV4LnNodXRkb3duKCkNCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICByb3dzID0gW3BpeC5zY2FuX3NlcmllcyhpKSBmb3IgaSBpbiBpdGVtc10NCiAgICBkdCA9IHRpbWUudGltZSgpIC0gdA0KICAgIGRmID0gcGl4LmF0dGFjaF9jc3ZfZmxhZ3MocGQuRGF0YUZyYW1lKHJvd3MpLCByb290KQ0KICAgIGZpbGVzID0gaW50KGRmWyduX2ZpbGVzJ10uc3VtKCkpDQogICAgcmVzWydpbmRleF9zX3Blcl9zZXJpZXMnXSA9IGR0IC8gbGVuKGl0ZW1zKQ0KICAgIHJlc1snaW5kZXhfZmlsZXNfcGVyX3MnXSA9IGZpbGVzIC8gZHQNCiAgICByZXNbJ2luZGV4X21pbnV0ZXNfZnVsbCddID0gcmVzWydpbmRleF9zX3Blcl9zZXJpZXMnXSAqIG5fdG90YWxfc2VyaWVzIC8gNjANCiAgICAjIC0tLS0gMi4gY2FjaGUgYnVpbGQNCiAgICBhbm4sIHRhYiA9IHBzbG90cy5hc3NpZ25fYWxsKGRmLCBjZmcuc2xvdF9wcmVmZXJfMmQsIGNmZy5zbG90X2ZzX3ByaW9yaXR5LCBjZmcuc2xvdF9jc3ZfZmFsbGJhY2spDQogICAgc2lkZXMgPSBwaXBlbGluZS5zdHVkeV9zaWRlcyhkZikNCiAgICByZWNvcmRzID0gcGlwZWxpbmUuaW5kZXhfdG9fcmVjb3Jkcyhhbm4pDQogICAgc2xvdF9yb3dzID0ge3M6IHRhYi5sb2Nbc10udG9fZGljdCgpIGZvciBzIGluIHN0dWRpZXMgaWYgcyBpbiB0YWIuaW5kZXh9DQogICAgcHJlZml4ID0gb3MucGF0aC5qb2luKHNjcmF0Y2gsICdjYWxpYl9jYWNoZScpDQogICAgY2FjaGUsIHN0ID0gcGNhY2hlLmJ1aWxkX2NhY2hlKHByZWZpeCwgc3R1ZGllcywgc2xvdF9yb3dzLCByZWNvcmRzLCBzaWRlcywgY2ZnLCB3b3JrZXJzLCByZXN1bWU9RmFsc2UsIHByb2dyZXNzPUZhbHNlKQ0KICAgIHJlc1snY2FjaGVfc19wZXJfc3R1ZHlfd2FsbCddID0gc3RbJ3NlY29uZHMnXSAvIGxlbihzdHVkaWVzKQ0KICAgIHJlc1snY2FjaGVfY3B1X3NfcGVyX3N0dWR5J10gPSBzdFsnc2Vjb25kcyddICogd29ya2VycyAvIGxlbihzdHVkaWVzKQ0KICAgIHJlc1snY2FjaGVfbWludXRlc19mdWxsJ10gPSByZXNbJ2NhY2hlX3NfcGVyX3N0dWR5X3dhbGwnXSAqIG5fdG90YWxfc3R1ZGllcyAvIDYwDQogICAgcmVzWydjYWNoZV9nYl9mdWxsJ10gPSBydW5uZXIuZXN0aW1hdGVfY2FjaGVfZ2Iobl90b3RhbF9zdHVkaWVzLCBjZmcpDQogICAgIyAtLS0tIDMuIHRyYWluaW5nIGxvb3AgZGF0YSByYXRlIChzaW5nbGUgcHJvY2Vzcywgd2l0aCBhdWdtZW50YXRpb24pDQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApDQogICAgaWR4ID0gW2kgZm9yIGkgaW4gcmFuZ2UobGVuKHN0dWRpZXMpKSBpZiBjYWNoZS5zbG90W2ldLmFueSgpXQ0KICAgIHQgPSB0aW1lLnRpbWUoKQ0KICAgIHJlcHMgPSAzDQogICAgZm9yIF8gaW4gcmFuZ2UocmVwcyk6DQogICAgICAgIGZvciBpIGluIGlkeDoNCiAgICAgICAgICAgIGxvYWRlci5tYWtlX3NhbXBsZShjYWNoZSwgaSwgY2ZnLCBUcnVlLCBuX3VzZSwgcm5nLCBUcnVlKQ0KICAgIHBlciA9ICh0aW1lLnRpbWUoKSAtIHQpIC8gKHJlcHMgKiBsZW4oaWR4KSkNCiAgICByZXNbJ2xvYWRlcl9tc19wZXJfc2FtcGxlXzFwcm9jJ10gPSBwZXIgKiAxMDAwDQogICAgcmVzWydsb2FkZXJfc2FtcGxlc19wZXJfc18xcHJvYyddID0gMS4wIC8gcGVyDQogICAgbmVlZCA9IGJhdGNoX3NpemUgLyBzdGVwX3NlY29uZHMNCiAgICByZXNbJ2xvYWRlcl9uZWVkZWRfc2FtcGxlc19wZXJfcyddID0gbmVlZA0KICAgIHJlc1snbG9hZGVyX2hlYWRyb29tX3gnXSA9ICgxLjAgLyBwZXIpICogbWluKDQsIHdvcmtlcnMpIC8gbmVlZA0KICAgIGZyZWUgPSBwY2FjaGUuc2h1dGlsLmRpc2tfdXNhZ2Uoc2NyYXRjaCkuZnJlZSAvIDFlOQ0KICAgIHJlc1snc2NyYXRjaF9mcmVlX2diJ10gPSBmcmVlDQogICAgcmVzWydzdHVkaWVzX3NhbXBsZWQnXSA9IGxlbihzdHVkaWVzKQ0KICAgIHJlc1snc2VyaWVzX3NhbXBsZWQnXSA9IGxlbihpdGVtcykNCiAgICBmb3IgayBpbiAoJ2NhbGliX2NhY2hlLmltYWdlcy51OCcsICdjYWxpYl9jYWNoZS52YWxpZC51MScsICdjYWxpYl9jYWNoZS5zbG90LnUxJywgJ2NhbGliX2NhY2hlLmRvbmUudTEnLCAnY2FsaWJfY2FjaGUubWV0YS5qc29uJywgJ2NhbGliX2NhY2hlLm1ldGEuY3N2Jyk6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIG9zLnJlbW92ZShvcy5wYXRoLmpvaW4oc2NyYXRjaCwgaykpDQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgcGFzcw0KICAgIHByaW50KGYic2FtcGxlOiB7bGVuKHN0dWRpZXMpfSBzdHVkaWVzIC8ge2xlbihpdGVtcyl9IHNlcmllcyAvIHtmaWxlc30gc2xpY2VzIG9uIHt3b3JrZXJzfSB3b3JrZXJzIChwcmVzZXQge2NmZy5uYW1lfSkiKQ0KICAgIHByaW50KGYiICBpbmRleCBzY2FuICAgOiB7cmVzWydpbmRleF9maWxlc19wZXJfcyddOi4wZn0gZmlsZXMvcyAgLT4gZnVsbCB0cmFpbiBzY2FuIH4ge3Jlc1snaW5kZXhfbWludXRlc19mdWxsJ106LjFmfSBtaW4iKQ0KICAgIHByaW50KGYiICBjYWNoZSBidWlsZCAgOiB7cmVzWydjYWNoZV9zX3Blcl9zdHVkeV93YWxsJ106LjJmfSBzL3N0dWR5IHdhbGwgKHtyZXNbJ2NhY2hlX2NwdV9zX3Blcl9zdHVkeSddOi4xZn0gY3B1LXMpIg0KICAgICAgICAgIGYiICAtPiBmdWxsIHRyYWluIGNhY2hlIH4ge3Jlc1snY2FjaGVfbWludXRlc19mdWxsJ106LjBmfSBtaW4sIHtyZXNbJ2NhY2hlX2diX2Z1bGwnXTouMWZ9IEdCIChzY3JhdGNoIGZyZWUge2ZyZWU6LjBmfSBHQikiKQ0KICAgIHByaW50KGYiICB0cmFpbiBsb2FkZXIgOiB7cmVzWydsb2FkZXJfbXNfcGVyX3NhbXBsZV8xcHJvYyddOi4wZn0gbXMvc2FtcGxlL3Byb2MgLT4ge3Jlc1snbG9hZGVyX2hlYWRyb29tX3gnXTouMWZ9eCBoZWFkcm9vbSAiDQogICAgICAgICAgZiJvdmVyIHRoZSB7bmVlZDouMGZ9IHNhbXBsZXMvcyBhIHtzdGVwX3NlY29uZHM6LjFmfXMgc3RlcCB3aXRoIGJhdGNoIHtiYXRjaF9zaXplfSBuZWVkcyAoNCB3b3JrZXJzKSIpDQogICAgaWYgcmVzWydjYWNoZV9nYl9mdWxsJ10gPiBmcmVlICogMC45MjoNCiAgICAgICAgcHJpbnQoJyAgISEgY2FjaGUgZG9lcyBub3QgZml0IHRoZSBzY3JhdGNoIGRpc2s6IHVzZSBydW5uZXIuZml0X2NhY2hlX2NmZyguLi4pIHRvIHBpY2sgYSBzbWFsbGVyIGltZ19zaXplL3N0YWNrX2RlcHRoJykNCiAgICByZXR1cm4gcmVzDQo=',
    'src/data/preprocess/cache.py': 'IiIiTWVtb3J5LW1hcHBlZCB1aW50OCBzdHVkeSBjYWNoZSArIGEgcGFyYWxsZWwgYnVpbGRlciB3aXRoIG5vIGludGVyLXByb2Nlc3MgYXJyYXkgdHJhZmZpYy4NCg0KTGF5b3V0IChvbmUgY29udGlndW91cyAxNiBNQiBibG9jayBwZXIgc3R1ZHkgYXQgRD0yNCwgUz02LCAzMzZeMiAtPiBvbmUgc2VxdWVudGlhbCByZWFkIHBlciBzYW1wbGUpOg0KICAgIDxwcmVmaXg+LmltYWdlcy51OCAgIFtOLCBTLCBELCBILCBXXSB1aW50OCAgICAgICAgPHByZWZpeD4udmFsaWQudTEgIFtOLCBTLCBEXQ0KICAgIDxwcmVmaXg+LnNsb3QudTEgICAgIFtOLCBTXSAgICAgICAgICAgICAgICAgICAgICAgPHByZWZpeD4uZG9uZS51MSAgIFtOXQ0KICAgIDxwcmVmaXg+Lm1ldGEuanNvbiAgIHN0dWR5IGlkcywgY2ZnLCBzaGFwZXMgICAgICAgPHByZWZpeD4ubWV0YS5jc3YgIHBlci1zdHVkeSBzaWRlL3NvdXJjZS9xYw0KDQpUaHJvdWdocHV0IGRlc2lnbjoNCiAgKiB3b3JrZXJzIHdyaXRlIHN0cmFpZ2h0IGludG8gdGhlIHNoYXJlZCBtZW1tYXAgKE1BUF9TSEFSRUQpIC0+IG5vdGhpbmcgYnV0IGEgdGlueSBkaWN0IGlzIHBpY2tsZWQgYmFjaw0KICAqIGxvbmdlc3QtcHJvY2Vzc2luZy10aW1lLWZpcnN0IHNjaGVkdWxpbmcgKDMtRCBzZXJpZXMgYXJlIH41eCBhIDItRCBvbmUpIHJlbW92ZXMgdGhlIHN0cmFnZ2xlciB0YWlsDQogICogcmVzdW1hYmxlOiBgZG9uZWAgZmxhZ3MgbGl2ZSBpbiBhIG1lbW1hcCwgc28gYSBraWxsZWQgcnVuIHJlc3RhcnRzIHdoZXJlIGl0IHN0b3BwZWQNCiAgKiBvcHRpb25hbCBzdHJlYW1pbmcgaG9vayAoYG9uX3JlYWR5YCkgbGV0cyBpbmZlcmVuY2UgcnVuIHRoZSBHUFUgb24gY2h1bmsgayB3aGlsZSBDUFVzIGJ1aWxkIGNodW5rIGsrMQ0KIiIiDQppbXBvcnQgb3MNCmltcG9ydCBqc29uDQppbXBvcnQgdGltZQ0KaW1wb3J0IHNodXRpbA0KaW1wb3J0IG11bHRpcHJvY2Vzc2luZyBhcyBtcA0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0DQpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgUHJvY2Vzc1Bvb2xFeGVjdXRvciwgVGhyZWFkUG9vbEV4ZWN1dG9yLCBhc19jb21wbGV0ZWQNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQpmcm9tIC4gaW1wb3J0IHBpcGVsaW5lDQoNCl9HID0ge30NCg0KDQpkZWYgX3BhdGhzKHByZWZpeCk6DQogICAgcmV0dXJuIHtrOiBmJ3twcmVmaXh9Lnt2fScgZm9yIGssIHYgaW4gZGljdChpbWFnZXM9J2ltYWdlcy51OCcsIHZhbGlkPSd2YWxpZC51MScsIHNsb3Q9J3Nsb3QudTEnLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRvbmU9J2RvbmUudTEnLCBlcnJvcnM9J2Vycm9ycy51NCcsIGZhaWxlZD0nZmFpbGVkLnUxJywNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwYWRkZWQ9J3BhZGRlZC51MScsIHBzX21pc3Npbmc9J3BzX21pc3NpbmcudTEnLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1ldGE9J21ldGEuanNvbicsIGNzdj0nbWV0YS5jc3YnKS5pdGVtcygpfQ0KDQoNCmNsYXNzIFN0dWR5Q2FjaGU6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByZWZpeCwgbW9kZT0ncicpOg0KICAgICAgICBzZWxmLnByZWZpeCwgc2VsZi5tb2RlID0gcHJlZml4LCBtb2RlDQogICAgICAgIHAgPSBfcGF0aHMocHJlZml4KQ0KICAgICAgICB3aXRoIG9wZW4ocFsnbWV0YSddKSBhcyBmaDoNCiAgICAgICAgICAgIHNlbGYubWV0YSA9IGpzb24ubG9hZChmaCkNCiAgICAgICAgaWYgc2VsZi5tZXRhLmdldCgic2NoZW1hX3ZlcnNpb24iKSAhPSAicHJlcHJvY2Vzcy1jYWNoZS12MiI6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgIGYiVW5zdXBwb3J0ZWQgb3IgbGVnYWN5IGNhY2hlIG1ldGFkYXRhIGF0IHtwWydtZXRhJ119OyByZWJ1aWxkIHdpdGggLS1mcmVzaF9wcmVwcm9jZXNzaW5nIg0KICAgICAgICAgICAgKQ0KICAgICAgICBzZWxmLnN0dWRpZXMgPSBzZWxmLm1ldGFbJ3N0dWRpZXMnXQ0KICAgICAgICBOLCBTLCBELCBIID0gc2VsZi5tZXRhWydOJ10sIHNlbGYubWV0YVsnUyddLCBzZWxmLm1ldGFbJ0QnXSwgc2VsZi5tZXRhWydIJ10NCiAgICAgICAgZXhwZWN0ZWRfc2l6ZXMgPSB7DQogICAgICAgICAgICAnaW1hZ2VzJzogTiAqIFMgKiBEICogSCAqIEgsDQogICAgICAgICAgICAndmFsaWQnOiBOICogUyAqIEQsDQogICAgICAgICAgICAnc2xvdCc6IE4gKiBTLA0KICAgICAgICAgICAgJ2RvbmUnOiBOLA0KICAgICAgICAgICAgJ2Vycm9ycyc6IE4gKiA0LA0KICAgICAgICAgICAgJ2ZhaWxlZCc6IE4gKiBTLA0KICAgICAgICAgICAgJ3BhZGRlZCc6IE4gKiBTLA0KICAgICAgICAgICAgJ3BzX21pc3NpbmcnOiBOICogUywNCiAgICAgICAgfQ0KICAgICAgICBmb3Iga2V5LCBleHBlY3RlZF9zaXplIGluIGV4cGVjdGVkX3NpemVzLml0ZW1zKCk6DQogICAgICAgICAgICBhY3R1YWxfc2l6ZSA9IG9zLnBhdGguZ2V0c2l6ZShwW2tleV0pIGlmIG9zLnBhdGguZXhpc3RzKHBba2V5XSkgZWxzZSAtMQ0KICAgICAgICAgICAgaWYgYWN0dWFsX3NpemUgIT0gZXhwZWN0ZWRfc2l6ZToNCiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgICAgICBmIkNhY2hlIHN0b3JhZ2UgbWlzbWF0Y2ggZm9yIHtwW2tleV19OiBleHBlY3RlZCB7ZXhwZWN0ZWRfc2l6ZX0gYnl0ZXMsICINCiAgICAgICAgICAgICAgICAgICAgZiJmb3VuZCB7YWN0dWFsX3NpemV9OyByZWJ1aWxkIHdpdGggLS1mcmVzaF9wcmVwcm9jZXNzaW5nIg0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgbSA9ICdyKycgaWYgbW9kZSA9PSAncisnIGVsc2UgJ3InDQogICAgICAgIHNlbGYuaW1hZ2VzID0gbnAubWVtbWFwKHBbJ2ltYWdlcyddLCBucC51aW50OCwgbSwgc2hhcGU9KE4sIFMsIEQsIEgsIEgpKQ0KICAgICAgICBzZWxmLnZhbGlkID0gbnAubWVtbWFwKHBbJ3ZhbGlkJ10sIG5wLnVpbnQ4LCBtLCBzaGFwZT0oTiwgUywgRCkpDQogICAgICAgIHNlbGYuc2xvdCA9IG5wLm1lbW1hcChwWydzbG90J10sIG5wLnVpbnQ4LCBtLCBzaGFwZT0oTiwgUykpDQogICAgICAgIHNlbGYuZG9uZSA9IG5wLm1lbW1hcChwWydkb25lJ10sIG5wLnVpbnQ4LCBtLCBzaGFwZT0oTiwpKQ0KICAgICAgICBzZWxmLmVycm9ycyA9IG5wLm1lbW1hcChwWydlcnJvcnMnXSwgbnAudWludDMyLCBtLCBzaGFwZT0oTiwpKQ0KICAgICAgICBzZWxmLmZhaWxlZCA9IG5wLm1lbW1hcChwWydmYWlsZWQnXSwgbnAudWludDgsIG0sIHNoYXBlPShOLCBTKSkNCiAgICAgICAgc2VsZi5wYWRkZWQgPSBucC5tZW1tYXAocFsncGFkZGVkJ10sIG5wLnVpbnQ4LCBtLCBzaGFwZT0oTiwgUykpDQogICAgICAgIHNlbGYucHNfbWlzc2luZyA9IG5wLm1lbW1hcChwWydwc19taXNzaW5nJ10sIG5wLnVpbnQ4LCBtLCBzaGFwZT0oTiwgUykpDQogICAgICAgIHNlbGYuaW5kZXggPSB7czogaSBmb3IgaSwgcyBpbiBlbnVtZXJhdGUoc2VsZi5zdHVkaWVzKX0NCiAgICAgICAgIyBIaW50IHJhbmRvbS1hY2Nlc3MgbWVtbWFwcyB0byBhdm9pZCBleGNlc3NpdmUgcmVhZC1haGVhZDsgdGhpcyBpcyBhZHZpc29yeS4NCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaW1wb3J0IG1tYXAgYXMgX3B5X21tYXANCiAgICAgICAgICAgIGFkdl9mbGFnID0gZ2V0YXR0cihfcHlfbW1hcCwgJ01BRFZfUkFORE9NJywgZ2V0YXR0cihfcHlfbW1hcCwgJ01BRFZfTk9STUFMJywgTm9uZSkpIGlmIG1vZGUgPT0gJ3InIGVsc2UgZ2V0YXR0cihfcHlfbW1hcCwgJ01BRFZfTk9STUFMJywgTm9uZSkNCiAgICAgICAgICAgIGlmIGFkdl9mbGFnIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgICAgIGZvciBhIGluIChzZWxmLmltYWdlcywgc2VsZi52YWxpZCwgc2VsZi5zbG90LCBzZWxmLmRvbmUpOg0KICAgICAgICAgICAgICAgICAgICBtbSA9IGdldGF0dHIoYSwgJ19tbWFwJywgTm9uZSkgb3IgZ2V0YXR0cihnZXRhdHRyKGEsICdiYXNlJywgTm9uZSksICdfbW1hcCcsIE5vbmUpDQogICAgICAgICAgICAgICAgICAgIGlmIG1tIGFuZCBoYXNhdHRyKG1tLCAnbWFkdmlzZScpOg0KICAgICAgICAgICAgICAgICAgICAgICAgbW0ubWFkdmlzZShhZHZfZmxhZykNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIHBhc3MNCg0KICAgIGRlZiBfX2xlbl9fKHNlbGYpOg0KICAgICAgICByZXR1cm4gbGVuKHNlbGYuc3R1ZGllcykNCg0KICAgIEBzdGF0aWNtZXRob2QNCiAgICBkZWYgY3JlYXRlKHByZWZpeCwgc3R1ZGllcywgY2ZnLCBzb3VyY2Vfc2lnbmF0dXJlPU5vbmUpOg0KICAgICAgICBzdHVkaWVzID0gbGlzdChzdHVkaWVzKQ0KICAgICAgICBpZiBub3Qgc3R1ZGllcyBvciBsZW4oc3R1ZGllcykgIT0gbGVuKHNldChzdHVkaWVzKSk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYWNoZSBzdHVkaWVzIG11c3QgYmUgYSBub24tZW1wdHkgbGlzdCBvZiB1bmlxdWUgU3R1ZHlJbnN0YW5jZVVJRCB2YWx1ZXMiKQ0KICAgICAgICBOLCBTLCBELCBIID0gbGVuKHN0dWRpZXMpLCBjb25maWcuTl9TTE9UUywgY2ZnLnN0YWNrX2RlcHRoLCBjZmcuaW1nX3NpemUNCiAgICAgICAgbmVlZCA9IE4gKiBTICogRCAqIEggKiBIICsgTiAqIFMgKiBEICsgTiAqIFMgKyBOICsgNCAqIE4gKyAzICogTiAqIFMNCiAgICAgICAgZCA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgocHJlZml4KSkNCiAgICAgICAgb3MubWFrZWRpcnMoZCwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgZnJlZSA9IHNodXRpbC5kaXNrX3VzYWdlKGQpLmZyZWUNCiAgICAgICAgaWYgbmVlZCAqIDEuMDIgPiBmcmVlOg0KICAgICAgICAgICAgcmFpc2UgT1NFcnJvcihmJ2NhY2hlIG5lZWRzIHtuZWVkIC8gMWU5Oi4xZn0gR0IgYnV0IG9ubHkge2ZyZWUgLyAxZTk6LjFmfSBHQiBmcmVlIGluIHtkfScpDQogICAgICAgIHAgPSBfcGF0aHMocHJlZml4KQ0KICAgICAgICBucC5tZW1tYXAocFsnaW1hZ2VzJ10sIG5wLnVpbnQ4LCAndysnLCBzaGFwZT0oTiwgUywgRCwgSCwgSCkpLmZsdXNoKCkNCiAgICAgICAgbnAubWVtbWFwKHBbJ3ZhbGlkJ10sIG5wLnVpbnQ4LCAndysnLCBzaGFwZT0oTiwgUywgRCkpLmZsdXNoKCkNCiAgICAgICAgbnAubWVtbWFwKHBbJ3Nsb3QnXSwgbnAudWludDgsICd3KycsIHNoYXBlPShOLCBTKSkuZmx1c2goKQ0KICAgICAgICBucC5tZW1tYXAocFsnZG9uZSddLCBucC51aW50OCwgJ3crJywgc2hhcGU9KE4sKSkuZmx1c2goKQ0KICAgICAgICBucC5tZW1tYXAocFsnZXJyb3JzJ10sIG5wLnVpbnQzMiwgJ3crJywgc2hhcGU9KE4sKSkuZmx1c2goKQ0KICAgICAgICBucC5tZW1tYXAocFsnZmFpbGVkJ10sIG5wLnVpbnQ4LCAndysnLCBzaGFwZT0oTiwgUykpLmZsdXNoKCkNCiAgICAgICAgbnAubWVtbWFwKHBbJ3BhZGRlZCddLCBucC51aW50OCwgJ3crJywgc2hhcGU9KE4sIFMpKS5mbHVzaCgpDQogICAgICAgIG5wLm1lbW1hcChwWydwc19taXNzaW5nJ10sIG5wLnVpbnQ4LCAndysnLCBzaGFwZT0oTiwgUykpLmZsdXNoKCkNCiAgICAgICAgbWV0YSA9IGRpY3QoDQogICAgICAgICAgICBzY2hlbWFfdmVyc2lvbj0icHJlcHJvY2Vzcy1jYWNoZS12MiIsDQogICAgICAgICAgICBzdHVkaWVzPXN0dWRpZXMsDQogICAgICAgICAgICBOPU4sDQogICAgICAgICAgICBTPVMsDQogICAgICAgICAgICBEPUQsDQogICAgICAgICAgICBIPUgsDQogICAgICAgICAgICBjZmc9YXNkaWN0KGNmZyksDQogICAgICAgICAgICBzb3VyY2Vfc2lnbmF0dXJlPXNvdXJjZV9zaWduYXR1cmUsDQogICAgICAgICkNCiAgICAgICAgdGVtcG9yYXJ5ID0gZiJ7cFsnbWV0YSddfS50bXAiDQogICAgICAgIHdpdGggb3Blbih0ZW1wb3JhcnksICd3JywgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6DQogICAgICAgICAgICBqc29uLmR1bXAobWV0YSwgZmgsIHNvcnRfa2V5cz1UcnVlKQ0KICAgICAgICAgICAgZmguZmx1c2goKQ0KICAgICAgICAgICAgb3MuZnN5bmMoZmguZmlsZW5vKCkpDQogICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBwWydtZXRhJ10pDQogICAgICAgIHJldHVybiBTdHVkeUNhY2hlKHByZWZpeCwgJ3IrJykNCg0KICAgIGRlZiBzbG90X3N0YWNrKHNlbGYsIGksIHMpOg0KICAgICAgICByZXR1cm4gc2VsZi5pbWFnZXNbaSwgc10NCg0KICAgIGRlZiBmbHVzaChzZWxmKToNCiAgICAgICAgZm9yIGEgaW4gKA0KICAgICAgICAgICAgc2VsZi5pbWFnZXMsIHNlbGYudmFsaWQsIHNlbGYuc2xvdCwgc2VsZi5kb25lLCBzZWxmLmVycm9ycywNCiAgICAgICAgICAgIHNlbGYuZmFpbGVkLCBzZWxmLnBhZGRlZCwgc2VsZi5wc19taXNzaW5nLA0KICAgICAgICApOg0KICAgICAgICAgICAgYS5mbHVzaCgpDQoNCg0KZGVmIGNmZ19vZihwcmVmaXgpOg0KICAgICIiIlRoZSBQcmVDZmcgYSBjYWNoZSB3YXMgYnVpbHQgd2l0aCAoc2luZ2xlIHNvdXJjZSBvZiB0cnV0aCBmb3IgdHJhaW5pbmcvaW5mZXJlbmNlIG9uIHRoYXQgY2FjaGUpLiIiIg0KICAgIHdpdGggb3BlbihfcGF0aHMocHJlZml4KVsnbWV0YSddKSBhcyBmaDoNCiAgICAgICAgcmV0dXJuIGNvbmZpZy5jZmdfZnJvbV9kaWN0KGpzb24ubG9hZChmaClbJ2NmZyddKQ0KDQoNCmRlZiBfd29yayhpKToNCiAgICBnID0gX0cNCiAgICBzdCA9IGdbJ3N0dWRpZXMnXVtpXQ0KICAgIHJlcyA9IHBpcGVsaW5lLnByZXBhcmVfc3R1ZHkoc3QsIGdbJ3Nsb3Rfcm93cyddLmdldChzdCwge30pLCBnWydyZWNvcmRzJ10sIGdbJ3NpZGVzJ10uZ2V0KHN0LCB7fSkuZ2V0KCdzaWRlJywgJ1UnKSwgZ1snY2ZnJ10pDQogICAgYyA9IGdbJ2NhY2hlJ10NCiAgICBjLmltYWdlc1tpXSA9IHJlc1snc3RhY2snXQ0KICAgIGMudmFsaWRbaV0gPSByZXNbJ3ZhbGlkJ10NCiAgICBjLnNsb3RbaV0gPSByZXNbJ3Nsb3RfbWFzayddDQogICAgZXhwZWN0ZWQgPSBucC5hcnJheSgNCiAgICAgICAgW2Jvb2woZ1snc2xvdF9yb3dzJ10uZ2V0KHN0LCB7fSkuZ2V0KG5hbWUpKSBmb3IgbmFtZSwgXywgXywgXyBpbiBjb25maWcuU0xPVFNdLA0KICAgICAgICBkdHlwZT1ib29sLA0KICAgICkNCiAgICBzbG90X2Vycm9ycyA9IG5wLmFycmF5KA0KICAgICAgICBbYm9vbChyZXNbJ2luZm8nXS5nZXQobmFtZSwge30pLmdldCgnZXJyb3JzJykpIGZvciBuYW1lLCBfLCBfLCBfIGluIGNvbmZpZy5TTE9UU10sDQogICAgICAgIGR0eXBlPWJvb2wsDQogICAgKQ0KICAgIGZhaWxlZCA9IGV4cGVjdGVkICYgKChyZXNbJ3Nsb3RfbWFzayddID09IDApIHwgc2xvdF9lcnJvcnMpDQogICAgZXJycyA9IHN1bShsZW4odi5nZXQoJ2Vycm9ycycsIFtdKSkgZm9yIHYgaW4gcmVzWydpbmZvJ10udmFsdWVzKCkpDQogICAgYy5lcnJvcnNbaV0gPSBlcnJzDQogICAgYy5mYWlsZWRbaV0gPSBmYWlsZWQuYXN0eXBlKG5wLnVpbnQ4KQ0KICAgIGMucGFkZGVkW2ldID0gbnAuYXJyYXkoDQogICAgICAgIFtib29sKHJlc1snaW5mbyddLmdldChuYW1lLCB7fSkuZ2V0KCdwYWRkZWQnKSkgZm9yIG5hbWUsIF8sIF8sIF8gaW4gY29uZmlnLlNMT1RTXSwNCiAgICAgICAgZHR5cGU9bnAudWludDgsDQogICAgKQ0KICAgIGMucHNfbWlzc2luZ1tpXSA9IG5wLmFycmF5KA0KICAgICAgICBbYm9vbChyZXNbJ2luZm8nXS5nZXQobmFtZSwge30pLmdldCgncHNfbWlzc2luZycpKSBmb3IgbmFtZSwgXywgXywgXyBpbiBjb25maWcuU0xPVFNdLA0KICAgICAgICBkdHlwZT1ucC51aW50OCwNCiAgICApDQogICAgYy5kb25lW2ldID0gMQ0KICAgIHBhZCA9IHN1bShib29sKHYuZ2V0KCdwYWRkZWQnKSkgZm9yIHYgaW4gcmVzWydpbmZvJ10udmFsdWVzKCkpDQogICAgbWlzcyA9IHN1bShib29sKHYuZ2V0KCdwc19taXNzaW5nJykpIGZvciB2IGluIHJlc1snaW5mbyddLnZhbHVlcygpKQ0KICAgIHJldHVybiBpLCBpbnQocmVzWydzbG90X21hc2snXS5zdW0oKSksIGVycnMsIHBhZCwgbWlzcywgaW50KGZhaWxlZC5zdW0oKSkNCg0KDQpkZWYgX2Nvc3Qoc3R1ZHksIHNsb3Rfcm93cywgcmVjb3Jkcyk6DQogICAgYyA9IDANCiAgICBmb3IgdWlkIGluIHNsb3Rfcm93cy5nZXQoc3R1ZHksIHt9KS52YWx1ZXMoKToNCiAgICAgICAgciA9IHJlY29yZHMuZ2V0KHVpZCkgaWYgdWlkIGVsc2UgTm9uZQ0KICAgICAgICBpZiByOg0KICAgICAgICAgICAgYyArPSAoci5nZXQoJ25fc2xpY2VzJykgb3IgMCkgKiAoNSBpZiBzdHIoci5nZXQoJ01SQWNxdWlzaXRpb25UeXBlJykpID09ICczRCcgZWxzZSAxKQ0KICAgIHJldHVybiBjDQoNCg0KIyBGSVggQTogSW5pdGlhbGl6ZSBnbG9iYWxzIGZvciBQcm9jZXNzUG9vbEV4ZWN1dG9yIHVuZGVyICJzcGF3biIgY29udGV4dA0KZGVmIF9pbml0X3dvcmtlcihzdHVkaWVzLCBzbG90X3Jvd3MsIHJlY29yZHMsIHNpZGVzLCBjZmcsIHByZWZpeCk6DQogICAgZ2xvYmFsIF9HDQogICAgX0cudXBkYXRlKHN0dWRpZXM9c3R1ZGllcywgc2xvdF9yb3dzPXNsb3Rfcm93cywgcmVjb3Jkcz1yZWNvcmRzLCBzaWRlcz1zaWRlcywgY2ZnPWNmZywgY2FjaGU9U3R1ZHlDYWNoZShwcmVmaXgsICdyKycpKQ0KDQoNCmRlZiBidWlsZF9jYWNoZShwcmVmaXgsIHN0dWRpZXMsIHNsb3Rfcm93cywgcmVjb3Jkcywgc2lkZXMsIGNmZywgd29ya2Vycz1Ob25lLCByZXN1bWU9VHJ1ZSwgb3JkZXI9J2xwdCcsDQogICAgICAgICAgICAgICAgY2h1bms9NjQsIG9uX3JlYWR5PU5vbmUsIHByb2dyZXNzPVRydWUsIHNvdXJjZV9zaWduYXR1cmU9Tm9uZSk6DQogICAgIiIiQnVpbGQgKG9yIHJlc3VtZSkgdGhlIGNhY2hlLiBSZXR1cm5zIHRoZSBvcGVuIFN0dWR5Q2FjaGUgKG1vZGUgcispIGFuZCBhIHN0YXRzIGRpY3QuDQoNCiAgICBzdHVkaWVzOiBvcmRlcmVkIGxpc3Qgb2YgU3R1ZHlJbnN0YW5jZVVJRC4gc2xvdF9yb3dzOiB7c3R1ZHk6IHtzbG90OiBzZXJpZXMgdWlkfE5vbmV9fS4NCiAgICBvbl9yZWFkeShjYWNoZSwgYSwgYik6IGNhbGxlZCBpbiB0aGUgbWFpbiB0aHJlYWQgd2hlbmV2ZXIgc3R1ZGllcyBbYSxiKSBhcmUgYWxsIGRvbmUgKHVzZSBvcmRlcj0nc2VxJykuIiIiDQogICAgd29ya2VycyA9IHdvcmtlcnMgb3IgbWF4KDIsIG9zLmNwdV9jb3VudCgpIG9yIDIpDQogICAgcCA9IF9wYXRocyhwcmVmaXgpDQogICAgaWYgc291cmNlX3NpZ25hdHVyZSBpcyBOb25lOg0KICAgICAgICBpbXBvcnQgaGFzaGxpYg0KICAgICAgICBpbXBvcnQganNvbiBhcyBfanNvbg0KICAgICAgICBzZWxlY3Rpb24gPSB7DQogICAgICAgICAgICBzdHVkeTogc2xvdF9yb3dzLmdldChzdHVkeSwge30pDQogICAgICAgICAgICBmb3Igc3R1ZHkgaW4gc3R1ZGllcw0KICAgICAgICB9DQogICAgICAgIHNvdXJjZV9zaWduYXR1cmUgPSBoYXNobGliLnNoYTI1NigNCiAgICAgICAgICAgIF9qc29uLmR1bXBzKHNlbGVjdGlvbiwgc29ydF9rZXlzPVRydWUsIGRlZmF1bHQ9c3RyKS5lbmNvZGUoInV0Zi04IikNCiAgICAgICAgKS5oZXhkaWdlc3QoKQ0KICAgIGlmIHJlc3VtZSBhbmQgb3MucGF0aC5leGlzdHMocFsnbWV0YSddKToNCiAgICAgICAgY2FjaGUgPSBTdHVkeUNhY2hlKHByZWZpeCwgJ3IrJykNCiAgICAgICAgaWYgKA0KICAgICAgICAgICAgY2FjaGUuc3R1ZGllcyAhPSBsaXN0KHN0dWRpZXMpDQogICAgICAgICAgICBvciBjYWNoZS5tZXRhWydjZmcnXSAhPSBqc29uLmxvYWRzKGpzb24uZHVtcHMoYXNkaWN0KGNmZykpKQ0KICAgICAgICAgICAgb3IgY2FjaGUubWV0YS5nZXQoInNvdXJjZV9zaWduYXR1cmUiKSAhPSBzb3VyY2Vfc2lnbmF0dXJlDQogICAgICAgICk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgICJleGlzdGluZyBjYWNoZSB3YXMgYnVpbHQgZnJvbSBkaWZmZXJlbnQgc3R1ZGllcywgc2xvdHMsIERJQ09Ncywgb3IgcHJlcHJvY2Vzc2luZyBjb25maWc7ICINCiAgICAgICAgICAgICAgICAidXNlIC0tZnJlc2hfcHJlcHJvY2Vzc2luZyBvciBhbm90aGVyIGNhY2hlIHByZWZpeCINCiAgICAgICAgICAgICkNCiAgICBlbHNlOg0KICAgICAgICBjYWNoZSA9IFN0dWR5Q2FjaGUuY3JlYXRlKHByZWZpeCwgc3R1ZGllcywgY2ZnLCBzb3VyY2Vfc2lnbmF0dXJlKQ0KICAgIHRvZG8gPSBbaSBmb3IgaSBpbiByYW5nZShsZW4oc3R1ZGllcykpIGlmIG5vdCBjYWNoZS5kb25lW2ldXQ0KICAgIGlmIG9yZGVyID09ICdscHQnOg0KICAgICAgICB0b2RvLnNvcnQoa2V5PWxhbWJkYSBpOiAtX2Nvc3Qoc3R1ZGllc1tpXSwgc2xvdF9yb3dzLCByZWNvcmRzKSkNCiAgICBfRy51cGRhdGUoc3R1ZGllcz1saXN0KHN0dWRpZXMpLCBzbG90X3Jvd3M9c2xvdF9yb3dzLCByZWNvcmRzPXJlY29yZHMsIHNpZGVzPXNpZGVzLCBjZmc9Y2ZnLCBjYWNoZT1jYWNoZSkNCiAgICB0cnk6DQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz13b3JrZXJzLCBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdmb3JrJykpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgIyBGSVggQTogVXNlIHNwYXduIGluc3RlYWQgb2YgR0lMLWJsb2NrZWQgVGhyZWFkUG9vbEV4ZWN1dG9yDQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcigNCiAgICAgICAgICAgIG1heF93b3JrZXJzPXdvcmtlcnMsDQogICAgICAgICAgICBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdzcGF3bicpLA0KICAgICAgICAgICAgaW5pdGlhbGl6ZXI9X2luaXRfd29ya2VyLA0KICAgICAgICAgICAgaW5pdGFyZ3M9KGxpc3Qoc3R1ZGllcyksIHNsb3Rfcm93cywgcmVjb3Jkcywgc2lkZXMsIGNmZywgcHJlZml4KQ0KICAgICAgICApDQogICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgIHN0YXRzID0gZGljdChzdHVkaWVzPWxlbihzdHVkaWVzKSwgYnVpbHQ9MCwgcmVzdW1lZD1sZW4oc3R1ZGllcykgLSBsZW4odG9kbykpDQogICAgdHJ5Og0KICAgICAgICBmdXRzID0ge2V4LnN1Ym1pdChfd29yaywgaSk6IGkgZm9yIGkgaW4gdG9kb30NCiAgICAgICAgcGVuZGluZyA9IHt9DQogICAgICAgIGZvciBhIGluIHJhbmdlKDAsIGxlbihzdHVkaWVzKSwgY2h1bmspOg0KICAgICAgICAgICAgcGVuZGluZ1thXSA9IHN1bSgxIGZvciBpIGluIHJhbmdlKGEsIG1pbihhICsgY2h1bmssIGxlbihzdHVkaWVzKSkpIGlmIG5vdCBjYWNoZS5kb25lW2ldKQ0KICAgICAgICBmb3IgYSwgbGVmdCBpbiBsaXN0KHBlbmRpbmcuaXRlbXMoKSk6DQogICAgICAgICAgICBpZiBsZWZ0ID09IDAgYW5kIG9uX3JlYWR5Og0KICAgICAgICAgICAgICAgIG9uX3JlYWR5KGNhY2hlLCBhLCBtaW4oYSArIGNodW5rLCBsZW4oc3R1ZGllcykpKQ0KICAgICAgICAgICAgaWYgbGVmdCA9PSAwOg0KICAgICAgICAgICAgICAgIHBlbmRpbmcucG9wKGEpDQogICAgICAgIGZvciBrLCBmIGluIGVudW1lcmF0ZShhc19jb21wbGV0ZWQoZnV0cykpOg0KICAgICAgICAgICAgaSwgXywgXywgXywgXywgXyA9IGYucmVzdWx0KCkNCiAgICAgICAgICAgIHN0YXRzWydidWlsdCddICs9IDENCiAgICAgICAgICAgIGEgPSAoaSAvLyBjaHVuaykgKiBjaHVuaw0KICAgICAgICAgICAgaWYgYSBpbiBwZW5kaW5nOg0KICAgICAgICAgICAgICAgIHBlbmRpbmdbYV0gLT0gMQ0KICAgICAgICAgICAgICAgIGlmIHBlbmRpbmdbYV0gPT0gMDoNCiAgICAgICAgICAgICAgICAgICAgaWYgb25fcmVhZHk6DQogICAgICAgICAgICAgICAgICAgICAgICBvbl9yZWFkeShjYWNoZSwgYSwgbWluKGEgKyBjaHVuaywgbGVuKHN0dWRpZXMpKSkNCiAgICAgICAgICAgICAgICAgICAgcGVuZGluZy5wb3AoYSkNCiAgICAgICAgICAgIGlmIHByb2dyZXNzIGFuZCAoayArIDEpICUgMjAwID09IDA6DQogICAgICAgICAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gdDANCiAgICAgICAgICAgICAgICB0aHJvdWdocHV0ID0gKGsgKyAxKSAvIGVsYXBzZWQNCiAgICAgICAgICAgICAgICBsZWZ0ID0gbGVuKHRvZG8pIC0gKGsgKyAxKQ0KICAgICAgICAgICAgICAgIGV0YSA9IGxlZnQgLyB0aHJvdWdocHV0IGlmIHRocm91Z2hwdXQgPiAwIGVsc2UgMA0KICAgICAgICAgICAgICAgIGV0YV9tLCBldGFfcyA9IGRpdm1vZChpbnQoZXRhKSwgNjApDQogICAgICAgICAgICAgICAgZXRhX2gsIGV0YV9tID0gZGl2bW9kKGV0YV9tLCA2MCkNCiAgICAgICAgICAgICAgICBwcmludCgNCiAgICAgICAgICAgICAgICAgICAgZicgIGNhY2hlIHtrICsgMX0ve2xlbih0b2RvKX0gfCB7dGhyb3VnaHB1dDouMWZ9IHN0dWRpZXMvcyB8ICcNCiAgICAgICAgICAgICAgICAgICAgZidFVEE6IHtldGFfaDowMmR9OntldGFfbTowMmR9OntldGFfczowMmR9JywNCiAgICAgICAgICAgICAgICAgICAgZmx1c2g9VHJ1ZSwNCiAgICAgICAgICAgICAgICApDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgZXguc2h1dGRvd24od2FpdD1UcnVlKQ0KICAgICAgICBjYWNoZS5mbHVzaCgpDQogICAgICAgIF9HLmNsZWFyKCkNCiAgICAgICAgcmFpc2UNCiAgICBlbHNlOg0KICAgICAgICBleC5zaHV0ZG93bih3YWl0PVRydWUpDQogICAgY2FjaGUuZmx1c2goKQ0KICAgIHN0YXRzLnVwZGF0ZSgNCiAgICAgICAgZGVjb2RlX2Vycm9ycz1pbnQobnAuYXNhcnJheShjYWNoZS5lcnJvcnMpLnN1bSgpKSwNCiAgICAgICAgZmFpbGVkX3NlbGVjdGVkX3Nsb3RzPWludChucC5hc2FycmF5KGNhY2hlLmZhaWxlZCkuc3VtKCkpLA0KICAgICAgICBwYWRkZWQ9aW50KG5wLmFzYXJyYXkoY2FjaGUucGFkZGVkKS5zdW0oKSksDQogICAgICAgIHBzX21pc3Npbmc9aW50KG5wLmFzYXJyYXkoY2FjaGUucHNfbWlzc2luZykuc3VtKCkpLA0KICAgICAgICBlbXB0eV9zdHVkaWVzPWludCgobnAuYXNhcnJheShjYWNoZS5zbG90KS5zdW0oYXhpcz0xKSA9PSAwKS5zdW0oKSksDQogICAgICAgIGluY29tcGxldGVfc3R1ZGllcz1pbnQoKG5wLmFzYXJyYXkoY2FjaGUuZG9uZSkgIT0gMSkuc3VtKCkpLA0KICAgICAgICBzb3VyY2Vfc2lnbmF0dXJlPXNvdXJjZV9zaWduYXR1cmUsDQogICAgKQ0KICAgIHN0YXRzWydzZWNvbmRzJ10gPSB0aW1lLnRpbWUoKSAtIHQwDQogICAgbWV0YV9mcmFtZSA9IHBkLkRhdGFGcmFtZShkaWN0KA0KICAgICAgICBTdHVkeUluc3RhbmNlVUlEPXN0dWRpZXMsDQogICAgICAgIHNpZGU9W3NpZGVzLmdldChzLCB7fSkuZ2V0KCdzaWRlJywgJ1UnKSBmb3IgcyBpbiBzdHVkaWVzXSwNCiAgICAgICAgc2lkZV9zb3VyY2U9W3NpZGVzLmdldChzLCB7fSkuZ2V0KCdzb3VyY2UnLCAnbm9uZScpIGZvciBzIGluIHN0dWRpZXNdLA0KICAgICAgICBuX3Nsb3RzPW5wLmFzYXJyYXkoY2FjaGUuc2xvdCkuc3VtKDEpLA0KICAgICAgICBkZWNvZGVfZXJyb3JzPW5wLmFzYXJyYXkoY2FjaGUuZXJyb3JzKSwNCiAgICAgICAgZmFpbGVkX3NlbGVjdGVkX3Nsb3RzPW5wLmFzYXJyYXkoY2FjaGUuZmFpbGVkKS5zdW0oMSksDQogICAgICAgIHBhZGRlZD1ucC5hc2FycmF5KGNhY2hlLnBhZGRlZCkuc3VtKDEpLA0KICAgICAgICBwc19taXNzaW5nPW5wLmFzYXJyYXkoY2FjaGUucHNfbWlzc2luZykuc3VtKDEpLA0KICAgICkpDQogICAgbWV0YV90bXAgPSBmIntwWydjc3YnXX0udG1wIg0KICAgIG1ldGFfZnJhbWUudG9fY3N2KG1ldGFfdG1wLCBpbmRleD1GYWxzZSkNCiAgICBvcy5yZXBsYWNlKG1ldGFfdG1wLCBwWydjc3YnXSkNCiAgICAjIE1FTU9SWSBBVURJVCBGSVg6IFB1cmdlIG1vZHVsZS1sZXZlbCBnbG9iYWwgZGljdCB0byByZWxlYXNlIGFsbCByZWNvcmRzIGFuZCBzZXJpZXMgbWV0YWRhdGENCiAgICBfRy5jbGVhcigpDQogICAgaW1wb3J0IGdjDQogICAgZ2MuY29sbGVjdCgpDQogICAgcmV0dXJuIGNhY2hlLCBzdGF0cw0K',
    'src/data/preprocess/dicomio.py': 'IiIiUm9idXN0IERJQ09NIGhlYWRlciAvIHBpeGVsIGFjY2Vzcy4NCg0KRGF0YXNldCBmYWN0cyB0aGlzIG1vZHVsZSBpcyBidWlsdCBhcm91bmQgKHZlcmlmaWVkIG9uIGFsbCAyNCwzNzEgdHJhaW4gc2VyaWVzKToNCiAgKiBldmVyeSBmaWxlIGlzIHVuY29tcHJlc3NlZCBFeHBsaWNpdCBWUiBMRSwgYnV0IHRoZSBoaWRkZW4gdGVzdCBzZXQgaXMgbm90IGd1YXJhbnRlZWQgdG8gYmUgLT4gZGVjb2RlIGVycm9ycw0KICAgIGFyZSBjYXVnaHQgcGVyIHNsaWNlLCBuZXZlciBmYXRhbA0KICAqIDI2JSBvZiBzZXJpZXMgYXJlIHNpZ25lZCBpbnQxNiwgQ2Fub24gc2VyaWVzIGNvbnRhaW4gbmVnYXRpdmUgcGl4ZWxzIChkb3duIHRvIC0xODcxKQ0KICAqIFJlc2NhbGVTbG9wZS9JbnRlcmNlcHQgYXJlIGNvbnN0YW50IGluc2lkZSBhIHNlcmllcyAoUGhpbGlwcyBzbG9wZSByZWFjaGVzIDI2MikNCiAgKiBESUNPTSBXaW5kb3dDZW50ZXIvV2lkdGggdmFyeSBwZXIgc2xpY2UgaW4gNTIlIG9mIHNlcmllcyAtPiBuZXZlciB1c2VkDQoiIiINCmltcG9ydCBvcw0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcHlkaWNvbQ0KDQpPUkRFUl9UQUdTID0gWydJbWFnZVBvc2l0aW9uUGF0aWVudCcsICdJbWFnZU9yaWVudGF0aW9uUGF0aWVudCcsICdJbnN0YW5jZU51bWJlcicsDQogICAgICAgICAgICAgICdSb3dzJywgJ0NvbHVtbnMnLCAnUGl4ZWxTcGFjaW5nJ10NCg0KUkVQX1RBR1MgPSBbJ1Nlcmllc0Rlc2NyaXB0aW9uJywgJ1NlcXVlbmNlTmFtZScsICdTY2FuT3B0aW9ucycsICdTY2FubmluZ1NlcXVlbmNlJywgJ1JlcGV0aXRpb25UaW1lJywgJ0VjaG9UaW1lJywNCiAgICAgICAgICAgICdJbnZlcnNpb25UaW1lJywgJ01SQWNxdWlzaXRpb25UeXBlJywgJ0xhdGVyYWxpdHknLCAnSW1hZ2VMYXRlcmFsaXR5JywgJ1NsaWNlVGhpY2tuZXNzJywNCiAgICAgICAgICAgICdTcGFjaW5nQmV0d2VlblNsaWNlcycsICdNYW51ZmFjdHVyZXInLCAnTWFudWZhY3R1cmVyTW9kZWxOYW1lJywgJ01hZ25ldGljRmllbGRTdHJlbmd0aCcsDQogICAgICAgICAgICAnQml0c1N0b3JlZCcsICdQaXhlbFJlcHJlc2VudGF0aW9uJywgJ1Bob3RvbWV0cmljSW50ZXJwcmV0YXRpb24nLCAnQm9keVBhcnRFeGFtaW5lZCddDQoNCg0KZGVmIGYxKHYpOg0KICAgICIiIkZpcnN0IGZsb2F0IG9mIGEgc2NhbGFyIC8gbXVsdGktdmFsdWUsIGVsc2UgTm9uZS4iIiINCiAgICB0cnk6DQogICAgICAgIHJldHVybiBmbG9hdCh2KQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJldHVybiBmbG9hdChsaXN0KHYpWzBdKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgcmV0dXJuIE5vbmUNCg0KDQpkZWYgZmxvYXRzKHYsIG4pOg0KICAgIHRyeToNCiAgICAgICAgYSA9IFtmbG9hdCh4KSBmb3IgeCBpbiB2XQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmV0dXJuIGEgaWYgbGVuKGEpID09IG4gYW5kIGFsbChucC5pc2Zpbml0ZShhKSkgZWxzZSBOb25lDQoNCg0KZGVmIHN2YWwodik6DQogICAgIiIiSGVhZGVyIHZhbHVlIC0+IHBsYWluIHN0ciAoJ3wnIGpvaW5lZCBmb3IgbXVsdGktdmFsdWVzKSwgTm9uZSBpZiBhYnNlbnQuIiIiDQogICAgaWYgdiBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGlmIGlzaW5zdGFuY2UodiwgKGxpc3QsIHR1cGxlKSkgb3Igdi5fX2NsYXNzX18uX19uYW1lX18gPT0gJ011bHRpVmFsdWUnOg0KICAgICAgICByZXR1cm4gJ3wnLmpvaW4oc3RyKHgpIGZvciB4IGluIHYpDQogICAgcmV0dXJuIHN0cih2KQ0KDQoNCmRlZiByZWFkX3RhZ3MocGF0aCwgdGFncyk6DQogICAgdHJ5Og0KICAgICAgICBkcyA9IHB5ZGljb20uZGNtcmVhZChwYXRoLCBzdG9wX2JlZm9yZV9waXhlbHM9VHJ1ZSwgZm9yY2U9VHJ1ZSwgc3BlY2lmaWNfdGFncz1saXN0KHRhZ3MpKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmV0dXJuIGRzDQoNCg0KZGVmIHJlcHJlc2VudGF0aXZlX2hlYWRlcihwYXRoKToNCiAgICAiIiJGdWxsIChwaXhlbC1sZXNzKSBoZWFkZXIgb2Ygb25lIGZpbGUgLT4gZGljdCBvZiBSRVBfVEFHUyAoc3RyaW5ncykgKyBudW1lcmljIGhlbHBlcnMuIiIiDQogICAgdHJ5Og0KICAgICAgICBkcyA9IHB5ZGljb20uZGNtcmVhZChwYXRoLCBzdG9wX2JlZm9yZV9waXhlbHM9VHJ1ZSwgZm9yY2U9VHJ1ZSkNCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIG91dCA9IHt0OiBzdmFsKGdldGF0dHIoZHMsIHQsIE5vbmUpKSBmb3IgdCBpbiBSRVBfVEFHU30NCiAgICBwcyA9IGZsb2F0cyhnZXRhdHRyKGRzLCAnUGl4ZWxTcGFjaW5nJywgTm9uZSksIDIpDQogICAgb3V0Wydwc19yb3cnXSwgb3V0Wydwc19jb2wnXSA9IChwc1swXSwgcHNbMV0pIGlmIHBzIGVsc2UgKG5wLm5hbiwgbnAubmFuKQ0KICAgIG91dFsndHJhbnNmZXJfc3ludGF4J10gPSBzdHIoZ2V0YXR0cihnZXRhdHRyKGRzLCAnZmlsZV9tZXRhJywgTm9uZSksICdUcmFuc2ZlclN5bnRheFVJRCcsICcnKSkNCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIGRlY29kZV9zbGljZShwYXRoKToNCiAgICAiIiJEZWNvZGUgb25lIHNsaWNlIC0+IGZsb2F0MzIgMi1EIGFycmF5IHdpdGggUmVzY2FsZVNsb3BlL0ludGVyY2VwdCBhcHBsaWVkLg0KDQogICAgUmFpc2VzIG9uIGFueSBmYWlsdXJlIChjYWxsZXIgZGVjaWRlcyB0aGUgZmFsbGJhY2spLiBNT05PQ0hST01FMSBpcyBpbnZlcnRlZC4gTm9uLWZpbml0ZSB2YWx1ZXMgLT4gMC4NCiAgICAiIiINCiAgICAjIE9QVElNSVpBVElPTjogQnlwYXNzaW5nIGhlYXZ5IG1ldGFkYXRhIHBhcnNpbmcuDQogICAgIyBPbmx5IHBhcnNlIHRoZSBleGFjdCB0YWdzIG5lZWRlZCB0byBkZWNvbXByZXNzIHBpeGVscyBhbmQgYXBwbHkgbGluZWFyIHJlc2NhbGUuDQogICAgIyBTYXZlcyB+MzAtNTAlIENQVSBwYXJzaW5nIHRpbWUgcGVyIHNsaWNlIChjcml0aWNhbCBmb3IgREdYIExpbnV4IG11bHRpLWNvcmUgc2NhbGluZykuDQogICAgdGFncyA9IFsnUGl4ZWxEYXRhJywgJ1Jvd3MnLCAnQ29sdW1ucycsICdCaXRzQWxsb2NhdGVkJywgJ0JpdHNTdG9yZWQnLCAnSGlnaEJpdCcsDQogICAgICAgICAgICAnUGl4ZWxSZXByZXNlbnRhdGlvbicsICdTYW1wbGVzUGVyUGl4ZWwnLCAnUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbicsDQogICAgICAgICAgICAnUmVzY2FsZVNsb3BlJywgJ1Jlc2NhbGVJbnRlcmNlcHQnXQ0KICAgIGRzID0gcHlkaWNvbS5kY21yZWFkKHBhdGgsIGZvcmNlPVRydWUsIHNwZWNpZmljX3RhZ3M9dGFncykNCiAgICBhID0gZHMucGl4ZWxfYXJyYXkNCiAgICBpZiBhLm5kaW0gIT0gMjoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ2V4cGVjdGVkIGEgMi1EIHNsaWNlLCBnb3QgbmRpbT17YS5uZGltfScpDQogICAgYSA9IGEuYXN0eXBlKG5wLmZsb2F0MzIpDQogICAgc2xvcGUgPSBmMShnZXRhdHRyKGRzLCAnUmVzY2FsZVNsb3BlJywgTm9uZSkpDQogICAgaWNwdCA9IGYxKGdldGF0dHIoZHMsICdSZXNjYWxlSW50ZXJjZXB0JywgTm9uZSkpDQogICAgYSA9IGEgKiAoc2xvcGUgaWYgc2xvcGUgZWxzZSAxLjApICsgKGljcHQgaWYgaWNwdCBlbHNlIDAuMCkNCiAgICBpZiBzdHIoZ2V0YXR0cihkcywgJ1Bob3RvbWV0cmljSW50ZXJwcmV0YXRpb24nLCAnJykpLnVwcGVyKCkgPT0gJ01PTk9DSFJPTUUxJzoNCiAgICAgICAgYSA9IGEubWF4KCkgLSBhDQogICAgaWYgbm90IG5wLmlzZmluaXRlKGEpLmFsbCgpOg0KICAgICAgICBhID0gbnAubmFuX3RvX251bShhLCBuYW49MC4wLCBwb3NpbmY9MC4wLCBuZWdpbmY9MC4wKQ0KICAgIHJldHVybiBhDQoNCg0KZGVmIGxpc3RfZGljb21zKHNlcmllc19kaXIpOg0KICAgIHRyeToNCiAgICAgICAgd2l0aCBvcy5zY2FuZGlyKHNlcmllc19kaXIpIGFzIGl0Og0KICAgICAgICAgICAgcmV0dXJuIHNvcnRlZChlLm5hbWUgZm9yIGUgaW4gaXQgaWYgZS5uYW1lLmxvd2VyKCkuZW5kc3dpdGgoJy5kY20nKSkNCiAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgcmV0dXJuIFtdDQoNCg0KZGVmIGRlY29kZV9yYXcocGF0aCk6DQogICAgIiIiRmFzdCBkZWNvZGU6IFJBVyBpbnRlZ2VyIHBpeGVscyArIHJlc2NhbGUgcGFyYW1zLg0KICAgIE9wdGltaXplZCB0byBkeW5hbWljYWxseSB1c2UgZGljb21zZGwgKEMrKyBiYXNlZCkgb24gREdYIGlmIGF2YWlsYWJsZSwgcHJvdmlkaW5nIGEgbWFzc2l2ZSBzcGVlZHVwICh+MW1zIHZzIDEybXMpLiIiIg0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IGRpY29tc2RsDQogICAgICAgIGRzID0gZGljb21zZGwub3BlbihwYXRoKQ0KICAgICAgICAjIFNPVEEgRml4OiBNdXN0IHVzZSBzdG9yZWR2YWx1ZT1UcnVlIHRvIGdldCByYXcgaW50ZWdlcnMsIG90aGVyd2lzZSBkaWNvbXNkbCBkb3VibGUtYXBwbGllcyBzbG9wZS9pY3B0IQ0KICAgICAgICBhID0gZHMucGl4ZWxEYXRhKHN0b3JlZHZhbHVlPVRydWUpDQogICAgICAgIA0KICAgICAgICAjIFBob3RvbWV0cmljIEludGVycHJldGF0aW9uIGNoZWNrIGlzIENSSVRJQ0FMDQogICAgICAgIHRyeToNCiAgICAgICAgICAgICMgU09UQSBGaXg6IGRpY29tc2RsIGRvZXMgbm90IGhhdmUgLmluZm8oKS4gVXNlIGRpcmVjdCBhdHRyaWJ1dGUgYWNjZXNzLg0KICAgICAgICAgICAgcGhvdG8gPSBnZXRhdHRyKGRzLCAnUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbicsICcnKQ0KICAgICAgICAgICAgaWYgJ01PTk9DSFJPTUUxJyBpbiBzdHIocGhvdG8pLnVwcGVyKCk6DQogICAgICAgICAgICAgICAgYSA9IGEubWF4KCkgLSBhDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBwYXNzDQogICAgICAgICAgICANCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgc2xvcGUgPSBmbG9hdChkcy5SZXNjYWxlU2xvcGUpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBzbG9wZSA9IDEuMA0KICAgICAgICAgICAgDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGljcHQgPSBmbG9hdChkcy5SZXNjYWxlSW50ZXJjZXB0KQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgaWNwdCA9IDAuMA0KICAgICAgICAgICAgDQogICAgICAgIHJldHVybiBhLCBzbG9wZSwgaWNwdA0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHBhc3MNCg0KICAgICMgRmFsbGJhY2sgdG8gc3BlY2lmaWNfdGFncyBvcHRpbWl6ZWQgcHlkaWNvbQ0KICAgIHRhZ3MgPSBbJ1BpeGVsRGF0YScsICdSb3dzJywgJ0NvbHVtbnMnLCAnQml0c0FsbG9jYXRlZCcsICdCaXRzU3RvcmVkJywgJ0hpZ2hCaXQnLA0KICAgICAgICAgICAgJ1BpeGVsUmVwcmVzZW50YXRpb24nLCAnU2FtcGxlc1BlclBpeGVsJywgJ1Bob3RvbWV0cmljSW50ZXJwcmV0YXRpb24nLA0KICAgICAgICAgICAgJ1Jlc2NhbGVTbG9wZScsICdSZXNjYWxlSW50ZXJjZXB0J10NCiAgICBkcyA9IHB5ZGljb20uZGNtcmVhZChwYXRoLCBmb3JjZT1UcnVlLCBzcGVjaWZpY190YWdzPXRhZ3MpDQogICAgYSA9IGRzLnBpeGVsX2FycmF5DQogICAgaWYgYS5uZGltICE9IDI6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidleHBlY3RlZCBhIDItRCBzbGljZSwgZ290IG5kaW09e2EubmRpbX0nKQ0KICAgIGlmIHN0cihnZXRhdHRyKGRzLCAnUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbicsICcnKSkudXBwZXIoKSA9PSAnTU9OT0NIUk9NRTEnOg0KICAgICAgICBhID0gYS5tYXgoKSAtIGENCiAgICBzbG9wZSA9IGdldGF0dHIoZHMsICdSZXNjYWxlU2xvcGUnLCAxLjApDQogICAgaWNwdCA9IGdldGF0dHIoZHMsICdSZXNjYWxlSW50ZXJjZXB0JywgMC4wKQ0KICAgIHJldHVybiBhLCBmbG9hdChzbG9wZSkgaWYgc2xvcGUgaXMgbm90IE5vbmUgZWxzZSAxLjAsIGZsb2F0KGljcHQpIGlmIGljcHQgaXMgbm90IE5vbmUgZWxzZSAwLjANCg==',
    'src/data/preprocess/geometry.py': 'IiIiU2xpY2Ugb3JkZXJpbmcsIG9yaWVudGF0aW9uLCBsYXRlcmFsaXR5Lg0KDQpXaHkgdGhpcyBleGlzdHMgKHZlcmlmaWVkIG9uIHRoZSBkYXRhc2V0KTogZmlsZSBuYW1lcyBhcmUgcHlkaWNvbS1nZW5lcmF0ZWQgcmFuZG9tIFVJRHMsIHNvIG5laXRoZXINCmBzb3J0ZWQob3MubGlzdGRpcigpKWAgbm9yIGFueSBmaWxlbmFtZSBvcmRlciBjYXJyaWVzIGFuYXRvbWljYWwgcG9zaXRpb24gKG9ic2VydmVkIFAoZmlyc3QgZmlsZSA9PSBzcGF0aWFsDQpjZW50cmUpID0gMy42NyUgdnMgMy41MSUgZXhwZWN0ZWQgZm9yIHJhbmRvbSBvcmRlciwgeiA9IDEuMzYsIG9uIGFsbCAyNCwzNzEgc2VyaWVzKS4gU2xpY2Ugb3JkZXIgbXVzdCBjb21lIGZyb20NCkltYWdlUG9zaXRpb25QYXRpZW50IHByb2plY3RlZCBvbiB0aGUgc2xpY2Ugbm9ybWFsLg0KIiIiDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCg0KX0xFVCA9IHsoMCwgMSk6ICdMJywgKDAsIC0xKTogJ1InLCAoMSwgMSk6ICdQJywgKDEsIC0xKTogJ0EnLCAoMiwgMSk6ICdTJywgKDIsIC0xKTogJ0knfQ0KX0FYSVMgPSB7J0wnOiAwLCAnUic6IDAsICdQJzogMSwgJ0EnOiAxLCAnUyc6IDIsICdJJzogMn0NClBMQU5FUyA9ICgnU2FnaXR0YWwnLCAnQ29yb25hbCcsICdBeGlhbCcpDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gb3JpZW50YXRpb24NCmRlZiBvcmllbnRfaW5mbyhpb3ApOg0KICAgICIiIi0+IChwbGFuZSBmcm9tIHNsaWNlIG5vcm1hbCwgfG5vcm1hbHwgYWxpZ25tZW50IHdpdGggaXRzIGF4aXMsIDItbGV0dGVyIExQUyBjb2RlKS4NCg0KICAgIENvZGUgbGV0dGVyIDEgPSBkaXJlY3Rpb24gaW1hZ2UgQ09MVU1OUyBpbmNyZWFzZSB0b3dhcmQ7IGxldHRlciAyID0gZGlyZWN0aW9uIGltYWdlIFJPV1MgaW5jcmVhc2UgdG93YXJkLg0KICAgIERhdGFzZXQgc3RhbmRhcmQ6IFNhZ2l0dGFsICdQSScsIENvcm9uYWwgJ0xJJywgQXhpYWwgJ0xQJy4NCiAgICAiIiINCiAgICBpZiBpb3AgaXMgTm9uZSBvciBsZW4oaW9wKSAhPSA2Og0KICAgICAgICByZXR1cm4gTm9uZSwgZmxvYXQoJ25hbicpLCBOb25lDQogICAgciwgYyA9IG5wLmFzYXJyYXkoaW9wWzozXSwgZmxvYXQpLCBucC5hc2FycmF5KGlvcFszOl0sIGZsb2F0KQ0KICAgIG4gPSBucC5jcm9zcyhyLCBjKQ0KICAgIG5uID0gbnAubGluYWxnLm5vcm0obikNCiAgICBpZiBub3QgbnAuaXNmaW5pdGUobm4pIG9yIG5uIDwgMWUtNjoNCiAgICAgICAgcmV0dXJuIE5vbmUsIGZsb2F0KCduYW4nKSwgTm9uZQ0KICAgIG4gPSBuIC8gbm4NCiAgICBwbGFuZSA9IFBMQU5FU1tpbnQobnAuYXJnbWF4KG5wLmFicyhuKSkpXQ0KICAgIGNvZGUgPSAnJw0KICAgIGZvciB2IGluIChyLCBjKToNCiAgICAgICAgYXggPSBpbnQobnAuYXJnbWF4KG5wLmFicyh2KSkpDQogICAgICAgIGNvZGUgKz0gX0xFVFsoYXgsIDEgaWYgdltheF0gPj0gMCBlbHNlIC0xKV0NCiAgICByZXR1cm4gcGxhbmUsIGZsb2F0KG5wLmFicyhuKS5tYXgoKSksIGNvZGUNCg0KDQpkZWYgc2xpY2Vfbm9ybWFsKGlvcCk6DQogICAgciwgYyA9IG5wLmFzYXJyYXkoaW9wWzozXSwgZmxvYXQpLCBucC5hc2FycmF5KGlvcFszOl0sIGZsb2F0KQ0KICAgIG4gPSBucC5jcm9zcyhyLCBjKQ0KICAgIHJldHVybiBuIC8gKG5wLmxpbmFsZy5ub3JtKG4pICsgMWUtMTIpDQoNCg0KZGVmIGNlbnRlcl94eXooaXBwLCBpb3AsIHBzLCByb3dzLCBjb2xzKToNCiAgICAiIiJQYXRpZW50LXNwYWNlIGNlbnRyZSBvZiBhIHNsaWNlIChtbSkuIiIiDQogICAgaXBwLCBpb3AgPSBucC5hc2FycmF5KGlwcCwgZmxvYXQpLCBucC5hc2FycmF5KGlvcCwgZmxvYXQpDQogICAgcmV0dXJuIGlwcCArIGlvcFs6M10gKiBwc1sxXSAqIChjb2xzIC0gMSkgLyAyLjAgKyBpb3BbMzpdICogcHNbMF0gKiAocm93cyAtIDEpIC8gMi4wDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gb3JkZXJpbmcNCmRlZiBvcmRlcl9yZWNvcmRzKHJlY3MsIG9yZGVyX21vZGU9J3Bvc2l0aW9uJyk6DQogICAgIiIiT3JkZXIgb25lIHNlcmllcycgc2xpY2VzIGJ5IGFuYXRvbWljYWwgcG9zaXRpb24uDQoNCiAgICByZWNzOiBsaXN0IG9mIGRpY3RzIHtuYW1lLCBpcHAsIGlvcCwgaW5zdCwgcm93cywgY29scywgcHN9ICAoTm9uZSB3aGVyZSBhIHRhZyBpcyBtaXNzaW5nKQ0KICAgIG9yZGVyX21vZGU6DQogICAgICAncG9zaXRpb24nIDoga2V5ID0gc2lnbihuW2F4aXNdKSAqIGRvdChpcHAsIG4pLCBhc2NlbmRpbmcgIC0+IGluY3JlYXNpbmcgYWxvbmcgK0wgLyArUCAvICtTDQogICAgICAgICAgICAgICAgICAgKGRpcmVjdGlvbiBpcyBpbmRlcGVuZGVudCBvZiB0aGUgdmVuZG9yJ3MgSU9QIHNpZ24gY29udmVudGlvbikNCiAgICAgICdub3JtYWwnICAgOiBrZXkgPSBkb3QoaXBwLCByIHggYykgYXNjZW5kaW5nIChwdWJsaWMtbm90ZWJvb2sgY29udmVudGlvbiwgbm8gc2lnbiBub3JtYWxpc2F0aW9uKQ0KICAgIEZhbGxzIGJhY2sgdG8gSW5zdGFuY2VOdW1iZXIsIHRoZW4gbmF0dXJhbCBuYW1lIG9yZGVyLCBhbmQgcmVwb3J0cyB3aGljaCBvbmUgd2FzIHVzZWQuDQogICAgRmlsZXMgd2hvc2UgbWF0cml4IGRpZmZlcnMgZnJvbSB0aGUgbWFqb3JpdHkgc2hhcGUgYXJlIGRyb3BwZWQgKGFuZCBjb3VudGVkKS4NCiAgICBSZXR1cm5zIGRpY3Qob3JkZXJlZCwgcG9zLCBtZXRob2QsIG5fZHJvcHBlZF9zaGFwZSwgcGxhbmUsIGNvZGUsIGFsaWduLCBzcGFjaW5nX21lZCwgel9leHRlbnQsIHNoYXBlLCBwcywNCiAgICAgICAgICAgICAgICAgY2VudGVyX3gsIHJlcF9pbmRleCkgb3IgTm9uZSB3aGVuIHRoZXJlIGFyZSBubyByZWFkYWJsZSBmaWxlcy4NCiAgICAiIiINCiAgICByZWNzID0gW3IgZm9yIHIgaW4gcmVjcyBpZiByIGlzIG5vdCBOb25lXQ0KICAgIGlmIG5vdCByZWNzOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHNoYXBlcyA9IENvdW50ZXIoKHJbJ3Jvd3MnXSwgclsnY29scyddKSBmb3IgciBpbiByZWNzKQ0KICAgIHNoYXBlID0gc2hhcGVzLm1vc3RfY29tbW9uKDEpWzBdWzBdDQogICAgbl9kcm9wID0gc3VtKDEgZm9yIHIgaW4gcmVjcyBpZiAoclsncm93cyddLCByWydjb2xzJ10pICE9IHNoYXBlKQ0KICAgIHJlY3MgPSBbciBmb3IgciBpbiByZWNzIGlmIChyWydyb3dzJ10sIHJbJ2NvbHMnXSkgPT0gc2hhcGVdDQogICAgaW9wcyA9IENvdW50ZXIodHVwbGUobnAucm91bmQoclsnaW9wJ10sIDMpKSBmb3IgciBpbiByZWNzIGlmIHJbJ2lvcCddIGlzIG5vdCBOb25lKQ0KICAgIGlvcCA9IG5wLmFycmF5KGlvcHMubW9zdF9jb21tb24oMSlbMF1bMF0pIGlmIGlvcHMgZWxzZSBOb25lDQogICAgcGxhbmUsIGFsaWduLCBjb2RlID0gb3JpZW50X2luZm8oaW9wKSBpZiBpb3AgaXMgbm90IE5vbmUgZWxzZSAoTm9uZSwgZmxvYXQoJ25hbicpLCBOb25lKQ0KDQogICAgaGF2ZV9wb3MgPSBbciBmb3IgciBpbiByZWNzIGlmIHJbJ2lwcCddIGlzIG5vdCBOb25lXQ0KICAgIG4gPSBsZW4ocmVjcykNCiAgICBuc2lnbiA9IDEuMA0KICAgICMgZXZlcnkgYnJhbmNoIGJ1aWxkcyB0dXBsZXMgKGtleSwgdGllYnJlYWssIG5hbWUsIHJlYyk7IGBwb3NgIGlzIGluIG1tIG9ubHkgZm9yIG1ldGhvZCA9PSAncG9zaXRpb24nDQogICAgaWYgaW9wIGlzIG5vdCBOb25lIGFuZCBsZW4oaGF2ZV9wb3MpID49IG1heCgyLCBpbnQoMC44ICogbikpOg0KICAgICAgICBucm0gPSBzbGljZV9ub3JtYWwoaW9wKQ0KICAgICAgICBheCA9IGludChucC5hcmdtYXgobnAuYWJzKG5ybSkpKQ0KICAgICAgICBuc2lnbiA9IDEuMCBpZiBucm1bYXhdID49IDAgZWxzZSAtMS4wDQogICAgICAgIHNnbiA9IG5zaWduIGlmIG9yZGVyX21vZGUgPT0gJ3Bvc2l0aW9uJyBlbHNlIDEuMA0KICAgICAgICBzcGFyZSA9IGZsb2F0KG5wLm1lZGlhbihbc2duICogZmxvYXQobnAuZG90KHJbJ2lwcCddLCBucm0pKSBmb3IgciBpbiBoYXZlX3Bvc10pKQ0KICAgICAgICBrZXllZCA9IFsoc2duICogZmxvYXQobnAuZG90KHJbJ2lwcCddLCBucm0pKSBpZiByWydpcHAnXSBpcyBub3QgTm9uZSBlbHNlIHNwYXJlLA0KICAgICAgICAgICAgICAgICAgclsnaW5zdCddIGlmIHJbJ2luc3QnXSBpcyBub3QgTm9uZSBlbHNlIGZsb2F0KCdpbmYnKSwgclsnbmFtZSddLCByKSBmb3IgciBpbiByZWNzXQ0KICAgICAgICBtZXRob2QgPSAncG9zaXRpb24nDQogICAgZWxpZiBzdW0oclsnaW5zdCddIGlzIG5vdCBOb25lIGZvciByIGluIHJlY3MpID49IG1heCgyLCBpbnQoMC44ICogbikpOg0KICAgICAgICBrZXllZCA9IFsoclsnaW5zdCddIGlmIHJbJ2luc3QnXSBpcyBub3QgTm9uZSBlbHNlIGZsb2F0KCdpbmYnKSwgMC4wLCByWyduYW1lJ10sIHIpIGZvciByIGluIHJlY3NdDQogICAgICAgIG1ldGhvZCA9ICdpbnN0YW5jZScNCiAgICBlbHNlOg0KICAgICAgICBrZXllZCA9IFsoMC4wLCAwLjAsIHJbJ25hbWUnXSwgcikgZm9yIHIgaW4gcmVjc10NCiAgICAgICAgbWV0aG9kID0gJ25hbWUnDQogICAga2V5ZWQuc29ydChrZXk9bGFtYmRhIHQ6ICh0WzBdLCB0WzFdLCB0WzJdKSkNCiAgICBvcmRlcmVkID0gW2tbMl0gZm9yIGsgaW4ga2V5ZWRdDQogICAgcG9zID0gbnAuYXJyYXkoW2tbMF0gZm9yIGsgaW4ga2V5ZWRdLCBkdHlwZT1ucC5mbG9hdDY0KSBpZiBtZXRob2QgIT0gJ25hbWUnIGVsc2UgbnAuYXJhbmdlKG4sIGR0eXBlPW5wLmZsb2F0NjQpDQogICAgZHAgPSBucC5kaWZmKHBvcykgaWYgbGVuKHBvcykgPiAxIGVsc2UgbnAuYXJyYXkoW10pDQogICAgc3BhY2luZyA9IGZsb2F0KG5wLm1lZGlhbihucC5hYnMoZHApKSkgaWYgbGVuKGRwKSBlbHNlIGZsb2F0KCduYW4nKQ0KICAgIHJlcCA9IGxlbihvcmRlcmVkKSAvLyAyDQogICAgcmVwX3JlYyA9IGtleWVkW3JlcF1bM10NCiAgICBwcyA9IHJlcF9yZWNbJ3BzJ10gaWYgcmVwX3JlY1sncHMnXSBlbHNlIChmbG9hdCgnbmFuJyksIGZsb2F0KCduYW4nKSkNCiAgICBjeCA9IGZsb2F0KCduYW4nKQ0KICAgIGlmIHJlcF9yZWNbJ2lwcCddIGlzIG5vdCBOb25lIGFuZCBpb3AgaXMgbm90IE5vbmUgYW5kIHJlcF9yZWNbJ3BzJ106DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGN4ID0gZmxvYXQoY2VudGVyX3h5eihyZXBfcmVjWydpcHAnXSwgaW9wLCByZXBfcmVjWydwcyddLCBzaGFwZVswXSwgc2hhcGVbMV0pWzBdKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgY3ggPSBmbG9hdCgnbmFuJykNCiAgICByZXR1cm4gZGljdChvcmRlcmVkPW9yZGVyZWQsIHBvcz1wb3MsIG1ldGhvZD1tZXRob2QsIG5fZHJvcHBlZF9zaGFwZT1uX2Ryb3AsIHBsYW5lPXBsYW5lLCBjb2RlPWNvZGUsDQogICAgICAgICAgICAgICAgYWxpZ249YWxpZ24sIHNwYWNpbmdfbWVkPXNwYWNpbmcsIHpfZXh0ZW50PWZsb2F0KHBvc1stMV0gLSBwb3NbMF0pIGlmIGxlbihwb3MpID4gMSBlbHNlIDAuMCwNCiAgICAgICAgICAgICAgICBzaGFwZT1zaGFwZSwgcHM9cHMsIGNlbnRlcl94PWN4LCByZXBfaW5kZXg9cmVwLCBuc2lnbj1uc2lnbikNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBsYXRlcmFsaXR5DQpkZWYgX3NpZGVfZnJvbV90YWcodik6DQogICAgaWYgdiBpcyBOb25lOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHMgPSBzdHIodikuc3RyaXAoKS51cHBlcigpDQogICAgcmV0dXJuIHNbMF0gaWYgcyBhbmQgc1swXSBpbiAoJ0wnLCAnUicpIGVsc2UgTm9uZQ0KDQoNCmRlZiByZXNvbHZlX2xhdGVyYWxpdHkocm93cywgbWluX29mZnNldD1Ob25lKToNCiAgICAiIiJTdHVkeS1sZXZlbCBzaWRlIG9mIHRoZSBrbmVlLg0KDQogICAgcm93czogaXRlcmFibGUgb2YgZGljdHMgd2l0aCBvcHRpb25hbCAnTGF0ZXJhbGl0eScsICdJbWFnZUxhdGVyYWxpdHknIChzdHJpbmdzKSBhbmQgJ2NlbnRlcl94JyAobW0pLg0KICAgIFByaW9yaXR5OiBESUNPTSB0YWcgKG1ham9yaXR5IHZvdGUpIC0+IHBhdGllbnQteCBvZiB0aGUgaW1hZ2UgY2VudHJlIChtZWRpYW4gb3ZlciBzZXJpZXMsIHx4fCA+PSBtaW5fb2Zmc2V0Og0KICAgIHg8MCAtPiBSaWdodCwgZWxzZSBMZWZ0KSAtPiB1bnJlc29sdmVkICgnVScpLg0KICAgIFJldHVybnMgZGljdChzaWRlIGluIHsnTCcsJ1InLCdVJ30sIHNvdXJjZSwgeF9tZWQsIGRpc2FncmVlKS4NCiAgICAiIiINCiAgICBtaW5fb2Zmc2V0ID0gY29uZmlnLkxBVF9NSU5fT0ZGU0VUX01NIGlmIG1pbl9vZmZzZXQgaXMgTm9uZSBlbHNlIG1pbl9vZmZzZXQNCiAgICB0YWdzID0gW10NCiAgICB4cyA9IFtdDQogICAgZm9yIHIgaW4gcm93czoNCiAgICAgICAgZm9yIGsgaW4gKCdMYXRlcmFsaXR5JywgJ0ltYWdlTGF0ZXJhbGl0eScpOg0KICAgICAgICAgICAgcyA9IF9zaWRlX2Zyb21fdGFnKHIuZ2V0KGspKQ0KICAgICAgICAgICAgaWYgczoNCiAgICAgICAgICAgICAgICB0YWdzLmFwcGVuZChzKQ0KICAgICAgICB4ID0gci5nZXQoJ2NlbnRlcl94JykNCiAgICAgICAgaWYgeCBpcyBub3QgTm9uZSBhbmQgbnAuaXNmaW5pdGUoeCk6DQogICAgICAgICAgICB4cy5hcHBlbmQoZmxvYXQoeCkpDQogICAgeG0gPSBmbG9hdChucC5tZWRpYW4oeHMpKSBpZiB4cyBlbHNlIGZsb2F0KCduYW4nKQ0KICAgIGdlbyA9IE5vbmUNCiAgICBpZiBucC5pc2Zpbml0ZSh4bSkgYW5kIGFicyh4bSkgPj0gbWluX29mZnNldDoNCiAgICAgICAgZ2VvID0gJ1InIGlmIHhtIDwgMCBlbHNlICdMJw0KICAgIGlmIHRhZ3M6DQogICAgICAgIHNpZGUgPSBDb3VudGVyKHRhZ3MpLm1vc3RfY29tbW9uKDEpWzBdWzBdDQogICAgICAgIHJldHVybiBkaWN0KHNpZGU9c2lkZSwgc291cmNlPSd0YWcnLCB4X21lZD14bSwgZGlzYWdyZWU9Ym9vbChnZW8gYW5kIGdlbyAhPSBzaWRlKSkNCiAgICBpZiBnZW86DQogICAgICAgIHJldHVybiBkaWN0KHNpZGU9Z2VvLCBzb3VyY2U9J2dlb21ldHJ5JywgeF9tZWQ9eG0sIGRpc2FncmVlPUZhbHNlKQ0KICAgIHJldHVybiBkaWN0KHNpZGU9J1UnLCBzb3VyY2U9J25vbmUnLCB4X21lZD14bSwgZGlzYWdyZWU9RmFsc2UpDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY2Fub25pY2FsaXNhdGlvbg0KZGVmIHJlb3JpZW50X3N0YWNrKHN0YWNrLCBjb2RlLCBwbGFuZSk6DQogICAgIiIiTWFwIGEgW0QsSCxXXSBzdGFjayB0byB0aGUgZGF0YXNldC1zdGFuZGFyZCBvcmllbnRhdGlvbiBjb2RlIG9mIGBwbGFuZWAgKENBTk9OX09SSUVOVCkuDQoNCiAgICBIYW5kbGVzIGF4aXMgc3dhcHMgKHRyYW5zcG9zZSkgYW5kIHNpZ24gZmxpcHMuIEEgbm8tb3AgZm9yIGFsbCAyNCwzNzEgdHJhaW4gc2VyaWVzIChzaW5nbGUgY29kZSBwZXIgcGxhbmUpLiIiIg0KICAgIHRhcmdldCA9IGNvbmZpZy5DQU5PTl9PUklFTlQuZ2V0KHBsYW5lKQ0KICAgIGlmIG5vdCBjb2RlIG9yIGxlbihjb2RlKSAhPSAyIG9yIG5vdCB0YXJnZXQgb3IgY29kZSA9PSB0YXJnZXQ6DQogICAgICAgIHJldHVybiBzdGFjaywgRmFsc2UNCiAgICByLCBjID0gY29kZQ0KICAgIHRyLCB0YyA9IHRhcmdldA0KICAgIGlmIF9BWElTW3JdICE9IF9BWElTW3RyXToNCiAgICAgICAgc3RhY2sgPSBzdGFjay50cmFuc3Bvc2UoMCwgMiwgMSkNCiAgICAgICAgciwgYyA9IGMsIHINCiAgICBpZiByICE9IHRyOg0KICAgICAgICBzdGFjayA9IHN0YWNrWzosIDosIDo6LTFdDQogICAgaWYgYyAhPSB0YzoNCiAgICAgICAgc3RhY2sgPSBzdGFja1s6LCA6Oi0xLCA6XQ0KICAgIHJldHVybiBucC5hc2NvbnRpZ3VvdXNhcnJheShzdGFjayksIFRydWUNCg0KDQpkZWYgY2Fub25pY2FsX3NpZGUoc3RhY2ssIHBsYW5lLCBzaWRlKToNCiAgICAiIiJNaXJyb3IgcmlnaHQga25lZXMgc28gdGhleSBsb29rIGxpa2UgbGVmdCBrbmVlcy4NCg0KICAgIENvcm9uYWwvQXhpYWw6IGZsaXAgaW1hZ2UgY29sdW1ucy4gU2FnaXR0YWw6IHJldmVyc2Ugc2xpY2Ugb3JkZXIgKHNsaWNlcyBydW4gbWVkaWFsPC0+bGF0ZXJhbCkuIiIiDQogICAgaWYgc2lkZSAhPSAnUic6DQogICAgICAgIHJldHVybiBzdGFjaw0KICAgIGlmIHBsYW5lIGluICgnQ29yb25hbCcsICdBeGlhbCcpOg0KICAgICAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkoc3RhY2tbOiwgOiwgOjotMV0pDQogICAgcmV0dXJuIG5wLmFzY29udGlndW91c2FycmF5KHN0YWNrWzo6LTFdKQ0K',
    'src/data/preprocess/index.py': 'IiIiU2VyaWVzIGluZGV4OiBvbmUgcm93IHBlciBzZXJpZXMgb24gZGlzaywgd2l0aCBhbmF0b21pY2FsIHNsaWNlIG9yZGVyLCBnZW9tZXRyeSBhbmQgYSByZXByZXNlbnRhdGl2ZSBoZWFkZXIuDQoNCkRlc2lnbiBydWxlcyAoZnJvbSB0aGUgZGF0YXNldCBhdWRpdCArIHRoZSBoaWRkZW4tcmVydW4gZmFpbHVyZXMgZG9jdW1lbnRlZCBpbiB0aGUgcHVibGljIG5vdGVib29rcyk6DQogICogdGhlIERJUkVDVE9SWSBUUkVFIGlzIHRoZSB0cnV0aCBmb3Igd2hpY2ggc2VyaWVzL2ZpbGVzIGV4aXN0OyB0aGUgQ1NWcyBvbmx5IGNvbnRyaWJ1dGUgZmxhZ3MNCiAgICAodGVzdF9zZXJpZXMuY3N2IGluIHRoZSBoaWRkZW4gc2V0IG1heSBkaWZmZXIgZnJvbSB0aGUgcGxhY2Vob2xkZXIgLT4gbmV2ZXIgYXNzdW1lIGl0IGlzIGNvbXBsZXRlKQ0KICAqIGEgZmFpbHVyZSBpbnNpZGUgb25lIHNlcmllcyBuZXZlciBhYm9ydHMgdGhlIHJ1bjogdGhlIHJvdyBpcyBrZXB0IHdpdGggYGVycmAgc2V0IGFuZCBhbiBlbXB0eSBmaWxlIGxpc3QNCiAgKiB0aGUgYnVpbGQgaXMgY2h1bmtlZCArIGNoZWNrcG9pbnRlZCwgc28gYSA0MC1taW51dGUgc2NhbiBzdXJ2aXZlcyBhIHNlc3Npb24gcmVzdGFydA0KIiIiDQppbXBvcnQgb3MNCmltcG9ydCB0aW1lDQppbXBvcnQgcGlja2xlDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IG11bHRpcHJvY2Vzc2luZyBhcyBtcA0KZnJvbSBjb25jdXJyZW50LmZ1dHVyZXMgaW1wb3J0IFByb2Nlc3NQb29sRXhlY3V0b3IsIFRocmVhZFBvb2xFeGVjdXRvcg0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gLiBpbXBvcnQgZGljb21pbyBhcyBkaW8NCmZyb20gLiBpbXBvcnQgZ2VvbWV0cnkgYXMgZ2VvDQoNClNQTElUX0RJUlMgPSB7J3RyYWluJzogJ3RyYWluX3NlcmllcycsICd0ZXN0JzogJ3Rlc3Rfc2VyaWVzJ30NCg0KDQpkZWYgZGlzY292ZXJfcm9vdChleHBsaWNpdD1Ob25lKToNCiAgICAiIiJGaW5kIHRoZSBjb21wZXRpdGlvbiBmb2xkZXIgKGhhbmRsZXMgL2thZ2dsZS9pbnB1dC9jb21wZXRpdGlvbnMvPHNsdWc+IGFuZCAva2FnZ2xlL2lucHV0LzxzbHVnPikuIiIiDQogICAgY2FuZHMgPSBbZXhwbGljaXQsIG9zLmVudmlyb24uZ2V0KCdLTkVFX0RBVEEnKV0gKyBsaXN0KGNvbmZpZy5ST09UX0NBTkRJREFURVMpDQogICAgZm9yIGMgaW4gY2FuZHM6DQogICAgICAgIGlmIGMgYW5kIG9zLnBhdGguaXNkaXIoYykgYW5kIChvcy5wYXRoLmlzZGlyKG9zLnBhdGguam9pbihjLCAndHJhaW5fc2VyaWVzJykpIG9yDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvcy5wYXRoLmlzZGlyKG9zLnBhdGguam9pbihjLCAndGVzdF9zZXJpZXMnKSkpOg0KICAgICAgICAgICAgcmV0dXJuIGMNCiAgICBiYXNlID0gJy9rYWdnbGUvaW5wdXQnDQogICAgaWYgb3MucGF0aC5pc2RpcihiYXNlKToNCiAgICAgICAgZm9yIHAgaW4gc29ydGVkKG9zLmxpc3RkaXIoYmFzZSkpOg0KICAgICAgICAgICAgZm9yIHEgaW4gKG9zLnBhdGguam9pbihiYXNlLCBwKSwgKltvcy5wYXRoLmpvaW4oYmFzZSwgcCwgeCkgZm9yIHggaW4gc29ydGVkKG9zLmxpc3RkaXIob3MucGF0aC5qb2luKGJhc2UsIHApKSlbOjIwXQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguaXNkaXIob3MucGF0aC5qb2luKGJhc2UsIHAsIHgpKV0pOg0KICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguaXNkaXIob3MucGF0aC5qb2luKHEsICd0ZXN0X3NlcmllcycpKSBvciBvcy5wYXRoLmlzZGlyKG9zLnBhdGguam9pbihxLCAndHJhaW5fc2VyaWVzJykpOg0KICAgICAgICAgICAgICAgICAgICByZXR1cm4gcQ0KICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKCdjb21wZXRpdGlvbiByb290IG5vdCBmb3VuZDsgcGFzcyBpdCBleHBsaWNpdGx5IG9yIHNldCBLTkVFX0RBVEEnKQ0KDQoNCmRlZiBsaXN0X3Nlcmllc19kaXJzKHJvb3QsIHNwbGl0KToNCiAgICBiYXNlcyA9IGRpY3QuZnJvbWtleXMoDQogICAgICAgIG9zLnBhdGguam9pbihyb290LCBuYW1lKQ0KICAgICAgICBmb3IgbmFtZSBpbiAoU1BMSVRfRElSUy5nZXQoc3BsaXQsIHNwbGl0KSwgc3BsaXQsIGYie3NwbGl0fV9zZXJpZXMiLCBmIntzcGxpdH1faW1hZ2VzIikNCiAgICApDQogICAgZm9yIGJhc2UgaW4gYmFzZXM6DQogICAgICAgIGlmIG5vdCBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgaXRlbXMgPSBbXQ0KICAgICAgICB3aXRoIG9zLnNjYW5kaXIoYmFzZSkgYXMgaXQ6DQogICAgICAgICAgICBzdHVkaWVzID0gc29ydGVkKGUubmFtZSBmb3IgZSBpbiBpdCBpZiBlLmlzX2RpcigpKQ0KICAgICAgICBmb3Igc3QgaW4gc3R1ZGllczoNCiAgICAgICAgICAgIHNkID0gb3MucGF0aC5qb2luKGJhc2UsIHN0KQ0KICAgICAgICAgICAgd2l0aCBvcy5zY2FuZGlyKHNkKSBhcyBpdDI6DQogICAgICAgICAgICAgICAgZm9yIGUgaW4gc29ydGVkKGl0Miwga2V5PWxhbWJkYSB4OiB4Lm5hbWUpOg0KICAgICAgICAgICAgICAgICAgICBpZiBlLmlzX2RpcigpOg0KICAgICAgICAgICAgICAgICAgICAgICAgaXRlbXMuYXBwZW5kKChzcGxpdCwgc3QsIGUubmFtZSwgZS5wYXRoKSkNCiAgICAgICAgaWYgaXRlbXM6DQogICAgICAgICAgICByZXR1cm4gaXRlbXMNCiAgICByZXR1cm4gW10NCg0KDQpkZWYgc2Nhbl9zZXJpZXMoaXRlbSk6DQogICAgIiIiRnVsbCBzbGljZSBvcmRlcmluZyArIHJlcHJlc2VudGF0aXZlIGhlYWRlciBmb3Igb25lIHNlcmllcyBkaXJlY3RvcnkuIiIiDQogICAgc3BsaXQsIHN0dWR5LCBzZXJpZXMsIGQgPSBpdGVtDQogICAgcm93ID0gZGljdChzcGxpdD1zcGxpdCwgU3R1ZHlJbnN0YW5jZVVJRD1zdHVkeSwgU2VyaWVzSW5zdGFuY2VVSUQ9c2VyaWVzLCBkaXI9ZCwgbl9maWxlcz0wLA0KICAgICAgICAgICAgICAgb3JkZXJlZF9maWxlcz1bXSwgcG9zaXRpb25zPW5wLnplcm9zKDApLCBlcnI9Tm9uZSkNCiAgICB0cnk6DQogICAgICAgIG5hbWVzID0gZGlvLmxpc3RfZGljb21zKGQpDQogICAgICAgIHJvd1snbl9maWxlcyddID0gbGVuKG5hbWVzKQ0KICAgICAgICByZWNzLCBiYWQgPSBbXSwgMA0KICAgICAgICBmb3Igbm0gaW4gbmFtZXM6DQogICAgICAgICAgICBkcyA9IGRpby5yZWFkX3RhZ3Mob3MucGF0aC5qb2luKGQsIG5tKSwgZGlvLk9SREVSX1RBR1MpDQogICAgICAgICAgICBpZiBkcyBpcyBOb25lIG9yICdSb3dzJyBub3QgaW4gZHMgb3IgJ0NvbHVtbnMnIG5vdCBpbiBkczoNCiAgICAgICAgICAgICAgICBiYWQgKz0gMQ0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBwcyA9IGRpby5mbG9hdHMoZ2V0YXR0cihkcywgJ1BpeGVsU3BhY2luZycsIE5vbmUpLCAyKQ0KICAgICAgICAgICAgcmVjcy5hcHBlbmQoZGljdChuYW1lPW5tLCBpcHA9ZGlvLmZsb2F0cyhnZXRhdHRyKGRzLCAnSW1hZ2VQb3NpdGlvblBhdGllbnQnLCBOb25lKSwgMyksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlvcD1kaW8uZmxvYXRzKGdldGF0dHIoZHMsICdJbWFnZU9yaWVudGF0aW9uUGF0aWVudCcsIE5vbmUpLCA2KSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5zdD1kaW8uZjEoZ2V0YXR0cihkcywgJ0luc3RhbmNlTnVtYmVyJywgTm9uZSkpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICByb3dzPWludChkcy5Sb3dzKSwgY29scz1pbnQoZHMuQ29sdW1ucyksIHBzPXR1cGxlKHBzKSBpZiBwcyBlbHNlIE5vbmUpKQ0KICAgICAgICByb3dbJ25fYmFkJ10gPSBiYWQNCiAgICAgICAgbyA9IGdlby5vcmRlcl9yZWNvcmRzKHJlY3MsICdwb3NpdGlvbicpDQogICAgICAgIGlmIG8gaXMgTm9uZToNCiAgICAgICAgICAgIHJvd1snZXJyJ10gPSAnbm9fcmVhZGFibGVfc2xpY2VzJw0KICAgICAgICAgICAgcmV0dXJuIHJvdw0KICAgICAgICByb3cudXBkYXRlKG9yZGVyZWRfZmlsZXM9b1snb3JkZXJlZCddLCBwb3NpdGlvbnM9b1sncG9zJ10uYXN0eXBlKG5wLmZsb2F0MzIpLCBvcmRlcl9tZXRob2Q9b1snbWV0aG9kJ10sDQogICAgICAgICAgICAgICAgICAgbl9kcm9wcGVkX3NoYXBlPW9bJ25fZHJvcHBlZF9zaGFwZSddLCBwbGFuZV9nZW89b1sncGxhbmUnXSwgb3JpZW50X2NvZGU9b1snY29kZSddLA0KICAgICAgICAgICAgICAgICAgIHBsYW5lX2FsaWduPW9bJ2FsaWduJ10sIHNwYWNpbmdfbWVkPW9bJ3NwYWNpbmdfbWVkJ10sIHpfZXh0ZW50PW9bJ3pfZXh0ZW50J10sDQogICAgICAgICAgICAgICAgICAgcm93cz1vWydzaGFwZSddWzBdLCBjb2xzPW9bJ3NoYXBlJ11bMV0sIHBzX3Jvdz1vWydwcyddWzBdLCBwc19jb2w9b1sncHMnXVsxXSwNCiAgICAgICAgICAgICAgICAgICBjZW50ZXJfeD1vWydjZW50ZXJfeCddLCBuc2lnbj1vWyduc2lnbiddLCBuX3NsaWNlcz1sZW4ob1snb3JkZXJlZCddKSkNCiAgICAgICAgcmVwID0gZGlvLnJlcHJlc2VudGF0aXZlX2hlYWRlcihvcy5wYXRoLmpvaW4oZCwgb1snb3JkZXJlZCddW29bJ3JlcF9pbmRleCddXSkpDQogICAgICAgIGlmIHJlcDoNCiAgICAgICAgICAgIHJvdy51cGRhdGUoe2s6IHYgZm9yIGssIHYgaW4gcmVwLml0ZW1zKCkgaWYgayBub3QgaW4gKCdwc19yb3cnLCAncHNfY29sJyl9KQ0KICAgICAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKHJvd1sncHNfcm93J10pIGFuZCBucC5pc2Zpbml0ZShyZXBbJ3BzX3JvdyddKToNCiAgICAgICAgICAgICAgICByb3dbJ3BzX3JvdyddLCByb3dbJ3BzX2NvbCddID0gcmVwWydwc19yb3cnXSwgcmVwWydwc19jb2wnXQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBuZXZlciBmYXRhbA0KICAgICAgICByb3dbJ2VyciddID0gZid7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjgwXX0nDQogICAgcmV0dXJuIHJvdw0KDQoNCmRlZiBzb3VyY2Vfc2lnbmF0dXJlKGl0ZW1zLCBjaHVuayk6DQogICAgIiIiRmluZ2VycHJpbnQgdGhlIGV4YWN0IERJQ09NIGZpbGUgc2V0IGFuZCBtdGltZXMgdXNlZCBieSByZXN1bWFibGUgaW5kZXggY2h1bmtzLiIiIg0KICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkNCiAgICBkaWdlc3QudXBkYXRlKGYiY2h1bms9e2NodW5rfVxuIi5lbmNvZGUoInV0Zi04IikpDQogICAgZm9yIHNwbGl0LCBzdHVkeSwgc2VyaWVzLCBkaXJlY3RvcnkgaW4gaXRlbXM6DQogICAgICAgIGRpZ2VzdC51cGRhdGUoZiJ7c3BsaXR9XDB7c3R1ZHl9XDB7c2VyaWVzfVxuIi5lbmNvZGUoInV0Zi04IikpDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGVudHJpZXMgPSBzb3J0ZWQoDQogICAgICAgICAgICAgICAgKGVudHJ5IGZvciBlbnRyeSBpbiBvcy5zY2FuZGlyKGRpcmVjdG9yeSkgaWYgZW50cnkuaXNfZmlsZSgpIGFuZCBlbnRyeS5uYW1lLmxvd2VyKCkuZW5kc3dpdGgoIi5kY20iKSksDQogICAgICAgICAgICAgICAga2V5PWxhbWJkYSBlbnRyeTogZW50cnkubmFtZSwNCiAgICAgICAgICAgICkNCiAgICAgICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOg0KICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShmIkVSUk9SXDB7dHlwZShleGMpLl9fbmFtZV9ffVwwe2V4Y31cbiIuZW5jb2RlKCJ1dGYtOCIpKQ0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgZm9yIGVudHJ5IGluIGVudHJpZXM6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgc3RhdCA9IGVudHJ5LnN0YXQoZm9sbG93X3N5bWxpbmtzPUZhbHNlKQ0KICAgICAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoDQogICAgICAgICAgICAgICAgICAgIGYie2VudHJ5Lm5hbWV9XDB7c3RhdC5zdF9zaXplfVwwe3N0YXQuc3RfbXRpbWVfbnN9XG4iLmVuY29kZSgidXRmLTgiKQ0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yIGFzIGV4YzoNCiAgICAgICAgICAgICAgICBkaWdlc3QudXBkYXRlKGYie2VudHJ5Lm5hbWV9XDBFUlJPUlwwe3R5cGUoZXhjKS5fX25hbWVfX31cbiIuZW5jb2RlKCJ1dGYtOCIpKQ0KICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkNCg0KDQpkZWYgYnVpbGRfaW5kZXgocm9vdCwgc3BsaXRzPSgndHJhaW4nLCAndGVzdCcpLCB3b3JrZXJzPU5vbmUsIGNodW5rPTEwMDAsIGNhY2hlX2Rpcj1Ob25lLCBsaW1pdD0wLCBmb3JjZT1GYWxzZSwNCiAgICAgICAgICAgICAgICBwcm9ncmVzcz1UcnVlKToNCiAgICAiIiJTY2FuIGV2ZXJ5IHNlcmllcyBvbiBkaXNrLiBSZXR1cm5zIGEgRGF0YUZyYW1lIChvYmplY3QgY29sdW1uczogb3JkZXJlZF9maWxlcywgcG9zaXRpb25zKS4iIiINCiAgICB3b3JrZXJzID0gd29ya2VycyBvciBtYXgoMiwgKG9zLmNwdV9jb3VudCgpIG9yIDQpICogMikNCiAgICBpdGVtcyA9IFtdDQogICAgZm9yIHNwIGluIHNwbGl0czoNCiAgICAgICAgaXRlbXMgKz0gbGlzdF9zZXJpZXNfZGlycyhyb290LCBzcCkNCiAgICBpZiBsaW1pdDoNCiAgICAgICAgaXRlbXMgPSBpdGVtc1s6bGltaXRdDQogICAgY2h1bmtzID0gW2l0ZW1zW2k6aSArIGNodW5rXSBmb3IgaSBpbiByYW5nZSgwLCBsZW4oaXRlbXMpLCBjaHVuayldDQogICAgc2lnbmF0dXJlID0gc291cmNlX3NpZ25hdHVyZShpdGVtcywgY2h1bmspDQogICAgaWYgY2FjaGVfZGlyOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgc3RhbGUtY2hlY2twb2ludCBndWFyZDogY2h1bmtzIGFyZSBvbmx5IHZhbGlkIGZvciBUSElTIGl0ZW0gbGlzdA0KICAgICAgICBpbXBvcnQgZ2xvYg0KICAgICAgICBzcCA9IG9zLnBhdGguam9pbihjYWNoZV9kaXIsICdzaWduYXR1cmUudHh0JykNCiAgICAgICAgb3MubWFrZWRpcnMoY2FjaGVfZGlyLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICBvbGRfc2lnbmF0dXJlID0gb3BlbihzcCkucmVhZCgpLnN0cmlwKCkgaWYgb3MucGF0aC5leGlzdHMoc3ApIGVsc2UgTm9uZQ0KICAgICAgICBpZiBmb3JjZSBvciBvbGRfc2lnbmF0dXJlICE9IHNpZ25hdHVyZToNCiAgICAgICAgICAgIGZvciBmIGluIGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oY2FjaGVfZGlyLCAnaW5kZXhfKi5wa2wnKSk6DQogICAgICAgICAgICAgICAgb3MucmVtb3ZlKGYpDQogICAgICAgICAgICB0ZW1wb3JhcnkgPSBmIntzcH0udG1wIg0KICAgICAgICAgICAgd2l0aCBvcGVuKHRlbXBvcmFyeSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06DQogICAgICAgICAgICAgICAgc3RyZWFtLndyaXRlKHNpZ25hdHVyZSkNCiAgICAgICAgICAgICAgICBzdHJlYW0uZmx1c2goKQ0KICAgICAgICAgICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkNCiAgICAgICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBzcCkNCiAgICBwYXJ0cywgdDAgPSBbXSwgdGltZS50aW1lKCkNCiAgICBleCA9IE5vbmUNCiAgICB0cnk6DQogICAgICAgIGV4ID0gUHJvY2Vzc1Bvb2xFeGVjdXRvcihtYXhfd29ya2Vycz13b3JrZXJzLCBtcF9jb250ZXh0PW1wLmdldF9jb250ZXh0KCdmb3JrJykpDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgZXggPSBUaHJlYWRQb29sRXhlY3V0b3IobWF4X3dvcmtlcnM9d29ya2VycykNCiAgICBmb3IgaywgY2ggaW4gZW51bWVyYXRlKGNodW5rcyk6DQogICAgICAgIHAgPSBvcy5wYXRoLmpvaW4oY2FjaGVfZGlyLCBmJ2luZGV4X3trOjA0ZH0ucGtsJykgaWYgY2FjaGVfZGlyIGVsc2UgTm9uZQ0KICAgICAgICBpZiBwIGFuZCBvcy5wYXRoLmV4aXN0cyhwKSBhbmQgbm90IGZvcmNlOg0KICAgICAgICAgICAgcGFydHMuYXBwZW5kKHBkLnJlYWRfcGlja2xlKHApKQ0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgZGYgPSBwZC5EYXRhRnJhbWUobGlzdChleC5tYXAoc2Nhbl9zZXJpZXMsIGNoLCBjaHVua3NpemU9OCkpKQ0KICAgICAgICBpZiBwOg0KICAgICAgICAgICAgb3MubWFrZWRpcnMoY2FjaGVfZGlyLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICAgICAgdGVtcG9yYXJ5ID0gZiJ7cH0udG1wIg0KICAgICAgICAgICAgZGYudG9fcGlja2xlKHRlbXBvcmFyeSkNCiAgICAgICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBwKQ0KICAgICAgICBwYXJ0cy5hcHBlbmQoZGYpDQogICAgICAgIGlmIHByb2dyZXNzOg0KICAgICAgICAgICAgcHJpbnQoZicgIGluZGV4IGNodW5rIHtrICsgMX0ve2xlbihjaHVua3MpfSAgeyhrICsgMSkgKiBjaHVuayAvIG1heCh0aW1lLnRpbWUoKSAtIHQwLCAxZS05KTouMWZ9IHNlcmllcy9zJywgZmx1c2g9VHJ1ZSkNCiAgICBleC5zaHV0ZG93bigpDQogICAgb3V0ID0gcGQuY29uY2F0KHBhcnRzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkgaWYgcGFydHMgZWxzZSBwZC5EYXRhRnJhbWUoKQ0KICAgIG91dC5hdHRyc1sic291cmNlX3NpZ25hdHVyZSJdID0gc2lnbmF0dXJlDQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBhdHRhY2hfY3N2X2ZsYWdzKGluZGV4X2RmLCByb290KToNCiAgICAiIiJMZWZ0LWpvaW4gdGhlIGNvbXBldGl0aW9uIENTViBmbGFncyAocGxhbmUgLyBmbHVpZCAvIGZhdC1zYXQpIG9udG8gdGhlIG9uLWRpc2sgaW5kZXguIiIiDQogICAgaWYgaW5kZXhfZGYuZW1wdHkgb3IgJ1Nlcmllc0luc3RhbmNlVUlEJyBub3QgaW4gaW5kZXhfZGYuY29sdW1uczoNCiAgICAgICAgZm9yIGMgaW4gKCdTZXJpZXNJbnN0YW5jZVVJRCcsICdBbmF0b21pY2FsX1BsYW5lJywgJ0ZsdWlkX1NlbnNpdGl2ZScsICdGYXRfU3VwcHJlc3Npb24nKToNCiAgICAgICAgICAgIGlmIGMgbm90IGluIGluZGV4X2RmLmNvbHVtbnM6DQogICAgICAgICAgICAgICAgaW5kZXhfZGZbY10gPSBbXSBpZiBpbmRleF9kZi5lbXB0eSBlbHNlIChucC5uYW4gaWYgYyAhPSAnQW5hdG9taWNhbF9QbGFuZScgZWxzZSBOb25lKQ0KICAgICAgICByZXR1cm4gaW5kZXhfZGYNCiAgICBmcmFtZXMgPSBbXQ0KICAgIGZvciBuYW1lIGluICgndHJhaW5fc2VyaWVzLmNzdicsICd0ZXN0X3Nlcmllcy5jc3YnKToNCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihyb290LCBuYW1lKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToNCiAgICAgICAgICAgIGZyYW1lcy5hcHBlbmQocGQucmVhZF9jc3YocCkpDQogICAgaWYgbm90IGZyYW1lczoNCiAgICAgICAgZm9yIGMgaW4gKCdBbmF0b21pY2FsX1BsYW5lJywgJ0ZsdWlkX1NlbnNpdGl2ZScsICdGYXRfU3VwcHJlc3Npb24nKToNCiAgICAgICAgICAgIGluZGV4X2RmW2NdID0gbnAubmFuIGlmIGMgIT0gJ0FuYXRvbWljYWxfUGxhbmUnIGVsc2UgTm9uZQ0KICAgICAgICByZXR1cm4gaW5kZXhfZGYNCiAgICBjc3YgPSBwZC5jb25jYXQoZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkuZHJvcF9kdXBsaWNhdGVzKCdTZXJpZXNJbnN0YW5jZVVJRCcpDQogICAga2VlcCA9IFtjIGZvciBjIGluICgnU2VyaWVzSW5zdGFuY2VVSUQnLCAnQW5hdG9taWNhbF9QbGFuZScsICdGbHVpZF9TZW5zaXRpdmUnLCAnRmF0X1N1cHByZXNzaW9uJykgaWYgYyBpbiBjc3YuY29sdW1uc10NCiAgICBvdXQgPSBpbmRleF9kZi5tZXJnZShjc3Zba2VlcF0sIG9uPSdTZXJpZXNJbnN0YW5jZVVJRCcsIGhvdz0nbGVmdCcpDQogICAgZm9yIGMgaW4gKCdBbmF0b21pY2FsX1BsYW5lJywgJ0ZsdWlkX1NlbnNpdGl2ZScsICdGYXRfU3VwcHJlc3Npb24nKToNCiAgICAgICAgaWYgYyBub3QgaW4gb3V0LmNvbHVtbnM6DQogICAgICAgICAgICBvdXRbY10gPSBucC5uYW4gaWYgYyAhPSAnQW5hdG9taWNhbF9QbGFuZScgZWxzZSBOb25lDQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBzYXZlX2luZGV4KGRmLCBwYXRoKToNCiAgICB0ZW1wb3JhcnkgPSBmIntwYXRofS50bXAiDQogICAgd2l0aCBvcGVuKHRlbXBvcmFyeSwgJ3diJykgYXMgZmg6DQogICAgICAgIHBpY2tsZS5kdW1wKGRmLCBmaCwgcHJvdG9jb2w9NCkNCiAgICAgICAgZmguZmx1c2goKQ0KICAgICAgICBvcy5mc3luYyhmaC5maWxlbm8oKSkNCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkNCg0KDQpkZWYgbG9hZF9pbmRleChwYXRoKToNCiAgICB3aXRoIG9wZW4ocGF0aCwgJ3JiJykgYXMgZmg6DQogICAgICAgIHJldHVybiBwaWNrbGUubG9hZChmaCkNCg0KDQpkZWYgaW5kZXhfcmVwb3J0KGRmKToNCiAgICAiIiJIdW1hbi1yZWFkYWJsZSBkYXRhLWNvbnRyYWN0IHN0YXRpc3RpY3MgKHByaW50ZWQgYnkgdG9vbHMvYnVpbGRfaW5kZXgucHkpLiIiIg0KICAgIHJlcCA9IHt9DQogICAgaWYgZGYuZW1wdHk6DQogICAgICAgIHJlcFsnc2VyaWVzJ10gPSAwDQogICAgICAgIHJlcFsnc2VyaWVzX3dpdGhfZXJyb3InXSA9IDANCiAgICAgICAgcmVwWydlbXB0eV9zZXJpZXMnXSA9IDANCiAgICAgICAgcmV0dXJuIHJlcA0KICAgIG9rID0gZGZbZGYuZXJyLmlzbmEoKV0gaWYgJ2VycicgaW4gZGYgZWxzZSBkZg0KICAgIHJlcFsnc2VyaWVzJ10gPSBsZW4oZGYpDQogICAgcmVwWydzZXJpZXNfd2l0aF9lcnJvciddID0gaW50KGRmLmVyci5ub3RuYSgpLnN1bSgpKSBpZiAnZXJyJyBpbiBkZiBlbHNlIDANCiAgICByZXBbJ2VtcHR5X3NlcmllcyddID0gaW50KChkZi5uX2ZpbGVzID09IDApLnN1bSgpKSBpZiAnbl9maWxlcycgaW4gZGYgZWxzZSAwDQogICAgaWYgJ29yZGVyX21ldGhvZCcgaW4gb2s6DQogICAgICAgIHJlcFsnb3JkZXJfbWV0aG9kJ10gPSBvay5vcmRlcl9tZXRob2QudmFsdWVfY291bnRzKCkudG9fZGljdCgpDQogICAgaWYgJ29yaWVudF9jb2RlJyBpbiBvayBhbmQgJ3BsYW5lX2dlbycgaW4gb2s6DQogICAgICAgIHJlcFsnb3JpZW50X2NvZGVzJ10gPSB7Zid7cH06e2N9JzogaW50KG4pIGZvciAocCwgYyksIG4gaW4gb2suZ3JvdXBieSgncGxhbmVfZ2VvJykub3JpZW50X2NvZGUudmFsdWVfY291bnRzKCkuaXRlbXMoKX0NCiAgICAgICAgYmFkID0gMA0KICAgICAgICBmb3IgcGwsIGNvZGUgaW4gY29uZmlnLkNBTk9OX09SSUVOVC5pdGVtcygpOg0KICAgICAgICAgICAgc3ViID0gb2tbb2sucGxhbmVfZ2VvID09IHBsXQ0KICAgICAgICAgICAgYmFkICs9IGludCgoc3ViLm9yaWVudF9jb2RlICE9IGNvZGUpLnN1bSgpKQ0KICAgICAgICByZXBbJ25vbl9jYW5vbmljYWxfb3JpZW50YXRpb24nXSA9IGJhZA0KICAgIGlmICdBbmF0b21pY2FsX1BsYW5lJyBpbiBvazoNCiAgICAgICAgbSA9IG9rW29rLkFuYXRvbWljYWxfUGxhbmUubm90bmEoKSAmIG9rLnBsYW5lX2dlby5ub3RuYSgpXQ0KICAgICAgICByZXBbJ2Nzdl9wbGFuZV92c19nZW9tZXRyeV9taXNtYXRjaCddID0gaW50KChtLkFuYXRvbWljYWxfUGxhbmUgIT0gbS5wbGFuZV9nZW8pLnN1bSgpKQ0KICAgIGlmICduX2Ryb3BwZWRfc2hhcGUnIGluIG9rOg0KICAgICAgICByZXBbJ3Nlcmllc193aXRoX21peGVkX3NoYXBlcyddID0gaW50KChvay5uX2Ryb3BwZWRfc2hhcGUgPiAwKS5zdW0oKSkNCiAgICBpZiAnbl9iYWQnIGluIG9rOg0KICAgICAgICByZXBbJ3Nlcmllc193aXRoX3VucmVhZGFibGVfc2xpY2VzJ10gPSBpbnQoKG9rLm5fYmFkID4gMCkuc3VtKCkpDQogICAgcmV0dXJuIHJlcA0K',
    'src/data/preprocess/loader.py': 'IiIiQ2FjaGUgLT4gdHJhaW5pbmcvZXZhbCBzYW1wbGUgKG51bXB5IG9ubHksIHNvIGl0IGlzIHRlc3RhYmxlIHdpdGhvdXQgdG9yY2g7IGRhdGFzZXQucHkganVzdCB3cmFwcyBpdCkuIiIiDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gLiBpbXBvcnQgc2FtcGxpbmcNCg0KDQpkZWYgbWFrZV9zYW1wbGUoY2FjaGUsIGksIGNmZywgdHJhaW49RmFsc2UsIG5fdXNlPU5vbmUsIHJuZz1Ob25lLCBhdWc9Tm9uZSk6DQogICAgIiIiT25lIHN0dWR5IGZyb20gdGhlIG1lbW1hcCBjYWNoZS4NCg0KICAgIFJldHVybnMgaW1ncyB1aW50OCBbUywgVywgRywgSCwgV10sIHNsb3RfbWFzayB1aW50OCBbU10sIHdpbl9tYXNrIGJvb2wgW1MsIFddLg0KICAgIHRyYWluOiBpbmRlcGVuZGVudCBzdHJhdGlmaWVkLXJhbmRvbSB3aW5kb3dzIHBlciBzbG90ICsgb25lIGdlb21ldHJpYy9pbnRlbnNpdHkgYXVnbWVudGF0aW9uIHBlciBzbG90Lg0KICAgIGV2YWwgOiB0aGUgc2FtZSBldmVubHkgc3BhY2VkIHdpbmRvd3MgZm9yIGV2ZXJ5IHNsb3QgKG5fdXNlPU5vbmUgLT4gYWxsIHdpbmRvd3MpLg0KICAgIGF1ZyAgOiBpZiBOb25lLCBkZWZhdWx0cyB0byB0cmFpbi4gSWYgZXhwbGljaXRseSBib29sLCBmb3JjZXMgYXVnbWVudGF0aW9uIG9uIG9yIG9mZiAoZS5nLiBmb3IgVFRBKS4iIiINCiAgICBTLCBHLCBIID0gY29uZmlnLk5fU0xPVFMsIGNmZy5ncm91cCwgY2ZnLmltZ19zaXplDQogICAgRCwgc3RyaWRlID0gY2ZnLnN0YWNrX2RlcHRoLCBjZmcud2luX3N0cmlkZQ0KICAgIFcgPSBzYW1wbGluZy5uX3dpbmRvd3MoRCwgRywgc3RyaWRlKSBpZiBuX3VzZSBpcyBOb25lIGVsc2UgbWluKG5fdXNlLCBzYW1wbGluZy5uX3dpbmRvd3MoRCwgRywgc3RyaWRlKSkNCiAgICBpbWdzID0gbnAuemVyb3MoKFMsIFcsIEcsIEgsIEgpLCBucC51aW50OCkNCiAgICB3bWFzayA9IG5wLnplcm9zKChTLCBXKSwgYm9vbCkNCiAgICBybmcgPSBybmcgaWYgcm5nIGlzIG5vdCBOb25lIGVsc2UgbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApDQogICAgc2hvdWxkX2F1ZyA9IGF1ZyBpZiBhdWcgaXMgbm90IE5vbmUgZWxzZSB0cmFpbg0KICAgIGZvciBzIGluIHJhbmdlKFMpOg0KICAgICAgICBpZiBub3QgY2FjaGUuc2xvdFtpLCBzXToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHN0YXJ0cyA9IHNhbXBsaW5nLndpbmRvd19zdGFydHMoRCwgRywgc3RyaWRlLCBXLCB0cmFpbiwgcm5nKQ0KICAgICAgICB3aW4sIHd2ID0gc2FtcGxpbmcuZ2F0aGVyX3dpbmRvd3MoY2FjaGUuaW1hZ2VzW2ksIHNdLCBjYWNoZS52YWxpZFtpLCBzXSwgc3RhcnRzLCBHKQ0KICAgICAgICBpZiBzaG91bGRfYXVnOg0KICAgICAgICAgICAgd2luID0gc2FtcGxpbmcuYXVnbWVudF9zbG90KHdpbiwgcm5nLCBjb25maWcuQVVHX1JPVF9ERUcsIGNvbmZpZy5BVUdfU0NBTEUsIGNvbmZpZy5BVUdfU0hJRlQsIGNvbmZpZy5BVUdfSU5URU5TSVRZKQ0KICAgICAgICBuID0gbGVuKHN0YXJ0cykNCiAgICAgICAgaW1nc1tzLCA6bl0sIHdtYXNrW3MsIDpuXSA9IHdpbiwgd3YNCiAgICBzbG90ID0gKG5wLmFzYXJyYXkoY2FjaGUuc2xvdFtpXSkgPiAwKSAmIHdtYXNrLmFueShheGlzPTEpDQogICAgcmV0dXJuIGltZ3MsIHNsb3QuYXN0eXBlKG5wLnVpbnQ4KSwgd21hc2sNCg0KDQpkZWYgbGFiZWxfYXJyYXlzKGRmLCBzdHVkaWVzX2luZGV4KToNCiAgICAiIiJUYXJnZXRzL3dlaWdodHMgYXMgZGVuc2UgZmxvYXQzMiBhcnJheXMgYWxpZ25lZCB0byBkZiByb3dzIChubyBwYW5kYXMgaW4gdGhlIGhvdCBsb29wKS4NCg0KICAgIE1pc3NpbmcgbGFiZWwgKE5hTikgLT4gdGFyZ2V0IDAgYW5kIHdlaWdodCAwIChtYXNrZWQpLCBmaXhpbmcgdGhlIE5hTi1sb3NzIGJ1ZyAoOTguNyUgb2Ygc3R1ZGllcyBhcmUgdW5sYWJlbGVkKS4NCiAgICBBbiBvcHRpb25hbCAnPHRhcmdldD5fd2VpZ2h0JyBjb2x1bW4gc2NhbGVzIGxhYmVsbGVkIGVudHJpZXMgKGUuZy4gcmVwb3J0LWV4dHJhY3RvciBjb25maWRlbmNlKS4iIiINCiAgICB2YWxpZF9tYXNrID0gZGZbJ1N0dWR5SW5zdGFuY2VVSUQnXS5pc2luKHN0dWRpZXNfaW5kZXgpDQogICAgaWYgbm90IHZhbGlkX21hc2suYWxsKCk6DQogICAgICAgIGRmID0gZGZbdmFsaWRfbWFza10ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQ0KICAgIFkgPSBucC56ZXJvcygobGVuKGRmKSwgbGVuKGNvbmZpZy5UQVJHRVRTKSksIG5wLmZsb2F0MzIpDQogICAgV3QgPSBucC56ZXJvc19saWtlKFkpDQogICAgZm9yIGosIHQgaW4gZW51bWVyYXRlKGNvbmZpZy5UQVJHRVRTKToNCiAgICAgICAgaWYgdCBub3QgaW4gZGYuY29sdW1uczoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHkgPSBkZlt0XS5hc3R5cGUoZmxvYXQpLnZhbHVlcw0KICAgICAgICBpZiBucC5pc2luZih5KS5hbnkoKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJsYWJlbHMgZm9yIHt0IXJ9IG11c3QgYmUgZmluaXRlIG9yIG1pc3NpbmciKQ0KICAgICAgICBsYWIgPSB+bnAuaXNuYW4oeSkNCiAgICAgICAgaWYgbm90IG5wLmxvZ2ljYWxfYW5kKHlbbGFiXSA+PSAwLjAsIHlbbGFiXSA8PSAxLjApLmFsbCgpOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImxhYmVscyBmb3Ige3Qhcn0gbXVzdCBiZSBpbiBbMCwgMV0iKQ0KICAgICAgICBZWzosIGpdID0gbnAud2hlcmUobGFiLCB5LCAwLjApDQogICAgICAgIGlmIGYne3R9X3dlaWdodCcgaW4gZGYuY29sdW1uczoNCiAgICAgICAgICAgIHcgPSBkZltmJ3t0fV93ZWlnaHQnXS5hc3R5cGUoZmxvYXQpLnZhbHVlcw0KICAgICAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKHdbbGFiXSkuYWxsKCkgb3IgKHdbbGFiXSA8IDApLmFueSgpIG9yICh3W2xhYl0gPiAxKS5hbnkoKToNCiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgICAgICBmImNvbmZpZGVuY2Ugd2VpZ2h0cyBmb3Ige3Qhcn0gbXVzdCBiZSBmaW5pdGUgYW5kIGluIFswLCAxXSINCiAgICAgICAgICAgICAgICApDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICB3ID0gbnAub25lcyhsZW4oZGYpLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgICAgICBXdFs6LCBqXSA9IG5wLndoZXJlKGxhYiwgdywgMC4wKQ0KICAgIHJvd3MgPSBucC5hcnJheShbc3R1ZGllc19pbmRleFtzXSBmb3IgcyBpbiBkZlsnU3R1ZHlJbnN0YW5jZVVJRCddXSwgZHR5cGU9bnAuaW50NjQpDQogICAgcmV0dXJuIHJvd3MsIFksIFd0DQo=',
    'src/data/preprocess/nlp_extractor.py': '...',
    'src/data/preprocess/pipeline.py': 'IiIiT25lIGNvZGUgcGF0aCBmb3IgZXZlcnl0aGluZzogb2ZmbGluZSBjYWNoZSBidWlsZCwgdHJhaW5pbmcgZmFsbGJhY2ssIGFuZCB0aGUgaW5mZXJlbmNlIG5vdGVib29rLg0KDQpwcmVwYXJlX3NlcmllcyAvIHByZXBhcmVfc3R1ZHkgTkVWRVIgcmFpc2UgKGhpZGRlbi1yZXJ1biBsZXNzb246IGRhdGEtZGVwZW5kZW50IGFib3J0cyBhcmUgd2hhdCBicm9rZSBlYXJsaWVyDQpwdWJsaWMgdmVyc2lvbnMpLiBFdmVyeSBwcm9ibGVtIGJlY29tZXMgYW4gZW50cnkgaW4gYGluZm9gIGFuZCBhIG1hc2tlZCBzbG90LCBuZXZlciBhIGNyYXNoIG9yIGEgTmFOLg0KIiIiDQppbXBvcnQgbnVtcHkgYXMgbnANCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCmZyb20gLiBpbXBvcnQgZ2VvbWV0cnkgYXMgZ2VvDQpmcm9tIC4gaW1wb3J0IHBpeGVscw0KDQoNCmRlZiBpbmRleF90b19yZWNvcmRzKGRmKToNCiAgICAiIiJEYXRhRnJhbWUgLT4ge1Nlcmllc0luc3RhbmNlVUlEOiBkaWN0fS4gRG9uZSBvbmNlOyBhdm9pZHMgcGFuZGFzIGluIHRoZSBob3QgcGF0aC4iIiINCiAgICByZXR1cm4ge3JbJ1Nlcmllc0luc3RhbmNlVUlEJ106IHIgZm9yIHIgaW4gZGYudG9fZGljdCgncmVjb3JkcycpfSBpZiBsZW4oZGYpIGVsc2Uge30NCg0KDQpkZWYgc3R1ZHlfc2lkZXMoaW5kZXhfZGYpOg0KICAgICIiIntzdHVkeTogbGF0ZXJhbGl0eSBkaWN0fSB1c2luZyB0aGUgRElDT00gdGFnIGZpcnN0LCBwYXRpZW50LXggZ2VvbWV0cnkgc2Vjb25kLiIiIg0KICAgIGlmIGluZGV4X2RmLmVtcHR5IG9yICdTdHVkeUluc3RhbmNlVUlEJyBub3QgaW4gaW5kZXhfZGYuY29sdW1uczoNCiAgICAgICAgcmV0dXJuIHt9DQogICAgb3V0ID0ge30NCiAgICBjb2xzID0gW2MgZm9yIGMgaW4gKCdTdHVkeUluc3RhbmNlVUlEJywgJ0xhdGVyYWxpdHknLCAnSW1hZ2VMYXRlcmFsaXR5JywgJ2NlbnRlcl94JykgaWYgYyBpbiBpbmRleF9kZi5jb2x1bW5zXQ0KICAgIGZvciBzdCwgZyBpbiBpbmRleF9kZltjb2xzXS5ncm91cGJ5KCdTdHVkeUluc3RhbmNlVUlEJywgc29ydD1GYWxzZSk6DQogICAgICAgIG91dFtzdF0gPSBnZW8ucmVzb2x2ZV9sYXRlcmFsaXR5KGcudG9fZGljdCgncmVjb3JkcycpKQ0KICAgIHJldHVybiBvdXQNCg0KDQpkZWYgcHJlcGFyZV9zZXJpZXMocmVjLCBwbGFuZSwgc2lkZSwgY2ZnKToNCiAgICAiIiItPiAodWludDggW0QsUyxTXSwgdmFsaWQgW0RdLCBpbmZvKSBvciAoTm9uZSwgTm9uZSwgaW5mbykuIiIiDQogICAgc3RhY2ssIHZhbGlkLCBpbmZvID0gcGl4ZWxzLmJ1aWxkX3N0YWNrKHJlYywgY2ZnKQ0KICAgIGlmIHN0YWNrIGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lLCBOb25lLCBpbmZvDQogICAgaW5mb1sncmVvcmllbnRlZCddID0gRmFsc2UNCiAgICBpZiBjZmcucmVvcmllbnQ6DQogICAgICAgIHN0YWNrLCBjaGFuZ2VkID0gZ2VvLnJlb3JpZW50X3N0YWNrKHN0YWNrLCByZWMuZ2V0KCdvcmllbnRfY29kZScpLCBwbGFuZSkNCiAgICAgICAgaW5mb1sncmVvcmllbnRlZCddID0gYm9vbChjaGFuZ2VkKSAgICAgICAgICAjIChhIHRyYW5zcG9zaXRpb24gbGVhdmVzIHRoZSBkZXB0aCBheGlzIHVudG91Y2hlZCkNCiAgICBpZiBjZmcubGF0X2Nhbm9uIGFuZCBzaWRlID09ICdSJzoNCiAgICAgICAgc3RhY2sgPSBnZW8uY2Fub25pY2FsX3NpZGUoc3RhY2ssIHBsYW5lLCAnUicpDQogICAgICAgIGlmIHBsYW5lID09ICdTYWdpdHRhbCc6DQogICAgICAgICAgICB2YWxpZCA9IHZhbGlkWzo6LTFdLmNvcHkoKQ0KICAgICAgICBpbmZvWydtaXJyb3JlZCddID0gVHJ1ZQ0KICAgIHJldHVybiBzdGFjaywgdmFsaWQsIGluZm8NCg0KDQpkZWYgcHJlcGFyZV9zdHVkeShzdHVkeSwgc2xvdF9yb3csIHJlY29yZHMsIHNpZGUsIGNmZyk6DQogICAgIiIic2xvdF9yb3c6IHtzbG90X25hbWU6IFNlcmllc0luc3RhbmNlVUlEIG9yIE5vbmV9Lg0KDQogICAgLT4gZGljdChzdGFjayB1aW50OCBbUyxELEgsV10sIHZhbGlkIGJvb2wgW1MsRF0sIHNsb3RfbWFzayB1aW50OCBbU10sIGluZm8ge3Nsb3Q6IGluZm99LCBzaWRlKSIiIg0KICAgIFMsIEQsIEggPSBjb25maWcuTl9TTE9UUywgY2ZnLnN0YWNrX2RlcHRoLCBjZmcuaW1nX3NpemUNCiAgICBzdGFjayA9IG5wLnplcm9zKChTLCBELCBILCBIKSwgbnAudWludDgpDQogICAgdmFsaWQgPSBucC56ZXJvcygoUywgRCksIGJvb2wpDQogICAgbWFzayA9IG5wLnplcm9zKFMsIG5wLnVpbnQ4KQ0KICAgIGluZm9zID0ge30NCiAgICBmb3IgaywgKG5hbWUsIHBsYW5lLCBfLCBfKSBpbiBlbnVtZXJhdGUoY29uZmlnLlNMT1RTKToNCiAgICAgICAgdWlkID0gc2xvdF9yb3cuZ2V0KG5hbWUpDQogICAgICAgIGlmIG5vdCB1aWQgb3IgdWlkIG5vdCBpbiByZWNvcmRzOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgc3QsIHYsIGluZm8gPSBwcmVwYXJlX3NlcmllcyhyZWNvcmRzW3VpZF0sIHBsYW5lLCBzaWRlLCBjZmcpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgYmVsdCBhbmQgYnJhY2VzDQogICAgICAgICAgICBzdCwgdiwgaW5mbyA9IE5vbmUsIE5vbmUsIGRpY3QoZXJyb3JzPVtmJ3t0eXBlKGUpLl9fbmFtZV9ffToge3N0cihlKVs6NjBdfSddKQ0KICAgICAgICBpbmZvc1tuYW1lXSA9IGluZm8NCiAgICAgICAgaWYgc3QgaXMgTm9uZToNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHN0YWNrW2tdLCB2YWxpZFtrXSwgbWFza1trXSA9IHN0LCB2LCAxDQogICAgcmV0dXJuIGRpY3Qoc3RhY2s9c3RhY2ssIHZhbGlkPXZhbGlkLCBzbG90X21hc2s9bWFzaywgaW5mbz1pbmZvcywgc2lkZT1zaWRlKQ0K',
    'src/data/preprocess/pixels.py': 'IiIiUGl4ZWwgcGlwZWxpbmU6IGRlY29kZSAtPiBwaHlzaWNhbCBjcm9wIC0+IHJvYnVzdCBub3JtYWxpc2UgLT4gcmVzYW1wbGUgLT4gdWludDggc3RhY2suDQoNClRocm91Z2hwdXQgZGVzaWduIChlYWNoIGNob2ljZSBtZWFzdXJlZC9qdXN0aWZpZWQgaW4gdG9vbHMvYmVuY2gucHkpOg0KICAqIGRlY29kZSBSQVcgaW50ZWdlcnMsIGNyb3AgRklSU1QsIGNvbnZlcnQgb25seSB0aGUgY3JvcHBlZCB3aW5kb3cgdG8gZmxvYXQzMg0KICAqIHNsYWIgYXZlcmFnaW5nIG9mIHRoaW4gMy1EIHNsaWNlcyBoYXBwZW5zIGF0IGNyb3AgcmVzb2x1dGlvbiwgb25jZSBwZXIgdGFyZ2V0IHNsaWNlDQogICogcGVyY2VudGlsZXMgb24gYSAyeDItc3Vic2FtcGxlZCBzdGFjayAoZXhhY3QgcGVyY2VudGlsZXMgZm9yIHRoZSAncHVibGljJyBwcmVzZXQpDQogICogY3YyIHJlc2l6ZSAoU0lNRCkgaW5zdGVhZCBvZiBwZXItc2xpY2UgdG9yY2ggY2FsbHM7IGN2Mi9CTEFTIHRocmVhZHMgcGlubmVkIHRvIDEgaW5zaWRlIHdvcmtlcnMNCkRhdGFzZXQtZHJpdmVuIHJ1bGVzIChhdWRpdCk6IHBoeXNpY2FsIG1tIGNyb3Agd2l0aCBlYWNoIHNlcmllcycgb3duIHJvdy9jb2x1bW4gc3BhY2luZyAoNy4zJSBvZiBtYXRyaWNlcyBhcmUNCm5vbi1zcXVhcmUsIDAuMzklIGhhdmUgbm9uLXNxdWFyZSBwaXhlbHMsIDY0MHgxMjgwIHdpZGUgZnJhbWVzIGV4aXN0KSwgYm9yZGVyLW1lZGlhbiBwYWRkaW5nICg3MiUgb2Ygc2VyaWVzIGhhdmUgYQ0Kbm9uLXplcm8gbm9pc2UgZmxvb3IpLCBuZWdhdGl2ZXMgY2xpcHBlZCAoMjQxIENhbm9uIHNlcmllcyksIHBlci1zZXJpZXMgcm9idXN0IHNjYWxlICgyMnggc3ByZWFkKSwgaGVhdnkgdGFpbHMNCig4LjIlIG9mIHNlcmllcyBoYXZlIHA5OS45ID4gMnggcDk5KS4NCiIiIg0KaW1wb3J0IG9zDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBjdjINCg0KZnJvbSAuIGltcG9ydCBkaWNvbWlvIGFzIGRpbw0KDQpjdjIuc2V0TnVtVGhyZWFkcygwKSAgICAgICAgICAgICAgICAgICAgIyBwYXJhbGxlbGlzbSBjb21lcyBmcm9tIHdvcmtlciBwcm9jZXNzZXM7IGF2b2lkIG92ZXJzdWJzY3JpcHRpb24NClBTX0ZBTExCQUNLID0gMC4zMTI1ICAgICAgICAgICAgICAgICAgICAjIGRhdGFzZXQgbWVkaWFuIGluLXBsYW5lIHNwYWNpbmcgKG1tKSwgb25seSBpZiB0aGUgaGVhZGVyIGhhcyBub25lDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0geiBzYW1wbGluZw0KZGVmIHpfZ3JvdXBzKHBvcywgY2ZnLCBzcGFjaW5nLCBtZXRob2Q9J3Bvc2l0aW9uJyk6DQogICAgIiIiV2hpY2ggbmF0aXZlIHNsaWNlcyBmZWVkIGVhY2ggb2YgdGhlIGNmZy5zdGFja19kZXB0aCBzdG9yZWQgc2xpY2VzLg0KDQogICAgUmV0dXJucyAoZ3JvdXBzOiBsaXN0W2xpc3RbaW50XV0sIHZhbGlkOiBib29sW0RdKS4NCiAgICAgICdpbmRleCcgKHB1YmxpYyk6IGxpbnNwYWNlIG92ZXIgY2ZnLmJhbmQgb2YgdGhlIG9yZGVyZWQgc2xpY2UgaW5kaWNlcywgZHVwbGljYXRlcyBwYWRkZWQNCiAgICAgICdtbSc6IHBoeXNpY2FsIGdyaWQgY2ZnLnpfc3RlcF9tbSBjZW50cmVkIG9uIHRoZSBzZXJpZXM7IG5lYXJlc3QgbmF0aXZlIHNsaWNlLCBvciB0aGUgbWVhbiBvZiBhbGwgbmF0aXZlDQogICAgICAgICAgICBzbGljZXMgaW5zaWRlICstc3RlcC8yIHdoZW4gdGhlIG5hdGl2ZSBzcGFjaW5nIGlzIGZpbmVyIHRoYW4gMC43NSpzdGVwICgzLUQgLyB0aGluLXNsaWNlIHNlcmllcykNCiAgICAiIiINCiAgICBuID0gbGVuKHBvcykNCiAgICBEID0gY2ZnLnN0YWNrX2RlcHRoDQogICAgaWYgY2ZnLnpfbW9kZSA9PSAnaW5kZXgnIG9yIG1ldGhvZCAhPSAncG9zaXRpb24nOg0KICAgICAgICAjIHB1YmxpYyBsaW5zcGFjZSBvdmVyIHRoZSBiYW5kOyBhbHNvIHRoZSBzYWZlIGZhbGxiYWNrIHdoZW4gc2xpY2Ugb3JkZXIgY2FtZSBmcm9tIEluc3RhbmNlTnVtYmVyL25hbWUNCiAgICAgICAgIyAocG9zaXRpb25zIGFyZSB0aGVuIGluZGV4IHVuaXRzLCBub3QgbW0sIHNvIGEgbWlsbGltZXRyZSBncmlkIHdvdWxkIGJlIG1lYW5pbmdsZXNzKQ0KICAgICAgICBiYW5kID0gY2ZnLmJhbmQgaWYgY2ZnLnpfbW9kZSA9PSAnaW5kZXgnIGVsc2UgKDAuMCwgMS4wKQ0KICAgICAgICBsbywgaGkgPSBpbnQoYmFuZFswXSAqIChuIC0gMSkpLCBpbnQoYmFuZFsxXSAqIChuIC0gMSkpDQogICAgICAgIGlkeCA9IG5wLnVuaXF1ZShucC5saW5zcGFjZShsbywgaGksIEQpLmFzdHlwZShpbnQpKSBpZiBoaSA+IGxvIGVsc2UgbnAuYXJyYXkoW24gLy8gMl0pDQogICAgICAgIHdoaWxlIGxlbihpZHgpIDwgRDoNCiAgICAgICAgICAgIGlkeCA9IG5wLmFwcGVuZChpZHgsIGlkeFstMV0pDQogICAgICAgIHJldHVybiBbW2ludChpKV0gZm9yIGkgaW4gaWR4WzpEXV0sIG5wLm9uZXMoRCwgYm9vbCkNCiAgICBwb3MgPSBucC5hc2FycmF5KHBvcywgbnAuZmxvYXQ2NCkNCiAgICBzdGVwID0gY2ZnLnpfc3RlcF9tbQ0KICAgIG1pZCA9IDAuNSAqIChwb3NbMF0gKyBwb3NbLTFdKQ0KICAgIHRhcmdldHMgPSBtaWQgKyAobnAuYXJhbmdlKEQpIC0gKEQgLSAxKSAvIDIuMCkgKiBzdGVwDQogICAgc3AgPSBzcGFjaW5nIGlmIG5wLmlzZmluaXRlKHNwYWNpbmcpIGFuZCBzcGFjaW5nID4gMCBlbHNlIHN0ZXANCiAgICB0b2wgPSAwLjc1ICogbWF4KHNwLCBzdGVwKQ0KICAgIG5lYXIgPSBucC5jbGlwKG5wLnNlYXJjaHNvcnRlZChwb3MsIHRhcmdldHMpLCAxLCBtYXgobiAtIDEsIDEpKQ0KICAgIGlmIG4gPT0gMToNCiAgICAgICAgbmVhciA9IG5wLnplcm9zKEQsIGludCkNCiAgICBlbHNlOg0KICAgICAgICBsZWZ0LCByaWdodCA9IHBvc1tuZWFyIC0gMV0sIHBvc1tuZWFyXQ0KICAgICAgICBuZWFyID0gbnAud2hlcmUobnAuYWJzKHRhcmdldHMgLSBsZWZ0KSA8PSBucC5hYnMocmlnaHQgLSB0YXJnZXRzKSwgbmVhciAtIDEsIG5lYXIpDQogICAgZGlzdCA9IG5wLmFicyhwb3NbbmVhcl0gLSB0YXJnZXRzKQ0KICAgIHZhbGlkID0gZGlzdCA8PSB0b2wNCiAgICBncm91cHMgPSBbXQ0KICAgIHNsYWIgPSBzcCA8IDAuNzUgKiBzdGVwDQogICAgZm9yIHQsIGsgaW4gemlwKHRhcmdldHMsIG5lYXIpOg0KICAgICAgICBpZiBzbGFiOg0KICAgICAgICAgICAgbG9faSA9IGludChucC5zZWFyY2hzb3J0ZWQocG9zLCB0IC0gc3RlcCAvIDIuMCwgJ2xlZnQnKSkNCiAgICAgICAgICAgIGhpX2kgPSBpbnQobnAuc2VhcmNoc29ydGVkKHBvcywgdCArIHN0ZXAgLyAyLjAsICdyaWdodCcpKQ0KICAgICAgICAgICAgZ3JvdXBzLmFwcGVuZChsaXN0KHJhbmdlKGxvX2ksIGhpX2kpKSBpZiBoaV9pID4gbG9faSBlbHNlIFtpbnQoayldKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgZ3JvdXBzLmFwcGVuZChbaW50KGspXSkNCiAgICByZXR1cm4gZ3JvdXBzLCB2YWxpZA0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGNyb3AgZ2VvbWV0cnkNCmRlZiBfYm9yZGVyX21lZGlhbihhKToNCiAgICBoLCB3ID0gYS5zaGFwZQ0KICAgIGIgPSBtYXgoMiwgaW50KDAuMDIgKiBtaW4oaCwgdykpKQ0KICAgIHJpbmcgPSBucC5jb25jYXRlbmF0ZShbYVs6Yl0ucmF2ZWwoKSwgYVstYjpdLnJhdmVsKCksIGFbYjotYiwgOmJdLnJhdmVsKCksIGFbYjotYiwgLWI6XS5yYXZlbCgpXSkNCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKHJpbmcpKQ0KDQoNCmRlZiBfZmdfY2VudGVyKHJhd3MpOg0KICAgICIiIkZvcmVncm91bmQgYmJveCBjZW50cmUgKHJvdywgY29sKSBpbiBmdWxsLWZyYW1lIHBpeGVscywgZnJvbSB0aGUgbWVhbiBvZiB0aGUgZGVjb2RlZCBzbGljZXMuIiIiDQogICAgcmVmID0gbnAubWVhbihbbnAubWF4aW11bShhWzo6NCwgOjo0XS5hc3R5cGUobnAuZmxvYXQzMiksIDApIGZvciBhIGluIHJhd3NdLCBheGlzPTApDQogICAgcDk5ID0gbnAucGVyY2VudGlsZShyZWYsIDk5KQ0KICAgIGlmIHA5OSA8PSAwOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIGZnID0gcmVmID4gMC4xICogcDk5DQogICAgaWYgbm90IGZnLmFueSgpOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHJyLCBjYyA9IG5wLndoZXJlKGZnLmFueSgxKSlbMF0sIG5wLndoZXJlKGZnLmFueSgwKSlbMF0NCiAgICByZXR1cm4gNC4wICogKHJyWzBdICsgcnJbLTFdKSAvIDIuMCwgNC4wICogKGNjWzBdICsgY2NbLTFdKSAvIDIuMA0KDQoNCmRlZiBjcm9wX3BsYW4oc2hhcGUsIHBzX3JvdywgcHNfY29sLCBjZmcsIHJhd3M9Tm9uZSk6DQogICAgIiIiLT4gKHkwLCB4MCwgSGMsIFdjKSBvZiB0aGUgcGh5c2ljYWwgY3JvcCB3aW5kb3cgaW4gZnVsbC1mcmFtZSBwaXhlbHMgKG1heSBleGNlZWQgdGhlIGZyYW1lIC0+IHBhZGRlZCksDQogICAgb3IgTm9uZSB3aGVuIHRoZSBwdWJsaWMgcnVsZSBza2lwcyB0aGUgY3JvcC4iIiINCiAgICBILCBXID0gc2hhcGUNCiAgICBvayA9IG5wLmlzZmluaXRlKHBzX3JvdykgYW5kIG5wLmlzZmluaXRlKHBzX2NvbCkgYW5kIHBzX3JvdyA+IDAgYW5kIHBzX2NvbCA+IDANCiAgICBpZiBjZmcucGFkX21vZGUgPT0gJ3B1YmxpYyc6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIHB1YmxpYyBydWxlLCB2ZXJiYXRpbQ0KICAgICAgICBpZiBub3QgKG9rIGFuZCBwc19yb3cgPiAwKToNCiAgICAgICAgICAgIHJldHVybiBOb25lDQogICAgICAgIHdhbnQgPSBpbnQocm91bmQoY2ZnLmNyb3BfbW0gLyBwc19yb3cpKQ0KICAgICAgICBpZiAxNiA8IHdhbnQgPCBtaW4oSCwgVyk6DQogICAgICAgICAgICBjeSwgY3ggPSBIIC8vIDIsIFcgLy8gMg0KICAgICAgICAgICAgaGFsZiA9IHdhbnQgLy8gMg0KICAgICAgICAgICAgcmV0dXJuIG1heCgwLCBjeSAtIGhhbGYpLCBtYXgoMCwgY3ggLSBoYWxmKSwgbWluKEgsIGN5ICsgaGFsZikgLSBtYXgoMCwgY3kgLSBoYWxmKSwgXA0KICAgICAgICAgICAgICAgIG1pbihXLCBjeCArIGhhbGYpIC0gbWF4KDAsIGN4IC0gaGFsZikNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBpZiBub3Qgb2s6DQogICAgICAgIHBzX3JvdyA9IHBzX2NvbCA9IFBTX0ZBTExCQUNLDQogICAgSGMgPSBtYXgoMTYsIGludChyb3VuZChjZmcuY3JvcF9tbSAvIHBzX3JvdykpKQ0KICAgIFdjID0gbWF4KDE2LCBpbnQocm91bmQoY2ZnLmNyb3BfbW0gLyBwc19jb2wpKSkNCiAgICBjeSwgY3ggPSAoSCAtIDEpIC8gMi4wLCAoVyAtIDEpIC8gMi4wDQogICAgYXNwZWN0ID0gKFcgKiBwc19jb2wpIC8gKEggKiBwc19yb3cpDQogICAgaWYgY2ZnLndpZGVfY2VudGVyIGFuZCByYXdzIGFuZCAoYXNwZWN0ID4gY2ZnLndpZGVfcmF0aW8gb3IgYXNwZWN0IDwgMS4wIC8gY2ZnLndpZGVfcmF0aW8pOg0KICAgICAgICBjID0gX2ZnX2NlbnRlcihyYXdzKQ0KICAgICAgICBpZiBjIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgaWYgYXNwZWN0ID4gMToNCiAgICAgICAgICAgICAgICBjeCA9IGNbMV0NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgY3kgPSBjWzBdDQogICAgeTAgPSBpbnQocm91bmQoY3kgLSAoSGMgLSAxKSAvIDIuMCkpDQogICAgeDAgPSBpbnQocm91bmQoY3ggLSAoV2MgLSAxKSAvIDIuMCkpDQogICAgcmV0dXJuIHkwLCB4MCwgSGMsIFdjDQoNCg0KZGVmIF9jcm9wX2Zsb2F0KHJhdywgc2xvcGUsIGljcHQsIHBsYW4sIGJnX2ZuKToNCiAgICAiIiJDcm9wICh3aXRoIGJvcmRlci1tZWRpYW4gcGFkZGluZykgYW5kIGNvbnZlcnQgT05MWSB0aGUgd2luZG93IHRvIGZsb2F0MzIuIiIiDQogICAgSCwgVyA9IHJhdy5zaGFwZQ0KICAgIGlmIHBsYW4gaXMgTm9uZToNCiAgICAgICAgcmV0dXJuIHJhdy5hc3R5cGUobnAuZmxvYXQzMikgKiBzbG9wZSArIGljcHQNCiAgICB5MCwgeDAsIEhjLCBXYyA9IHBsYW4NCiAgICB5czAsIHhzMCwgeXMxLCB4czEgPSBtYXgoeTAsIDApLCBtYXgoeDAsIDApLCBtaW4oeTAgKyBIYywgSCksIG1pbih4MCArIFdjLCBXKQ0KICAgIHdpbiA9IHJhd1t5czA6eXMxLCB4czA6eHMxXS5hc3R5cGUobnAuZmxvYXQzMikgKiBzbG9wZSArIGljcHQNCiAgICBpZiAoeXMxIC0geXMwLCB4czEgLSB4czApID09IChIYywgV2MpOg0KICAgICAgICByZXR1cm4gd2luDQogICAgb3V0ID0gbnAuZnVsbCgoSGMsIFdjKSwgYmdfZm4ocmF3LCBzbG9wZSwgaWNwdCksIG5wLmZsb2F0MzIpDQogICAgb3V0W3lzMCAtIHkwOnlzMSAtIHkwLCB4czAgLSB4MDp4czEgLSB4MF0gPSB3aW4NCiAgICByZXR1cm4gb3V0DQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbm9ybWFsaXNlIC8gcmVzaXplDQpkZWYgbm9ybWFsaXNlKHZvbCwgdmFsaWQsIGNmZyk6DQogICAgIiIiU3RhY2std2lkZSByb2J1c3Qgc2NhbGluZyB0byBbMCwxXSAoYWxsIHNsaWNlcyBzaGFyZSBvbmUgd2luZG93OyBpbnZhbGlkIHNsaWNlcyB1bnRvdWNoZWQpLiIiIg0KICAgIGlmIGNmZy5jbGlwX25lZ2F0aXZlOg0KICAgICAgICBucC5tYXhpbXVtKHZvbCwgMCwgb3V0PXZvbCkNCiAgICB2ID0gdm9sW3ZhbGlkXQ0KICAgIHN1YiA9IHYgaWYgY2ZnLm5hbWUgPT0gJ3B1YmxpYycgZWxzZSB2WzosIDo6MiwgOjoyXQ0KICAgIGxvLCBoaSA9IG5wLnBlcmNlbnRpbGUoc3ViLCBbY2ZnLm5vcm1fbG8sIGNmZy5ub3JtX2hpXSkNCiAgICB2b2wgLT0gbG8NCiAgICB2b2wgLz0gbWF4KGZsb2F0KGhpIC0gbG8pLCAxZS02KQ0KICAgIG5wLmNsaXAodm9sLCAwLjAsIDEuMCwgb3V0PXZvbCkNCiAgICByZXR1cm4gdm9sDQoNCg0KZGVmIHJlc2l6ZV9zdGFjayh2b2wsIHNpemUsIG1vZGUpOg0KICAgICIiIltELGgsd10gZmxvYXQgLT4gW0Qsc2l6ZSxzaXplXSBmbG9hdC4gJ2FyZWEnID0gYW50aWFsaWFzZWQgd2hlbiBzaHJpbmtpbmcsIGJpbGluZWFyIHdoZW4gZW5sYXJnaW5nLiIiIg0KICAgIEQsIGgsIHcgPSB2b2wuc2hhcGUNCiAgICBpZiBtb2RlID09ICdhcmVhJyBhbmQgKGggPiBzaXplIG9yIHcgPiBzaXplKToNCiAgICAgICAgaW50ZXJwID0gY3YyLklOVEVSX0FSRUENCiAgICBlbHNlOg0KICAgICAgICBpbnRlcnAgPSBjdjIuSU5URVJfTElORUFSDQogICAgb3V0ID0gbnAuZW1wdHkoKEQsIHNpemUsIHNpemUpLCBucC5mbG9hdDMyKQ0KICAgIGZvciBpIGluIHJhbmdlKEQpOg0KICAgICAgICBvdXRbaV0gPSBjdjIucmVzaXplKHZvbFtpXSwgKHNpemUsIHNpemUpLCBpbnRlcnBvbGF0aW9uPWludGVycCkNCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIHRvX3VpbnQ4KHgpOg0KICAgIHggKj0gMjU1LjANCiAgICBucC5yaW50KHgsIG91dD14KQ0KICAgIHJldHVybiB4LmFzdHlwZShucC51aW50OCkNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBvbmUgc2VyaWVzIC0+IHN0YWNrDQpkZWYgYnVpbGRfc3RhY2socmVjLCBjZmcpOg0KICAgICIiInJlYzogaW5kZXggcm93IChkaWN0LWxpa2UpIHdpdGggZGlyLCBvcmRlcmVkX2ZpbGVzLCBwb3NpdGlvbnMsIG5zaWduLCBwc19yb3csIHBzX2NvbCwgc3BhY2luZ19tZWQuDQoNCiAgICAtPiAodWludDggW0QsUyxTXSwgdmFsaWQgYm9vbCBbRF0sIGluZm8pIG9yIChOb25lLCBOb25lLCBpbmZvKSB3aGVuIG5vdGhpbmcgY291bGQgYmUgZGVjb2RlZC4NCiAgICBOZXZlciByYWlzZXM6IGV2ZXJ5IHNsaWNlIGZhaWx1cmUgaXMgcmVjb3JkZWQgaW4gaW5mb1snZXJyb3JzJ10uIiIiDQogICAgaW5mbyA9IGRpY3QoZXJyb3JzPVtdLCBwYWRkZWQ9RmFsc2UsIHBzX21pc3Npbmc9RmFsc2UsIG5fZGVjb2RlZD0wKQ0KICAgIGZpbGVzID0gbGlzdChyZWNbJ29yZGVyZWRfZmlsZXMnXSkNCiAgICBwb3MgPSBucC5hc2FycmF5KHJlY1sncG9zaXRpb25zJ10sIG5wLmZsb2F0NjQpDQogICAgaWYgbm90IGZpbGVzOg0KICAgICAgICBpbmZvWydlcnJvcnMnXS5hcHBlbmQoJ25vX2ZpbGVzJykNCiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUsIGluZm8NCiAgICBpZiBjZmcub3JkZXJfbW9kZSA9PSAnbm9ybWFsJyBhbmQgZmxvYXQocmVjLmdldCgnbnNpZ24nLCAxLjApKSA8IDA6ICAgICAjIHB1YmxpYyBzaWduIGNvbnZlbnRpb24NCiAgICAgICAgZmlsZXMsIHBvcyA9IGZpbGVzWzo6LTFdLCAtcG9zWzo6LTFdDQogICAgbWV0aG9kID0gc3RyKHJlYy5nZXQoJ29yZGVyX21ldGhvZCcsICdwb3NpdGlvbicpKQ0KICAgIGdyb3VwcywgdmFsaWQgPSB6X2dyb3Vwcyhwb3MsIGNmZywgZmxvYXQocmVjLmdldCgnc3BhY2luZ19tZWQnLCBucC5uYW4pKSwgbWV0aG9kKQ0KICAgIGluZm9bJ3pfZmFsbGJhY2snXSA9IGNmZy56X21vZGUgPT0gJ21tJyBhbmQgbWV0aG9kICE9ICdwb3NpdGlvbicNCiAgICBuZWVkID0gc29ydGVkKHtpIGZvciBnIGluIGdyb3VwcyBmb3IgaSBpbiBnfSkNCiAgICByYXdzID0ge30NCiAgICBmb3IgaSBpbiBuZWVkOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICByYXdzW2ldID0gZGlvLmRlY29kZV9yYXcob3MucGF0aC5qb2luKHJlY1snZGlyJ10sIGZpbGVzW2ldKSkNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgaW5mb1snZXJyb3JzJ10uYXBwZW5kKGYne2ZpbGVzW2ldfToge3R5cGUoZSkuX19uYW1lX199JykNCiAgICBpbmZvWyduX2RlY29kZWQnXSA9IGxlbihyYXdzKQ0KICAgIGlmIG5vdCByYXdzOg0KICAgICAgICByZXR1cm4gTm9uZSwgTm9uZSwgaW5mbw0KICAgIHNoYXBlID0gbWF4KHtyWzBdLnNoYXBlIGZvciByIGluIHJhd3MudmFsdWVzKCl9LCBrZXk9bGFtYmRhIHM6IHN1bShyWzBdLnNoYXBlID09IHMgZm9yIHIgaW4gcmF3cy52YWx1ZXMoKSkpDQogICAgcmF3cyA9IHtpOiByIGZvciBpLCByIGluIHJhd3MuaXRlbXMoKSBpZiByWzBdLnNoYXBlID09IHNoYXBlfQ0KICAgIHBzX3JvdywgcHNfY29sID0gZmxvYXQocmVjLmdldCgncHNfcm93JywgbnAubmFuKSksIGZsb2F0KHJlYy5nZXQoJ3BzX2NvbCcsIG5wLm5hbikpDQogICAgaW5mb1sncHNfbWlzc2luZyddID0gbm90IChucC5pc2Zpbml0ZShwc19yb3cpIGFuZCBucC5pc2Zpbml0ZShwc19jb2wpIGFuZCBwc19yb3cgPiAwIGFuZCBwc19jb2wgPiAwKQ0KICAgIHBsYW4gPSBjcm9wX3BsYW4oc2hhcGUsIHBzX3JvdywgcHNfY29sLCBjZmcsIFtyWzBdIGZvciByIGluIHJhd3MudmFsdWVzKCldIGlmIGNmZy53aWRlX2NlbnRlciBlbHNlIE5vbmUpDQogICAgaWYgcGxhbiBpcyBub3QgTm9uZToNCiAgICAgICAgaW5mb1sncGFkZGVkJ10gPSBwbGFuWzBdIDwgMCBvciBwbGFuWzFdIDwgMCBvciBwbGFuWzBdICsgcGxhblsyXSA+IHNoYXBlWzBdIG9yIHBsYW5bMV0gKyBwbGFuWzNdID4gc2hhcGVbMV0NCiAgICBkZWYgX2JnKHJhdywgc2xvcGUsIGljcHQpOg0KICAgICAgICAjIEZJWCBCOiBNZWRpYW5zIGNvbW11dGUgd2l0aCBtb25vdG9uaWMgbGluZWFyIHRyYW5zZm9ybWF0aW9ucy4NCiAgICAgICAgIyBFeHRyYWN0IGJvcmRlciBmcm9tIFJBVyBpbnRlZ2VyIGFycmF5LCBjb21wdXRlIG1lZGlhbiwgVEhFTiBhcHBseSBzbG9wZS9pY3B0Lg0KICAgICAgICAjIFNhdmVzIGNvbnZlcnRpbmcgdGhlIGVudGlyZSA4MDAsMDAwKyBwaXhlbCBpbWFnZSB0byBmbG9hdDMyLg0KICAgICAgICByZXR1cm4gZmxvYXQoX2JvcmRlcl9tZWRpYW4ocmF3KSkgKiBzbG9wZSArIGljcHQNCg0KICAgIEQgPSBsZW4oZ3JvdXBzKQ0KICAgIGNyb3BzID0ge30NCiAgICBmb3IgaSwgKHJhdywgc2xvcGUsIGljcHQpIGluIHJhd3MuaXRlbXMoKToNCiAgICAgICAgY3JvcHNbaV0gPSBfY3JvcF9mbG9hdChyYXcsIHNsb3BlLCBpY3B0LCBwbGFuLCBfYmcpDQogICAgcmVmX2kgPSBuZXh0KGl0ZXIoY3JvcHMpKQ0KICAgIHZvbCA9IG5wLnplcm9zKChELCkgKyBjcm9wc1tyZWZfaV0uc2hhcGUsIG5wLmZsb2F0MzIpDQogICAgb2sgPSBucC56ZXJvcyhELCBib29sKQ0KICAgIGZvciBkLCBnIGluIGVudW1lcmF0ZShncm91cHMpOg0KICAgICAgICBnb3QgPSBbY3JvcHNbaV0gZm9yIGkgaW4gZyBpZiBpIGluIGNyb3BzXQ0KICAgICAgICBpZiBub3QgZ290Og0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgdm9sW2RdID0gZ290WzBdIGlmIGxlbihnb3QpID09IDEgZWxzZSBucC5tZWFuKGdvdCwgYXhpcz0wLCBkdHlwZT1ucC5mbG9hdDMyKQ0KICAgICAgICBva1tkXSA9IFRydWUNCiAgICB2YWxpZCA9IHZhbGlkICYgb2sNCiAgICBpZiBjZmcuel9tb2RlID09ICdpbmRleCcgb3IgbWV0aG9kICE9ICdwb3NpdGlvbic6ICAgIyBmYWlsZWQgc2xpY2VzIGJvcnJvdyB0aGUgbmVhcmVzdCBkZWNvZGVkIG5laWdoYm91cg0KICAgICAgICBmb3IgZCBpbiBucC53aGVyZSh+b2spWzBdOg0KICAgICAgICAgICAgaiA9IGludChucC5hcmdtaW4obnAud2hlcmUob2ssIG5wLmFicyhucC5hcmFuZ2UoRCkgLSBkKSwgMTAgKiogNikpKQ0KICAgICAgICAgICAgdm9sW2RdID0gdm9sW2pdDQogICAgICAgIHZhbGlkID0gbnAub25lcyhELCBib29sKQ0KICAgIGlmIG5vdCB2YWxpZC5hbnkoKToNCiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUsIGluZm8NCiAgICB2b2wgPSBub3JtYWxpc2Uodm9sLCB2YWxpZCwgY2ZnKQ0KICAgIHZvbCA9IHJlc2l6ZV9zdGFjayh2b2wsIGNmZy5pbWdfc2l6ZSwgY2ZnLnJlc2l6ZV9tb2RlKQ0KICAgIG91dCA9IHRvX3VpbnQ4KHZvbCkNCiAgICBvdXRbfnZhbGlkXSA9IDANCiAgICByZXR1cm4gb3V0LCB2YWxpZCwgaW5mbw0K',
    'src/data/preprocess/qc.py': 'IiIiRGF0YS1jb250cmFjdCBjaGVja3MuIENoZWFwIGVub3VnaCB0byBydW4gb24gZXZlcnkgYnVpbGQ7IHRoZWlyIG51bWJlcnMgYXJlIHRoZSBhY2NlcHRhbmNlIGdhdGVzLiIiIg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQoNCg0KZGVmIGNhY2hlX3Nhbml0eShjYWNoZSwgbl9zYW1wbGU9MjAwLCBzZWVkPTAsIGV4cGVjdGVkX3Nsb3RfbWFzaz1Ob25lKToNCiAgICAiIiJTYW1wbGUgYnVpbHQgc3R1ZGllcyBhbmQgdmVyaWZ5IHRoZSBjYWNoZSBpcyBub3Qgc2lsZW50bHkgYnJva2VuIChjb25zdGFudCBpbWFnZXMsIGVtcHR5LWJ1dC1mbGFnZ2VkIHNsb3RzLA0KICAgIGFsbC1pbnZhbGlkIGRlcHRoLCB3cm9uZyBkdHlwZSkuIiIiDQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpDQogICAgZG9uZSA9IG5wLndoZXJlKG5wLmFzYXJyYXkoY2FjaGUuZG9uZSkgPT0gMSlbMF0NCiAgICBpZiBsZW4oZG9uZSkgPT0gMDoNCiAgICAgICAgcmV0dXJuIGRpY3Qoc3R1ZGllc19jaGVja2VkPTAsIGFjY2VwdGVkPUZhbHNlLCByZWFzb249Im5vIGNvbXBsZXRlZCBzdHVkaWVzIikNCiAgICBub3RfZG9uZSA9IG5wLndoZXJlKG5wLmFzYXJyYXkoY2FjaGUuZG9uZSkgIT0gMSlbMF0NCiAgICBmYWlsZWQgPSBucC5hc2FycmF5KGNhY2hlLmZhaWxlZCkNCiAgICBkZWNvZGVfZXJyb3JzID0gbnAuYXNhcnJheShjYWNoZS5lcnJvcnMpDQogICAgc2xvdF9hcnJheSA9IG5wLmFzYXJyYXkoY2FjaGUuc2xvdCkNCiAgICBkb25lX2FycmF5ID0gbnAuYXNhcnJheShjYWNoZS5kb25lKQ0KICAgIHZhbGlkX2FycmF5ID0gbnAuYXNhcnJheShjYWNoZS52YWxpZCkNCiAgICBleHBlY3RlZF9mYWlsdXJlcyA9IFtdDQogICAgaWYgZXhwZWN0ZWRfc2xvdF9tYXNrIGlzIG5vdCBOb25lOg0KICAgICAgICBleHBlY3RlZF9zbG90X21hc2sgPSBucC5hc2FycmF5KGV4cGVjdGVkX3Nsb3RfbWFzaywgZHR5cGU9Ym9vbCkNCiAgICAgICAgaWYgZXhwZWN0ZWRfc2xvdF9tYXNrLnNoYXBlICE9IGNhY2hlLnNsb3Quc2hhcGU6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgIGYiRXhwZWN0ZWQgc2xvdCBtYXNrIHNoYXBlIHtleHBlY3RlZF9zbG90X21hc2suc2hhcGV9IGRvZXMgbm90IG1hdGNoIGNhY2hlIHtjYWNoZS5zbG90LnNoYXBlfSINCiAgICAgICAgICAgICkNCiAgICAgICAgZXhwZWN0ZWRfZmFpbHVyZXMgPSBucC5hcmd3aGVyZShleHBlY3RlZF9zbG90X21hc2sgJiAobnAuYXNhcnJheShjYWNoZS5zbG90KSA9PSAwKSkNCiAgICBlbXB0eV9zdHVkaWVzID0gaW50KChzbG90X2FycmF5LnN1bShheGlzPTEpID09IDApLnN1bSgpKQ0KICAgIGludmFsaWRfZG9uZV9mbGFncyA9IGludCgofm5wLmlzaW4oZG9uZV9hcnJheSwgWzAsIDFdKSkuc3VtKCkpDQogICAgaW52YWxpZF9zbG90X2ZsYWdzID0gaW50KCh+bnAuaXNpbihzbG90X2FycmF5LCBbMCwgMV0pKS5zdW0oKSkNCiAgICBpbnZhbGlkX3ZhbGlkX2ZsYWdzID0gaW50KCh+bnAuaXNpbih2YWxpZF9hcnJheSwgWzAsIDFdKSkuc3VtKCkpDQogICAgdW5mbGFnZ2VkX3ZhbGlkX3Nsb3RzID0gaW50KA0KICAgICAgICAoKHNsb3RfYXJyYXkgPT0gMCkgJiAodmFsaWRfYXJyYXkuc3VtKGF4aXM9MikgPiAwKSkuc3VtKCkNCiAgICApDQogICAgcGljayA9IHJuZy5jaG9pY2UoZG9uZSwgc2l6ZT1taW4obl9zYW1wbGUsIGxlbihkb25lKSksIHJlcGxhY2U9RmFsc2UpDQogICAgYmFkID0gZGljdChjb25zdGFudF9zbG90PVtdLCBmbGFnZ2VkX2J1dF9ibGFuaz1bXSwgbG93X3ZhbGlkX2RlcHRoPVtdLCB1bmZsYWdnZWRfYnV0X2ZpbGxlZD1bXSkNCiAgICBuX3Nsb3RzID0gMA0KICAgIHN0ZHMsIG1lYW5zLCB2ZnJhYyA9IFtdLCBbXSwgW10NCiAgICBmb3IgaSBpbiBwaWNrOg0KICAgICAgICBmb3IgcyBpbiByYW5nZShjb25maWcuTl9TTE9UUyk6DQogICAgICAgICAgICBmaWxsZWQgPSBib29sKGNhY2hlLnNsb3RbaSwgc10pDQogICAgICAgICAgICBzdCA9IG5wLmFzYXJyYXkoY2FjaGUuaW1hZ2VzW2ksIHMsIDo6MiwgOjo0LCA6OjRdKSAgICAgICAgICAjIGNoZWFwIHN1YnNhbXBsZQ0KICAgICAgICAgICAgaWYgZmlsbGVkOg0KICAgICAgICAgICAgICAgIG5fc2xvdHMgKz0gMQ0KICAgICAgICAgICAgICAgIHYgPSBmbG9hdChucC5tZWFuKGNhY2hlLnZhbGlkW2ksIHNdKSkNCiAgICAgICAgICAgICAgICB2ZnJhYy5hcHBlbmQodikNCiAgICAgICAgICAgICAgICBpZiBzdC5tYXgoKSA9PSAwOg0KICAgICAgICAgICAgICAgICAgICBiYWRbJ2ZsYWdnZWRfYnV0X2JsYW5rJ10uYXBwZW5kKChjYWNoZS5zdHVkaWVzW2ldLCBzKSkNCiAgICAgICAgICAgICAgICBpZiBzdC5zdGQoKSA8IDEuMDoNCiAgICAgICAgICAgICAgICAgICAgYmFkWydjb25zdGFudF9zbG90J10uYXBwZW5kKChjYWNoZS5zdHVkaWVzW2ldLCBzKSkNCiAgICAgICAgICAgICAgICBpZiB2IDwgMC41Og0KICAgICAgICAgICAgICAgICAgICBiYWRbJ2xvd192YWxpZF9kZXB0aCddLmFwcGVuZCgoY2FjaGUuc3R1ZGllc1tpXSwgcykpDQogICAgICAgICAgICAgICAgc3Rkcy5hcHBlbmQoZmxvYXQoc3Quc3RkKCkpKQ0KICAgICAgICAgICAgICAgIG1lYW5zLmFwcGVuZChmbG9hdChzdC5tZWFuKCkpKQ0KICAgICAgICAgICAgZWxpZiBzdC5tYXgoKSA+IDA6DQogICAgICAgICAgICAgICAgYmFkWyd1bmZsYWdnZWRfYnV0X2ZpbGxlZCddLmFwcGVuZCgoY2FjaGUuc3R1ZGllc1tpXSwgcykpDQogICAgb3V0ID0gZGljdChzdHVkaWVzX2NoZWNrZWQ9bGVuKHBpY2spLCBzbG90c19jaGVja2VkPW5fc2xvdHMsDQogICAgICAgICAgICAgICB0b3RhbF9zdHVkaWVzPWxlbihjYWNoZS5zdHVkaWVzKSwNCiAgICAgICAgICAgICAgIGluY29tcGxldGVfc3R1ZGllcz1sZW4obm90X2RvbmUpLA0KICAgICAgICAgICAgICAgZW1wdHlfc3R1ZGllcz1lbXB0eV9zdHVkaWVzLA0KICAgICAgICAgICAgICAgaW52YWxpZF9kb25lX2ZsYWdzPWludmFsaWRfZG9uZV9mbGFncywNCiAgICAgICAgICAgICAgIGludmFsaWRfc2xvdF9mbGFncz1pbnZhbGlkX3Nsb3RfZmxhZ3MsDQogICAgICAgICAgICAgICBpbnZhbGlkX3ZhbGlkX2ZsYWdzPWludmFsaWRfdmFsaWRfZmxhZ3MsDQogICAgICAgICAgICAgICB1bmZsYWdnZWRfdmFsaWRfc2xvdHM9dW5mbGFnZ2VkX3ZhbGlkX3Nsb3RzLA0KICAgICAgICAgICAgICAgZGVjb2RlX2Vycm9ycz1pbnQoZGVjb2RlX2Vycm9ycy5zdW0oKSksDQogICAgICAgICAgICAgICBmYWlsZWRfc2VsZWN0ZWRfc2xvdHM9aW50KGZhaWxlZC5zdW0oKSksDQogICAgICAgICAgICAgICBtaXNzaW5nX2V4cGVjdGVkX3Nsb3RzPWxlbihleHBlY3RlZF9mYWlsdXJlcyksDQogICAgICAgICAgICAgICBtZWFuX3ZhbGlkX2RlcHRoPWZsb2F0KG5wLm1lYW4odmZyYWMpKSBpZiB2ZnJhYyBlbHNlIGZsb2F0KCduYW4nKSwNCiAgICAgICAgICAgICAgIHNsb3RfbWVhbl9wNV9wOTU9W2Zsb2F0KG5wLnBlcmNlbnRpbGUobWVhbnMsIDUpKSwgZmxvYXQobnAucGVyY2VudGlsZShtZWFucywgOTUpKV0gaWYgbWVhbnMgZWxzZSBOb25lLA0KICAgICAgICAgICAgICAgc2xvdF9zdGRfcDVfcDk1PVtmbG9hdChucC5wZXJjZW50aWxlKHN0ZHMsIDUpKSwgZmxvYXQobnAucGVyY2VudGlsZShzdGRzLCA5NSkpXSBpZiBzdGRzIGVsc2UgTm9uZSkNCiAgICBvdXQudXBkYXRlKHtrOiBsZW4odikgZm9yIGssIHYgaW4gYmFkLml0ZW1zKCl9KQ0KICAgIG91dFsnZmFpbHVyZXMnXSA9IHtrOiB2Wzo1XSBmb3IgaywgdiBpbiBiYWQuaXRlbXMoKSBpZiB2fQ0KICAgIG91dFsnZXhwZWN0ZWRfc2xvdF9mYWlsdXJlX2V4YW1wbGVzJ10gPSBbDQogICAgICAgIChjYWNoZS5zdHVkaWVzW2ludChpKV0sIGludChzKSkgZm9yIGksIHMgaW4gZXhwZWN0ZWRfZmFpbHVyZXNbOjEwXQ0KICAgIF0NCiAgICBibG9ja2luZ19waXhlbF9lcnJvcnMgPSBhbnkoDQogICAgICAgIGJhZFtrZXldDQogICAgICAgIGZvciBrZXkgaW4gKCJjb25zdGFudF9zbG90IiwgImZsYWdnZWRfYnV0X2JsYW5rIiwgInVuZmxhZ2dlZF9idXRfZmlsbGVkIikNCiAgICApDQogICAgb3V0WydhY2NlcHRlZCddID0gYm9vbCgNCiAgICAgICAgbGVuKG5vdF9kb25lKSA9PSAwDQogICAgICAgIGFuZCBlbXB0eV9zdHVkaWVzID09IDANCiAgICAgICAgYW5kIGludmFsaWRfZG9uZV9mbGFncyA9PSAwDQogICAgICAgIGFuZCBpbnZhbGlkX3Nsb3RfZmxhZ3MgPT0gMA0KICAgICAgICBhbmQgaW52YWxpZF92YWxpZF9mbGFncyA9PSAwDQogICAgICAgIGFuZCB1bmZsYWdnZWRfdmFsaWRfc2xvdHMgPT0gMA0KICAgICAgICBhbmQgZGVjb2RlX2Vycm9ycy5zdW0oKSA9PSAwDQogICAgICAgIGFuZCBmYWlsZWQuc3VtKCkgPT0gMA0KICAgICAgICBhbmQgbGVuKGV4cGVjdGVkX2ZhaWx1cmVzKSA9PSAwDQogICAgICAgIGFuZCBub3QgYmxvY2tpbmdfcGl4ZWxfZXJyb3JzDQogICAgICAgIGFuZCBvdXRbJ21lYW5fdmFsaWRfZGVwdGgnXSA+IDAuOA0KICAgICkNCiAgICByZXR1cm4gb3V0DQoNCg0KZGVmIGxhdGVyYWxpdHlfcmVwb3J0KHNpZGVzKToNCiAgICBpZiBub3Qgc2lkZXM6DQogICAgICAgIHJldHVybiBkaWN0KHN0dWRpZXM9MCwgYnlfc2lkZT17fSwgYnlfc291cmNlPXt9LCB0YWdfZ2VvbWV0cnlfZGlzYWdyZWU9MCwgdW5yZXNvbHZlZF9mcmFjPTAuMCkNCiAgICBkZiA9IHBkLkRhdGFGcmFtZShzaWRlcykuVA0KICAgIHJlcCA9IGRpY3Qoc3R1ZGllcz1sZW4oZGYpLCBieV9zaWRlPWRmWydzaWRlJ10udmFsdWVfY291bnRzKCkudG9fZGljdCgpIGlmICdzaWRlJyBpbiBkZiBlbHNlIHt9LA0KICAgICAgICAgICAgICAgYnlfc291cmNlPWRmWydzb3VyY2UnXS52YWx1ZV9jb3VudHMoKS50b19kaWN0KCkgaWYgJ3NvdXJjZScgaW4gZGYgZWxzZSB7fSwNCiAgICAgICAgICAgICAgIHRhZ19nZW9tZXRyeV9kaXNhZ3JlZT1pbnQoZGZbJ2Rpc2FncmVlJ10uc3VtKCkpIGlmICdkaXNhZ3JlZScgaW4gZGYgZWxzZSAwKQ0KICAgIHJlcFsndW5yZXNvbHZlZF9mcmFjJ10gPSBmbG9hdCgoZGZbJ3NpZGUnXSA9PSAnVScpLm1lYW4oKSkgaWYgJ3NpZGUnIGluIGRmIGVsc2UgMC4wDQogICAgcmV0dXJuIHJlcA0K',
    'src/data/preprocess/runner.py': 'IiIiTGlicmFyeS1sZXZlbCBlbnRyeSBwb2ludHMgdXNlZCBieSB0aGUgbm90ZWJvb2s6IGluZGV4IC0+IHNsb3RzL2xhdGVyYWxpdHkgLT4gY2FjaGUgLT4gUUMuDQoNCkV2ZXJ5dGhpbmcgaXMgY2hlY2twb2ludGVkL3Jlc3VtYWJsZTsgZXZlcnkgc3RlcCBwcmludHMgdGhlIG51bWJlcnMgdGhhdCBzZXJ2ZSBhcyBhY2NlcHRhbmNlIGdhdGVzLiIiIg0KaW1wb3J0IG9zDQppbXBvcnQganNvbg0KaW1wb3J0IGhhc2hsaWINCmltcG9ydCB0aW1lDQppbXBvcnQgc2h1dGlsDQppbXBvcnQgdGVtcGZpbGUNCg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQpmcm9tIC4gaW1wb3J0IGluZGV4IGFzIHBpeA0KZnJvbSAuIGltcG9ydCBzbG90cyBhcyBwc2xvdHMNCmZyb20gLiBpbXBvcnQgcGlwZWxpbmUNCmZyb20gLiBpbXBvcnQgY2FjaGUgYXMgcGNhY2hlDQpmcm9tIC4gaW1wb3J0IHFjDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBlbnZpcm9ubWVudA0KZGVmIHBpY2tfc2NyYXRjaChjYW5kaWRhdGVzPU5vbmUsIHZlcmJvc2U9VHJ1ZSk6DQogICAgIiIiTGFyZ2VzdCB3cml0YWJsZSBzY3JhdGNoIGRpc2sgT1VUU0lERSAva2FnZ2xlL3dvcmtpbmcgKHdoaWNoIGlzIGNhcHBlZCBhdCB+MjAgR0IgYW5kIHBlcnNpc3RlZCBhcyBvdXRwdXQpLiIiIg0KICAgIGNhbmRzID0gY2FuZGlkYXRlcyBvciBbJy9rYWdnbGUvdGVtcCcsICcva2FnZ2xlL3RtcCcsICcvdG1wJywgdGVtcGZpbGUuZ2V0dGVtcGRpcigpXQ0KICAgIGJlc3QsIHRhYmxlID0gTm9uZSwgW10NCiAgICBmb3IgYyBpbiBkaWN0LmZyb21rZXlzKGNhbmRzKToNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaWYgbm90IG9zLnBhdGguaXNkaXIoYykgb3Igbm90IG9zLmFjY2VzcyhjLCBvcy5XX09LKTogICAgICAjIG9ubHkgZXhpc3RpbmcsIHdyaXRhYmxlIGRpcnMgKG5vIHNpZGUgZWZmZWN0cykNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgZnJlZSA9IHNodXRpbC5kaXNrX3VzYWdlKGMpLmZyZWUgLyAxZTkNCiAgICAgICAgICAgIHRhYmxlLmFwcGVuZCgoYywgcm91bmQoZnJlZSwgMSkpKQ0KICAgICAgICAgICAgaWYgYmVzdCBpcyBOb25lIG9yIGZyZWUgPiBiZXN0WzFdOg0KICAgICAgICAgICAgICAgIGJlc3QgPSAoYywgZnJlZSkNCiAgICAgICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgIGlmIHZlcmJvc2U6DQogICAgICAgIHByaW50KCdzY3JhdGNoIGNhbmRpZGF0ZXMgKHBhdGgsIGZyZWUgR0IpOicsIHRhYmxlKQ0KICAgIGlmIGJlc3QgaXMgTm9uZToNCiAgICAgICAgcmFpc2UgT1NFcnJvcignbm8gd3JpdGFibGUgc2NyYXRjaCBkaXJlY3RvcnkgZm91bmQnKQ0KICAgIHJldHVybiBiZXN0WzBdDQoNCg0KZGVmIGVzdGltYXRlX2NhY2hlX2diKG5fc3R1ZGllcywgY2ZnKToNCiAgICByZXR1cm4gbl9zdHVkaWVzICogY29uZmlnLk5fU0xPVFMgKiBjZmcuc3RhY2tfZGVwdGggKiBjZmcuaW1nX3NpemUgKiogMiAvIDFlOQ0KDQoNCmRlZiBmaXRfY2FjaGVfY2ZnKG5fc3R1ZGllcywgZnJlZV9nYiwgcHJlc2V0PSd2MicsIG1hcmdpbj0wLjkyKToNCiAgICAiIiJMYXJnZXN0IHN0YW5kYXJkIChpbWdfc2l6ZSwgc3RhY2tfZGVwdGgpIHRoYXQgZml0cyB0aGUgZnJlZSBkaXNrLiBUaGUgJ3B1YmxpYycgcHJlc2V0IGlzIGZpeGVkIChEPTEyLCAzMzYgcHgpLiIiIg0KICAgIGlmIHByZXNldCA9PSAncHVibGljJzoNCiAgICAgICAgcmV0dXJuIGNvbmZpZy5nZXRfY2ZnKCdwdWJsaWMnKQ0KICAgIGxhZGRlciA9IFsoMzM2LCAyNCksICgzMzYsIDIwKSwgKDMyMCwgMjApLCAoMjg4LCAyNCksICgyODgsIDIwKSwgKDI1NiwgMjApLCAoMjU2LCAxNiksICgyMjQsIDE2KV0NCiAgICBmb3IgaW1nLCBkZXB0aCBpbiBsYWRkZXI6DQogICAgICAgIGNmZyA9IGNvbmZpZy5nZXRfY2ZnKHByZXNldCwgaW1nX3NpemU9aW1nLCBzdGFja19kZXB0aD1kZXB0aCkNCiAgICAgICAgaWYgZXN0aW1hdGVfY2FjaGVfZ2Iobl9zdHVkaWVzLCBjZmcpIDw9IGZyZWVfZ2IgKiBtYXJnaW46DQogICAgICAgICAgICByZXR1cm4gY2ZnDQogICAgcmV0dXJuIGNvbmZpZy5nZXRfY2ZnKHByZXNldCwgaW1nX3NpemU9MjI0LCBzdGFja19kZXB0aD0xMikNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIDEuIGluZGV4DQpkZWYgX3NoYTI1Nl9maWxlKHBhdGgpOg0KICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkNCiAgICB3aXRoIG9wZW4ocGF0aCwgJ3JiJykgYXMgc3RyZWFtOg0KICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IHN0cmVhbS5yZWFkKDEwMjQgKiAxMDI0KSwgYicnKToNCiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoY2h1bmspDQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBfYXRvbWljX2pzb24ocGF0aCwgcGF5bG9hZCk6DQogICAgdGVtcG9yYXJ5ID0gZiJ7cGF0aH0udG1wIg0KICAgIHdpdGggb3Blbih0ZW1wb3JhcnksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgc3RyZWFtOg0KICAgICAgICBqc29uLmR1bXAocGF5bG9hZCwgc3RyZWFtLCBpbmRlbnQ9MSwgc29ydF9rZXlzPVRydWUsIGRlZmF1bHQ9c3RyKQ0KICAgICAgICBzdHJlYW0uZmx1c2goKQ0KICAgICAgICBvcy5mc3luYyhzdHJlYW0uZmlsZW5vKCkpDQogICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIHBhdGgpDQoNCg0KZGVmIHJ1bl9pbmRleChyb290LCBvdXRfZGlyLCBzcGxpdHM9KCd0cmFpbicsICd0ZXN0JyksIHdvcmtlcnM9Tm9uZSwgbGltaXQ9MCwgcHJlc2V0PSd2MicsIHByb2dyZXNzPVRydWUsIGZvcmNlPUZhbHNlKToNCiAgICAiIiJTY2FuIGV2ZXJ5IHNlcmllcyBvbiBkaXNrLCBqb2luIENTViBmbGFncywgYXNzaWduIHNsb3RzLCByZXNvbHZlIGxhdGVyYWxpdHkuIFNhdmVzIHRvIG91dF9kaXI6DQogICAgaW5kZXgucGtsLCBzbG90cy5wa2wsIHNpZGVzLmpzb24sIHNlcmllc190YWJsZS5jc3YuZ3osIHJlcG9ydC5qc29uLiAtPiBkaWN0KGFubiwgdGFiLCBzaWRlcywgcmVwb3J0KS4iIiINCiAgICBjZmcgPSBjb25maWcuZ2V0X2NmZyhwcmVzZXQpDQogICAgb3MubWFrZWRpcnMob3V0X2RpciwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB0MCA9IHRpbWUudGltZSgpDQogICAgZGYgPSBwaXguYnVpbGRfaW5kZXgoDQogICAgICAgIHJvb3QsIHR1cGxlKHNwbGl0cyksIHdvcmtlcnMsIDEwMDAsIG9zLnBhdGguam9pbihvdXRfZGlyLCAnY2h1bmtzJyksIGxpbWl0LA0KICAgICAgICBmb3JjZT1mb3JjZSwgcHJvZ3Jlc3M9cHJvZ3Jlc3MsDQogICAgKQ0KICAgIGRpY29tX3NpZ25hdHVyZSA9IGRmLmF0dHJzLmdldCgic291cmNlX3NpZ25hdHVyZSIpDQogICAgZGYgPSBwaXguYXR0YWNoX2Nzdl9mbGFncyhkZiwgcm9vdCkNCiAgICByZWxldmFudF9jc3ZzID0gWw0KICAgICAgICBvcy5wYXRoLmpvaW4ocm9vdCwgbmFtZSkNCiAgICAgICAgZm9yIG5hbWUgaW4gKCJ0cmFpbl9zZXJpZXMuY3N2IiwgInRlc3Rfc2VyaWVzLmNzdiIpDQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihyb290LCBuYW1lKSkNCiAgICAgICAgYW5kIChuYW1lLnN0YXJ0c3dpdGgoInRyYWluIikgb3IgInRlc3QiIGluIHNwbGl0cykNCiAgICBdDQogICAgY3N2X3NpZ25hdHVyZXMgPSB7b3MucGF0aC5iYXNlbmFtZShwYXRoKTogX3NoYTI1Nl9maWxlKHBhdGgpIGZvciBwYXRoIGluIHJlbGV2YW50X2NzdnN9DQogICAgc291cmNlX3NpZ25hdHVyZSA9IGhhc2hsaWIuc2hhMjU2KA0KICAgICAgICBqc29uLmR1bXBzKA0KICAgICAgICAgICAgew0KICAgICAgICAgICAgICAgICJkaWNvbV9zaWduYXR1cmUiOiBkaWNvbV9zaWduYXR1cmUsDQogICAgICAgICAgICAgICAgImNzdl9zaWduYXR1cmVzIjogY3N2X3NpZ25hdHVyZXMsDQogICAgICAgICAgICAgICAgInByZXByb2Nlc3Nfc2NoZW1hIjogImluZGV4LXNsb3RzLXYyIiwNCiAgICAgICAgICAgICAgICAicHJlc2V0IjogY2ZnLm5hbWUsDQogICAgICAgICAgICAgICAgInNsb3RfcHJlZmVyXzJkIjogY2ZnLnNsb3RfcHJlZmVyXzJkLA0KICAgICAgICAgICAgICAgICJzbG90X2ZzX3ByaW9yaXR5IjogY2ZnLnNsb3RfZnNfcHJpb3JpdHksDQogICAgICAgICAgICAgICAgInNsb3RfY3N2X2ZhbGxiYWNrIjogY2ZnLnNsb3RfY3N2X2ZhbGxiYWNrLA0KICAgICAgICAgICAgfSwNCiAgICAgICAgICAgIHNvcnRfa2V5cz1UcnVlLA0KICAgICAgICApLmVuY29kZSgidXRmLTgiKQ0KICAgICkuaGV4ZGlnZXN0KCkNCiAgICBkZi5hdHRyc1sic291cmNlX3NpZ25hdHVyZSJdID0gc291cmNlX3NpZ25hdHVyZQ0KICAgIHJlcCA9IHBpeC5pbmRleF9yZXBvcnQoZGYpDQogICAgcmVwWydzY2FuX21pbnV0ZXMnXSA9IHJvdW5kKCh0aW1lLnRpbWUoKSAtIHQwKSAvIDYwLCAyKQ0KICAgIHJlcFsnc291cmNlX3NpZ25hdHVyZSddID0gc291cmNlX3NpZ25hdHVyZQ0KICAgIHJlcFsnY3N2X3NpZ25hdHVyZXMnXSA9IGNzdl9zaWduYXR1cmVzDQogICAgYW5uLCB0YWIgPSBwc2xvdHMuYXNzaWduX2FsbChkZiwgY2ZnLnNsb3RfcHJlZmVyXzJkLCBjZmcuc2xvdF9mc19wcmlvcml0eSwgY2ZnLnNsb3RfY3N2X2ZhbGxiYWNrKQ0KICAgIHNpZGVzID0gcGlwZWxpbmUuc3R1ZHlfc2lkZXMoZGYpDQogICAgcmVwWydzbG90X2NvdmVyYWdlX3BjdCddID0gKHRhYi5ub3RuYSgpLm1lYW4oKSAqIDEwMCkucm91bmQoMSkudG9fZGljdCgpDQogICAgcmVwWydzbG90c19maWxsZWRfcGVyX3N0dWR5J10gPSB7aW50KGspOiBpbnQodikgZm9yIGssIHYgaW4gdGFiLm5vdG5hKCkuc3VtKGF4aXM9MSkudmFsdWVfY291bnRzKCkuc29ydF9pbmRleCgpLml0ZW1zKCl9DQogICAgcmVwWydsYXRlcmFsaXR5J10gPSBxYy5sYXRlcmFsaXR5X3JlcG9ydChzaWRlcykNCiAgICByZXBbJ2hlYWRlcl92c19jc3ZfZmF0c2F0X2NvbmZsaWN0cyddID0gaW50KGFublsnZnNfY29uZmxpY3QnXS5zdW0oKSkgaWYgJ2ZzX2NvbmZsaWN0JyBpbiBhbm4gZWxzZSAwDQogICAgcGl4LnNhdmVfaW5kZXgoYW5uLCBvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ2luZGV4LnBrbCcpKQ0KICAgIHNsb3RzX3BhdGggPSBvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ3Nsb3RzLnBrbCcpDQogICAgcGQudG9fcGlja2xlKHRhYiwgZiJ7c2xvdHNfcGF0aH0udG1wIikNCiAgICBvcy5yZXBsYWNlKGYie3Nsb3RzX3BhdGh9LnRtcCIsIHNsb3RzX3BhdGgpDQogICAgX2F0b21pY19qc29uKG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2lkZXMuanNvbicpLCBzaWRlcykNCiAgICBhbm4uZHJvcChjb2x1bW5zPVsnb3JkZXJlZF9maWxlcycsICdwb3NpdGlvbnMnXSwgZXJyb3JzPSdpZ25vcmUnKS50b19jc3YoDQogICAgICAgIG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2VyaWVzX3RhYmxlLmNzdi5nei50bXAnKSwgaW5kZXg9RmFsc2UsIGNvbXByZXNzaW9uPSJnemlwIg0KICAgICkNCiAgICBvcy5yZXBsYWNlKA0KICAgICAgICBvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ3Nlcmllc190YWJsZS5jc3YuZ3oudG1wJyksDQogICAgICAgIG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2VyaWVzX3RhYmxlLmNzdi5neicpLA0KICAgICkNCiAgICBfYXRvbWljX2pzb24ob3MucGF0aC5qb2luKG91dF9kaXIsICdyZXBvcnQuanNvbicpLCByZXApDQogICAgX2F0b21pY19qc29uKA0KICAgICAgICBvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ2luZGV4X21hbmlmZXN0Lmpzb24nKSwNCiAgICAgICAgew0KICAgICAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogImluZGV4LXYyIiwNCiAgICAgICAgICAgICJzb3VyY2Vfc2lnbmF0dXJlIjogc291cmNlX3NpZ25hdHVyZSwNCiAgICAgICAgICAgICJkaWNvbV9zaWduYXR1cmUiOiBkaWNvbV9zaWduYXR1cmUsDQogICAgICAgICAgICAiY3N2X3NpZ25hdHVyZXMiOiBjc3Zfc2lnbmF0dXJlcywNCiAgICAgICAgICAgICJzZXJpZXMiOiBsZW4oYW5uKSwNCiAgICAgICAgICAgICJzdHVkaWVzIjogaW50KGFublsnU3R1ZHlJbnN0YW5jZVVJRCddLm51bmlxdWUoKSkgaWYgbm90IGFubi5lbXB0eSBlbHNlIDAsDQogICAgICAgIH0sDQogICAgKQ0KICAgIGxhdCA9IHJlcFsnbGF0ZXJhbGl0eSddDQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXAsIGluZGVudD0xLCBkZWZhdWx0PXN0cikpDQogICAgcHJpbnQoJ1xuR0FURVMgKGluZGV4KTonKQ0KICAgIHByaW50KGYiICBzZXJpZXMgd2l0aCByZWFkIGVycm9ycyAgICAgICAgICA6IHtyZXBbJ3Nlcmllc193aXRoX2Vycm9yJ119ICAoZXhwZWN0IH4wKSIpDQogICAgcHJpbnQoZiIgIG5vbi1jYW5vbmljYWwgb3JpZW50YXRpb24gc2VyaWVzIDoge3JlcC5nZXQoJ25vbl9jYW5vbmljYWxfb3JpZW50YXRpb24nKX0gIChleHBlY3QgMCkiKQ0KICAgIHByaW50KGYiICBDU1YgcGxhbmUgdnMgRElDT00gZ2VvbWV0cnkgICAgICA6IHtyZXAuZ2V0KCdjc3ZfcGxhbmVfdnNfZ2VvbWV0cnlfbWlzbWF0Y2gnKX0gbWlzbWF0Y2hlcyAoZXhwZWN0IDApIikNCiAgICBwcmludChmIiAgbGF0ZXJhbGl0eSB1bnJlc29sdmVkICAgICAgICAgICAgOiB7bGF0Wyd1bnJlc29sdmVkX2ZyYWMnXTouMiV9ICAoZ2F0ZSA8IDElKSIpDQogICAgcHJpbnQoZiIgIGxhdGVyYWxpdHkgdGFnLXZzLWdlb21ldHJ5IGNsYXNoIDoge2xhdFsndGFnX2dlb21ldHJ5X2Rpc2FncmVlJ119IHN0dWRpZXMgIChnYXRlIDwgMSUgb2YgdGFnZ2VkKSIpDQogICAgcmV0dXJuIGRpY3QoYW5uPWFubiwgdGFiPXRhYiwgc2lkZXM9c2lkZXMsIHJlcG9ydD1yZXAsIHNvdXJjZV9zaWduYXR1cmU9c291cmNlX3NpZ25hdHVyZSkNCg0KDQpkZWYgZ2V0X2luZGV4KHJvb3QsIG91dF9kaXIsIHNwbGl0cz0oJ3RyYWluJywgJ3Rlc3QnKSwgd29ya2Vycz1Ob25lLCBmb3JjZT1GYWxzZSwgKiprdyk6DQogICAgIiIiUmV2YWxpZGF0ZSBjdXJyZW50IERJQ09NL0NTViBmaW5nZXJwcmludHMsIHRoZW4gcmV1c2Ugb25seSBtYXRjaGluZyBpbmRleCBjaHVua3MuIiIiDQogICAgcmV0dXJuIHJ1bl9pbmRleChyb290LCBvdXRfZGlyLCBzcGxpdHMsIHdvcmtlcnMsIGZvcmNlPWZvcmNlLCAqKmt3KQ0KDQoNCmRlZiBsb2FkX2luZGV4X2RpcihvdXRfZGlyKToNCiAgICByZXR1cm4gKHBpeC5sb2FkX2luZGV4KG9zLnBhdGguam9pbihvdXRfZGlyLCAnaW5kZXgucGtsJykpLCBwZC5yZWFkX3BpY2tsZShvcy5wYXRoLmpvaW4ob3V0X2RpciwgJ3Nsb3RzLnBrbCcpKSwNCiAgICAgICAgICAgIGpzb24ubG9hZChvcGVuKG9zLnBhdGguam9pbihvdXRfZGlyLCAnc2lkZXMuanNvbicpKSkpDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAyLiBjYWNoZQ0KZGVmIHJ1bl9jYWNoZShpbmRleCwgc3BsaXQsIHByZWZpeCwgcHJlc2V0PSd2MicsIHdvcmtlcnM9Tm9uZSwgbGltaXRfc3R1ZGllcz0wLCBmcmVzaD1GYWxzZSwgY2ZnPU5vbmUsIHN0dWRpZXM9Tm9uZSwNCiAgICAgICAgICAgICAgKipjZmdfb3ZlcnJpZGVzKToNCiAgICAiIiJCdWlsZC9yZXN1bWUgdGhlIG1lbW1hcCBjYWNoZSBmb3IgYSBzcGxpdC4gYGluZGV4YCBpcyB0aGUgcnVuX2luZGV4KCkgZGljdCBvciBhbiBpbmRleCBkaXJlY3RvcnkuDQogICAgLT4gKGNhY2hlLCBzdGF0cykuIiIiDQogICAgaWYgaXNpbnN0YW5jZShpbmRleCwgZGljdCk6DQogICAgICAgIGFubiwgdGFiLCBzaWRlcyA9IGluZGV4Wydhbm4nXSwgaW5kZXhbJ3RhYiddLCBpbmRleFsnc2lkZXMnXQ0KICAgICAgICBzb3VyY2Vfc2lnbmF0dXJlID0gaW5kZXguZ2V0KCJzb3VyY2Vfc2lnbmF0dXJlIikNCiAgICBlbHNlOg0KICAgICAgICBhbm4sIHRhYiwgc2lkZXMgPSBsb2FkX2luZGV4X2RpcihpbmRleCkNCiAgICAgICAgbWFuaWZlc3RfcGF0aCA9IG9zLnBhdGguam9pbihpbmRleCwgImluZGV4X21hbmlmZXN0Lmpzb24iKQ0KICAgICAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMobWFuaWZlc3RfcGF0aCk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWlzc2luZyBpbmRleCBwcm92ZW5hbmNlIG1hbmlmZXN0OiB7bWFuaWZlc3RfcGF0aH07IHJlYnVpbGQgdGhlIGluZGV4IikNCiAgICAgICAgd2l0aCBvcGVuKG1hbmlmZXN0X3BhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIHN0cmVhbToNCiAgICAgICAgICAgIHNvdXJjZV9zaWduYXR1cmUgPSBqc29uLmxvYWQoc3RyZWFtKS5nZXQoInNvdXJjZV9zaWduYXR1cmUiKQ0KICAgICAgICBpZiBub3Qgc291cmNlX3NpZ25hdHVyZToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIGluZGV4IHByb3ZlbmFuY2UgbWFuaWZlc3Q6IHttYW5pZmVzdF9wYXRofSIpDQogICAgY2ZnID0gY2ZnIG9yIGNvbmZpZy5nZXRfY2ZnKHByZXNldCwgKipjZmdfb3ZlcnJpZGVzKQ0KICAgIGlmIHN0dWRpZXMgaXMgTm9uZToNCiAgICAgICAgc3R1ZGllcyA9IHNvcnRlZChhbm5bYW5uWydzcGxpdCddID09IHNwbGl0XVsnU3R1ZHlJbnN0YW5jZVVJRCddLnVuaXF1ZSgpKQ0KICAgIGlmIGxpbWl0X3N0dWRpZXM6DQogICAgICAgIHN0dWRpZXMgPSBzdHVkaWVzWzpsaW1pdF9zdHVkaWVzXQ0KICAgIG5lZWQgPSBlc3RpbWF0ZV9jYWNoZV9nYihsZW4oc3R1ZGllcyksIGNmZykNCiAgICBwcmludChmJ3tsZW4oc3R1ZGllcyl9IHN0dWRpZXMgLT4gY2FjaGUge25lZWQ6LjFmfSBHQiAgKHByZXNldD17Y2ZnLm5hbWV9LCBEPXtjZmcuc3RhY2tfZGVwdGh9LCB7Y2ZnLmltZ19zaXplfXB4KSBhdCB7cHJlZml4fScpDQogICAgc3ViID0gYW5uW2FublsnU3R1ZHlJbnN0YW5jZVVJRCddLmlzaW4oc2V0KHN0dWRpZXMpKV0gaWYgKCdTdHVkeUluc3RhbmNlVUlEJyBpbiBhbm4uY29sdW1ucykgZWxzZSBhbm4NCiAgICByZWNvcmRzID0gcGlwZWxpbmUuaW5kZXhfdG9fcmVjb3JkcyhzdWIpDQogICAgc2xvdF9yb3dzID0gew0KICAgICAgICBzOiB7DQogICAgICAgICAgICBuYW1lOiAoDQogICAgICAgICAgICAgICAgc3RyKHZhbHVlKS5zdHJpcCgpDQogICAgICAgICAgICAgICAgaWYgcGQubm90bmEodmFsdWUpIGFuZCBzdHIodmFsdWUpLnN0cmlwKCkNCiAgICAgICAgICAgICAgICBlbHNlIE5vbmUNCiAgICAgICAgICAgICkNCiAgICAgICAgICAgIGZvciBuYW1lLCB2YWx1ZSBpbiB0YWIubG9jW3NdLnRvX2RpY3QoKS5pdGVtcygpDQogICAgICAgIH0NCiAgICAgICAgZm9yIHMgaW4gc3R1ZGllcw0KICAgICAgICBpZiBzIGluIHRhYi5pbmRleA0KICAgIH0NCiAgICBjYWNoZSwgc3RhdHMgPSBwY2FjaGUuYnVpbGRfY2FjaGUoDQogICAgICAgIHByZWZpeCwNCiAgICAgICAgc3R1ZGllcywNCiAgICAgICAgc2xvdF9yb3dzLA0KICAgICAgICByZWNvcmRzLA0KICAgICAgICBzaWRlcywNCiAgICAgICAgY2ZnLA0KICAgICAgICB3b3JrZXJzLA0KICAgICAgICByZXN1bWU9bm90IGZyZXNoLA0KICAgICAgICBzb3VyY2Vfc2lnbmF0dXJlPXNvdXJjZV9zaWduYXR1cmUsDQogICAgKQ0KICAgIHByaW50KGpzb24uZHVtcHMoc3RhdHMsIGluZGVudD0xKSkNCiAgICByZXR1cm4gY2FjaGUsIHN0YXRzDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAzLiBRQw0KZGVmIHJ1bl9xYyhwcmVmaXgsIG91dF9kaXI9Tm9uZSwgbj0zMDAsIG1vbnRhZ2U9VHJ1ZSwgZXhwZWN0ZWRfc2xvdF9tYXNrPU5vbmUsIHJlcXVpcmVfYWNjZXB0YW5jZT1UcnVlKToNCiAgICAiIiJBY2NlcHRhbmNlIGdhdGVzIG9uIGEgYnVpbHQgY2FjaGUgKCsgb3B0aW9uYWwgbW9udGFnZSBQTkc6IG9uZSByb3cgcGVyIHNsb3QgdHlwZSkuIiIiDQogICAgYyA9IHBjYWNoZS5TdHVkeUNhY2hlKHByZWZpeCkNCiAgICByZXAgPSBxYy5jYWNoZV9zYW5pdHkoYywgbiwgZXhwZWN0ZWRfc2xvdF9tYXNrPWV4cGVjdGVkX3Nsb3RfbWFzaykNCiAgICByZXBbInNvdXJjZV9zaWduYXR1cmUiXSA9IGMubWV0YS5nZXQoInNvdXJjZV9zaWduYXR1cmUiKQ0KICAgIHByaW50KGpzb24uZHVtcHMocmVwLCBpbmRlbnQ9MSwgZGVmYXVsdD1zdHIpKQ0KICAgIHByaW50KGYiXG5HQVRFUyAoY2FjaGUpOiBhY2NlcHRlZD17cmVwLmdldCgnYWNjZXB0ZWQnKX0gIg0KICAgICAgICAgIGYiaW5jb21wbGV0ZT17cmVwLmdldCgnaW5jb21wbGV0ZV9zdHVkaWVzJyl9IGRlY29kZV9lcnJvcnM9e3JlcC5nZXQoJ2RlY29kZV9lcnJvcnMnKX0gIg0KICAgICAgICAgIGYiZmFpbGVkX3NlbGVjdGVkX3Nsb3RzPXtyZXAuZ2V0KCdmYWlsZWRfc2VsZWN0ZWRfc2xvdHMnKX0gIg0KICAgICAgICAgIGYibWlzc2luZ19leHBlY3RlZF9zbG90cz17cmVwLmdldCgnbWlzc2luZ19leHBlY3RlZF9zbG90cycpfSAiDQogICAgICAgICAgZiJtZWFuX3ZhbGlkX2RlcHRoPXtyZXAuZ2V0KCdtZWFuX3ZhbGlkX2RlcHRoJywgZmxvYXQoJ25hbicpKTouMmZ9IikNCiAgICBpZiBvdXRfZGlyOg0KICAgICAgICBvcy5tYWtlZGlycyhvdXRfZGlyLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICBfYXRvbWljX2pzb24ob3MucGF0aC5qb2luKG91dF9kaXIsICdjYWNoZV9zYW5pdHkuanNvbicpLCByZXApDQogICAgaWYgbW9udGFnZSBhbmQgb3V0X2RpcjoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaW1wb3J0IG1hdHBsb3RsaWINCiAgICAgICAgICAgIG1hdHBsb3RsaWIudXNlKCdBZ2cnKQ0KICAgICAgICAgICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdA0KICAgICAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApDQogICAgICAgICAgICBkb25lID0gbnAud2hlcmUobnAuYXNhcnJheShjLmRvbmUpID09IDEpWzBdDQogICAgICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGNvbmZpZy5OX1NMT1RTLCA2LCBmaWdzaXplPSgxNCwgMi40ICogY29uZmlnLk5fU0xPVFMpKQ0KICAgICAgICAgICAgZm9yIHMgaW4gcmFuZ2UoY29uZmlnLk5fU0xPVFMpOg0KICAgICAgICAgICAgICAgIGNhbmQgPSBbaSBmb3IgaSBpbiBkb25lIGlmIGMuc2xvdFtpLCBzXV0NCiAgICAgICAgICAgICAgICBwaWNrID0gcm5nLmNob2ljZShjYW5kLCBtaW4oNiwgbGVuKGNhbmQpKSwgcmVwbGFjZT1GYWxzZSkgaWYgY2FuZCBlbHNlIFtdDQogICAgICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2UoNik6DQogICAgICAgICAgICAgICAgICAgIGF4W3MsIGpdLmF4aXMoJ29mZicpDQogICAgICAgICAgICAgICAgZm9yIGosIGkgaW4gZW51bWVyYXRlKHBpY2spOg0KICAgICAgICAgICAgICAgICAgICBheFtzLCBqXS5pbXNob3coYy5pbWFnZXNbaSwgcywgYy5pbWFnZXMuc2hhcGVbMl0gLy8gMl0sIGNtYXA9J2dyYXknLCB2bWluPTAsIHZtYXg9MjU1KQ0KICAgICAgICAgICAgICAgICAgICBheFtzLCBqXS5zZXRfdGl0bGUoY29uZmlnLlNMT1RTW3NdWzBdLCBmb250c2l6ZT03KQ0KICAgICAgICAgICAgcGx0LnRpZ2h0X2xheW91dCgpDQogICAgICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKG91dF9kaXIsICdtb250YWdlLnBuZycpDQogICAgICAgICAgICBwbHQuc2F2ZWZpZyhwYXRoLCBkcGk9OTApDQogICAgICAgICAgICBwbHQuY2xvc2UoJ2FsbCcpDQogICAgICAgICAgICBwcmludCgnbW9udGFnZSAtPicsIHBhdGgpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGEgcGxvdHRpbmcgcHJvYmxlbSBtdXN0IG5ldmVyIGZhaWwgdGhlIHBpcGVsaW5lDQogICAgICAgICAgICBwcmludCgnbW9udGFnZSBza2lwcGVkOicsIHR5cGUoZSkuX19uYW1lX18sIGUpDQogICAgaWYgcmVxdWlyZV9hY2NlcHRhbmNlIGFuZCBub3QgcmVwLmdldCgiYWNjZXB0ZWQiLCBGYWxzZSk6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigNCiAgICAgICAgICAgIGYiUHJlcHJvY2Vzc2luZyBjYWNoZSBmYWlsZWQgYWNjZXB0YW5jZSBnYXRlczsgc2VlICINCiAgICAgICAgICAgIGYie29zLnBhdGguam9pbihvdXRfZGlyLCAnY2FjaGVfc2FuaXR5Lmpzb24nKSBpZiBvdXRfZGlyIGVsc2UgcHJlZml4ICsgJy5tZXRhLmNzdid9Ig0KICAgICAgICApDQogICAgcmV0dXJuIHJlcA0K',
    'src/data/preprocess/sampling.py': 'IiIiV2luZG93IHNlbGVjdGlvbiArIGF1Z21lbnRhdGlvbiAodGhlIHRyYWluaW5nIGhvdCBsb29wOyBudW1weS9PcGVuQ1Ygb25seSkuDQoNClRocm91Z2hwdXQgcnVsZXM6DQogICogY2hvb3NlIHRoZSB3aW5kb3dzIEZJUlNULCB0aGVuIGF1Z21lbnQgb25seSB0aGUgc2xpY2VzIHRob3NlIHdpbmRvd3MgdXNlICh0cmFpbmluZyB1c2VzIH40IG9mIDExIHdpbmRvd3MsDQogICAgaS5lLiB+MTIgb2YgdGhlIDI0IHN0b3JlZCBzbGljZXMgLT4gfjJ4IGxlc3Mgd2FycGluZykNCiAgKiBvbmUgYWZmaW5lIG1hdHJpeCBhbmQgb25lIDI1Ni1lbnRyeSBMVVQgcGVyIHNsb3QsIGFwcGxpZWQgd2l0aCBjdjIgKFNJTUQpIHRvIGFsbCBvZiBpdHMgc2xpY2VzDQogICogbm8gZmxpcHM6IGtuZWVzIGFyZSBjYW5vbmljYWxpc2VkIHRvICdsZWZ0JyBhdCBwcmVwcm9jZXNzaW5nIHRpbWUsIHNvIGEgaG9yaXpvbnRhbCBmbGlwIHdvdWxkIGRlc3Ryb3kgdGhlDQogICAgbWVkaWFsL2xhdGVyYWwgaW1hZ2Utc2lkZSBjb25zaXN0ZW5jeSB0aGF0IHRoZSBzbG90IGhlYWQgcmVsaWVzIG9uDQoiIiINCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IGN2Mg0KDQpjdjIuc2V0TnVtVGhyZWFkcygwKQ0KDQoNCmRlZiBuX3dpbmRvd3MoZGVwdGgsIGdyb3VwLCBzdHJpZGUpOg0KICAgIHJldHVybiBtYXgoMSwgKGRlcHRoIC0gZ3JvdXApIC8vIHN0cmlkZSArIDEpDQoNCg0KZGVmIHdpbmRvd19zdGFydHMoZGVwdGgsIGdyb3VwLCBzdHJpZGUsIG5fdXNlPU5vbmUsIHRyYWluPUZhbHNlLCBybmc9Tm9uZSk6DQogICAgIiIiU3RhcnQgaW5kaWNlcyAoaW50byB0aGUgc3RvcmVkIHN0YWNrKSBvZiB0aGUgd2luZG93cyB0byB1c2UuDQoNCiAgICBldmFsIDogZXZlbmx5IHNwYWNlZCBzdWJzZXQgKG9yIGFsbCkgICAgICAgICAgIHRyYWluOiBvbmUgcmFuZG9tIHdpbmRvdyBwZXIgZXF1YWwtd2lkdGggYmluIChzdHJhdGlmaWVkLCBzbw0KICAgIHRoZSB3aG9sZSBkZXB0aCBpcyBjb3ZlcmVkIGV2ZXJ5IHN0ZXApIHdpdGggYSByYW5kb20gcGhhc2UuIiIiDQogICAgVyA9IG5fd2luZG93cyhkZXB0aCwgZ3JvdXAsIHN0cmlkZSkNCiAgICBzdGFydHMgPSBucC5hcmFuZ2UoVykgKiBzdHJpZGUNCiAgICBpZiBuX3VzZSBpcyBOb25lIG9yIG5fdXNlID49IFc6DQogICAgICAgIHJldHVybiBzdGFydHMNCiAgICBpZiB0cmFpbjoNCiAgICAgICAgcm5nID0gcm5nIG9yIG5wLnJhbmRvbS5kZWZhdWx0X3JuZygpDQogICAgICAgICMgVVBHUkFERSAzOiBaLWF4aXMgaml0dGVyICh0aHJvdWdoLXBsYW5lIHRyYW5zbGF0aW9uIGF1Z21lbnRhdGlvbikuDQogICAgICAgICMgU2hpZnRzIHRoZSBlbnRpcmUgd2luZG93IGdyaWQgYnkgYSByYW5kb20gb2Zmc2V0IGJlZm9yZSBiaW5uaW5nLg0KICAgICAgICAjIEZvcmNlcyB0aGUgbW9kZWwgdG8gbGVhcm4gYW5hdG9teSByYXRoZXIgdGhhbiBtZW1vcml6aW5nIHNsaWNlIHBvc2l0aW9ucy4NCiAgICAgICAgel9qaXR0ZXIgPSBpbnQocm5nLmludGVnZXJzKDAsIG1heCgxLCBzdHJpZGUgLy8gMiArIDEpKSkNCiAgICAgICAgaml0dGVyZWQgPSBucC5jbGlwKHN0YXJ0cyArIHpfaml0dGVyLCAwLCBkZXB0aCAtIGdyb3VwKQ0KICAgICAgICBlZGdlcyA9IG5wLmxpbnNwYWNlKDAsIFcsIG5fdXNlICsgMSkNCiAgICAgICAgcGljayA9IFtpbnQocm5nLmludGVnZXJzKGludChucC5mbG9vcihlZGdlc1tpXSkpLCBtYXgoaW50KG5wLmZsb29yKGVkZ2VzW2ldKSkgKyAxLCBpbnQobnAuY2VpbChlZGdlc1tpICsgMV0pKSkpKSBmb3IgaSBpbiByYW5nZShuX3VzZSldDQogICAgICAgIHJldHVybiBqaXR0ZXJlZFtucC5jbGlwKHBpY2ssIDAsIFcgLSAxKV0NCiAgICByZXR1cm4gc3RhcnRzW25wLnVuaXF1ZShucC5yaW50KG5wLmxpbnNwYWNlKDAsIFcgLSAxLCBuX3VzZSkpLmFzdHlwZShpbnQpKV0NCg0KDQpkZWYgZ2F0aGVyX3dpbmRvd3Moc3RhY2ssIHZhbGlkLCBzdGFydHMsIGdyb3VwKToNCiAgICAiIiJzdGFjayBbRCxILFddIChtYXkgYmUgYSBtZW1tYXAgdmlldykgLT4gd2luZG93cyBbbixncm91cCxILFddIHVpbnQ4IGFuZCB3aW5kb3cgdmFsaWRpdHkgW25dLg0KDQogICAgQSB3aW5kb3cgaXMgdmFsaWQgb25seSBpZiBhbGwgb2YgaXRzIGNoYW5uZWxzIGFyZSB2YWxpZCBzbGljZXMuDQoNCiAgICBDT05USUdVT1VTIE1FTU1BUCBSRUFEIE9QVElNSVpBVElPTjoNCiAgICBJbnN0ZWFkIG9mIG5vbi1jb250aWd1b3VzIGluZGV4aW5nIHN0YWNrW2lkeF0gdGhhdCB0cmlnZ2VycyBzY2F0dGVyZWQgZGlzayBwYWdlIGZhdWx0cw0KICAgIGFjcm9zcyB0aGUgMjI3IEdCIG1lbW1hcCBmaWxlLCByZWFkIHRoZSBib3VuZGluZyBzbGljZSBzdGFja1tsbzpoaV0gYXMgYSBzaW5nbGUgY29udGlndW91cw0KICAgIGJsb2NrIGludG8gUkFNLCB0aGVuIGV4dHJhY3Qgd2luZG93cyBsb2NhbGx5IGF0IG1lbW9yeSBidXMgc3BlZWRzICg+MTAwIEdCL3MpLg0KICAgICIiIg0KICAgIGlmIGxlbihzdGFydHMpID09IDA6DQogICAgICAgIHJldHVybiBucC56ZXJvcygoMCwgZ3JvdXAsIHN0YWNrLnNoYXBlWzFdLCBzdGFjay5zaGFwZVsyXSksIGR0eXBlPXN0YWNrLmR0eXBlKSwgbnAuemVyb3MoMCwgZHR5cGU9Ym9vbCkNCiAgICBpZHggPSBucC5hc2FycmF5KHN0YXJ0cylbOiwgTm9uZV0gKyBucC5hcmFuZ2UoZ3JvdXApW05vbmUsIDpdDQogICAgbG8gPSBpbnQoaWR4Lm1pbigpKQ0KICAgIGhpID0gaW50KGlkeC5tYXgoKSkgKyAxDQogICAgYmxvY2sgPSBucC5hc2NvbnRpZ3VvdXNhcnJheShzdGFja1tsbzpoaV0pDQogICAgd2luID0gYmxvY2tbaWR4IC0gbG9dDQogICAgd3YgPSBucC5hc2FycmF5KHZhbGlkKVtpZHhdLmFsbChheGlzPTEpDQogICAgcmV0dXJuIHdpbiwgd3YNCg0KDQpkZWYgX2x1dChybmcsIGEpOg0KICAgIGMgPSAxLjAgKyBybmcudW5pZm9ybSgtYSwgYSkNCiAgICBiID0gcm5nLnVuaWZvcm0oLWEgLyAyLjAsIGEgLyAyLjApDQogICAgZyA9IDEuMCArIHJuZy51bmlmb3JtKC1hLCBhKQ0KICAgIHggPSBucC5hcmFuZ2UoMjU2LCBkdHlwZT1ucC5mbG9hdDMyKSAvIDI1NS4wDQogICAgeSA9IG5wLmNsaXAoKG5wLnBvd2VyKHgsIGcpIC0gMC41KSAqIGMgKyAwLjUgKyBiLCAwLjAsIDEuMCkNCiAgICByZXR1cm4gbnAucmludCh5ICogMjU1LjApLmFzdHlwZShucC51aW50OCkNCg0KDQpkZWYgYXVnbWVudF9zbG90KHdpbiwgcm5nLCByb3RfZGVnPTguMCwgc2NhbGU9MC4wOCwgc2hpZnQ9MC4wNSwgaW50ZW5zaXR5PTAuMSk6DQogICAgIiIid2luIFtuLGcsSCxXXSB1aW50OCAtPiBzYW1lIHNoYXBlLiBPbmUgcmFuZG9tIHJvdGF0aW9uL3NjYWxlL3NoaWZ0ICsgb25lIGludGVuc2l0eSBMVVQgZm9yIHRoZSB3aG9sZSBzbG90LiIiIg0KICAgIG4sIGcsIEgsIFcgPSB3aW4uc2hhcGUNCiAgICBhbmcgPSBybmcudW5pZm9ybSgtcm90X2RlZywgcm90X2RlZykNCiAgICBzYyA9IDEuMCArIHJuZy51bmlmb3JtKC1zY2FsZSwgc2NhbGUpDQogICAgTSA9IGN2Mi5nZXRSb3RhdGlvbk1hdHJpeDJEKCgoVyAtIDEpIC8gMi4wLCAoSCAtIDEpIC8gMi4wKSwgYW5nLCBzYykNCiAgICBNWzAsIDJdICs9IHJuZy51bmlmb3JtKC1zaGlmdCwgc2hpZnQpICogVw0KICAgIE1bMSwgMl0gKz0gcm5nLnVuaWZvcm0oLXNoaWZ0LCBzaGlmdCkgKiBIDQogICAgbHV0ID0gX2x1dChybmcsIGludGVuc2l0eSkgaWYgaW50ZW5zaXR5ID4gMCBlbHNlIE5vbmUNCiAgICBvdXQgPSBucC5lbXB0eV9saWtlKHdpbikNCiAgICBmb3IgaSBpbiByYW5nZShuKToNCiAgICAgICAgZm9yIGogaW4gcmFuZ2UoZyk6DQogICAgICAgICAgICB4ID0gY3YyLndhcnBBZmZpbmUod2luW2ksIGpdLCBNLCAoVywgSCksIGZsYWdzPWN2Mi5JTlRFUl9MSU5FQVIsIGJvcmRlck1vZGU9Y3YyLkJPUkRFUl9SRUZMRUNUXzEwMSkNCiAgICAgICAgICAgIG91dFtpLCBqXSA9IGN2Mi5MVVQoeCwgbHV0KSBpZiBsdXQgaXMgbm90IE5vbmUgZWxzZSB4DQogICAgcmV0dXJuIG91dA0K',
    'src/data/preprocess/slots.py': 'IiIiU2VyaWVzIC0+IGFjcXVpc2l0aW9uLXNsb3QgYXNzaWdubWVudC4NCg0KQW5ub3RhdGlvbiBjb25zdGFudHMgYXJlIHRoZSBwdWJsaWMgMC45NDMgbm90ZWJvb2sncyAocmVncmVzc2lvbi10ZXN0ZWQgYWdhaW5zdCBpdCBvbiBhbGwgNCw0MDcgdHJhaW4gc3R1ZGllczoNClNBRy9DT1IvQVggZmx1aWQtRlMgc2xvdHMgZmlsbGVkIGZvciA5NC4zIC8gOTUuNSAvIDk5LjQgJSBvZiBzdHVkaWVzKS4NCg0KV2hhdCB0aGUgYXVkaXQgYWRkZWQ6DQogICogdGhlIENTViBgRmF0X1N1cHByZXNzaW9uYCBmbGFnIGFncmVlcyB3aXRoIHRoZSBoZWFkZXItZGVyaXZlZCBmYXQtc2F0IGV2aWRlbmNlIGZvciA5OS4xNCUgb2Ygc2VyaWVzLCBhbmQNCiAgICBgRmx1aWRfU2Vuc2l0aXZlYCBpcyBJREVOVElDQUwgdG8gaXQgaW4gdHJhaW4gKGl0IHRyYWNrcyBmYXQtc3VwcHJlc3Npb24sIG5vdCBQRC9UMiBjb250cmFzdCksIHNvIHRoZSBDU1YgZmx1aWQNCiAgICBmbGFnIGlzIG9ubHkgdXNlZCBhcyBhIGxhc3QtcmVzb3J0IHByb3h5Ow0KICAqIHRoZSBtZXRhZGF0YS1wb29yIGNvaG9ydCAoMjM4IHN0dWRpZXMsIG5vIFRSL1RFL1NjYW5PcHRpb25zKSBzdGlsbCBoYXMgZGVzY3JpcHRpb25zLCBzbyBoZWFkZXIgcnVsZXMgd29yazsNCiAgKiBoaWRkZW4tc2V0IHJvYnVzdG5lc3M6IGV2ZXJ5IG1pc3NpbmcgaW5wdXQgZGVncmFkZXMgdG8gYSB3ZWFrZXIgc2lnbmFsIGluc3RlYWQgb2YgYW4gZXhjZXB0aW9uOw0KICAqIDMuMCUgb2YgJ21vc3Qgc2xpY2VzJyBwaWNrcyBhcmUgMy1EIHZvbHVtZXMgLT4gYHByZWZlcl8yZGAgcGlja3MgYSAyLUQgc2VyaWVzIHdoZW5ldmVyIG9uZSBleGlzdHMuDQoiIiINCmltcG9ydCByZQ0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQoNCmltcG9ydCBzcmMuY29yZS5jb25maWcgYXMgY29uZmlnDQoNCkZBVFNBVF9PUFRTID0geydGUycsICdGQVRTQVQnLCAnRkFUX1NBVCcsICdGU0FUJ30NCl9GQVRTQVRfUlggPSByZS5jb21waWxlKCdcXGJmc1xcYnxmYXRzYXR8ZmF0IHNhdHxcXGJzdGlyXFxifFxcYnNwYWlyXFxifFxcYnNwaXJcXGJ8XFxid2VcXGJ8d2F0ZXIgZXhjaXR8XFxidGlybVxcYnwnDQogICAgICAgICAgICAgICAgICAgICAgICAnXFxic3RpbmdcXGJ8XFxiZmF0c3VwXFxifHNtYXJ0IGZhdHxcXGJ3YXRlclxcYicpDQpfVDFfUlggPSByZS5jb21waWxlKCdcXGJ0MVxcYnxcXGJ0MXdcXGInKQ0KX1QyX1JYID0gcmUuY29tcGlsZSgnXFxidDJcXGJ8XFxidDJ3XFxiJykNCl9QRF9SWCA9IHJlLmNvbXBpbGUoJ1xcYnBkXFxifFxcYnBkd1xcYnxwcm90b258XFxiZHBcXGJ8ZGVucycpDQpfU0VQID0gcmUuY29tcGlsZSgnW19cXC0uXScpDQoNCk5FRURFRCA9IFsnU2VyaWVzRGVzY3JpcHRpb24nLCAnU2VxdWVuY2VOYW1lJywgJ1NjYW5PcHRpb25zJywgJ1NjYW5uaW5nU2VxdWVuY2UnLCAnUmVwZXRpdGlvblRpbWUnLCAnRWNob1RpbWUnXQ0KDQoNCmRlZiB0b19mbGFnKHgpOg0KICAgICIiIlJvYnVzdCAwLzEvTmFOIGZyb20gQ1NWIHZhbHVlcyAoaGlkZGVuLXNldCBDU1YgZm9ybWF0dGluZyBpcyBub3QgZ3VhcmFudGVlZCkuIiIiDQogICAgaWYgeCBpcyBOb25lOg0KICAgICAgICByZXR1cm4gbnAubmFuDQogICAgaWYgaXNpbnN0YW5jZSh4LCAoYm9vbCwgbnAuYm9vbF8pKToNCiAgICAgICAgcmV0dXJuIGZsb2F0KGJvb2woeCkpDQogICAgaWYgaXNpbnN0YW5jZSh4LCAoaW50LCBmbG9hdCwgbnAuaW50ZWdlciwgbnAuZmxvYXRpbmcpKToNCiAgICAgICAgcmV0dXJuIGZsb2F0KHgpIGlmIHggaW4gKDAsIDEpIGVsc2UgbnAubmFuDQogICAgcyA9IHN0cih4KS5zdHJpcCgpLmxvd2VyKCkNCiAgICBpZiBzIGluICgnMScsICd0cnVlJywgJ3llcycsICd5JywgJ3QnLCAnMS4wJyk6DQogICAgICAgIHJldHVybiAxLjANCiAgICBpZiBzIGluICgnMCcsICdmYWxzZScsICdubycsICduJywgJ2YnLCAnMC4wJyk6DQogICAgICAgIHJldHVybiAwLjANCiAgICByZXR1cm4gbnAubmFuDQoNCg0KZGVmIGFubm90YXRlKGRmLCBmc19wcmlvcml0eT0naGRyJywgY3N2X2ZhbGxiYWNrPVRydWUpOg0KICAgICIiIkFkZCBmYXRzYXQgLyB3ZWlnaHQgLyBmbHVpZCAvIGluZm9ybWF0aXZlIC8gaXMzZCBjb2x1bW5zLiBSZXR1cm5zIGEgY29weS4NCg0KICAgIGZhdHNhdDogaGVhZGVyIGV2aWRlbmNlIChkZXNjcmlwdGlvbiByZWdleCBvciBTY2FuT3B0aW9ucyB0b2tlbikgYXMgaW4gdGhlIHB1YmxpYyBwaXBlbGluZS4gV2hlbiB0aGUgaGVhZGVyDQogICAgaGFzIG5vIHVzYWJsZSB0ZXh0IGF0IGFsbCAoZHVtbXkvbWlzc2luZyBkZXNjcmlwdGlvbiBBTkQgbm8gU2Nhbk9wdGlvbnMpIHRoZSBDU1YgZmxhZyBpcyB1c2VkIGluc3RlYWQ7IHdpdGgNCiAgICBmc19wcmlvcml0eT0nY3N2JyB0aGUgQ1NWIGZsYWcgd2lucyB3aGVuZXZlciBwcmVzZW50Lg0KICAgICIiIg0KICAgIGRmID0gZGYuY29weSgpDQogICAgaWYgJ3BsYW5lJyBub3QgaW4gZGYuY29sdW1uczoNCiAgICAgICAgZGZbJ3BsYW5lJ10gPSBwbGFuZV9vZihkZikNCiAgICBmb3IgYyBpbiBORUVERUQ6DQogICAgICAgIGlmIGMgbm90IGluIGRmLmNvbHVtbnM6DQogICAgICAgICAgICBkZltjXSA9IE5vbmUNCiAgICByYXcgPSBkZlsnU2VyaWVzRGVzY3JpcHRpb24nXS5maWxsbmEoJycpLmFzdHlwZShzdHIpDQogICAgZHVtbXkgPSByYXcuc3RyLmNvbnRhaW5zKCdkdW1teXNlcmllc2Rlc2MnLCBjYXNlPUZhbHNlKQ0KICAgIGRlc2MgPSAocmF3LndoZXJlKH5kdW1teSwgJycpICsgJyAnICsgZGZbJ1NlcXVlbmNlTmFtZSddLmZpbGxuYSgnJykuYXN0eXBlKHN0cikpLnN0ci5sb3dlcigpDQogICAgZGVzYyA9IGRlc2Muc3RyLnJlcGxhY2UoX1NFUCwgJyAnLCByZWdleD1UcnVlKQ0KICAgIG9wdHMgPSBkZlsnU2Nhbk9wdGlvbnMnXS5maWxsbmEoJycpLmFzdHlwZShzdHIpLnN0ci51cHBlcigpLnN0ci5zcGxpdCgnfCcpDQogICAgb3B0c19mcyA9IG9wdHMuYXBwbHkobGFtYmRhIHRzOiBhbnkodC5zdHJpcCgpIGluIEZBVFNBVF9PUFRTIGZvciB0IGluIHRzKSkNCiAgICBmc19oZHIgPSBkZXNjLnN0ci5jb250YWlucyhfRkFUU0FUX1JYKSB8IG9wdHNfZnMNCiAgICBpbmZvcm1hdGl2ZSA9IChkZXNjLnN0ci5zdHJpcCgpICE9ICcnKSB8IChkZlsnU2Nhbk9wdGlvbnMnXS5maWxsbmEoJycpLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpICE9ICcnKQ0KDQogICAgY3N2X2ZzID0gcGQuU2VyaWVzKG5wLm5hbiwgaW5kZXg9ZGYuaW5kZXgpDQogICAgaWYgJ0ZhdF9TdXBwcmVzc2lvbicgaW4gZGYuY29sdW1uczoNCiAgICAgICAgY3N2X2ZzID0gZGZbJ0ZhdF9TdXBwcmVzc2lvbiddLm1hcCh0b19mbGFnKQ0KICAgIGVsaWYgJ0ZsdWlkX1NlbnNpdGl2ZScgaW4gZGYuY29sdW1uczoNCiAgICAgICAgIyBGSVg6IFRlc3Qgc2V0IG1pZ2h0IGRyb3AgRmF0X1N1cHByZXNzaW9uOyB1c2UgRmx1aWRfU2Vuc2l0aXZlIGFzIGlkZW50aWNhbCBwcm94eSAocGVyIEVEQSkNCiAgICAgICAgY3N2X2ZzID0gZGZbJ0ZsdWlkX1NlbnNpdGl2ZSddLm1hcCh0b19mbGFnKQ0KICAgIGlmIGZzX3ByaW9yaXR5ID09ICdjc3YnOg0KICAgICAgICBmcyA9IG5wLndoZXJlKGNzdl9mcy5ub3RuYSgpLCBjc3ZfZnMgPT0gMS4wLCBmc19oZHIpDQogICAgICAgIHNyYyA9IG5wLndoZXJlKGNzdl9mcy5ub3RuYSgpLCAnY3N2JywgJ2hkcicpDQogICAgZWxzZToNCiAgICAgICAgdXNlX2NzdiA9ICgofmluZm9ybWF0aXZlKSAmIGNzdl9mcy5ub3RuYSgpKSBpZiBjc3ZfZmFsbGJhY2sgZWxzZSBwZC5TZXJpZXMoRmFsc2UsIGluZGV4PWRmLmluZGV4KQ0KICAgICAgICBmcyA9IG5wLndoZXJlKHVzZV9jc3YsIGNzdl9mcyA9PSAxLjAsIGZzX2hkcikNCiAgICAgICAgc3JjID0gbnAud2hlcmUodXNlX2NzdiwgJ2NzdicsICdoZHInKQ0KICAgIGRmWydmYXRzYXQnXSA9IGZzLmFzdHlwZShib29sKQ0KICAgIGRmWydmc19zb3VyY2UnXSA9IHNyYw0KICAgIGRmWydmc19jb25mbGljdCddID0gY3N2X2ZzLm5vdG5hKCkgJiAoKGNzdl9mcy5maWxsbmEoMCkuYXN0eXBlKGZsb2F0KSA9PSAxLjApICE9IGRmWydmYXRzYXQnXSkNCg0KICAgIHRyID0gcGQudG9fbnVtZXJpYyhkZlsnUmVwZXRpdGlvblRpbWUnXSwgZXJyb3JzPSdjb2VyY2UnKQ0KICAgIHRlID0gcGQudG9fbnVtZXJpYyhkZlsnRWNob1RpbWUnXSwgZXJyb3JzPSdjb2VyY2UnKQ0KICAgIGdyZSA9IGRmWydTY2FubmluZ1NlcXVlbmNlJ10uZmlsbG5hKCcnKS5hc3R5cGUoc3RyKS5zdHIudXBwZXIoKS5zdHIuY29udGFpbnMoJ0dSJykNCiAgICB0MSwgdDIsIHBkdyA9IGRlc2Muc3RyLmNvbnRhaW5zKF9UMV9SWCksIGRlc2Muc3RyLmNvbnRhaW5zKF9UMl9SWCksIGRlc2Muc3RyLmNvbnRhaW5zKF9QRF9SWCkNCiAgICB3ID0gbnAud2hlcmUodDEgJiB+dDIgJiB+cGR3LCAnVDEnLCBucC53aGVyZSh0MiAmIH5wZHcsICdUMicsIG5wLndoZXJlKHBkdywgJ1BEJywgbnAud2hlcmUoDQogICAgICAgIGdyZSwgJ0dSRScsIG5wLndoZXJlKHRyIDwgODAwLCAnVDEnLCBucC53aGVyZSh0ZSA+IDYwLCAnVDInLCBucC53aGVyZSh0ciA+PSA4MDAsICdQRCcsICdVTksnKSkpKSkpKQ0KICAgIGRmWyd3ZWlnaHQnXSA9IHcNCiAgICBmbHVpZCA9IG5wLmlzaW4odywgWydQRCcsICdUMiddKQ0KICAgICMgbGFzdCByZXNvcnQ6IG5vIGhlYWRlciBjb250cmFzdCBhdCBhbGwgLT4gdGhlIENTViBmbHVpZCBmbGFnICg9PSBmYXQtc2F0IGZsYWcgaW4gdHJhaW4pIGFzIGEgcHJveHkNCiAgICBpZiBjc3ZfZmFsbGJhY2sgYW5kICdGbHVpZF9TZW5zaXRpdmUnIGluIGRmLmNvbHVtbnM6DQogICAgICAgIGNmID0gZGZbJ0ZsdWlkX1NlbnNpdGl2ZSddLm1hcCh0b19mbGFnKQ0KICAgICAgICB1bmsgPSAodyA9PSAnVU5LJykgJiBjZi5ub3RuYSgpDQogICAgICAgIGZsdWlkID0gbnAud2hlcmUodW5rLCBjZiA9PSAxLjAsIGZsdWlkKQ0KICAgIGRmWydmbHVpZCddID0gZmx1aWQuYXN0eXBlKGJvb2wpDQoNCiAgICBhY3EgPSBkZlsnTVJBY3F1aXNpdGlvblR5cGUnXS5maWxsbmEoJycpLmFzdHlwZShzdHIpLnN0ci51cHBlcigpIGlmICdNUkFjcXVpc2l0aW9uVHlwZScgaW4gZGYuY29sdW1ucyBlbHNlICcnDQogICAgaWYgJ25fc2xpY2VzJyBpbiBkZi5jb2x1bW5zOg0KICAgICAgICBucyA9IHBkLnRvX251bWVyaWMoZGZbJ25fc2xpY2VzJ10sIGVycm9ycz0nY29lcmNlJykuZmlsbG5hKDApDQogICAgZWxpZiAnbl9maWxlcycgaW4gZGYuY29sdW1uczoNCiAgICAgICAgbnMgPSBwZC50b19udW1lcmljKGRmWyduX2ZpbGVzJ10sIGVycm9ycz0nY29lcmNlJykuZmlsbG5hKDApDQogICAgZWxzZToNCiAgICAgICAgbnMgPSBwZC5TZXJpZXMoMCwgaW5kZXg9ZGYuaW5kZXgpDQogICAgdGggPSBwZC50b19udW1lcmljKGRmLmdldCgnU2xpY2VUaGlja25lc3MnLCBucC5uYW4pLCBlcnJvcnM9J2NvZXJjZScpDQogICAgZGZbJ2lzM2QnXSA9IChhY3EgPT0gJzNEJykgfCAoKG5zID4gMTIwKSAmICh0aCA8IDEuNSkpDQogICAgcmV0dXJuIGRmDQoNCg0KZGVmIHBsYW5lX29mKGRmKToNCiAgICAiIiJDU1YgcGxhbmUgKHZlcmlmaWVkIDEwMCUgPSBESUNPTSBnZW9tZXRyeSkgd2l0aCBnZW9tZXRyeSBhcyB0aGUgZmFsbGJhY2suIiIiDQogICAgcCA9IGRmWydBbmF0b21pY2FsX1BsYW5lJ10gaWYgJ0FuYXRvbWljYWxfUGxhbmUnIGluIGRmLmNvbHVtbnMgZWxzZSBwZC5TZXJpZXMoTm9uZSwgaW5kZXg9ZGYuaW5kZXgpDQogICAgZyA9IGRmWydwbGFuZV9nZW8nXSBpZiAncGxhbmVfZ2VvJyBpbiBkZi5jb2x1bW5zIGVsc2UgcGQuU2VyaWVzKE5vbmUsIGluZGV4PWRmLmluZGV4KQ0KICAgIG9rID0gcC5pc2luKFsnU2FnaXR0YWwnLCAnQ29yb25hbCcsICdBeGlhbCddKQ0KICAgIHJldHVybiBwLndoZXJlKG9rLCBnKQ0KDQoNCmRlZiBhc3NpZ25fc2xvdHMoZywgcHJlZmVyXzJkPVRydWUsIHNsb3RzPU5vbmUpOg0KICAgICIiImc6IGFubm90YXRlZCByb3dzIG9mIE9ORSBzdHVkeSAobmVlZHMgcGxhbmUsIGZhdHNhdCwgZmx1aWQsIGlzM2QsIG5fc2xpY2VzLCBTZXJpZXNJbnN0YW5jZVVJRCkuDQoNCiAgICBSZXR1cm5zIChjaG9zZW4sIGFsdHMsIGV4dHJhcyk6DQogICAgICBjaG9zZW46IHtzbG90X25hbWU6IFNlcmllc0luc3RhbmNlVUlEfQ0KICAgICAgYWx0czogICB7c2xvdF9uYW1lOiBbb3RoZXIgY2FuZGlkYXRlcyBvZiB0aGUgc2FtZSBzbG90IHR5cGUsIGJlc3QgZmlyc3RdfSAgICh2aWV3LXN3YXAgYXVnbWVudGF0aW9uIHBvb2wpDQogICAgICBleHRyYXM6IHNlcmllcyBub3QgdXNlZCBieSBhbnkgc2xvdA0KICAgIFJ1bGUgKHB1YmxpYyk6IG1vc3Qgc2xpY2VzIHdpbnMuIFdpdGggcHJlZmVyXzJkIGEgMi1EIHNlcmllcyBhbHdheXMgYmVhdHMgYSAzLUQgdm9sdW1lLg0KICAgICIiIg0KICAgIGlmICdwbGFuZScgbm90IGluIGcuY29sdW1ucyBvciAnZmF0c2F0JyBub3QgaW4gZy5jb2x1bW5zIG9yICdmbHVpZCcgbm90IGluIGcuY29sdW1ucyBvciAnaXMzZCcgbm90IGluIGcuY29sdW1uczoNCiAgICAgICAgZyA9IGFubm90YXRlKGcpDQogICAgc2xvdHMgPSBzbG90cyBvciBjb25maWcuU0xPVFMNCiAgICBjaG9zZW4sIGFsdHMsIHVzZWQgPSB7fSwge30sIHNldCgpDQogICAgaWYgJ25fc2xpY2VzJyBpbiBnLmNvbHVtbnM6DQogICAgICAgIG5zID0gcGQudG9fbnVtZXJpYyhnWyduX3NsaWNlcyddLCBlcnJvcnM9J2NvZXJjZScpLmZpbGxuYSgwKQ0KICAgIGVsaWYgJ25fZmlsZXMnIGluIGcuY29sdW1uczoNCiAgICAgICAgbnMgPSBwZC50b19udW1lcmljKGdbJ25fZmlsZXMnXSwgZXJyb3JzPSdjb2VyY2UnKS5maWxsbmEoMCkNCiAgICBlbHNlOg0KICAgICAgICBucyA9IHBkLlNlcmllcygwLCBpbmRleD1nLmluZGV4KQ0KICAgIGdnID0gZy5hc3NpZ24oX249bnMpDQogICAgZm9yIG5hbWUsIHBsYW5lLCBmbHVpZCwgZnMgaW4gc2xvdHM6DQogICAgICAgIHNlbCA9IChnZ1sncGxhbmUnXSA9PSBwbGFuZSkgJiAoZ2dbJ2ZhdHNhdCddID09IGZzKQ0KICAgICAgICBpZiBmbHVpZCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHNlbCAmPSAoZ2dbJ2ZsdWlkJ10gPT0gZmx1aWQpDQogICAgICAgIGMgPSBnZ1tzZWxdDQogICAgICAgIGlmIGxlbihjKSA9PSAwOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgaWYgcHJlZmVyXzJkOg0KICAgICAgICAgICAgYyA9IGMuYXNzaWduKF9rPWNbJ2lzM2QnXS5maWxsbmEoRmFsc2UpLmFzdHlwZShpbnQpKQ0KICAgICAgICAgICAgYyA9IGMuc29ydF92YWx1ZXMoWydfaycsICdfbicsICdTZXJpZXNJbnN0YW5jZVVJRCddLCBhc2NlbmRpbmc9W1RydWUsIEZhbHNlLCBUcnVlXSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGMgPSBjLnNvcnRfdmFsdWVzKFsnX24nLCAnU2VyaWVzSW5zdGFuY2VVSUQnXSwgYXNjZW5kaW5nPVtGYWxzZSwgVHJ1ZV0pDQogICAgICAgIGNob3NlbltuYW1lXSA9IGMuaWxvY1swXVsnU2VyaWVzSW5zdGFuY2VVSUQnXQ0KICAgICAgICBhbHRzW25hbWVdID0gW3MgZm9yIHMgaW4gY1snU2VyaWVzSW5zdGFuY2VVSUQnXS50b2xpc3QoKVsxOl1dDQogICAgICAgIHVzZWQuYWRkKGNob3NlbltuYW1lXSkNCiAgICBleHRyYXMgPSBbcyBmb3IgcyBpbiBnZ1snU2VyaWVzSW5zdGFuY2VVSUQnXSBpZiBzIG5vdCBpbiB1c2VkXQ0KICAgIHJldHVybiBjaG9zZW4sIGFsdHMsIGV4dHJhcw0KDQoNCmRlZiBhc3NpZ25fYWxsKGluZGV4X2RmLCBwcmVmZXJfMmQ9VHJ1ZSwgZnNfcHJpb3JpdHk9J2hkcicsIGNzdl9mYWxsYmFjaz1UcnVlKToNCiAgICAiIiJBbm5vdGF0ZSB0aGUgd2hvbGUgaW5kZXggYW5kIGFzc2lnbiBzbG90cyBmb3IgZXZlcnkgc3R1ZHkgKGZ1bGx5IHZlY3RvcmlzZWQ6IH5PKG4gbG9nIG4pLCBubyBwZXItc3R1ZHkgbG9vcCkuDQoNCiAgICBSZXR1cm5zIChhbm5vdGF0ZWRfZGYsIHNsb3RfdGFibGUpOg0KICAgICAgc2xvdF90YWJsZTogRGF0YUZyYW1lIGluZGV4ZWQgYnkgU3R1ZHlJbnN0YW5jZVVJRCwgb25lIGNvbHVtbiBwZXIgc2xvdCBob2xkaW5nIGEgU2VyaWVzSW5zdGFuY2VVSUQgb3IgTm9uZTsNCiAgICAgIHNsb3RfdGFibGUuYXR0cnNbJ2FsdHMnXToge3Nsb3Q6IHtzdHVkeTogW2FsdGVybmF0ZSBTZXJpZXNJbnN0YW5jZVVJRHNdfX0gZm9yIHZpZXctc3dhcCBhdWdtZW50YXRpb24uDQogICAgVGllLWJyZWFrIGlzIGRldGVybWluaXN0aWMgKHNlcmllcyBVSUQpLCB1bmxpa2UgYSBzdGFibGUgc29ydCBvdmVyIGRpcmVjdG9yeSBvcmRlci4iIiINCiAgICBpZiBpbmRleF9kZi5lbXB0eSBvciAnU3R1ZHlJbnN0YW5jZVVJRCcgbm90IGluIGluZGV4X2RmLmNvbHVtbnM6DQogICAgICAgIHN0dWRpZXMgPSBwZC5JbmRleChbXSwgbmFtZT0nU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgICAgIHRhYiA9IHBkLkRhdGFGcmFtZShpbmRleD1zdHVkaWVzLCBjb2x1bW5zPVtuIGZvciBuLCAqXyBpbiBjb25maWcuU0xPVFNdLCBkdHlwZT1vYmplY3QpDQogICAgICAgIHRhYi5hdHRyc1snYWx0cyddID0ge30NCiAgICAgICAgcmV0dXJuIGluZGV4X2RmLmNvcHkoKSwgdGFiDQogICAgZGYgPSBpbmRleF9kZi5jb3B5KCkNCiAgICBkZlsncGxhbmUnXSA9IHBsYW5lX29mKGRmKQ0KICAgIGRmID0gYW5ub3RhdGUoZGYsIGZzX3ByaW9yaXR5LCBjc3ZfZmFsbGJhY2spDQogICAgaGFzX2NvdW50ID0gKCduX3NsaWNlcycgaW4gZGYuY29sdW1ucykgb3IgKCduX2ZpbGVzJyBpbiBkZi5jb2x1bW5zKQ0KICAgIGlmICduX3NsaWNlcycgaW4gZGYuY29sdW1uczoNCiAgICAgICAgbnMgPSBwZC50b19udW1lcmljKGRmWyduX3NsaWNlcyddLCBlcnJvcnM9J2NvZXJjZScpLmZpbGxuYSgwKQ0KICAgIGVsaWYgJ25fZmlsZXMnIGluIGRmLmNvbHVtbnM6DQogICAgICAgIG5zID0gcGQudG9fbnVtZXJpYyhkZlsnbl9maWxlcyddLCBlcnJvcnM9J2NvZXJjZScpLmZpbGxuYSgwKQ0KICAgIGVsc2U6DQogICAgICAgIG5zID0gcGQuU2VyaWVzKDEsIGluZGV4PWRmLmluZGV4KQ0KICAgIHVzYWJsZSA9IGRmWydwbGFuZSddLmlzaW4oWydTYWdpdHRhbCcsICdDb3JvbmFsJywgJ0F4aWFsJ10pICYgKChucyA+IDApIGlmIGhhc19jb3VudCBlbHNlIFRydWUpDQogICAgd29yayA9IGRmLmFzc2lnbihfbj1ucywgX2s9KGRmWydpczNkJ10uYXN0eXBlKGludCkgaWYgcHJlZmVyXzJkIGVsc2UgMCkpW3VzYWJsZV0NCiAgICB3b3JrID0gd29yay5zb3J0X3ZhbHVlcyhbJ1N0dWR5SW5zdGFuY2VVSUQnLCAnX2snLCAnX24nLCAnU2VyaWVzSW5zdGFuY2VVSUQnXSwgYXNjZW5kaW5nPVtUcnVlLCBUcnVlLCBGYWxzZSwgVHJ1ZV0pDQogICAgc3R1ZGllcyA9IHBkLkluZGV4KGRmWydTdHVkeUluc3RhbmNlVUlEJ10udW5pcXVlKCksIG5hbWU9J1N0dWR5SW5zdGFuY2VVSUQnKQ0KICAgIHRhYiA9IHBkLkRhdGFGcmFtZShpbmRleD1zdHVkaWVzLCBjb2x1bW5zPVtuIGZvciBuLCAqXyBpbiBjb25maWcuU0xPVFNdLCBkdHlwZT1vYmplY3QpDQogICAgYWx0cyA9IHt9DQogICAgZm9yIG5hbWUsIHBsYW5lLCBmbHVpZCwgZnMgaW4gY29uZmlnLlNMT1RTOg0KICAgICAgICBtID0gKHdvcmtbJ3BsYW5lJ10gPT0gcGxhbmUpICYgKHdvcmtbJ2ZhdHNhdCddID09IGZzKQ0KICAgICAgICBpZiBmbHVpZCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIG0gJj0gKHdvcmtbJ2ZsdWlkJ10gPT0gZmx1aWQpDQogICAgICAgIGMgPSB3b3JrW21dDQogICAgICAgIGZpcnN0ID0gfmMuZHVwbGljYXRlZCgnU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgICAgIHRhYi5sb2NbYy5sb2NbZmlyc3QsICdTdHVkeUluc3RhbmNlVUlEJ10udmFsdWVzLCBuYW1lXSA9IGMubG9jW2ZpcnN0LCAnU2VyaWVzSW5zdGFuY2VVSUQnXS52YWx1ZXMNCiAgICAgICAgcmVzdCA9IGNbfmZpcnN0XQ0KICAgICAgICBhbHRzW25hbWVdID0gcmVzdC5ncm91cGJ5KCdTdHVkeUluc3RhbmNlVUlEJylbJ1Nlcmllc0luc3RhbmNlVUlEJ10uYXBwbHkobGlzdCkudG9fZGljdCgpIGlmIGxlbihyZXN0KSBlbHNlIHt9DQogICAgdGFiID0gdGFiLndoZXJlKHRhYi5ub3RuYSgpLCBOb25lKQ0KICAgIHRhYi5hdHRyc1snYWx0cyddID0gYWx0cw0KICAgIHJldHVybiBkZiwgdGFiDQo=',
    'src/data/preprocess/splits.py': 'IiIiTGVha2FnZS1zYWZlIGdyb3VwIGZvbGRzLg0KDQpHcm91cHMgPSAocmVwb3J0IGxhbmd1YWdlIHwgc2Nhbm5lciBtb2RlbCk6IGxhbmd1YWdlIGlzIGEgc3Ryb25nIHNpdGUgcHJveHkgaW4gdGhpcyBkYXRhc2V0IChlLmcuIGV2ZXJ5IEN5cmlsbGljDQpyZXBvcnQgaXMgYSBQaGlsaXBzIEluZ2VuaWEsIGV2ZXJ5IEdlcm1hbi9GcmVuY2gvR3JlZWsvRHV0Y2ggcmVwb3J0IGlzIFNpZW1lbnMpIGFuZCBzY2FubmVyIG1vZGVsIHNlcGFyYXRlcyB0aGUNCm1hbnkgc2l0ZXMgaGlkaW5nIGluc2lkZSB0aGUgMSw3MTcgRW5nbGlzaCByZXBvcnRzLiBBdWRpdCBudW1iZXJzOiA3MiBncm91cHMsIGxhcmdlc3QgNS45JSBvZiBzdHVkaWVzLCAyNCBncm91cHMNCndpdGggPDEwIHN0dWRpZXMuIFJhcmUgZ3JvdXBzIGFyZSBtZXJnZWQgaW50byAncmFyZXw8dmVuZG9yPicuIENvcnJvYm9yYXRlZCBkdXBsaWNhdGUgZXhhbXMgYXJlIHJlcHJlc2VudGVkIGFzDQpjb25uZWN0ZWQgY29tcG9uZW50cyBhbmQga2VwdCBpbiB0aGUgc2FtZSBmb2xkLg0KSWRlbnRpY2FsIHJlcG9ydCBURVhUUyBhcmUgZGVsaWJlcmF0ZWx5IE5PVCB1bmlvbmVkIGludG8gZ3JvdXBzOiB0ZW1wbGF0ZWQgRW5nbGlzaCByZXBvcnRzIHJlY3VyIGFjcm9zcyBzaXRlcyBhbmQNCndvdWxkIGNoYWluIHVucmVsYXRlZCBzaXRlcyBpbnRvIG9uZSBnaWFudCBjb21wb25lbnQuDQoiIiINCmltcG9ydCBvcw0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgcGFuZGFzIGFzIHBkDQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdA0KDQojIHRoZSBvbmx5IHRydWUgZHVwbGljYXRlIGV4YW1zIGZvdW5kIGJ5IHRoZSBhdWRpdCAoNSBpZGVudGljYWwgc2VyaWVzIGVhY2gsIFNpZW1lbnMgQWVyYSwgbm9uLWdvbGQpDQpLTk9XTl9EVVBfUEFJUlMgPSBbKA0KICAgICcxLjIuODI2LjAuMS4zNjgwMDQzLjguNDk4Ljk5NzI1NTIzNTYwMjU0NzMyMDk0NDMyMjQ0NTU2NjIzMzc2NzExJywNCiAgICAnMS4yLjgyNi4wLjEuMzY4MDA0My44LjQ5OC45OTkyNjYyNDk2ODI0MDY4MTc3MjczNTEwMjMwMzUzMTYxMzIwOScsDQopXQ0KDQoNCmRlZiBkdXBsaWNhdGVfcGFpcnNfZnJvbV9zZXJpZXNfbWV0YShwYXRoLCBzdHVkeV91aWRzPU5vbmUsIG1pbl9zaGFyZWRfaGFzaGVzPTIpOg0KICAgICIiIlJldHVybiBvbmx5IGNyb3NzLXN0dWR5IHBhaXJzIGNvcnJvYm9yYXRlZCBieSBtdWx0aXBsZSBkaXN0aW5jdCBleGFjdCBzZXJpZXMgaGFzaGVzLg0KDQogICAgVGhlIGNvbXBhY3QgRURBIGRIYXNoIGlzIGNvbGxpc2lvbi1wcm9uZSBhdCBzaW5nbGUtaGFzaCByZXNvbHV0aW9uLiBTaW5nbGUgbWF0Y2hlcw0KICAgIGFyZSByZXBvcnRlZCBhcyBjYW5kaWRhdGVzIGJ1dCBuZXZlciB1c2VkIHRvIGNvbnN0cmFpbiBmb2xkcy4NCiAgICAiIiINCiAgICBpZiBub3QgcGF0aCBvciBub3Qgb3MucGF0aC5leGlzdHMocGF0aCk6DQogICAgICAgIGFsbG93ZWQgPSBzZXQobWFwKHN0ciwgc3R1ZHlfdWlkcykpIGlmIHN0dWR5X3VpZHMgaXMgbm90IE5vbmUgZWxzZSBOb25lDQogICAgICAgIHBhaXJzID0gWw0KICAgICAgICAgICAgcGFpciBmb3IgcGFpciBpbiBLTk9XTl9EVVBfUEFJUlMNCiAgICAgICAgICAgIGlmIGFsbG93ZWQgaXMgTm9uZSBvciAocGFpclswXSBpbiBhbGxvd2VkIGFuZCBwYWlyWzFdIGluIGFsbG93ZWQpDQogICAgICAgIF0NCiAgICAgICAgcmV0dXJuIHBhaXJzLCB7DQogICAgICAgICAgICAic291cmNlIjogImtub3duLXZlcmlmaWVkLWZhbGxiYWNrIiwNCiAgICAgICAgICAgICJjYW5kaWRhdGVfcGFpcnMiOiAwLA0KICAgICAgICAgICAgImFjY2VwdGVkX3BhaXJzIjogbGVuKHBhaXJzKSwNCiAgICAgICAgICAgICJhY2NlcHRlZF9wYWlyX2lkcyI6IFtsaXN0KHBhaXIpIGZvciBwYWlyIGluIHBhaXJzXSwNCiAgICAgICAgICAgICJtaW5fc2hhcmVkX2hhc2hlcyI6IG1pbl9zaGFyZWRfaGFzaGVzLA0KICAgICAgICB9DQoNCiAgICBtZXRhID0gcGQucmVhZF9jc3YocGF0aCkNCiAgICByZXF1aXJlZCA9IHsiU3R1ZHlJbnN0YW5jZVVJRCIsICJkaGFzaCJ9DQogICAgaWYgbm90IHJlcXVpcmVkLmlzc3Vic2V0KG1ldGEuY29sdW1ucyk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJEdXBsaWNhdGUgbWV0YWRhdGEgbXVzdCBjb250YWluIHtzb3J0ZWQocmVxdWlyZWQpfToge3BhdGh9IikNCiAgICBtZXRhID0gbWV0YVtbIlN0dWR5SW5zdGFuY2VVSUQiLCAiZGhhc2giXV0uZHJvcG5hKCkNCiAgICBtZXRhWyJTdHVkeUluc3RhbmNlVUlEIl0gPSBtZXRhWyJTdHVkeUluc3RhbmNlVUlEIl0uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkNCiAgICBtZXRhWyJkaGFzaCJdID0gbWV0YVsiZGhhc2giXS5hc3R5cGUoc3RyKS5zdHIuc3RyaXAoKS5zdHIubG93ZXIoKQ0KICAgIGlmIHN0dWR5X3VpZHMgaXMgbm90IE5vbmU6DQogICAgICAgIGFsbG93ZWQgPSBzZXQobWFwKHN0ciwgc3R1ZHlfdWlkcykpDQogICAgICAgIG1ldGEgPSBtZXRhW21ldGFbIlN0dWR5SW5zdGFuY2VVSUQiXS5pc2luKGFsbG93ZWQpXQ0KICAgIGhhc2hlc19ieV9zdHVkeSA9IG1ldGEuZ3JvdXBieSgiU3R1ZHlJbnN0YW5jZVVJRCIpWyJkaGFzaCJdLmFnZyhsYW1iZGEgdmFsdWVzOiBzZXQodmFsdWVzKSkNCiAgICBzdHVkaWVzX2J5X2hhc2ggPSBkZWZhdWx0ZGljdChsaXN0KQ0KICAgIGZvciBzdHVkeSwgaGFzaGVzIGluIGhhc2hlc19ieV9zdHVkeS5pdGVtcygpOg0KICAgICAgICBmb3IgdmFsdWUgaW4gaGFzaGVzOg0KICAgICAgICAgICAgaWYgdmFsdWUgYW5kIHZhbHVlIG5vdCBpbiB7Im5hbiIsICJub25lIn06DQogICAgICAgICAgICAgICAgc3R1ZGllc19ieV9oYXNoW3ZhbHVlXS5hcHBlbmQoc3R1ZHkpDQoNCiAgICBzaGFyZWQgPSBkZWZhdWx0ZGljdChzZXQpDQogICAgZm9yIHZhbHVlLCBzdHVkaWVzIGluIHN0dWRpZXNfYnlfaGFzaC5pdGVtcygpOg0KICAgICAgICBpZiBsZW4oc3R1ZGllcykgPCAyOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgc3R1ZGllcyA9IHNvcnRlZChzZXQoc3R1ZGllcykpDQogICAgICAgIGZvciBpLCBhIGluIGVudW1lcmF0ZShzdHVkaWVzKToNCiAgICAgICAgICAgIGZvciBiIGluIHN0dWRpZXNbaSArIDE6XToNCiAgICAgICAgICAgICAgICBzaGFyZWRbKGEsIGIpXS5hZGQodmFsdWUpDQogICAgYWNjZXB0ZWQgPSBzb3J0ZWQocGFpciBmb3IgcGFpciwgaGFzaGVzIGluIHNoYXJlZC5pdGVtcygpIGlmIGxlbihoYXNoZXMpID49IG1pbl9zaGFyZWRfaGFzaGVzKQ0KICAgIHJldHVybiBhY2NlcHRlZCwgew0KICAgICAgICAic291cmNlIjogb3MucGF0aC5hYnNwYXRoKHBhdGgpLA0KICAgICAgICAiY2FuZGlkYXRlX3BhaXJzIjogbGVuKHNoYXJlZCksDQogICAgICAgICJzaW5nbGVfaGFzaF9wYWlycyI6IHN1bShsZW4oaGFzaGVzKSA9PSAxIGZvciBoYXNoZXMgaW4gc2hhcmVkLnZhbHVlcygpKSwNCiAgICAgICAgImFjY2VwdGVkX3BhaXJzIjogbGVuKGFjY2VwdGVkKSwNCiAgICAgICAgImFjY2VwdGVkX3BhaXJfaWRzIjogW2xpc3QocGFpcikgZm9yIHBhaXIgaW4gYWNjZXB0ZWRdLA0KICAgICAgICAibWluX3NoYXJlZF9oYXNoZXMiOiBtaW5fc2hhcmVkX2hhc2hlcywNCiAgICB9DQoNCg0KZGVmIF9kdXBsaWNhdGVfY29tcG9uZW50cyhzdHVkeV91aWRzLCBkdXBfcGFpcnMpOg0KICAgIHBhcmVudCA9IHt1aWQ6IHVpZCBmb3IgdWlkIGluIHN0dWR5X3VpZHN9DQoNCiAgICBkZWYgZmluZCh1aWQpOg0KICAgICAgICB3aGlsZSBwYXJlbnRbdWlkXSAhPSB1aWQ6DQogICAgICAgICAgICBwYXJlbnRbdWlkXSA9IHBhcmVudFtwYXJlbnRbdWlkXV0NCiAgICAgICAgICAgIHVpZCA9IHBhcmVudFt1aWRdDQogICAgICAgIHJldHVybiB1aWQNCg0KICAgIGZvciBhLCBiIGluIGR1cF9wYWlyczoNCiAgICAgICAgaWYgYSBub3QgaW4gcGFyZW50IG9yIGIgbm90IGluIHBhcmVudDoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHJhLCByYiA9IGZpbmQoYSksIGZpbmQoYikNCiAgICAgICAgaWYgcmEgIT0gcmI6DQogICAgICAgICAgICBwYXJlbnRbbWF4KHJhLCByYildID0gbWluKHJhLCByYikNCiAgICByZXR1cm4ge3VpZDogZmluZCh1aWQpIGZvciB1aWQgaW4gc3R1ZHlfdWlkc30NCg0KDQpkZWYgbWFrZV9ncm91cHMoc3R1ZHlfbWV0YSwgbWluX2dyb3VwPTEwLCBkdXBfcGFpcnM9Tm9uZSwgbl9zcGxpdHM9NSk6DQogICAgc20gPSBzdHVkeV9tZXRhLmNvcHkoKQ0KICAgIGcgPSBzbVsnbGFuZyddLmZpbGxuYSgnPycpLmFzdHlwZShzdHIpICsgJ3wnICsgc21bJ21vZGVsJ10uZmlsbG5hKCc/JykuYXN0eXBlKHN0cikNCiAgICBjbnQgPSBnLm1hcChnLnZhbHVlX2NvdW50cygpKQ0KICAgIGcgPSBnLndoZXJlKGNudCA+PSBtaW5fZ3JvdXAsICdyYXJlfCcgKyBzbVsndmVuZG9yJ10uZmlsbG5hKCc/JykuYXN0eXBlKHN0cikpDQogICAgY250ID0gZy5tYXAoZy52YWx1ZV9jb3VudHMoKSkgICAgICAgICAgICAgICAgICAgICAgICMgYSAncmFyZXw8dmVuZG9yPicgYnVja2V0IGNhbiBpdHNlbGYgYmUgdG9vIHNtYWxsIHRvIHN0cmF0aWZ5DQogICAgZyA9IGcud2hlcmUoY250ID49IG5fc3BsaXRzLCAncmFyZXxBTEwnKQ0KICAgIHNtWydncm91cCddID0gZw0KICAgIHBhaXJzID0gZHVwX3BhaXJzIGlmIGR1cF9wYWlycyBpcyBub3QgTm9uZSBlbHNlIEtOT1dOX0RVUF9QQUlSUw0KICAgIGNvbXBvbmVudHMgPSBfZHVwbGljYXRlX2NvbXBvbmVudHMoc21bIlN0dWR5SW5zdGFuY2VVSUQiXS5hc3R5cGUoc3RyKS50b2xpc3QoKSwgcGFpcnMpDQogICAgc21bImR1cGxpY2F0ZV9ncm91cCJdID0gc21bIlN0dWR5SW5zdGFuY2VVSUQiXS5hc3R5cGUoc3RyKS5tYXAoY29tcG9uZW50cykNCiAgICByZXR1cm4gc20NCg0KDQpkZWYgZ3JvdXBfZm9sZHMoc3R1ZHlfbWV0YSwgbl9zcGxpdHM9NSwgc2VlZD0yMDI2LCBtaW5fZ3JvdXA9MTAsIGR1cF9wYWlycz1Ob25lLCBzY2hlbWU9J3NpdGUnKToNCiAgICAiIiItPiBEYXRhRnJhbWVbU3R1ZHlJbnN0YW5jZVVJRCwgZ3JvdXAsIGZvbGQsIC4uLl0uDQoNCiAgICBzY2hlbWU9J3NpdGUnICAoZGVmYXVsdCwgZm9yIG1vZGVsIHNlbGVjdGlvbik6IGV2ZXJ5IHNpdGUvc2Nhbm5lciBncm91cCBpcyBzcHJlYWQgZXZlbmx5IG92ZXIgYWxsIGZvbGRzDQogICAgICAgICAgICAgICAgICAgKHN0cmF0aWZpZWQgYnkgZ3JvdXApIC0gbWlycm9ycyB0aGUgaGlkZGVuIHRlc3QsIHdoaWNoIGlzIGRyYXduIGZyb20gdGhlIHNhbWUgc2l0ZXMgYXMgdHJhaW4uDQogICAgICAgICAgICAgICAgICAgRHVwbGljYXRlIGV4YW1zIGFyZSBrZXB0IHRvZ2V0aGVyLg0KICAgIHNjaGVtZT0nZ3JvdXAnIChyb2J1c3RuZXNzIGNoZWNrKTogd2hvbGUgZ3JvdXBzIGFyZSBoZWxkIG91dCB0b2dldGhlciA9IHVuc2Vlbi1zY2FubmVyIGdlbmVyYWxpc2F0aW9uLg0KICAgICAgICAgICAgICAgICAgIFBlc3NpbWlzdGljIGJ5IGNvbnN0cnVjdGlvbjogYXVkaXQgc2hvd3MgMjA1LzIzOCBtZXRhZGF0YS1wb29yIHN0dWRpZXMgbGFuZCBpbiBvbmUgZm9sZCBhbmQNCiAgICAgICAgICAgICAgICAgICBtb3N0IGxhbmd1YWdlcyBhcmUgYWJzZW50IGZyb20gc29tZSB0cmFpbmluZyBmb2xkcy4iIiINCiAgICBzbSA9IG1ha2VfZ3JvdXBzKHN0dWR5X21ldGEsIG1pbl9ncm91cCwgZHVwX3BhaXJzLCBuX3NwbGl0cykNCiAgICBmb2xkID0gbnAuZnVsbChsZW4oc20pLCAtMSkNCiAgICBjb21wb25lbnRzID0gc21bImR1cGxpY2F0ZV9ncm91cCJdLmFzdHlwZShzdHIpDQogICAgY29tcG9uZW50X2lkcyA9IGNvbXBvbmVudHMuZHJvcF9kdXBsaWNhdGVzKCkudG9saXN0KCkNCiAgICBpZiBsZW4oY29tcG9uZW50X2lkcykgPCAyICogbl9zcGxpdHM6DQogICAgICAgIG9yZGVyID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpLnBlcm11dGF0aW9uKGNvbXBvbmVudF9pZHMpDQogICAgICAgIGNvbXBvbmVudF9mb2xkID0ge2NvbXBvbmVudDogaSAlIG5fc3BsaXRzIGZvciBpLCBjb21wb25lbnQgaW4gZW51bWVyYXRlKG9yZGVyKX0NCiAgICAgICAgc21bImZvbGQiXSA9IGNvbXBvbmVudHMubWFwKGNvbXBvbmVudF9mb2xkKS5hc3R5cGUoaW50KQ0KICAgICAgICByZXR1cm4gc21bWydTdHVkeUluc3RhbmNlVUlEJywgJ2dyb3VwJywgJ2R1cGxpY2F0ZV9ncm91cCcsICdmb2xkJ10gKyBbYyBmb3IgYyBpbiAoJ2xhbmcnLCAnbW9kZWwnLCAndmVuZG9yJywgJ2dvbGQnKSBpZiBjIGluIHNtLmNvbHVtbnNdXQ0KICAgIGlmIHNjaGVtZSA9PSAnZ3JvdXAnOg0KICAgICAgICBzaXRlX2dyb3VwcyA9IHNtWyJncm91cCJdLmFzdHlwZShzdHIpLnRvbGlzdCgpDQogICAgICAgIHNpdGVfcGFyZW50ID0ge2Yic2l0ZTp7Z3JvdXB9IjogZiJzaXRlOntncm91cH0iIGZvciBncm91cCBpbiBzaXRlX2dyb3Vwc30NCg0KICAgICAgICBkZWYgZmluZF9zaXRlKGdyb3VwKToNCiAgICAgICAgICAgIHdoaWxlIHNpdGVfcGFyZW50W2dyb3VwXSAhPSBncm91cDoNCiAgICAgICAgICAgICAgICBzaXRlX3BhcmVudFtncm91cF0gPSBzaXRlX3BhcmVudFtzaXRlX3BhcmVudFtncm91cF1dDQogICAgICAgICAgICAgICAgZ3JvdXAgPSBzaXRlX3BhcmVudFtncm91cF0NCiAgICAgICAgICAgIHJldHVybiBncm91cA0KDQogICAgICAgIHVpZF90b19ncm91cCA9IGRpY3QoemlwKHNtWyJTdHVkeUluc3RhbmNlVUlEIl0uYXN0eXBlKHN0ciksIHNpdGVfZ3JvdXBzKSkNCiAgICAgICAgZm9yIGEsIGIgaW4gKGR1cF9wYWlycyBpZiBkdXBfcGFpcnMgaXMgbm90IE5vbmUgZWxzZSBLTk9XTl9EVVBfUEFJUlMpOg0KICAgICAgICAgICAgaWYgYSBub3QgaW4gdWlkX3RvX2dyb3VwIG9yIGIgbm90IGluIHVpZF90b19ncm91cDoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgZ2EsIGdiID0gZiJzaXRlOnt1aWRfdG9fZ3JvdXBbYV19IiwgZiJzaXRlOnt1aWRfdG9fZ3JvdXBbYl19Ig0KICAgICAgICAgICAgcmEsIHJiID0gZmluZF9zaXRlKGdhKSwgZmluZF9zaXRlKGdiKQ0KICAgICAgICAgICAgaWYgcmEgIT0gcmI6DQogICAgICAgICAgICAgICAgc2l0ZV9wYXJlbnRbbWF4KHJhLCByYildID0gbWluKHJhLCByYikNCiAgICAgICAgdmFsaWRhdGlvbl9ncm91cHMgPSBucC5hcnJheSgNCiAgICAgICAgICAgIFtmaW5kX3NpdGUoZiJzaXRlOntncm91cH0iKSBmb3IgZ3JvdXAgaW4gc2l0ZV9ncm91cHNdDQogICAgICAgICkNCiAgICAgICAgaWYgbGVuKHNldCh2YWxpZGF0aW9uX2dyb3VwcykpIDwgbl9zcGxpdHM6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKA0KICAgICAgICAgICAgICAgIGYiQ2Fubm90IGNyZWF0ZSB7bl9zcGxpdHN9IHNpdGUtaGVsZC1vdXQgZm9sZHMgZnJvbSAiDQogICAgICAgICAgICAgICAgZiJ7bGVuKHNldCh2YWxpZGF0aW9uX2dyb3VwcykpfSBpbmRlcGVuZGVudCBzaXRlL2R1cGxpY2F0ZSBncm91cHMiDQogICAgICAgICAgICApDQogICAgICAgIGZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IFN0cmF0aWZpZWRHcm91cEtGb2xkDQogICAgICAgIHkgPSBzbVsndmVuZG9yJ10uZmlsbG5hKCc/JykuYXN0eXBlKCdjYXRlZ29yeScpLmNhdC5jb2Rlcy52YWx1ZXMNCiAgICAgICAgZm9yIGssIChfLCB2YSkgaW4gZW51bWVyYXRlKA0KICAgICAgICAgICAgU3RyYXRpZmllZEdyb3VwS0ZvbGQobl9zcGxpdHM9bl9zcGxpdHMsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPXNlZWQpLnNwbGl0KA0KICAgICAgICAgICAgICAgIHNtLCB5LCB2YWxpZGF0aW9uX2dyb3Vwcw0KICAgICAgICAgICAgKQ0KICAgICAgICApOg0KICAgICAgICAgICAgZm9sZFt2YV0gPSBrDQogICAgZWxpZiBzY2hlbWUgPT0gJ3NpdGUnOg0KICAgICAgICBmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkS0ZvbGQNCiAgICAgICAgcmVwID0gc20uZHJvcF9kdXBsaWNhdGVzKCJkdXBsaWNhdGVfZ3JvdXAiKS5jb3B5KCkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQ0KICAgICAgICBjb3VudHMgPSByZXBbJ2dyb3VwJ10udmFsdWVfY291bnRzKCkNCiAgICAgICAgdG9vX3NtYWxsID0gY291bnRzW2NvdW50cyA8IG5fc3BsaXRzXS5pbmRleA0KICAgICAgICBpZiBsZW4odG9vX3NtYWxsKSA+IDAgYW5kIGxlbihjb3VudHMpID4gMDoNCiAgICAgICAgICAgIG1ham9yaXR5X2dycCA9IGNvdW50cy5pbmRleFswXQ0KICAgICAgICAgICAgcmVwWydncm91cCddID0gcmVwWydncm91cCddLnJlcGxhY2Uoe2c6IG1ham9yaXR5X2dycCBmb3IgZyBpbiB0b29fc21hbGx9KQ0KICAgICAgICB5ID0gcmVwWydncm91cCddLmFzdHlwZSgnY2F0ZWdvcnknKS5jYXQuY29kZXMudmFsdWVzDQogICAgICAgIHNrZiA9IFN0cmF0aWZpZWRLRm9sZChuX3NwbGl0cz1uX3NwbGl0cywgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9c2VlZCkNCiAgICAgICAgdWZvbGQgPSB7fQ0KICAgICAgICBmb3IgaywgKF8sIHZhKSBpbiBlbnVtZXJhdGUoc2tmLnNwbGl0KHJlcCwgeSkpOg0KICAgICAgICAgICAgZm9yIGNvbXBvbmVudCBpbiByZXAubG9jW3ZhLCAnZHVwbGljYXRlX2dyb3VwJ106DQogICAgICAgICAgICAgICAgdWZvbGRbY29tcG9uZW50XSA9IGsNCiAgICAgICAgZm9sZCA9IGNvbXBvbmVudHMubWFwKHVmb2xkKS50b19udW1weShkdHlwZT1pbnQpDQogICAgZWxzZToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihzY2hlbWUpDQogICAgc21bJ2ZvbGQnXSA9IGZvbGQNCiAgICByZXR1cm4gc21bWydTdHVkeUluc3RhbmNlVUlEJywgJ2dyb3VwJywgJ2R1cGxpY2F0ZV9ncm91cCcsICdmb2xkJ10gKyBbYyBmb3IgYyBpbiAoJ2xhbmcnLCAnbW9kZWwnLCAndmVuZG9yJywgJ2dvbGQnKSBpZiBjIGluIHNtLmNvbHVtbnNdXQ0KDQoNCmRlZiBtYWtlX3N0dWR5X21ldGEoaW5kZXhfZGYsIHRyYWluX2Nzdj1Ob25lLCBsYWJlbHNfZGY9Tm9uZSk6DQogICAgIiIiU3R1ZHktbGV2ZWwgdGFibGUgKHZlbmRvciwgc2Nhbm5lciBtb2RlbCwgcmVwb3J0IGxhbmd1YWdlLCBnb2xkIGZsYWcsIG1ldGFkYXRhLXBvb3IgZmxhZykgc3RyYWlnaHQgZnJvbSB0aGUNCiAgICBpbmRleCArIHRyYWluLmNzdiwgc28gdGhlIG5vdGVib29rIG5ldmVyIG5lZWRzIHRoZSBhdWRpdCdzIHN0dWR5X21ldGEuY3N2LiIiIg0KICAgIGZyb20gLnRleHQgaW1wb3J0IGd1ZXNzX2xhbmcsIHZlbmRvcl9vZg0KICAgIGlmIGluZGV4X2RmIGlzIE5vbmUgb3IgbGVuKGluZGV4X2RmKSA9PSAwOg0KICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZSgpDQogICAgZWxzZToNCiAgICAgICAgZGYgPSBpbmRleF9kZltpbmRleF9kZlsnc3BsaXQnXSA9PSAndHJhaW4nXSBpZiAnc3BsaXQnIGluIGluZGV4X2RmLmNvbHVtbnMgZWxzZSBpbmRleF9kZg0KICAgIGlmIGRmLmVtcHR5IG9yICdTdHVkeUluc3RhbmNlVUlEJyBub3QgaW4gZGYuY29sdW1uczoNCiAgICAgICAgaWYgbGFiZWxzX2RmIGlzIG5vdCBOb25lIGFuZCAnU3R1ZHlJbnN0YW5jZVVJRCcgaW4gbGFiZWxzX2RmLmNvbHVtbnM6DQogICAgICAgICAgICBzdHVkaWVzID0gcGQuSW5kZXgobGFiZWxzX2RmWydTdHVkeUluc3RhbmNlVUlEJ10udW5pcXVlKCksIG5hbWU9J1N0dWR5SW5zdGFuY2VVSUQnKQ0KICAgICAgICBlbGlmIHRyYWluX2NzdiBpcyBub3QgTm9uZSBhbmQgJ1N0dWR5SW5zdGFuY2VVSUQnIGluIHRyYWluX2Nzdi5jb2x1bW5zOg0KICAgICAgICAgICAgc3R1ZGllcyA9IHBkLkluZGV4KHRyYWluX2NzdlsnU3R1ZHlJbnN0YW5jZVVJRCddLnVuaXF1ZSgpLCBuYW1lPSdTdHVkeUluc3RhbmNlVUlEJykNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHN0dWRpZXMgPSBwZC5JbmRleChbXSwgbmFtZT0nU3R1ZHlJbnN0YW5jZVVJRCcpDQogICAgICAgIHNtID0gcGQuRGF0YUZyYW1lKHsndmVuZG9yJzogJz8nLCAnbW9kZWwnOiAnPycsICdwb29yJzogRmFsc2V9LCBpbmRleD1zdHVkaWVzKS5yZXNldF9pbmRleCgpDQogICAgICAgIHNtWydsYW5nJ10gPSAnPycNCiAgICAgICAgaWYgdHJhaW5fY3N2IGlzIG5vdCBOb25lIGFuZCAnUmVwb3J0JyBpbiB0cmFpbl9jc3YuY29sdW1uczoNCiAgICAgICAgICAgIGxhbmcgPSB0cmFpbl9jc3Yuc2V0X2luZGV4KCdTdHVkeUluc3RhbmNlVUlEJylbJ1JlcG9ydCddLm1hcChndWVzc19sYW5nKQ0KICAgICAgICAgICAgc21bJ2xhbmcnXSA9IHNtWydTdHVkeUluc3RhbmNlVUlEJ10ubWFwKGxhbmcpLmZpbGxuYSgnPycpDQogICAgICAgIHNtWydnb2xkJ10gPSBGYWxzZQ0KICAgICAgICBpZiBsYWJlbHNfZGYgaXMgbm90IE5vbmU6DQogICAgICAgICAgICBpbXBvcnQgc3JjLmNvcmUuY29uZmlnIGFzIGNvbmZpZw0KICAgICAgICAgICAgaWYgJ3NvdXJjZScgaW4gbGFiZWxzX2RmLmNvbHVtbnM6DQogICAgICAgICAgICAgICAgaGFzID0gbGFiZWxzX2RmWydzb3VyY2UnXSA9PSAnZ29sZCcNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgaGFzID0gbGFiZWxzX2RmW1t0IGZvciB0IGluIGNvbmZpZy5UQVJHRVRTIGlmIHQgaW4gbGFiZWxzX2RmLmNvbHVtbnNdXS5ub3RuYSgpLmFueShheGlzPTEpDQogICAgICAgICAgICBnb2xkID0gbGFiZWxzX2RmLmFzc2lnbihfZz1oYXMpLnNldF9pbmRleCgnU3R1ZHlJbnN0YW5jZVVJRCcpWydfZyddDQogICAgICAgICAgICBzbVsnZ29sZCddID0gc21bJ1N0dWR5SW5zdGFuY2VVSUQnXS5tYXAoZ29sZCkuZmlsbG5hKEZhbHNlKQ0KICAgICAgICByZXR1cm4gc20NCiAgICBtb2RlID0gbGFtYmRhIHM6IHMuZHJvcG5hKCkubW9kZSgpLmlsb2NbMF0gaWYgcy5ub3RuYSgpLmFueSgpIGVsc2UgbnAubmFuDQogICAgZyA9IGRmLmdyb3VwYnkoJ1N0dWR5SW5zdGFuY2VVSUQnKQ0KICAgIHN0dWRpZXMgPSBwZC5JbmRleChkZlsnU3R1ZHlJbnN0YW5jZVVJRCddLnVuaXF1ZSgpLCBuYW1lPSdTdHVkeUluc3RhbmNlVUlEJykNCiAgICB2ZW5kb3IgPSBnWydNYW51ZmFjdHVyZXInXS5hZ2cobGFtYmRhIHM6IG1vZGUocy5tYXAodmVuZG9yX29mKSkpIGlmICdNYW51ZmFjdHVyZXInIGluIGRmIGVsc2UgcGQuU2VyaWVzKCc/JywgaW5kZXg9c3R1ZGllcykNCiAgICBtb2RlbCA9IGdbJ01hbnVmYWN0dXJlck1vZGVsTmFtZSddLmFnZyhtb2RlKSBpZiAnTWFudWZhY3R1cmVyTW9kZWxOYW1lJyBpbiBkZiBlbHNlIHBkLlNlcmllcygnPycsIGluZGV4PXN0dWRpZXMpDQogICAgcG9vciA9IGdbJ1JlcGV0aXRpb25UaW1lJ10uYWdnKGxhbWJkYSBzOiBzLmlzbmEoKS5hbGwoKSkgaWYgJ1JlcGV0aXRpb25UaW1lJyBpbiBkZiBlbHNlIHBkLlNlcmllcyhGYWxzZSwgaW5kZXg9c3R1ZGllcykNCiAgICBzbSA9IHBkLkRhdGFGcmFtZSh7J3ZlbmRvcic6IHZlbmRvciwgJ21vZGVsJzogbW9kZWwsICdwb29yJzogcG9vcn0sIGluZGV4PXN0dWRpZXMpLnJlc2V0X2luZGV4KCkNCiAgICBzbVsnbGFuZyddID0gJz8nDQogICAgaWYgdHJhaW5fY3N2IGlzIG5vdCBOb25lIGFuZCAnUmVwb3J0JyBpbiB0cmFpbl9jc3YuY29sdW1uczoNCiAgICAgICAgbGFuZyA9IHRyYWluX2Nzdi5zZXRfaW5kZXgoJ1N0dWR5SW5zdGFuY2VVSUQnKVsnUmVwb3J0J10ubWFwKGd1ZXNzX2xhbmcpDQogICAgICAgIHNtWydsYW5nJ10gPSBzbVsnU3R1ZHlJbnN0YW5jZVVJRCddLm1hcChsYW5nKS5maWxsbmEoJz8nKQ0KICAgIGlmIGxhYmVsc19kZiBpcyBub3QgTm9uZToNCiAgICAgICAgaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCiAgICAgICAgaWYgJ3NvdXJjZScgaW4gbGFiZWxzX2RmLmNvbHVtbnM6ICAgICAgICAgICAgICAgICAgICAgICAjIGJ1aWxkX2xhYmVscygpIG1hcmtzIGdvbGQgLyBleHRyYSAvIG5vbmUNCiAgICAgICAgICAgIGhhcyA9IGxhYmVsc19kZlsnc291cmNlJ10gPT0gJ2dvbGQnDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBoYXMgPSBsYWJlbHNfZGZbW3QgZm9yIHQgaW4gY29uZmlnLlRBUkdFVFMgaWYgdCBpbiBsYWJlbHNfZGYuY29sdW1uc11dLm5vdG5hKCkuYW55KGF4aXM9MSkNCiAgICAgICAgZ29sZCA9IGxhYmVsc19kZi5hc3NpZ24oX2c9aGFzKS5zZXRfaW5kZXgoJ1N0dWR5SW5zdGFuY2VVSUQnKVsnX2cnXQ0KICAgICAgICBzbVsnZ29sZCddID0gc21bJ1N0dWR5SW5zdGFuY2VVSUQnXS5tYXAoZ29sZCkuZmlsbG5hKEZhbHNlKQ0KICAgIGVsc2U6DQogICAgICAgIHNtWydnb2xkJ10gPSBGYWxzZQ0KICAgIHJldHVybiBzbQ0K',
    'src/data/preprocess/text.py': 'IiIiVGlueSB0ZXh0L3ZlbmRvciBoZWxwZXJzIHVzZWQgdG8gYnVpbGQgbGVha2FnZS1zYWZlIGZvbGRzIChubyBleHRlcm5hbCBOTFAgZGVwZW5kZW5jeSkuIiIiCmltcG9ydCByZQoKX1NUT1AgPSB7CiAgICAnZW4nOiAndGhlIGFuZCBvZiB3aXRoIGlzIGFyZSB0aGVyZSBubyB3aXRob3V0IGluIHRvIGFuIGF0IGZyb20gd2hpY2ggc2VlbiBub3RlZCcsCiAgICAnZXMnOiAnZWwgbGEgbG9zIGxhcyBkZSBkZWwgY29uIHNpbiBzZSBlbiB5IHVuYSB1biBxdWUgZXMgaGF5IG5vIG9ic2VydmEnLAogICAgJ3B0JzogJ28gb3MgYXMgZGUgZGEgZG8gY29tIHNlbSBlbSBlIHVtYSB1bSBxdWUgbsOjbyBuYW8gaMOhIGhhJywKICAgICdmcic6ICdsZSBsYSBsZXMgZGVzIGR1IGRlIGV0IGF2ZWMgc2FucyB1biB1bmUgZXN0IHBhcyBpbCB5IGEgZW4gZGFucycsCiAgICAnaXQnOiAnaWwgbG8gZ2xpIGxlIGRpIGRlbCBkZWxsYSBjb24gc2VuemEgbm9uIMOoIHVuYSB1biBlIG5lbCBuZWxsYSBzaScsCiAgICAnZGUnOiAnZGVyIGRpZSBkYXMgdW5kIG1pdCBvaG5lIGtlaW4ga2VpbmUgaXN0IG5pY2h0IGVpbmUgaW0gZGVuIGRlcyB6ZWlndCcsCiAgICAndHInOiAndmUgaWxlIGJpciBvbGFyYWsgeW9rIGl6bGVuZGkgZ8O2csO8bG1la3RlZGlyIG5vcm1hbCBkZSBkYSBidWx1bm1ha3RhZMSxcicsCiAgICAnaHIvYnMvc3InOiAnaSB1IGplIG5hIHNlIGJleiBzIG5lbWEgbmlqZSB2aWRsaml2IHZpZGxqaXZhIHVyZWRhbiB1cmVkbmEnLAogICAgJ25sJzogJ2RlIGhldCBlZW4gZW4gbWV0IHpvbmRlciBnZWVuIGlzIG5pZXQgdmFuIGluIG9wIHRlJywKfQpfU1RPUCA9IHtrOiBzZXQodi5zcGxpdCgpKSBmb3IgaywgdiBpbiBfU1RPUC5pdGVtcygpfQoKCmRlZiBndWVzc19sYW5nKHRleHQpOgogICAgIiIiQ29hcnNlIHJlcG9ydC1sYW5ndWFnZSBwcm94eSAoc2NyaXB0IHJhbmdlcyArIHN0b3Atd29yZCBzY29yZXMpLiBPbmx5IHVzZWQgYXMgYSBTSVRFIHByb3h5IGZvciBmb2xkIGdyb3VwaW5nLAogICAgbmV2ZXIgYXMgYSBmZWF0dXJlLiBLbm93biBsaW1pdGF0aW9uOiB0ZW1wbGF0ZWQgRW5nbGlzaCByZXBvcnRzIGNhbiBiZSB0YWdnZWQgJ3RyJzsgcGFpcmVkIHdpdGggc2Nhbm5lciBtb2RlbAogICAgaW4gdGhlIGdyb3VwIGtleSwgdGhpcyBkb2VzIG5vdCBtYXR0ZXIuIiIiCiAgICBpZiBub3QgaXNpbnN0YW5jZSh0ZXh0LCBzdHIpIG9yIGxlbih0ZXh0LnN0cmlwKCkpIDwgNToKICAgICAgICByZXR1cm4gJ2VtcHR5JwogICAgdCA9IHRleHQubG93ZXIoKQogICAgbGV0dGVycyA9IFtjaCBmb3IgY2ggaW4gdCBpZiBjaC5pc2FscGhhKCldCiAgICBpZiBub3QgbGV0dGVyczoKICAgICAgICByZXR1cm4gJ2VtcHR5JwoKICAgIGRlZiBmcmFjKGEsIGIpOgogICAgICAgIHJldHVybiBzdW0oYSA8PSBjaCA8PSBiIGZvciBjaCBpbiBsZXR0ZXJzKSAvIGxlbihsZXR0ZXJzKQogICAgaWYgZnJhYygnXHUwNDAwJywgJ1x1MDRmZicpID4gLjM6CiAgICAgICAgcmV0dXJuICdjeXJpbGxpYycKICAgIGlmIGZyYWMoJ1x1MDM3MCcsICdcdTAzZmYnKSA+IC4zOgogICAgICAgIHJldHVybiAnZ3JlZWsnCiAgICBpZiBmcmFjKCdcdTBlMDAnLCAnXHUwZTdmJykgPiAuMzoKICAgICAgICByZXR1cm4gJ3RoYWknCiAgICBpZiBmcmFjKCdcdTRlMDAnLCAnXHU5ZmZmJykgPiAuMjoKICAgICAgICByZXR1cm4gJ2NqaycKICAgIGlmIGZyYWMoJ1x1MDYwMCcsICdcdTA2ZmYnKSA+IC4zOgogICAgICAgIHJldHVybiAnYXJhYmljJwogICAgdG9rcyA9IHJlLmZpbmRhbGwocidbXlxXXGRfXSsnLCB0KQogICAgc2MgPSB7azogc3VtKHcgaW4gdiBmb3IgdyBpbiB0b2tzKSBmb3IgaywgdiBpbiBfU1RPUC5pdGVtcygpfQogICAgc2NbJ3RyJ10gKz0gMiAqIHN1bShjaCBpbiAnxJ/EscWfxLAnIGZvciBjaCBpbiB0ZXh0KSAvIG1heChsZW4odG9rcyksIDEpICogMTAKICAgIGJlc3QgPSBtYXgoc2MsIGtleT1zYy5nZXQpCiAgICByZXR1cm4gYmVzdCBpZiBzY1tiZXN0XSA+PSAzIGVsc2UgJ3Vua25vd24nCgoKZGVmIHZlbmRvcl9vZihtYW51ZmFjdHVyZXIpOgogICAgIiIiTm9ybWFsaXNlIHRoZSBtYW55IHNwZWxsaW5ncyBzZWVuIGluIHRoZSBkYXRhIChTaWVtZW5zL1NJRU1FTlMvU2llbWVucyBIZWFsdGhpbmVlcnMsIEdFL0dFSEMsIC4uLikuIiIiCiAgICBtID0gc3RyKG1hbnVmYWN0dXJlcikubG93ZXIoKQogICAgZm9yIGssIHYgaW4gKCgnc2llbWVucycsICdTSUVNRU5TJyksICgnZ2UgbWVkaWNhbCcsICdHRScpLCAoJ2dlaGMnLCAnR0UnKSwgKCdwaGlsaXBzJywgJ1BISUxJUFMnKSwKICAgICAgICAgICAgICAgICAoJ3Rvc2hpYmEnLCAnQ0FOT04vVE9TSElCQScpLCAoJ2Nhbm9uJywgJ0NBTk9OL1RPU0hJQkEnKSwgKCdmdWppJywgJ0ZVSkkvSElUQUNISScpLAogICAgICAgICAgICAgICAgICgnaGl0YWNoaScsICdGVUpJL0hJVEFDSEknKSk6CiAgICAgICAgaWYgayBpbiBtOgogICAgICAgICAgICByZXR1cm4gdgogICAgcmV0dXJuICdPVEhFUicK',
    'src/data/preprocess/__init__.py': 'IiIiRGF0YXNldC12ZXJpZmllZCBwcmVwcm9jZXNzaW5nIGxheWVyIChudW1weS9PcGVuQ1Ygb25seTsgdG9yY2gtZnJlZSkuIiIiCmltcG9ydCBvcwpvcy5lbnZpcm9uWydPTVBfTlVNX1RIUkVBRFMnXSA9ICcxJwpvcy5lbnZpcm9uWydPUEVOQkxBU19OVU1fVEhSRUFEUyddID0gJzEnCm9zLmVudmlyb25bJ01LTF9OVU1fVEhSRUFEUyddID0gJzEnCm9zLmVudmlyb25bJ1ZFQ0xJQl9NQVhJTVVNX1RIUkVBRFMnXSA9ICcxJwpvcy5lbnZpcm9uWydOVU1FWFBSX05VTV9USFJFQURTJ10gPSAnMScKCmltcG9ydCBjdjIKY3YyLnNldE51bVRocmVhZHMoMCkK',
    'src/inference/inference.py': '...',
    'src/inference/__init__.py': '',
    'src/modeling/ensemble.py': 'IiIiRW5zZW1ibGUgdXRpbGl0aWVzLg0KDQpJbXBsZW1lbnRzOg0KICBUZW1wZXJhdHVyZUNhbGlicmF0aW9uICBQb3N0LWhvYyBwcm9iYWJpbGl0eSBjYWxpYnJhdGlvbiBwZXIgbW9kZWwgYXJtLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBHdW8gZXQgYWwuLCBJQ01MIDIwMTcg4oCUICJPbiBDYWxpYnJhdGlvbiBvZiBNb2Rlcm4gTmV1cmFsIE5ldHdvcmtzIi4NCiAgICAgICAgICAgICAgICAgICAgICAgICAgRmluZHMgc2NhbGFyIFQgcGVyIGFybSB0aGF0IG1pbmltaXNlcyBOTEwgb24gT09GIHByZWRpY3Rpb25zLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBQb3NpdGl2ZSBzY2FsYXIgdGVtcGVyYXR1cmUgZG9lcyBub3QgY2hhbmdlIHBlci10YXJnZXQgQVVDL3JhbmtzLg0KDQogIGdyZWVkeV9lbnNlbWJsZV9zZWxlY3QgIENhcnVhbmEgZXQgYWwuLCBJQ01MIDIwMDQg4oCUICJFbnNlbWJsZSBTZWxlY3Rpb24gZnJvbSBMaWJyYXJpZXMgb2YgTW9kZWxzIi4NCiAgICAgICAgICAgICAgICAgICAgICAgICAgRXhwZXJpbWVudGFsIGdyZWVkeSBzZWxlY3Rpb24gb24gT09GIHByZWRpY3Rpb25zOyBpdHMNCiAgICAgICAgICAgICAgICAgICAgICAgICAgaW4tc2FtcGxlIHJlc3VsdCBpcyBub3QgYW4gdW5iaWFzZWQgZ2VuZXJhbGl6YXRpb24gZXN0aW1hdGUuDQoNCiAgcmFua19lbnNlbWJsZV9uICAgICAgICAgTi1hcm0gcmFuay1wZXJjZW50aWxlIGJsZW5kaW5nIChnZW5lcmFsaXphdGlvbiBvZiBiYXNlbGluZSAyLWFybSBibGVuZCkuDQogICAgICAgICAgICAgICAgICAgICAgICAgIFBlci10YXJnZXQgQ29BdE5ldCB3ZWlnaHRpbmcga2VwdCBmcm9tIGJhc2VsaW5lICgwLjk0MyBzb3VyY2UpLg0KICAgICAgICAgICAgICAgICAgICAgICAgICBUaGlyZCBhcm0gKENvbnZOZVh0IC8gUmFkSW1hZ2VOZXQgQ05OKSBhZGRlZCB3aXRoIHBlci10YXJnZXQgd2VpZ2h0cy4NCiIiIg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbnVtcHkgYXMgbnANCmltcG9ydCBwYW5kYXMgYXMgcGQNCmZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IG1pbmltaXplX3NjYWxhcg0KDQppbXBvcnQgc3JjLmNvcmUuY29uZmlnIGFzIGNvbmZpZw0KDQoNCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAIFRlbXBlcmF0dXJlIENhbGlicmF0aW9uIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgA0KY2xhc3MgVGVtcGVyYXR1cmVDYWxpYnJhdGlvbjoNCiAgICAiIiJQZXItYXJtIHRlbXBlcmF0dXJlIHNjYWxpbmcgKEd1byBldCBhbC4sIElDTUwgMjAxNykuDQoNCiAgICBVc2FnZQ0KICAgIC0tLS0tDQogICAgY2FsID0gVGVtcGVyYXR1cmVDYWxpYnJhdGlvbigpDQogICAgY2FsLmZpdChvb2ZfbG9naXRzLCBvb2ZfbGFiZWxzKSAgICAgICAgICAjIG51bXB5IFtOLCBDXQ0KICAgIGNhbGlicmF0ZWRfcHJvYnMgPSBjYWwudHJhbnNmb3JtKGxvZ2l0cykgIyBudW1weSBbTiwgQ10NCiAgICAiIiINCg0KICAgIGRlZiBfX2luaXRfXyhzZWxmKToNCiAgICAgICAgc2VsZi5UOiBmbG9hdCA9IDEuMA0KDQogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0NCiAgICBkZWYgX25sbChzZWxmLCBUOiBmbG9hdCwgbG9naXRzOiBucC5uZGFycmF5LCBsYWJlbHM6IG5wLm5kYXJyYXkpIC0+IGZsb2F0Og0KICAgICAgICAiIiJCaW5hcnkgY3Jvc3MtZW50cm9weSBOTEwgZm9yIGEgZ2l2ZW4gdGVtcGVyYXR1cmUuIiIiDQogICAgICAgIHAgPSAxLjAgLyAoMS4wICsgbnAuZXhwKC1sb2dpdHMgLyBtYXgoVCwgMWUtNikpKQ0KICAgICAgICBwID0gbnAuY2xpcChwLCAxZS03LCAxLjAgLSAxZS03KQ0KICAgICAgICByZXR1cm4gLW5wLm1lYW4obGFiZWxzICogbnAubG9nKHApICsgKDEgLSBsYWJlbHMpICogbnAubG9nKDEgLSBwKSkNCg0KICAgIGRlZiBmaXQoc2VsZiwgb29mX2xvZ2l0czogbnAubmRhcnJheSwgb29mX2xhYmVsczogbnAubmRhcnJheSkgLT4gIlRlbXBlcmF0dXJlQ2FsaWJyYXRpb24iOg0KICAgICAgICAiIiJGaW5kIFQqID0gYXJnbWluIE5MTChUKSBvbiBPT0YgZGF0YSB2aWEgYm91bmRlZCBzY2FsYXIgc2VhcmNoLg0KDQogICAgICAgIGxvZ2l0czogcmF3IG1vZGVsIG91dHB1dCAocHJlLXNpZ21vaWQpIFtOLCBDXS4NCiAgICAgICAgbGFiZWxzOiBiaW5hcnkgW04sIENdLg0KICAgICAgICBPbmx5IHVzZXMgbGFiZWxsZWQgZW50cmllcyAod2VpZ2h0ID4gMCkgaWYgb29mX2xhYmVscyBjb250YWlucyBOYU5zLg0KICAgICAgICAiIiINCiAgICAgICAgbWFzayA9IG5wLmlzZmluaXRlKG9vZl9sYWJlbHMpICYgbnAuaXNmaW5pdGUob29mX2xvZ2l0cykNCiAgICAgICAgbG8gPSBvb2ZfbG9naXRzW21hc2tdLnJhdmVsKCkNCiAgICAgICAgbGEgPSBvb2ZfbGFiZWxzW21hc2tdLnJhdmVsKCkNCiAgICAgICAgaWYgbGVuKGxvKSA9PSAwIG9yIGxhLnN1bSgpID09IDAgb3IgbGEuc3VtKCkgPT0gbGVuKGxhKToNCiAgICAgICAgICAgIHNlbGYuVCA9IDEuMA0KICAgICAgICAgICAgcmV0dXJuIHNlbGYNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgcmVzID0gbWluaW1pemVfc2NhbGFyKA0KICAgICAgICAgICAgICAgIGxhbWJkYSBUOiBzZWxmLl9ubGwoVCwgbG8sIGxhKSwNCiAgICAgICAgICAgICAgICBib3VuZHM9KDAuMSwgMTAuMCksDQogICAgICAgICAgICAgICAgbWV0aG9kPSJib3VuZGVkIiwNCiAgICAgICAgICAgICkNCiAgICAgICAgICAgIHNlbGYuVCA9IGZsb2F0KHJlcy54KSBpZiAoaGFzYXR0cihyZXMsICJ4IikgYW5kIG5wLmlzZmluaXRlKHJlcy54KSkgZWxzZSAxLjANCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIHNlbGYuVCA9IDEuMA0KICAgICAgICByZXR1cm4gc2VsZg0KDQogICAgZGVmIHRyYW5zZm9ybShzZWxmLCBsb2dpdHM6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6DQogICAgICAgICIiIkFwcGx5IGNhbGlicmF0aW9uOiBzaWdtb2lkKGxvZ2l0cyAvIFQpLiIiIg0KICAgICAgICByZXR1cm4gMS4wIC8gKDEuMCArIG5wLmV4cCgtbG9naXRzIC8gbWF4KHNlbGYuVCwgMWUtNikpKQ0KDQogICAgZGVmIGZpdF90cmFuc2Zvcm0oc2VsZiwgb29mX2xvZ2l0cywgb29mX2xhYmVscywgdGVzdF9sb2dpdHM9Tm9uZSk6DQogICAgICAgIHNlbGYuZml0KG9vZl9sb2dpdHMsIG9vZl9sYWJlbHMpDQogICAgICAgIGlmIHRlc3RfbG9naXRzIGlzIE5vbmU6DQogICAgICAgICAgICByZXR1cm4gc2VsZi50cmFuc2Zvcm0ob29mX2xvZ2l0cykNCiAgICAgICAgcmV0dXJuIHNlbGYudHJhbnNmb3JtKHRlc3RfbG9naXRzKQ0KDQogICAgZGVmIF9fcmVwcl9fKHNlbGYpOg0KICAgICAgICByZXR1cm4gZiJUZW1wZXJhdHVyZUNhbGlicmF0aW9uKFQ9e3NlbGYuVDouNGZ9KSINCg0KDQojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgCBHcmVlZHkgRW5zZW1ibGUgU2VsZWN0aW9uIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgA0KZGVmIGdyZWVkeV9lbnNlbWJsZV9zZWxlY3QoDQogICAgb29mX3ByZWRzOiBsaXN0W25wLm5kYXJyYXldLCAgICMgbGlzdCBvZiBbTiwgQ10gcHJvYmFiaWxpdHkgYXJyYXlzIChwZXIgbW9kZWwpDQogICAgb29mX2xhYmVsczogbnAubmRhcnJheSwgICAgICAgICMgW04sIENdIGJpbmFyeSBncm91bmQgdHJ1dGgNCiAgICBuX3NlbGVjdDogaW50ID0gMTAsICAgICAgICAgICAgIyBtYXggZW5zZW1ibGUgbWVtYmVycyAod2l0aCByZXBsYWNlbWVudCkNCiAgICBtZXRyaWM6IHN0ciA9ICJhdWMiLCAgICAgICAgICAgIyAiYXVjIiBvciAiYmNlIg0KKSAtPiBsaXN0W2ludF06DQogICAgIiIiQ2FydWFuYSBldCBhbC4sIElDTUwgMjAwNCDigJQgZ3JlZWR5IGZvcndhcmQgc2VsZWN0aW9uIHdpdGggcmVwbGFjZW1lbnQuDQoNCiAgICBSZXR1cm5zIGEgbGlzdCBvZiBpbmRpY2VzICh3aXRoIHJlcGV0aXRpb25zKSBpbnRvIG9vZl9wcmVkcyByZXByZXNlbnRpbmcgdGhlDQogICAgc2VsZWN0ZWQgZW5zZW1ibGUgbWVtYmVycy4gV2VpZ2h0IG9mIGVhY2ggbW9kZWwgPSBjb3VudChpbmRleCkgLyBuX3NlbGVjdC4NCg0KICAgIFRoaXMgZG9lcyBub3QgZ3VhcmFudGVlIGltcHJvdmVtZW50IG92ZXIgdGhlIGJlc3Qgc2luZ2xlIG1vZGVsIGFuZCBjYW4NCiAgICBvdmVyZml0IHdoZW4gdGhlIE9PRiBzZXQgaXMgc21hbGwgb3IgcmV1c2VkIGZvciBtb2RlbCBzZWxlY3Rpb24uDQogICAgIiIiDQogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUNCg0KICAgIGRlZiBzY29yZShibGVuZDogbnAubmRhcnJheSkgLT4gZmxvYXQ6DQogICAgICAgIGlmIG1ldHJpYyA9PSAiYXVjIjoNCiAgICAgICAgICAgIGF1Y3MgPSBbXQ0KICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2UoYmxlbmQuc2hhcGVbMV0pOg0KICAgICAgICAgICAgICAgIG0gPSBucC5pc2Zpbml0ZShvb2ZfbGFiZWxzWzosIGpdKQ0KICAgICAgICAgICAgICAgIGlmIG0uc3VtKCkgPiAxIGFuZCAwIDwgb29mX2xhYmVsc1ttLCBqXS5zdW0oKSA8IG0uc3VtKCk6DQogICAgICAgICAgICAgICAgICAgIGF1Y3MuYXBwZW5kKHJvY19hdWNfc2NvcmUob29mX2xhYmVsc1ttLCBqXSwgYmxlbmRbbSwgal0pKQ0KICAgICAgICAgICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oYXVjcykpIGlmIGF1Y3MgZWxzZSAwLjANCiAgICAgICAgZWxzZTogICMgQkNFDQogICAgICAgICAgICBwID0gbnAuY2xpcChibGVuZCwgMWUtNywgMSAtIDFlLTcpDQogICAgICAgICAgICBtID0gbnAuaXNmaW5pdGUob29mX2xhYmVscykNCiAgICAgICAgICAgIHJldHVybiAtZmxvYXQobnAubWVhbigNCiAgICAgICAgICAgICAgICAtKG9vZl9sYWJlbHNbbV0gKiBucC5sb2cocFttXSkgKyAoMSAtIG9vZl9sYWJlbHNbbV0pICogbnAubG9nKDEgLSBwW21dKSkNCiAgICAgICAgICAgICkpDQoNCiAgICBzZWxlY3RlZDogbGlzdFtpbnRdID0gW10NCiAgICBjdXJyZW50X2JsZW5kID0gbnAuemVyb3NfbGlrZShvb2ZfcHJlZHNbMF0pDQoNCiAgICBmb3Igc3RlcCBpbiByYW5nZShuX3NlbGVjdCk6DQogICAgICAgIGJlc3RfaWR4LCBiZXN0X3Njb3JlID0gLTEsIC1ucC5pbmYNCiAgICAgICAgZm9yIGksIHByZWQgaW4gZW51bWVyYXRlKG9vZl9wcmVkcyk6DQogICAgICAgICAgICBjYW5kaWRhdGUgPSAoY3VycmVudF9ibGVuZCAqIHN0ZXAgKyBwcmVkKSAvIChzdGVwICsgMSkNCiAgICAgICAgICAgIHMgPSBzY29yZShjYW5kaWRhdGUpDQogICAgICAgICAgICBpZiBzID4gYmVzdF9zY29yZToNCiAgICAgICAgICAgICAgICBiZXN0X3Njb3JlLCBiZXN0X2lkeCA9IHMsIGkNCiAgICAgICAgc2VsZWN0ZWQuYXBwZW5kKGJlc3RfaWR4KQ0KICAgICAgICBjdXJyZW50X2JsZW5kID0gKGN1cnJlbnRfYmxlbmQgKiBzdGVwICsgb29mX3ByZWRzW2Jlc3RfaWR4XSkgLyAoc3RlcCArIDEpDQoNCiAgICByZXR1cm4gc2VsZWN0ZWQNCg0KDQpkZWYgYmxlbmRfZnJvbV9zZWxlY3Rpb24ocHJlZHM6IGxpc3RbbnAubmRhcnJheV0sIHNlbGVjdGVkOiBsaXN0W2ludF0pIC0+IG5wLm5kYXJyYXk6DQogICAgIiIiQXZlcmFnZSBwcmVkaWN0aW9ucyBhY2NvcmRpbmcgdG8gYSBncmVlZHktc2VsZWN0ZWQgaW5kZXggbGlzdCAod2l0aCByZXBldGl0aW9uKS4iIiINCiAgICBjb3VudHMgPSBucC5iaW5jb3VudChzZWxlY3RlZCwgbWlubGVuZ3RoPWxlbihwcmVkcykpLmFzdHlwZShmbG9hdCkNCiAgICBibGVuZCA9IHN1bShwICogYyBmb3IgcCwgYyBpbiB6aXAocHJlZHMsIGNvdW50cykpDQogICAgcmV0dXJuIGJsZW5kIC8gY291bnRzLnN1bSgpDQoNCg0KIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgTi1Bcm0gUmFuayBFbnNlbWJsZSDilIDilIDilIDilIDilIDilIDilIDilIDilIANCiMgUGVyLXRhcmdldCBibGVuZGluZyB3ZWlnaHRzIGZvciB0aGUgZXhpc3RpbmcgdHdvLWFybSBzeXN0ZW0uDQojIEtleXM6IHRhcmdldCBuYW1lIChjb25maWcuVEFSR0VUUykuICBWYWx1ZXM6IGZyYWN0aW9uIGFzc2lnbmVkIHRvIENvQXROZXQgYXJtLg0KX0NPQVRfV0VJR0hUUzogZGljdFtzdHIsIGZsb2F0XSA9IHt0OiAwLjYwIGZvciB0IGluIGNvbmZpZy5UQVJHRVRTfQ0KX0NPQVRfV0VJR0hUUy51cGRhdGUoew0KICAgICJBQ0wiOiAwLjc1LA0KICAgICJNZWRpYWwgTWVuaXNjdXMiOiAwLjgwLA0KICAgICJMYXRlcmFsIE1lbmlzY3VzIjogMS4wMCwNCiAgICAiTGF0ZXJhbCBPQSI6IDAuNzUsDQogICAgIkZyYWN0dXJlIjogMC43NSwNCn0pDQoNCiMgVGhpcmQgYXJtIChDb252TmVYdCAvIFJhZEltYWdlTmV0LXByZXRyYWluZWQgQ05OKS4NCiMgVmVyaWZpZWQgYmFzaXM6IE1laSBldCBhbC4gUmFkSW1hZ2VOZXQgMjAyMiDigJQgQUNMICs0LjglLCBNZW5pc2N1cyArNC41JS4NCiMgQ29udk5lWHQgdGV4dHVyZSBmb2N1cyDihpIgc3Ryb25nIGZvciBsb2NhbGlzZWQgZmluZGluZ3MgKEZyYWN0dXJlLCBPQSkuDQojIFN0YXJ0IGNvbnNlcnZhdGl2ZWx5IGF0IDAuMTUgYWNyb3NzIGJvYXJkOyBpbmNyZWFzZSBmb3IgQUNML01lbmlzY3VzIGVtcGlyaWNhbGx5Lg0KX0NPTlZORVhUX1dFSUdIVFM6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7dDogMC4xMCBmb3IgdCBpbiBjb25maWcuVEFSR0VUU30NCl9DT05WTkVYVF9XRUlHSFRTLnVwZGF0ZSh7DQogICAgIkFDTCI6IDAuMTUsDQogICAgIk1lZGlhbCBNZW5pc2N1cyI6IDAuMTUsDQogICAgIkxhdGVyYWwgTWVuaXNjdXMiOiAwLjIwLA0KICAgICJGcmFjdHVyZSI6IDAuMTUsDQogICAgIkNvbnR1c2lvbiI6IDAuMTAsDQp9KQ0KDQoNCmRlZiByYW5rX2Vuc2VtYmxlX24oDQogICAgZGZzOiBsaXN0W3R1cGxlW3N0ciwgcGQuRGF0YUZyYW1lXV0sDQogICAgY29hdF93ZWlnaHRzOiBkaWN0W3N0ciwgZmxvYXRdIHwgTm9uZSA9IE5vbmUsDQogICAgY29udm5leHRfd2VpZ2h0czogZGljdFtzdHIsIGZsb2F0XSB8IE5vbmUgPSBOb25lLA0KKSAtPiBwZC5EYXRhRnJhbWU6DQogICAgIiIiTi1hcm0gcmFuay1wZXJjZW50aWxlIGJsZW5kaW5nLiAgR2VuZXJhbGlzZXMgdGhlIGJhc2VsaW5lIDItYXJtIGJsZW5kLg0KDQogICAgUGFyYW1ldGVycw0KICAgIC0tLS0tLS0tLS0NCiAgICBkZnMgOiBsaXN0IG9mIChhcm1fbmFtZSwgRGF0YUZyYW1lKSBwYWlycy4NCiAgICAgICAgRWFjaCBEYXRhRnJhbWUgbXVzdCBoYXZlICdTdHVkeUluc3RhbmNlVUlEJyArIGNvbmZpZy5UQVJHRVRTIGNvbHVtbnMuDQogICAgICAgIFN1cHBvcnRlZCBhcm0gbmFtZXM6ICdkaW5vJywgJ2NvYXRuZXQnLCAnY29udm5leHQnLiBPdGhlcnMgdHJlYXRlZCBhcw0KICAgICAgICBhZGRpdGlvbmFsIERJTk92Mi1zdHlsZSBhcm1zIGFuZCBhZGRlZCB0byB0aGUgRElOT3YyIHdlaWdodC4NCiAgICBjb2F0X3dlaWdodHMgLyBjb252bmV4dF93ZWlnaHRzIDogb3B0aW9uYWwgb3ZlcnJpZGUgZGljdHMuDQoNCiAgICBSZXR1cm5zDQogICAgLS0tLS0tLQ0KICAgIERhdGFGcmFtZSB3aXRoICdTdHVkeUluc3RhbmNlVUlEJyArIGNvbmZpZy5UQVJHRVRTIGFzIHJhbmstcGVyY2VudGlsZSBibGVuZC4NCiAgICAiIiINCiAgICBjdyA9IGNvYXRfd2VpZ2h0cyBvciBfQ09BVF9XRUlHSFRTDQogICAgeHcgPSBjb252bmV4dF93ZWlnaHRzIG9yIF9DT05WTkVYVF9XRUlHSFRTDQoNCiAgICAjIEFsaWduIG9uIHRoZSBmaXJzdCBhcm0ncyBzdHVkeSBvcmRlcg0KICAgIHJlZl91aWQgPSBkZnNbMF1bMV1bIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgIHJhbmtzOiBkaWN0W3N0ciwgcGQuRGF0YUZyYW1lXSA9IHt9DQogICAgZm9yIGFybV9uYW1lLCBkZiBpbiBkZnM6DQogICAgICAgIGFsaWduZWQgPSBkZi5zZXRfaW5kZXgoIlN0dWR5SW5zdGFuY2VVSUQiKS5yZWluZGV4KHJlZl91aWQpLnJlc2V0X2luZGV4KCkNCiAgICAgICAgciA9IGFsaWduZWRbY29uZmlnLlRBUkdFVFNdLnJhbmsobWV0aG9kPSJhdmVyYWdlIiwgcGN0PVRydWUpDQogICAgICAgIHIuaW5kZXggPSByZWZfdWlkDQogICAgICAgIHJhbmtzW2FybV9uYW1lXSA9IHINCg0KICAgIGJsZW5kID0gcGQuRGF0YUZyYW1lKGluZGV4PXJlZl91aWQsIGNvbHVtbnM9Y29uZmlnLlRBUkdFVFMsIGR0eXBlPWZsb2F0KQ0KICAgIGZvciB0IGluIGNvbmZpZy5UQVJHRVRTOg0KICAgICAgICBjb2F0X3cgPSBjdy5nZXQodCwgMC42MCkNCiAgICAgICAgY29udl93ID0geHcuZ2V0KHQsIDAuMTApDQogICAgICAgIGRpbm9fdG90YWwgPSAxLjAgLSBjb2F0X3cgLSBjb252X3cgICMgcmVtYWluaW5nIHdlaWdodCB0byBESU5PdjIgYXJtcw0KDQogICAgICAgICMgU3VtIERJTk92Mi10eXBlIGFybSBjb250cmlidXRpb25zIChoYW5kbGVzIDUtZm9sZCBhdmVyYWdlcykNCiAgICAgICAgZGlub19jb250cmliID0gMC4wDQogICAgICAgIG5fZGlubyA9IDANCiAgICAgICAgZm9yIGFybV9uYW1lLCBfIGluIGRmczoNCiAgICAgICAgICAgIGlmIGFybV9uYW1lIG5vdCBpbiAoImNvYXRuZXQiLCAiY29udm5leHQiKToNCiAgICAgICAgICAgICAgICBkaW5vX2NvbnRyaWIgKz0gcmFua3NbYXJtX25hbWVdW3RdDQogICAgICAgICAgICAgICAgbl9kaW5vICs9IDENCiAgICAgICAgaWYgbl9kaW5vID4gMDoNCiAgICAgICAgICAgIGRpbm9fY29udHJpYiA9IChkaW5vX2NvbnRyaWIgLyBuX2Rpbm8pICogZGlub190b3RhbA0KDQogICAgICAgIGNvYXRfY29udHJpYiA9IHJhbmtzLmdldCgiY29hdG5ldCIsIHJhbmtzW2Rmc1swXVswXV0pW3RdICogY29hdF93IGlmICJjb2F0bmV0IiBpbiByYW5rcyBlbHNlIDAuMA0KICAgICAgICBjb252X2NvbnRyaWIgPSByYW5rcy5nZXQoImNvbnZuZXh0IiwgcmFua3NbZGZzWzBdWzBdXSlbdF0gKiBjb252X3cgaWYgImNvbnZuZXh0IiBpbiByYW5rcyBlbHNlIDAuMA0KDQogICAgICAgIGJsZW5kW3RdID0gZGlub19jb250cmliICsgY29hdF9jb250cmliICsgY29udl9jb250cmliDQoNCiAgICAjIEZpbmFsIHJlLXJhbmsgdG8gdW5pZm9ybSBbMCwgMV0gcGVyY2VudGlsZSBzcGFjZQ0KICAgIGJsZW5kID0gYmxlbmQucmFuayhtZXRob2Q9ImF2ZXJhZ2UiLCBwY3Q9VHJ1ZSkNCiAgICByZXN1bHQgPSBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogcmVmX3VpZH0pDQogICAgZm9yIHQgaW4gY29uZmlnLlRBUkdFVFM6DQogICAgICAgIHJlc3VsdFt0XSA9IGJsZW5kW3RdLnZhbHVlcw0KICAgIHJldHVybiByZXN1bHQNCg==',
    'src/modeling/losses.py': 'IiIiTG9zcyBmdW5jdGlvbnMgZm9yIG11bHRpLWxhYmVsIHRyYWluaW5nLgoKSW1wbGVtZW50YXRpb25zCi0tLS0tLS0tLS0tLS0tLQpBc3ltbWV0cmljTG9zcyAgUmlkbmlrIGV0IGFsLiwgSUNDViAyMDIxIChhclhpdjoyMDA5LjE0MTE5KS4KICAgICAgICAgICAgICAgIFVzZXMgc2VwYXJhdGUgcG9zaXRpdmUgYW5kIG5lZ2F0aXZlIGZvY3VzaW5nIGV4cG9uZW50cyBhbmQgYW4KICAgICAgICAgICAgICAgIG9wdGlvbmFsIG5lZ2F0aXZlLXByb2JhYmlsaXR5IGNsaXAuIEl0cyBlZmZlY3Qgb24gdGhpcyBkYXRhc2V0CiAgICAgICAgICAgICAgICBtdXN0IGJlIGNvbXBhcmVkIGVtcGlyaWNhbGx5IHdpdGggQkNFLgoKICAgICAgICAgICAgICAgIGdhbW1hX25lZyBtYXkgYmUgc2hhcmVkIGFjcm9zcyB0YXJnZXRzIG9yIHN1cHBsaWVkIHBlciB0YXJnZXQuCiAgICAgICAgICAgICAgICBBIGxhcmdlciBuZWdhdGl2ZSBmb2N1c2luZyBleHBvbmVudCBzdXBwcmVzc2VzIGVhc3kgbmVnYXRpdmUKICAgICAgICAgICAgICAgIGV4YW1wbGVzOyBpdCBkb2VzIG5vdCBkaXJlY3RseSB1cHdlaWdodCBwb3NpdGl2ZSBleGFtcGxlcy4KCldlaWdodGVkQkNFICAgICBMZWdhY3kgZm9yIEEvQiBhYmxhdGlvbjsga2VwdCBhcyBhIGRyb3AtaW4gYWx0ZXJuYXRpdmUuCgpSZWZlcmVuY2U6IGh0dHBzOi8vZ2l0aHViLmNvbS9BbGliYWJhLU1JSUwvQVNMCiIiIgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBBc3ltbWV0cmljTG9zcyhubi5Nb2R1bGUpOgogICAgIiIiUmlkbmlrIGV0IGFsLiwgSUNDViAyMDIxLgoKICAgIFBhcmFtZXRlcnMKICAgIC0tLS0tLS0tLS0KICAgIGdhbW1hX25lZyA6IGZsb2F0IG9yIGxpc3RbZmxvYXRdCiAgICAgICAgRm9jdXNpbmcgZXhwb25lbnQgZm9yIG5lZ2F0aXZlcy4gIFNjYWxhcjogYXBwbGllZCB1bmlmb3JtbHkgYWNyb3NzIGFsbAogICAgICAgIGNsYXNzZXMuICBMaXN0W0NdOiBwZXItdGFyZ2V0IGdhbW1hLCBlbmFibGluZyBwcmV2YWxlbmNlLW1hdGNoZWQgZm9jdXNpbmcKICAgICAgICAoaGlnaGVyIGdhbW1hIGZvciByYXJlIHRhcmdldHMsIGxvd2VyIGZvciBjb21tb24gb25lcykuCiAgICBnYW1tYV9wb3MgOiBmbG9hdAogICAgICAgIEZvY3VzaW5nIGV4cG9uZW50IGZvciBwb3NpdGl2ZXMuIERlZmF1bHQgMCDigJQgbmV2ZXIgZG93bi13ZWlnaHQgdHJ1ZQogICAgICAgIHBvc2l0aXZlcyAoY3JpdGljYWwgZm9yIHJhcmUsIGhpZ2gtc3Rha2VzIGZpbmRpbmdzKS4KICAgIGNsaXAgOiBmbG9hdAogICAgICAgIFByb2JhYmlsaXR5IHNoaWZ0IGZvciBuZWdhdGl2ZXMuIEVudHJpZXMgd2hlcmUgcF9uZWcgPCBjbGlwIGFyZSB6ZXJvZWQsCiAgICAgICAgZGlzY2FyZGluZyB0cml2aWFsbHkgZWFzeSAvIGxpa2VseSBtaXNsYWJlbGxlZCBuZWdhdGl2ZXMuCiAgICAgICAgU2V0IGNsaXA9MC4wIHRvIGRpc2FibGUgKGRlZ2VuZXJhdGVzIHRvIGFzeW1tZXRyaWMgRm9jYWwgTG9zcykuCiAgICBlcHMgOiBmbG9hdAogICAgICAgIExvZy1zdGFiaWxpdHkgZXBzaWxvbi4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBnYW1tYV9uZWc9Mi4wLCBnYW1tYV9wb3M6IGZsb2F0ID0gMC4wLAogICAgICAgICAgICAgICAgIGNsaXA6IGZsb2F0ID0gMC4wNSwgZXBzOiBmbG9hdCA9IDFlLTgpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGlzaW5zdGFuY2UoZ2FtbWFfbmVnLCAobGlzdCwgdHVwbGUpKToKICAgICAgICAgICAgIyBSZWdpc3RlciBhcyBidWZmZXIgc28gaXQgbW92ZXMgdG8gR1BVIHdpdGggLnRvKGRldmljZSkgYXV0b21hdGljYWxseQogICAgICAgICAgICBnYW1tYV90ZW5zb3IgPSB0b3JjaC50ZW5zb3IoZ2FtbWFfbmVnLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUoZ2FtbWFfdGVuc29yKS5hbGwoKSBvciAoZ2FtbWFfdGVuc29yIDwgMCkuYW55KCk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJnYW1tYV9uZWcgdmFsdWVzIG11c3QgYmUgZmluaXRlIGFuZCBub24tbmVnYXRpdmUiKQogICAgICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiZ2FtbWFfbmVnX3ZlYyIsIGdhbW1hX3RlbnNvcikKICAgICAgICAgICAgc2VsZi5fZ2FtbWFfbmVnX3NjYWxhciA9IE5vbmUKICAgICAgICBlbHNlOgogICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUodG9yY2gudGVuc29yKGZsb2F0KGdhbW1hX25lZykpKSBvciBnYW1tYV9uZWcgPCAwOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZ2FtbWFfbmVnIG11c3QgYmUgZmluaXRlIGFuZCBub24tbmVnYXRpdmUiKQogICAgICAgICAgICBzZWxmLmdhbW1hX25lZ192ZWMgPSBOb25lCiAgICAgICAgICAgIHNlbGYuX2dhbW1hX25lZ19zY2FsYXIgPSBmbG9hdChnYW1tYV9uZWcpCiAgICAgICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKHRvcmNoLnRlbnNvcihmbG9hdChnYW1tYV9wb3MpKSkgb3IgZ2FtbWFfcG9zIDwgMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZ2FtbWFfcG9zIG11c3QgYmUgZmluaXRlIGFuZCBub24tbmVnYXRpdmUiKQogICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZSh0b3JjaC50ZW5zb3IoZmxvYXQoY2xpcCkpKSBvciBub3QgMCA8PSBjbGlwIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2xpcCBtdXN0IGJlIGZpbml0ZSBhbmQgaW4gWzAsIDEpIikKICAgICAgICBzZWxmLmdhbW1hX3BvcyA9IGZsb2F0KGdhbW1hX3BvcykKICAgICAgICBzZWxmLmNsaXAgPSBmbG9hdChjbGlwKQogICAgICAgIHNlbGYuZXBzID0gZXBzCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBmb3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgbG9naXRzOiB0b3JjaC5UZW5zb3IsICAgICAgICAgICMgW0IsIENdIHJhdyAocHJlLXNpZ21vaWQpCiAgICAgICAgdGFyZ2V0czogdG9yY2guVGVuc29yLCAgICAgICAgICMgW0IsIENdIGluIFswLCAxXTsgc29mdCBsYWJlbHMgT0sKICAgICAgICB3ZWlnaHRzOiB0b3JjaC5UZW5zb3IgfCBOb25lID0gTm9uZSwgICMgW0IsIENdIHBlci1lbnRyeSB3ZWlnaHQKICAgICkgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHhzX3BvcyA9IHRvcmNoLnNpZ21vaWQobG9naXRzKQogICAgICAgIHhzX25lZyA9IDEuMCAtIHhzX3BvcwoKICAgICAgICAjIFByb2JhYmlsaXR5IHNoaWZ0IOKAlCBraWxscyBlYXN5IG5lZ2F0aXZlcyAmIG1pc2xhYmVsbGVkIGVudHJpZXMKICAgICAgICBpZiBzZWxmLmNsaXAgPiAwOgogICAgICAgICAgICB4c19uZWcgPSAoeHNfbmVnICsgc2VsZi5jbGlwKS5jbGFtcChtYXg9MS4wKQoKICAgICAgICAjIExvZyBsb3NzIHRlcm1zCiAgICAgICAgbG9zc19wb3MgPSB0YXJnZXRzICAgICAgICogdG9yY2gubG9nKHhzX3Bvcy5jbGFtcChtaW49c2VsZi5lcHMpKQogICAgICAgIGxvc3NfbmVnID0gKDEuMCAtIHRhcmdldHMpICogdG9yY2gubG9nKHhzX25lZy5jbGFtcChtaW49c2VsZi5lcHMpKQogICAgICAgIGxvc3MgPSBsb3NzX3BvcyArIGxvc3NfbmVnICAgIyBbQiwgQ10KCiAgICAgICAgIyBBc3ltbWV0cmljIGZvY3VzaW5nIOKAlCBkb3duLXdlaWdodCBlYXN5IGV4YW1wbGVzIHBlciBjbGFzcwogICAgICAgICMgZ2FtbWFfbmVnOiBzY2FsYXIgb3IgW0NdIHRlbnNvciAoYnJvYWRjYXN0cyBjb3JyZWN0bHkgZWl0aGVyIHdheSkKICAgICAgICAjIFNPVEEgQnVnZml4OiBlbnN1cmUgZ19uZWcgYWx3YXlzIG1hdGNoZXMgdGFyZ2V0cy5kZXZpY2UgYW5kIHRhcmdldHMuZHR5cGUKICAgICAgICBpZiBzZWxmLmdhbW1hX25lZ192ZWMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlmIHNlbGYuZ2FtbWFfbmVnX3ZlYy5udW1lbCgpICE9IGxvZ2l0cy5zaGFwZVsxXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAgICAgZiJnYW1tYV9uZWcgaGFzIHtzZWxmLmdhbW1hX25lZ192ZWMubnVtZWwoKX0gdmFsdWVzIGZvciAiCiAgICAgICAgICAgICAgICAgICAgZiJ7bG9naXRzLnNoYXBlWzFdfSB0YXJnZXRzIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBnX25lZyA9IHNlbGYuZ2FtbWFfbmVnX3ZlYy50byhkZXZpY2U9dGFyZ2V0cy5kZXZpY2UsIGR0eXBlPXRhcmdldHMuZHR5cGUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZ19uZWcgPSBsb2dpdHMubmV3X2Z1bGwoKDEsKSwgc2VsZi5fZ2FtbWFfbmVnX3NjYWxhciwgZGV2aWNlPXRhcmdldHMuZGV2aWNlLCBkdHlwZT10YXJnZXRzLmR0eXBlKQogICAgICAgIHB0ICAgID0geHNfcG9zICogdGFyZ2V0cyArIHhzX25lZyAqICgxLjAgLSB0YXJnZXRzKSAgICMgW0IsIENdCiAgICAgICAgZ2FtbWEgPSBzZWxmLmdhbW1hX3BvcyAqIHRhcmdldHMgKyBnX25lZyAqICgxLjAgLSB0YXJnZXRzKSAgIyBbQiwgQ10KICAgICAgICBsb3NzICA9IGxvc3MgKiAoKDEuMCAtIHB0KSAqKiBnYW1tYSkKCiAgICAgICAgbG9zcyA9IC1sb3NzICAjIG1pbmltaXNlCgogICAgICAgIGlmIHdlaWdodHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlmIHdlaWdodHMuc2hhcGUgIT0gbG9zcy5zaGFwZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAgICAgZiJ3ZWlnaHRzIHNoYXBlIHt0dXBsZSh3ZWlnaHRzLnNoYXBlKX0gbXVzdCBtYXRjaCBsb3NzIHNoYXBlICIKICAgICAgICAgICAgICAgICAgICBmInt0dXBsZShsb3NzLnNoYXBlKX0iCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGxvc3MgPSBsb3NzICogd2VpZ2h0cwogICAgICAgICAgICAjIE5vcm1hbGl6ZSBieSBsYWJlbGVkIGVudHJpZXMsIG5vdCBjb25maWRlbmNlLXdlaWdodCBzdW06IGNvbmZpZGVuY2UKICAgICAgICAgICAgIyB0aGVuIHNjYWxlcyB0aGUgbG9zcyBhcyBpbnRlbmRlZCByYXRoZXIgdGhhbiBjYW5jZWxpbmcgaXRzZWxmIG91dC4KICAgICAgICAgICAgbGFiZWxlZF9jb3VudCA9ICh3ZWlnaHRzID4gMCkuc3VtKGRpbT0wKS5jbGFtcF9taW4oMSkKICAgICAgICAgICAgY2xhc3NfbG9zc2VzID0gbG9zcy5zdW0oZGltPTApIC8gbGFiZWxlZF9jb3VudAogICAgICAgICAgICByZXR1cm4gY2xhc3NfbG9zc2VzLm1lYW4oKQogICAgICAgIHJldHVybiBsb3NzLm1lYW4oKQoKCmNsYXNzIFdlaWdodGVkQkNFKG5uLk1vZHVsZSk6CiAgICAiIiJMZWdhY3kgd2VpZ2h0ZWQgQkNFIOKAlCBrZXB0IGZvciBhYmxhdGlvbiBjb21wYXJpc29ucy4iIiIKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBsb2dpdHMsIHRhcmdldHMsIHdlaWdodHM9Tm9uZSk6CiAgICAgICAgbG9zcyA9IEYuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMobG9naXRzLmZsb2F0KCksIHRhcmdldHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVkdWN0aW9uPSJub25lIikKICAgICAgICBpZiB3ZWlnaHRzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBpZiB3ZWlnaHRzLnNoYXBlICE9IGxvc3Muc2hhcGU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgICAgIGYid2VpZ2h0cyBzaGFwZSB7dHVwbGUod2VpZ2h0cy5zaGFwZSl9IG11c3QgbWF0Y2ggbG9zcyBzaGFwZSAiCiAgICAgICAgICAgICAgICAgICAgZiJ7dHVwbGUobG9zcy5zaGFwZSl9IgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBsb3NzID0gbG9zcyAqIHdlaWdodHMKICAgICAgICAgICAgbGFiZWxlZF9jb3VudCA9ICh3ZWlnaHRzID4gMCkuc3VtKGRpbT0wKS5jbGFtcF9taW4oMSkKICAgICAgICAgICAgY2xhc3NfbG9zc2VzID0gbG9zcy5zdW0oZGltPTApIC8gbGFiZWxlZF9jb3VudAogICAgICAgICAgICByZXR1cm4gY2xhc3NfbG9zc2VzLm1lYW4oKQogICAgICAgIHJldHVybiBsb3NzLm1lYW4oKQoKCkxPU1NfUkVHSVNUUlk6IGRpY3Rbc3RyLCB0eXBlXSA9IHsiYXNsIjogQXN5bW1ldHJpY0xvc3MsICJiY2UiOiBXZWlnaHRlZEJDRX0KCgpkZWYgYnVpbGRfbG9zcyhuYW1lOiBzdHIgPSAiYXNsIiwgKiprd2FyZ3MpIC0+IG5uLk1vZHVsZToKICAgICIiIkZhY3RvcnkuICBuYW1lIGluIHsnYXNsJywgJ2JjZSd9LiAgRXh0cmEga3dhcmdzIGZvcndhcmRlZCB0byBfX2luaXRfXy4iIiIKICAgIGlmIG5hbWUgbm90IGluIExPU1NfUkVHSVNUUlk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gbG9zcyAne25hbWV9Jy4gQXZhaWxhYmxlOiB7bGlzdChMT1NTX1JFR0lTVFJZKX0iKQogICAgcmV0dXJuIExPU1NfUkVHSVNUUllbbmFtZV0oKiprd2FyZ3MpCg==',
    'src/modeling/model.py': '...',
    'src/modeling/__init__.py': '',
    'src/training/train.py': '...',
    'src/training/__init__.py': '',
    'src/modeling/mil.py': 'IiIiVGltbSBpbWFnZSBlbmNvZGVyIHdpdGggdGFyZ2V0LXNwZWNpZmljIGF0dGVudGlvbiBwb29saW5nIG92ZXIgc3R1ZHkgd2luZG93cy4iIiINCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYNCg0KaW1wb3J0IHNyYy5jb3JlLmNvbmZpZyBhcyBjb25maWcNCg0KDQpjbGFzcyBUaW1tQXR0ZW50aW9uTUlMKG5uLk1vZHVsZSk6DQogICAgIiIiRW5jb2RlIHZhbGlkIHdpbmRvd3Mgd2l0aCBmbGF0IG9yIHNsb3QtaGllcmFyY2hpY2FsIHRhcmdldCBhdHRlbnRpb24uIiIiDQoNCiAgICBkZWYgX19pbml0X18oDQogICAgICAgIHNlbGYsDQogICAgICAgIGJhY2tib25lOiBubi5Nb2R1bGUsDQogICAgICAgIGZlYXR1cmVfZGltOiBpbnQsDQogICAgICAgIGlucHV0X3NpemU6IGludCA9IGNvbmZpZy5DT0FUTkVUX0lOUFVUX1NJWkUsDQogICAgICAgIGVuY29kZV9jaHVua19zaXplOiBpbnQgPSBjb25maWcuQ09BVE5FVF9FTkNPREVfQ0hVTkssDQogICAgICAgIHBvb2xpbmdfbW9kZTogc3RyID0gImZsYXQiLA0KICAgICk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBpZiBpbnB1dF9zaXplIDw9IDAgb3IgZW5jb2RlX2NodW5rX3NpemUgPD0gMDoNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImlucHV0X3NpemUgYW5kIGVuY29kZV9jaHVua19zaXplIG11c3QgYmUgcG9zaXRpdmUiKQ0KICAgICAgICBpZiBwb29saW5nX21vZGUgbm90IGluIHsiZmxhdCIsICJoaWVyYXJjaGljYWwifToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnN1cHBvcnRlZCB0aW1tIE1JTCBwb29saW5nIG1vZGU6IHtwb29saW5nX21vZGUhcn0iKQ0KICAgICAgICBzZWxmLmJhY2tib25lID0gYmFja2JvbmUNCiAgICAgICAgc2VsZi5mZWF0dXJlX2RpbSA9IGludChmZWF0dXJlX2RpbSkNCiAgICAgICAgc2VsZi5pbnB1dF9zaXplID0gaW50KGlucHV0X3NpemUpDQogICAgICAgIHNlbGYuZW5jb2RlX2NodW5rX3NpemUgPSBpbnQoZW5jb2RlX2NodW5rX3NpemUpDQogICAgICAgIHNlbGYucG9vbGluZ19tb2RlID0gcG9vbGluZ19tb2RlDQogICAgICAgIHNlbGYubW9kZWxfdHlwZSA9ICJjb2F0bmV0X21pbCINCiAgICAgICAgc2VsZi5ub3JtID0gbm4uTGF5ZXJOb3JtKHNlbGYuZmVhdHVyZV9kaW0pDQogICAgICAgIHNlbGYuYXR0ZW50aW9uID0gbm4uU2VxdWVudGlhbCgNCiAgICAgICAgICAgIG5uLkxpbmVhcihzZWxmLmZlYXR1cmVfZGltLCAyNTYpLA0KICAgICAgICAgICAgbm4uVGFuaCgpLA0KICAgICAgICAgICAgbm4uRHJvcG91dCgwLjIpLA0KICAgICAgICAgICAgbm4uTGluZWFyKDI1NiwgbGVuKGNvbmZpZy5UQVJHRVRTKSksDQogICAgICAgICkNCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gbm4uUGFyYW1ldGVyKHRvcmNoLmVtcHR5KGxlbihjb25maWcuVEFSR0VUUyksIHNlbGYuZmVhdHVyZV9kaW0pKQ0KICAgICAgICBzZWxmLmJpYXMgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MobGVuKGNvbmZpZy5UQVJHRVRTKSkpDQogICAgICAgIGlmIHNlbGYucG9vbGluZ19tb2RlID09ICJoaWVyYXJjaGljYWwiOg0KICAgICAgICAgICAgc2VsZi5zbG90X2VtYmVkZGluZyA9IG5uLlBhcmFtZXRlcigNCiAgICAgICAgICAgICAgICB0b3JjaC5lbXB0eShjb25maWcuTl9TTE9UUywgc2VsZi5mZWF0dXJlX2RpbSkNCiAgICAgICAgICAgICkNCiAgICAgICAgICAgIHNlbGYuc2xvdF9xdWVyeSA9IG5uLlBhcmFtZXRlcigNCiAgICAgICAgICAgICAgICB0b3JjaC5lbXB0eShsZW4oY29uZmlnLlRBUkdFVFMpLCBzZWxmLmZlYXR1cmVfZGltKQ0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgc2VsZi5zbG90X2JpYXMgPSBubi5QYXJhbWV0ZXIoDQogICAgICAgICAgICAgICAgdG9yY2guemVyb3MobGVuKGNvbmZpZy5UQVJHRVRTKSwgY29uZmlnLk5fU0xPVFMpDQogICAgICAgICAgICApDQogICAgICAgICAgICBubi5pbml0LnRydW5jX25vcm1hbF8oc2VsZi5zbG90X2VtYmVkZGluZywgc3RkPTAuMDIpDQogICAgICAgICAgICBubi5pbml0LnRydW5jX25vcm1hbF8oc2VsZi5zbG90X3F1ZXJ5LCBzdGQ9MC4wMikNCiAgICAgICAgbm4uaW5pdC50cnVuY19ub3JtYWxfKHNlbGYuY2xhc3NpZmllciwgc3RkPTAuMDIpDQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKA0KICAgICAgICAgICAgIm1lYW4iLA0KICAgICAgICAgICAgdG9yY2gudGVuc29yKFswLjQ4NSwgMC40NTYsIDAuNDA2XSkudmlldygxLCAzLCAxLCAxKSwNCiAgICAgICAgKQ0KICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigNCiAgICAgICAgICAgICJzdGQiLA0KICAgICAgICAgICAgdG9yY2gudGVuc29yKFswLjIyOSwgMC4yMjQsIDAuMjI1XSkudmlldygxLCAzLCAxLCAxKSwNCiAgICAgICAgKQ0KDQogICAgZGVmIF9lbmNvZGUoc2VsZiwgaW1hZ2VzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoNCiAgICAgICAgZmVhdHVyZXMgPSBbXQ0KICAgICAgICBwYXJhbWV0ZXIgPSBuZXh0KHNlbGYuYmFja2JvbmUucGFyYW1ldGVycygpLCBOb25lKQ0KICAgICAgICBkdHlwZSA9IHBhcmFtZXRlci5kdHlwZSBpZiBwYXJhbWV0ZXIgaXMgbm90IE5vbmUgZWxzZSBpbWFnZXMuZHR5cGUNCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihpbWFnZXMpLCBzZWxmLmVuY29kZV9jaHVua19zaXplKToNCiAgICAgICAgICAgIGJhdGNoID0gaW1hZ2VzW3N0YXJ0OnN0YXJ0ICsgc2VsZi5lbmNvZGVfY2h1bmtfc2l6ZV0NCiAgICAgICAgICAgIGlmIGJhdGNoLnNoYXBlWy0yOl0gIT0gKHNlbGYuaW5wdXRfc2l6ZSwgc2VsZi5pbnB1dF9zaXplKToNCiAgICAgICAgICAgICAgICBiYXRjaCA9IEYuaW50ZXJwb2xhdGUoDQogICAgICAgICAgICAgICAgICAgIGJhdGNoLA0KICAgICAgICAgICAgICAgICAgICBzaXplPShzZWxmLmlucHV0X3NpemUsIHNlbGYuaW5wdXRfc2l6ZSksDQogICAgICAgICAgICAgICAgICAgIG1vZGU9ImJpbGluZWFyIiwNCiAgICAgICAgICAgICAgICAgICAgYWxpZ25fY29ybmVycz1GYWxzZSwNCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICBiYXRjaCA9IGJhdGNoLnRvKGR0eXBlPWR0eXBlKQ0KICAgICAgICAgICAgYmF0Y2ggPSAoYmF0Y2ggLSBzZWxmLm1lYW4udG8oYmF0Y2guZHR5cGUpKSAvIHNlbGYuc3RkLnRvKGJhdGNoLmR0eXBlKQ0KICAgICAgICAgICAgaWYgc2VsZi50cmFpbmluZzoNCiAgICAgICAgICAgICAgICBiYXRjaC5yZXF1aXJlc19ncmFkXyhUcnVlKQ0KICAgICAgICAgICAgZW5jb2RlZCA9IHNlbGYuYmFja2JvbmUoYmF0Y2gpDQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGVuY29kZWQsICh0dXBsZSwgbGlzdCkpOg0KICAgICAgICAgICAgICAgIGVuY29kZWQgPSBlbmNvZGVkWzBdDQogICAgICAgICAgICBpZiBlbmNvZGVkLm5kaW0gPiAyOg0KICAgICAgICAgICAgICAgIGVuY29kZWQgPSBlbmNvZGVkLmZsYXR0ZW4oMikubWVhbigtMSkNCiAgICAgICAgICAgIGlmIGVuY29kZWQubmRpbSAhPSAyIG9yIGVuY29kZWQuc2hhcGVbMV0gIT0gc2VsZi5mZWF0dXJlX2RpbToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoDQogICAgICAgICAgICAgICAgICAgIGYidGltbSBiYWNrYm9uZSByZXR1cm5lZCB7dHVwbGUoZW5jb2RlZC5zaGFwZSl9OyBleHBlY3RlZCAiDQogICAgICAgICAgICAgICAgICAgIGYiW04sIHtzZWxmLmZlYXR1cmVfZGltfV0gcG9vbGVkIGZlYXR1cmVzIg0KICAgICAgICAgICAgICAgICkNCiAgICAgICAgICAgIGZlYXR1cmVzLmFwcGVuZChlbmNvZGVkKQ0KICAgICAgICByZXR1cm4gdG9yY2guY2F0KGZlYXR1cmVzLCBkaW09MCkNCg0KICAgIGRlZiBmb3J3YXJkKA0KICAgICAgICBzZWxmLA0KICAgICAgICBpbWdzOiB0b3JjaC5UZW5zb3IsDQogICAgICAgIHNsb3RfbWFzazogdG9yY2guVGVuc29yLA0KICAgICAgICB3aW5kb3dfbWFzazogdG9yY2guVGVuc29yLA0KICAgICkgLT4gdG9yY2guVGVuc29yOg0KICAgICAgICBpZiBpbWdzLm5kaW0gIT0gNjoNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlY3RlZCBpbWFnZXMgW0IsUyxXLDMsSCxXXSwgcmVjZWl2ZWQge3R1cGxlKGltZ3Muc2hhcGUpfSIpDQogICAgICAgIGJhdGNoX3NpemUsIHNsb3RzLCB3aW5kb3dzID0gaW1ncy5zaGFwZVs6M10NCiAgICAgICAgaWYgaW1ncy5zaGFwZVszXSAhPSAzOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImV4cGVjdGVkIHRocmVlIGFkamFjZW50LXNsaWNlIGNoYW5uZWxzLCByZWNlaXZlZCB7aW1ncy5zaGFwZVszXX0iKQ0KICAgICAgICBpZiBzbG90X21hc2suc2hhcGUgIT0gKGJhdGNoX3NpemUsIHNsb3RzKToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoDQogICAgICAgICAgICAgICAgZiJzbG90X21hc2sgbXVzdCBoYXZlIHNoYXBlIHsoYmF0Y2hfc2l6ZSwgc2xvdHMpfSwgIg0KICAgICAgICAgICAgICAgIGYicmVjZWl2ZWQge3R1cGxlKHNsb3RfbWFzay5zaGFwZSl9Ig0KICAgICAgICAgICAgKQ0KICAgICAgICBpZiB3aW5kb3dfbWFzay5zaGFwZSAhPSAoYmF0Y2hfc2l6ZSwgc2xvdHMsIHdpbmRvd3MpOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigNCiAgICAgICAgICAgICAgICBmIndpbmRvd19tYXNrIG11c3QgaGF2ZSBzaGFwZSB7KGJhdGNoX3NpemUsIHNsb3RzLCB3aW5kb3dzKX0sICINCiAgICAgICAgICAgICAgICBmInJlY2VpdmVkIHt0dXBsZSh3aW5kb3dfbWFzay5zaGFwZSl9Ig0KICAgICAgICAgICAgKQ0KICAgICAgICB2YWxpZCA9IChzbG90X21hc2sudW5zcXVlZXplKC0xKSA+IDAuNSkgJiAod2luZG93X21hc2sgPiAwLjUpDQogICAgICAgIGZsYXRfaW1hZ2VzID0gaW1ncy5yZXNoYXBlKGJhdGNoX3NpemUgKiBzbG90cyAqIHdpbmRvd3MsICppbWdzLnNoYXBlWzM6XSkNCiAgICAgICAgdmFsaWRfaW5kaWNlcyA9IHZhbGlkLnJlc2hhcGUoLTEpLm5vbnplcm8oYXNfdHVwbGU9RmFsc2UpLmZsYXR0ZW4oKQ0KDQogICAgICAgIGZlYXR1cmVfZHR5cGUgPSBzZWxmLm5vcm0ud2VpZ2h0LmR0eXBlDQogICAgICAgIGZsYXRfZmVhdHVyZXMgPSBpbWdzLm5ld196ZXJvcygNCiAgICAgICAgICAgIChiYXRjaF9zaXplICogc2xvdHMgKiB3aW5kb3dzLCBzZWxmLmZlYXR1cmVfZGltKSwgZHR5cGU9ZmVhdHVyZV9kdHlwZQ0KICAgICAgICApDQogICAgICAgIGlmIHZhbGlkX2luZGljZXMubnVtZWwoKToNCiAgICAgICAgICAgIGVuY29kZWRfY2h1bmtzID0gW10NCiAgICAgICAgICAgIGluZGV4X2NodW5rcyA9IFtdDQogICAgICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgdmFsaWRfaW5kaWNlcy5udW1lbCgpLCBzZWxmLmVuY29kZV9jaHVua19zaXplKToNCiAgICAgICAgICAgICAgICBpbmRpY2VzID0gdmFsaWRfaW5kaWNlc1tzdGFydDpzdGFydCArIHNlbGYuZW5jb2RlX2NodW5rX3NpemVdDQogICAgICAgICAgICAgICAgc2VsZWN0ZWQgPSBmbGF0X2ltYWdlcy5pbmRleF9zZWxlY3QoMCwgaW5kaWNlcykuZmxvYXQoKS5kaXYoMjU1LjApDQogICAgICAgICAgICAgICAgZW5jb2RlZF9jaHVua3MuYXBwZW5kKHNlbGYuX2VuY29kZShzZWxlY3RlZCkpDQogICAgICAgICAgICAgICAgaW5kZXhfY2h1bmtzLmFwcGVuZChpbmRpY2VzKQ0KICAgICAgICAgICAgZW5jb2RlZCA9IHRvcmNoLmNhdChlbmNvZGVkX2NodW5rcywgZGltPTApDQogICAgICAgICAgICBpbmRpY2VzID0gdG9yY2guY2F0KGluZGV4X2NodW5rcywgZGltPTApDQogICAgICAgICAgICBmbGF0X2ZlYXR1cmVzID0gZmxhdF9mZWF0dXJlcy5pbmRleF9jb3B5KA0KICAgICAgICAgICAgICAgIDAsIGluZGljZXMsIGVuY29kZWQudG8oZmVhdHVyZV9kdHlwZSkNCiAgICAgICAgICAgICkNCg0KICAgICAgICBmZWF0dXJlcyA9IHNlbGYubm9ybSgNCiAgICAgICAgICAgIGZsYXRfZmVhdHVyZXMudmlldyhiYXRjaF9zaXplLCBzbG90cyAqIHdpbmRvd3MsIHNlbGYuZmVhdHVyZV9kaW0pDQogICAgICAgICkNCiAgICAgICAgdmFsaWRfd2luZG93cyA9IHZhbGlkLnJlc2hhcGUoYmF0Y2hfc2l6ZSwgc2xvdHMgKiB3aW5kb3dzKQ0KICAgICAgICBpZiBzZWxmLnBvb2xpbmdfbW9kZSA9PSAiaGllcmFyY2hpY2FsIjoNCiAgICAgICAgICAgIGlmIHNsb3RzICE9IGNvbmZpZy5OX1NMT1RTOg0KICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoDQogICAgICAgICAgICAgICAgICAgIGYiaGllcmFyY2hpY2FsIHBvb2xpbmcgcmVxdWlyZXMge2NvbmZpZy5OX1NMT1RTfSBhbmF0b21pY2FsIHNsb3RzLCAiDQogICAgICAgICAgICAgICAgICAgIGYicmVjZWl2ZWQge3Nsb3RzfSINCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICBmZWF0dXJlcyA9IGZlYXR1cmVzLnZpZXcoDQogICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZSwgc2xvdHMsIHdpbmRvd3MsIHNlbGYuZmVhdHVyZV9kaW0NCiAgICAgICAgICAgICkgKyBzZWxmLnNsb3RfZW1iZWRkaW5nLnZpZXcoMSwgc2xvdHMsIDEsIHNlbGYuZmVhdHVyZV9kaW0pDQogICAgICAgICAgICBzY29yZXMgPSBzZWxmLmF0dGVudGlvbihmZWF0dXJlcykNCiAgICAgICAgICAgIHNjb3JlcyA9IHNjb3Jlcy5tYXNrZWRfZmlsbCh+dmFsaWQudW5zcXVlZXplKC0xKSwgLTEwMDAwLjApDQogICAgICAgICAgICB3aW5kb3dfYXR0ZW50aW9uID0gc2NvcmVzLnNvZnRtYXgoZGltPTIpDQogICAgICAgICAgICBzbG90X2ZlYXR1cmVzID0gdG9yY2guZWluc3VtKA0KICAgICAgICAgICAgICAgICJic3djLGJzd2QtPmJzY2QiLCB3aW5kb3dfYXR0ZW50aW9uLCBmZWF0dXJlcw0KICAgICAgICAgICAgKQ0KDQogICAgICAgICAgICBzbG90X3ZhbGlkID0gdmFsaWQuYW55KGRpbT0yKQ0KICAgICAgICAgICAgc2xvdF9zY29yZXMgPSB0b3JjaC5laW5zdW0oDQogICAgICAgICAgICAgICAgImJzY2QsY2QtPmJjcyIsIHNsb3RfZmVhdHVyZXMsIHNlbGYuc2xvdF9xdWVyeQ0KICAgICAgICAgICAgKSAvIHNlbGYuZmVhdHVyZV9kaW0qKjAuNQ0KICAgICAgICAgICAgc2xvdF9zY29yZXMgPSBzbG90X3Njb3JlcyArIHNlbGYuc2xvdF9iaWFzLnVuc3F1ZWV6ZSgwKQ0KICAgICAgICAgICAgc2xvdF9zY29yZXMgPSBzbG90X3Njb3Jlcy5tYXNrZWRfZmlsbCgNCiAgICAgICAgICAgICAgICB+c2xvdF92YWxpZC51bnNxdWVlemUoMSksIC0xMDAwMC4wDQogICAgICAgICAgICApDQogICAgICAgICAgICBzbG90X2F0dGVudGlvbiA9IHNsb3Rfc2NvcmVzLnNvZnRtYXgoZGltPTIpDQogICAgICAgICAgICBzbG90X2F0dGVudGlvbiA9IHNsb3RfYXR0ZW50aW9uLm1hc2tlZF9maWxsKA0KICAgICAgICAgICAgICAgIH5zbG90X3ZhbGlkLmFueShkaW09MSwga2VlcGRpbT1UcnVlKS51bnNxdWVlemUoLTEpLCAwLjANCiAgICAgICAgICAgICkNCiAgICAgICAgICAgIHBvb2xlZCA9IHRvcmNoLmVpbnN1bSgNCiAgICAgICAgICAgICAgICAiYmNzLGJzY2QtPmJjZCIsIHNsb3RfYXR0ZW50aW9uLCBzbG90X2ZlYXR1cmVzDQogICAgICAgICAgICApDQogICAgICAgICAgICByZXR1cm4gKHBvb2xlZCAqIHNlbGYuY2xhc3NpZmllci51bnNxdWVlemUoMCkpLnN1bSgtMSkgKyBzZWxmLmJpYXMNCg0KICAgICAgICBzY29yZXMgPSBzZWxmLmF0dGVudGlvbihmZWF0dXJlcykNCiAgICAgICAgc2NvcmVzID0gc2NvcmVzLm1hc2tlZF9maWxsKH52YWxpZF93aW5kb3dzLnVuc3F1ZWV6ZSgtMSksIC0xMDAwMC4wKQ0KICAgICAgICBhdHRlbnRpb24gPSBzY29yZXMuc29mdG1heChkaW09MSkNCiAgICAgICAgYXR0ZW50aW9uID0gYXR0ZW50aW9uLm1hc2tlZF9maWxsKA0KICAgICAgICAgICAgfnZhbGlkX3dpbmRvd3MuYW55KGRpbT0xLCBrZWVwZGltPVRydWUpLnVuc3F1ZWV6ZSgtMSksIDAuMA0KICAgICAgICApDQogICAgICAgIHBvb2xlZCA9IHRvcmNoLmVpbnN1bSgiYm5jLGJuZi0+YmNmIiwgYXR0ZW50aW9uLCBmZWF0dXJlcykNCiAgICAgICAgcmV0dXJuIChwb29sZWQgKiBzZWxmLmNsYXNzaWZpZXIudW5zcXVlZXplKDApKS5zdW0oLTEpICsgc2VsZi5iaWFzDQoNCg0KZGVmIGJ1aWxkX3RpbW1fYXR0ZW50aW9uX21pbCgNCiAgICBhcmNoOiBzdHIsDQogICAgcHJldHJhaW5lZDogYm9vbCA9IEZhbHNlLA0KICAgIGlucHV0X3NpemU6IGludCA9IGNvbmZpZy5DT0FUTkVUX0lOUFVUX1NJWkUsDQogICAgZW5jb2RlX2NodW5rX3NpemU6IGludCA9IGNvbmZpZy5DT0FUTkVUX0VOQ09ERV9DSFVOSywNCiAgICBwb29saW5nX21vZGU6IHN0ciA9ICJmbGF0IiwNCiAgICBtb2RlbF90eXBlOiBzdHIgPSAiY29hdG5ldF9taWwiLA0KKSAtPiBUaW1tQXR0ZW50aW9uTUlMOg0KICAgIHRyeToNCiAgICAgICAgaW1wb3J0IHRpbW0NCiAgICBleGNlcHQgSW1wb3J0RXJyb3IgYXMgZXhjOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRoZSB0aW1tIHBhY2thZ2UgaXMgcmVxdWlyZWQgZm9yIHRoZSBDb0F0TmV0IE1JTCBtb2RlbCIpIGZyb20gZXhjDQoNCiAgICBiYWNrYm9uZSA9IHRpbW0uY3JlYXRlX21vZGVsKA0KICAgICAgICBhcmNoLA0KICAgICAgICBwcmV0cmFpbmVkPXByZXRyYWluZWQsDQogICAgICAgIG51bV9jbGFzc2VzPTAsDQogICAgICAgIGdsb2JhbF9wb29sPSJhdmciLA0KICAgICAgICBpbl9jaGFucz0zLA0KICAgICkNCiAgICBpZiBoYXNhdHRyKGJhY2tib25lLCAic2V0X2dyYWRfY2hlY2twb2ludGluZyIpOg0KICAgICAgICBiYWNrYm9uZS5zZXRfZ3JhZF9jaGVja3BvaW50aW5nKFRydWUpDQogICAgZmVhdHVyZV9kaW0gPSBnZXRhdHRyKGJhY2tib25lLCAibnVtX2ZlYXR1cmVzIiwgTm9uZSkNCiAgICBpZiBub3QgZmVhdHVyZV9kaW06DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ0aW1tIG1vZGVsIHthcmNoIXJ9IGRvZXMgbm90IGV4cG9zZSBudW1fZmVhdHVyZXMiKQ0KICAgIG1vZGVsID0gVGltbUF0dGVudGlvbk1JTCgNCiAgICAgICAgYmFja2JvbmUsDQogICAgICAgIGZlYXR1cmVfZGltLA0KICAgICAgICBpbnB1dF9zaXplPWlucHV0X3NpemUsDQogICAgICAgIGVuY29kZV9jaHVua19zaXplPWVuY29kZV9jaHVua19zaXplLA0KICAgICAgICBwb29saW5nX21vZGU9cG9vbGluZ19tb2RlLA0KICAgICkNCiAgICBtb2RlbC5tb2RlbF90eXBlID0gbW9kZWxfdHlwZQ0KICAgIHJldHVybiBtb2RlbA0K',
    'src/modeling/configs/config.json': 'ew0KICAiYXBwbHlfbGF5ZXJub3JtIjogdHJ1ZSwNCiAgImFyY2hpdGVjdHVyZXMiOiBbDQogICAgIkRpbm92Mk1vZGVsIg0KICBdLA0KICAiYXR0ZW50aW9uX3Byb2JzX2Ryb3BvdXRfcHJvYiI6IDAuMCwNCiAgImRyb3BfcGF0aF9yYXRlIjogMC4wLA0KICAiZHR5cGUiOiAiZmxvYXQzMiIsDQogICJoaWRkZW5fYWN0IjogImdlbHUiLA0KICAiaGlkZGVuX2Ryb3BvdXRfcHJvYiI6IDAuMCwNCiAgImhpZGRlbl9zaXplIjogNzY4LA0KICAiaW1hZ2Vfc2l6ZSI6IDUxOCwNCiAgImluaXRpYWxpemVyX3JhbmdlIjogMC4wMiwNCiAgImxheWVyX25vcm1fZXBzIjogMWUtMDYsDQogICJsYXllcnNjYWxlX3ZhbHVlIjogMS4wLA0KICAibWxwX3JhdGlvIjogNCwNCiAgIm1vZGVsX3R5cGUiOiAiZGlub3YyIiwNCiAgIm51bV9hdHRlbnRpb25faGVhZHMiOiAxMiwNCiAgIm51bV9jaGFubmVscyI6IDMsDQogICJudW1faGlkZGVuX2xheWVycyI6IDEyLA0KICAib3V0X2ZlYXR1cmVzIjogWw0KICAgICJzdGFnZTEyIg0KICBdLA0KICAib3V0X2luZGljZXMiOiBbDQogICAgMTINCiAgXSwNCiAgInBhdGNoX3NpemUiOiAxNCwNCiAgInFrdl9iaWFzIjogdHJ1ZSwNCiAgInJlc2hhcGVfaGlkZGVuX3N0YXRlcyI6IHRydWUsDQogICJzdGFnZV9uYW1lcyI6IFsNCiAgICAic3RlbSIsDQogICAgInN0YWdlMSIsDQogICAgInN0YWdlMiIsDQogICAgInN0YWdlMyIsDQogICAgInN0YWdlNCIsDQogICAgInN0YWdlNSIsDQogICAgInN0YWdlNiIsDQogICAgInN0YWdlNyIsDQogICAgInN0YWdlOCIsDQogICAgInN0YWdlOSIsDQogICAgInN0YWdlMTAiLA0KICAgICJzdGFnZTExIiwNCiAgICAic3RhZ2UxMiINCiAgXSwNCiAgInRyYW5zZm9ybWVyc192ZXJzaW9uIjogIjUuMTQuMSIsDQogICJ1c2VfbWFza190b2tlbiI6IHRydWUsDQogICJ1c2Vfc3dpZ2x1X2ZmbiI6IGZhbHNlDQp9DQo=',
}

for file_path, b64_data in PIPELINE_MODULES.items():
    full_path = os.path.join(REPO_ROOT, file_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'wb') as module_file:
        module_file.write(base64.b64decode(b64_data))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f'SUCCESS: Extracted {len(PIPELINE_MODULES)} current pipeline modules locally.')

In [ ]:
# ==============================================================================
# STEP 4: VERIFY ENVIRONMENT, ARCHITECTURE & GPU SETUP
# ==============================================================================
import torch
import src.core.config as config
from src.modeling.model import build_model, load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'[HARDWARE] Device: {device} | CUDA Available: {torch.cuda.is_available()} | GPU Count: {n_gpus}')
if torch.cuda.is_available():
    for g in range(n_gpus):
        print(f'   GPU {g}: {torch.cuda.get_device_name(g)} (Memory: {torch.cuda.get_device_properties(g).total_memory / 1e9:.2f} GB)')

# Test model instantiation using embedded offline configuration
print('[VERIFY] Testing DINOv2-Base + CrossSlotTransformer offline instantiation...')
test_model = build_model('dinov2-base', use_cross_slot=True, pretrained=False)
n_params = sum(p.numel() for p in test_model.parameters())
print(f'[SUCCESS] Model architecture verified! Total parameters: {n_params:,}')
del test_model


In [ ]:
# ==============================================================================
# STEP 5: LOAD AVAILABLE FOLD CHECKPOINTS BY MODEL FAMILY
# ==============================================================================
import re
import torch

# Prefer one best checkpoint per fold and family; use EMA/SWA only when best is absent.
checkpoint_priority = {'best': 0, 'ema': 1, 'swa': 2}
checkpoint_candidates = sorted(
    {os.path.abspath(path) for path in found_checkpoints if path.lower().endswith('.pt')},
    key=lambda path: (0 if os.path.commonpath([path, os.path.abspath(WEIGHTS_DIR)]) == os.path.abspath(WEIGHTS_DIR) else 1, path),
)
selected_checkpoints = {}
for checkpoint_path in checkpoint_candidates:
    match = re.search(r'fold(\d+)_(best|ema|swa)\.pt$', os.path.basename(checkpoint_path))
    if match is None:
        continue
    fold, checkpoint_kind = int(match.group(1)), match.group(2)
    metadata = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_config = metadata.get('model_config', {})
    model_type = model_config.get('model_type', metadata.get('model_type'))
    variant = model_config.get('variant', metadata.get('variant', 'dinov2-base'))
    if model_type is None:
        model_type = 'coatnet_mil' if str(variant).startswith('coatnet') else 'dinov2'
    if model_type not in {'dinov2', 'coatnet_mil', 'timm_mil'}:
        raise ValueError(f'Unsupported checkpoint family {model_type!r}: {checkpoint_path}')
    model_family = variant if model_type == 'timm_mil' else model_type
    key = (model_family, fold)
    candidate = (checkpoint_priority[checkpoint_kind], checkpoint_path)
    if key not in selected_checkpoints or candidate[0] < selected_checkpoints[key][0]:
        selected_checkpoints[key] = candidate

if not selected_checkpoints:
    raise RuntimeError('No supported fold checkpoints found in uploaded weights.')

selected_paths = [
    selected_checkpoints[key][1]
    for key in sorted(selected_checkpoints, key=lambda item: (item[0], item[1]))
]
models = []
model_types = []
model_families = []
for checkpoint_path in selected_paths:
    metadata = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model_config = metadata.get('model_config', {})
    model_type = model_config.get('model_type', metadata.get('model_type'))
    variant = model_config.get('variant', metadata.get('variant', 'dinov2-base'))
    if model_type is None:
        model_type = 'coatnet_mil' if str(variant).startswith('coatnet') else 'dinov2'
    print(f'[LOAD] {model_type} checkpoint: {checkpoint_path}')
    models.append(load_checkpoint(checkpoint_path, device='cpu'))
    model_types.append(model_type)
    model_families.append(models[-1]._rsna_model_family)

family_counts = {family: model_families.count(family) for family in sorted(set(model_families))}
print(f'[SUCCESS] Loaded checkpoints by family: {family_counts}')
if set(model_types) == {'dinov2', 'coatnet_mil'}:
    print('[BLEND] DINOv2 + CoAtNet detected; equal-family rank blending is the default. D4-derived weights are opt-in below.')
else:
    print('[BLEND] Equal-family rank blending will be used when multiple families are present.')

In [ ]:
# ==============================================================================
# STEP 6: RUN INFERENCE (TTA DISABLED BY DEFAULT)
# ==============================================================================
import pandas as pd
from src.inference.inference import run_inference

# Auto-locate competition test data directory
candidate_roots = [
    '/kaggle/input/rsna-knee-abnormality-detection',
    '/kaggle/input/rsna-knee-challenge',
    '/kaggle/input/rsna-2026'
]
DATA_ROOT = None
for cr in candidate_roots:
    if os.path.exists(os.path.join(cr, 'test.csv')) or os.path.exists(os.path.join(cr, 'test_series.csv')):
        DATA_ROOT = cr
        break

if DATA_ROOT is None:
    DATA_ROOT = '/kaggle/input/rsna-knee-abnormality-detection'

TEST_CSV = os.path.join(DATA_ROOT, 'test.csv')
OUT_CSV = '/kaggle/working/submission.csv'
CACHE_DIR = '/kaggle/temp/rsna-knee-test-cache' if os.path.isdir('/kaggle/temp') else '/kaggle/working/rsna-knee-test-cache'

print(f'[DATA] Data Root: {DATA_ROOT}')
print(f'[DATA] Test CSV : {TEST_CSV}')
USE_D4_TARGET_WEIGHTS = False
print('[EXEC] Starting inference with equal-family rank blending unless explicitly overridden...')

sub, stats = run_inference(
    root=DATA_ROOT,
    models=models,  # Use every selected fold checkpoint and available model family
    test_csv=TEST_CSV,
    cfg=cfg,
    out_csv=OUT_CSV,
    cache_dir=CACHE_DIR,
    workers=4,
    chunk=64,
    device=device,
    use_tta=False,  # Keep the time-bounded inference path; no horizontal flips
    n_tta=1,
    batch=8,
    use_d4_target_weights=USE_D4_TARGET_WEIGHTS,
)

# Clean up temporary cache to avoid Kaggle disk quota errors
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print('[CLEANUP] Temporary slice cache removed to conserve disk.')


In [ ]:
# ==============================================================================
# STEP 7: VALIDATE SUBMISSION INTEGRITY & DISPLAY PREVIEW
# ==============================================================================
assert os.path.exists(OUT_CSV), f'FATAL: {OUT_CSV} does not exist!'
sub_df = pd.read_csv(OUT_CSV)

print('=' * 80)
print('SUBMISSION VERIFICATION SUMMARY')
print('=' * 80)
print(f'1. Output File     : {OUT_CSV}')
print(f'2. Shape           : {sub_df.shape} (Expected: N x 13)')
print(f'3. Null Values     : {sub_df.isnull().sum().sum()} (Must be 0)')
print(f'4. Target Columns  : {list(sub_df.columns[1:])}')

# Verify matching row count with test.csv
if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    print(f'5. Row Match Test  : len(sub) == len(test) -> {len(sub_df)} == {len(test_df)}')
    assert len(sub_df) == len(test_df), 'Row count mismatch with test.csv!'
    assert (sub_df['StudyInstanceUID'].values == test_df['StudyInstanceUID'].values).all(), 'UID ordering mismatch!'
    print('   -> PERFECT: Exact 1-to-1 StudyInstanceUID match and ordering verified!')

# Verify probabilities are in [0, 1]
vals = sub_df.iloc[:, 1:].values
assert (vals >= 0.0).all() and (vals <= 1.0).all(), 'Probabilities out of [0, 1] range!'
print('6. Probability Range: All values within [0.0, 1.0] valid range')

print('\n[SUCCESS] submission.csv is verified and ready for Kaggle scoring!')
print('=' * 80)
display(sub_df.head(10))
